# Cognitive Rungs -- Modular Action Decision Architecture

**BitterTruth-AI** | ARC-AGI-3 Competition | Self-Contained Offline Notebook

A biologically-inspired decision system where **85 cognitive rungs** -- pluggable
analysis modules -- evaluate game states and vote on actions. Each rung specializes
in one aspect of reasoning: emergency safety, environmental survey, hypothesis
formation, knowledge exploitation, or exploration.

**This notebook plays real ARC games** using the rung cascade as the sole decision engine.
No BFS, no MCTS, no hardcoded solvers -- just 85 rungs deciding one action at a time.

```
Game State --> [Emergency] --> [Orientation] --> [Hypothesis] --> [Exploitation] --> [Filter] --> [Fallback] --> Action
                  |                |                 |                 |                |
             Loop breaker     Survey env       Form theories     Use knowledge    Safety gates
             Oscillation      Detect grids     Test beliefs      Replay wins      Death avoid
             detection        Frame parse      Two streams       Embeddings       Terminal patterns
```

### The PTMA Loop (Perceive -> Think -> Map -> Act)

Each game step runs the full cognitive loop:
1. **Perceive**: Visual analysis, grid detection, object identification
2. **Think**: Hypothesis formation, theory testing, belief updates
3. **Map**: Causal model updates, spatial mapping, knowledge consolidation
4. **Act**: Rung evaluation cascade -> weighted action selection

### Key Concepts

- **85 rungs** across 7 cognitive categories
- **6 decision strategies**: ladder, weighted, phased, parallel, cognitive, context-adaptive
- **13+ ordering presets**: efficiency, comprehensive, human_brain, action6_only, etc.
- **Knowledge provenance**: Every decision tracks HOW it knows what it knows
- **Temporal modulation**: Rung priorities shift based on experience over time

In [ ]:
# -- Write placeholder submission.parquet immediately -------------------------
# Kaggle requires a submission.parquet in /kaggle/working/ even if the notebook
# crashes later. This ensures a valid file always exists.
import os as _os
_kaggle = _os.path.exists('/kaggle')
if _kaggle:
    try:
        import pandas as _pd_early
        _placeholder = _pd_early.DataFrame(
            [{"row_id": "0_0", "game_id": "placeholder",
              "end_of_game": True, "score": 0.0}]
        )
        _placeholder.to_parquet('/kaggle/working/submission.parquet', index=False)
        print("[OK] submission.parquet placeholder written")
    except Exception as _ep:
        print(f"[WARN] could not write placeholder parquet: {_ep}")
else:
    print("[INFO] Not on Kaggle -- skipping submission.parquet placeholder")

In [ ]:
import sys, os, logging, random, json, re, hashlib, warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Set, Tuple
from collections import Counter, defaultdict
from datetime import datetime

sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
warnings.filterwarnings('ignore')

try:
    import numpy as np
except ImportError:
    # Minimal numpy stub for offline environments
    class _NpStub:
        float64 = float
        int64 = int
        def array(self, x, **kw): return x
        def zeros(self, shape, **kw): return [[0]*shape[1] for _ in range(shape[0])] if isinstance(shape, tuple) and len(shape)==2 else [0]*shape
        def mean(self, x, **kw): return sum(x)/max(len(x),1) if hasattr(x,'__len__') else x
        def std(self, x, **kw): return 0.0
        def argmax(self, x, **kw): return max(range(len(x)), key=lambda i: x[i]) if x else 0
        def sqrt(self, x): return x**0.5
        def log(self, x): return __import__('math').log(max(x, 1e-10))
        def exp(self, x): return __import__('math').exp(min(x, 700))
        def clip(self, x, lo, hi): return max(lo, min(x, hi))
    np = _NpStub()

logger = logging.getLogger('cognitive_rungs')
logging.basicConfig(level=logging.WARNING, format='[%(name)s] %(message)s')

# -- Universal module faker -------------------------------------------------
# Instead of stripping imports, we make ALL missing modules importable.
# Any 'from engines.X import Y' or 'from database_logger import Z' will succeed
# and return a stub that does nothing but does not crash.
import types

class _UniversalStub:
    """A stub that returns itself for any attribute access or call."""
    def __init__(self, *a, **kw): pass
    def __getattr__(self, name): return _UniversalStub
    def __call__(self, *a, **kw): return _UniversalStub()
    def __bool__(self): return False
    def __iter__(self): return iter([])
    def __len__(self): return 0
    def __repr__(self): return '<Stub>'
    def __enter__(self): return self
    def __exit__(self, *a): pass

class _StubModule(types.ModuleType):
    """A fake module where every attribute is a universal stub."""
    def __getattr__(self, name):
        return _UniversalStub

# Pre-register all engine/internal modules that rung code might import from
_STUB_PACKAGES = [
    'engines', 'engines.registry',
    'engines.perception', 'engines.perception.palette_detector',
    'engines.perception.grid_detector', 'engines.perception.object_detector',
    'engines.perception.sparse_grid_analyzer', 'engines.perception.visual_analyzer',
    'engines.perception.event_detector', 'engines.perception.object_tracker',
    'engines.perception.sparse_grid', 'engines.perception.spatial_learning',
    'engines.cognition', 'engines.cognition.rung_roles', 'engines.cognition.rung_affinity',
    'engines.cognition.cognitive_router', 'engines.cognition.blackboard',
    'engines.cognition.edge_inference', 'engines.cognition.epistemic_tracker',
    'engines.cognition.rule_induction', 'engines.cognition.shadow_testing',
    'engines.memory', 'engines.memory.temporal_integrator',
    'engines.reasoning', 'engines.reasoning.edge_inference',
    'engines.reasoning.shadow_tester', 'engines.reasoning.causal_reasoner',
    'engines.reasoning.deliberation_audit',
    'database_logger', 'database_interface', 'seed_primitives',
    'concept_discovery_engine', 'representation_learner',
    'multi_stage_matching_pipeline', 'network_intelligence_engine',
    'collective_reasoning_engine', 'mastery_system',
    'context_builder', 'learning_systems',
    'rungs', 'rungs.base',
]

for _pkg in _STUB_PACKAGES:
    if _pkg not in sys.modules:
        sys.modules[_pkg] = _StubModule(_pkg)

# -- Stub: EngineRegistry --------------------------------------------------
# Returns None for every engine attribute. Rungs already handle None gracefully.
class _StubRegistry:
    def __getattr__(self, name):
        return None
    def _get_db_interface(self):
        return None

_STUB_REGISTRY = _StubRegistry()

# -- Stub: database_logger -------------------------------------------------
def setup_database_logging(*a, **kw): pass

# -- Stub: seed_primitives -------------------------------------------------
def get_seed_primitives(): return None

print("[OK] Stubs and stdlib loaded. All rung code will run with engines=None (graceful fallback mode).")

## Environment & SDK

Locate game environment files and import the ARC-AGI-3 SDK.

In [ ]:
# -- Environment setup + SDK imports -----------------------------------------
import os, sys, time, glob, copy, subprocess

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

logging.basicConfig(level=logging.WARNING)
for noisy in ['arc_agi', 'arcengine', 'engines', 'rungs']:
    logging.getLogger(noisy).setLevel(logging.ERROR)

def _find_dir(name, fallback=None):
    """Search /kaggle/input recursively for a directory by name."""
    matches = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if matches:
        return matches[0]
    direct = f'/kaggle/input/{name}'
    if os.path.isdir(direct):
        return direct
    return fallback

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    CODE_DIR = _find_dir('bittertruth-ai', '/kaggle/input/bittertruth-ai')
    ENVS_DIR = _find_dir('environment_files',
                         '/kaggle/input/arc-prize-2026-arc-agi-3/environment_files')
    # Add code dir to path
    if CODE_DIR:
        sys.path.insert(0, CODE_DIR)

    # Install arc_agi SDK from competition wheels (required on Kaggle)
    wheels_dir = None
    for pattern in ['/kaggle/input/**/arc_agi_3_wheels',
                    '/kaggle/input/arc_agi_3_wheels']:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            wheels_dir = matches[0]
            break
    if wheels_dir:
        print(f'Installing SDK from {wheels_dir}')
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-index',
             '--find-links', wheels_dir, 'arc-agi', 'arcengine'],
            capture_output=True, text=True
        )
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        if result.returncode != 0:
            print('STDERR:', result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print('[WARN] arc_agi_3_wheels not found -- SDK may already be installed')
else:
    CODE_DIR = os.path.dirname(os.path.abspath('.'))
    ENVS_DIR = 'environment_files'

print(f'Running on: {"Kaggle" if KAGGLE else "Local"}')
print(f'Code:  {CODE_DIR} (exists={os.path.isdir(CODE_DIR) if CODE_DIR else False})')
print(f'Games: {ENVS_DIR} (exists={os.path.isdir(ENVS_DIR) if ENVS_DIR else False})')

# -- SDK imports ----------------------------------------------------------
from arc_agi import Arcade, OperationMode
from arcengine import GameAction, GameState
print('[OK] SDK imports OK')

## Arcade Initialization

Start the ARC game environments in offline mode. In competition rerun mode, connect to Kaggle's gateway instead.

In [ ]:
# -- Initialize Arcade in OFFLINE mode ----------------------------------------
# OFFLINE = reads from local environment_files/, no API calls needed.
# This satisfies the Kaggle "no internet" constraint.
import threading as _threading

_IS_COMP_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
if _IS_COMP_RERUN:
    print('COMPETITION RERUN: connecting to Kaggle gateway at gateway:8001...')
    import urllib.request as _ur
    for _attempt in range(24):
        try:
            _ur.urlopen('http://gateway:8001/api/games', timeout=5)
            break
        except Exception:
            time.sleep(5)
    os.environ['ARC_BASE_URL'] = 'http://gateway:8001'
    _arc_api_key = os.getenv('ARC_API_KEY', 'test-key-123')
    arcade = Arcade(
        operation_mode=OperationMode.ONLINE,
        arc_api_key=_arc_api_key,
        environments_dir=ENVS_DIR,
    )
    games = arcade.get_environments()
    print(f'Gateway ready. {len(games)} competition games available.')
else:
    print('Starting local game server on port 8001...')
    _srv_arcade = Arcade(
        operation_mode=OperationMode.OFFLINE,
        arc_api_key='',
        environments_dir=ENVS_DIR,
    )
    _srv_thread = _threading.Thread(
        target=lambda: _srv_arcade.listen_and_serve(competition_mode=True),
        daemon=True,
    )
    _srv_thread.start()
    time.sleep(3)
    os.environ['ARC_BASE_URL'] = 'http://localhost:8001'
    arcade = Arcade(
        operation_mode=OperationMode.COMPETITION,
        arc_api_key='local',
        environments_dir=ENVS_DIR,
    )
    games = arcade.get_environments()
    print(f'Server started. {len(games)} competition games available.')

for g in games:
    print(f'  {g.game_id}  tags={getattr(g, "tags", "?")}')

## Cell 3: Base Infrastructure

`rungs/base.py` -- DecisionRung ABC, RungResult, KnowledgeProvenance, Action6CoordinateProvider, and utility functions.

In [ ]:
"""
Rung Base Infrastructure
========================
Shared types, ABC, utilities, and Action6 coordinate system
used by all decision rung implementations.

Extracted from decision_rung_system.py Phase 4.2.
"""


import logging
import random
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import TYPE_CHECKING, Any, Dict, List, Optional, Tuple

# Set up logging with database support (Rule 2) + console output
logger = __import__("logging").getLogger(__name__)

# -- Lazy loaders (avoid circular imports) ------------------------------------

_primitives_loaded: bool = False
_seed_primitives: Any = None


def _load_primitives() -> Any:
    """Lazy-load seed primitives registry."""
    global _primitives_loaded, _seed_primitives
    if _primitives_loaded:
        return _seed_primitives

    try:
        _seed_primitives = get_seed_primitives()
        _primitives_loaded = True
        if _seed_primitives is not None:
            logger.debug(f"[RUNG-PRIMITIVES] Loaded {_seed_primitives.count()} seed primitives")
    except ImportError as e:
        logger.debug(f"[RUNG-PRIMITIVES] seed_primitives not available: {e}")
        _primitives_loaded = True
        _seed_primitives = None

    return _seed_primitives




# -- Enums --------------------------------------------------------------------

class DecisionStrategy(Enum):
    """How to combine rung outputs"""
    LADDER = "ladder"
    WEIGHTED = "weighted"
    PHASED = "phased"
    PARALLEL = "parallel"
    CONTEXT_ADAPTIVE = "context_adaptive"
    COGNITIVE = "cognitive"


# -- Data Classes -------------------------------------------------------------

@dataclass
class KnowledgeProvenance:
    """Tracks HOW knowledge became knowable - epistemological provenance.

    From 'Simultaneous Learning' theory: "Amplification != Validity"
    We need to distinguish between 'frequently tried' and 'actually validated'.

    Stages (from Knowledge Crystallization Pipeline):
    - detection: How was this pattern first identified?
    - classification: How was it named/bounded?
    - amplification: What drove its spread? (frequency vs outcome-based)
    - normalization: Is this now assumed/foundational knowledge?
    """
    detection_source: str = "unknown"
    sample_size: int = 0
    agent_diversity: int = 0
    temporal_spread_generations: float = 0.0

    validation_type: str = "frequency"
    positive_outcomes: int = 0
    negative_outcomes: int = 0

    crystallization_stage: int = 1

    resonance_games: int = 0
    resonance_score: float = 0.0

    def validity_score(self) -> float:
        """Calculate validity score - separating 'widely known' from 'actually true'."""
        total_outcomes = self.positive_outcomes + self.negative_outcomes
        outcome_ratio = self.positive_outcomes / max(1, total_outcomes)

        diversity_factor = min(1.0, self.agent_diversity / 5.0)
        temporal_factor = min(1.0, self.temporal_spread_generations / 20.0)
        resonance_factor = self.resonance_score * 0.5

        validation_multiplier = {
            'frequency': 0.5,
            'outcome_based': 0.8,
            'win_validated': 1.0,
            'cross_game': 1.2,
        }.get(self.validation_type, 0.5)

        return min(1.0, (
            outcome_ratio * 0.4 +
            diversity_factor * 0.2 +
            temporal_factor * 0.1 +
            resonance_factor * 0.3
        ) * validation_multiplier)

    def to_dict(self) -> Dict[str, Any]:
        return {
            'detection_source': self.detection_source,
            'sample_size': self.sample_size,
            'agent_diversity': self.agent_diversity,
            'temporal_spread_generations': self.temporal_spread_generations,
            'validation_type': self.validation_type,
            'positive_outcomes': self.positive_outcomes,
            'negative_outcomes': self.negative_outcomes,
            'crystallization_stage': self.crystallization_stage,
            'resonance_games': self.resonance_games,
            'resonance_score': self.resonance_score,
            'validity_score': self.validity_score(),
        }


@dataclass
class RungResult:
    """Standard output from a decision rung"""
    action: Optional[str] = None
    confidence: float = 0.0
    reason: str = ""
    weights: Optional[Dict[str, float]] = None
    metadata: Dict[str, Any] = field(default_factory=lambda: {})
    primitives_used: List[str] = field(default_factory=lambda: [])
    provenance: Optional[KnowledgeProvenance] = None
    resolved_questions: List[str] = field(default_factory=lambda: [])

    def has_suggestion(self, threshold: float = 0.0) -> bool:
        return self.action is not None and self.confidence > threshold

    def adjusted_confidence(self) -> float:
        """Confidence adjusted by provenance validity."""
        if self.provenance is None:
            return self.confidence
        return self.confidence * self.provenance.validity_score()


# -- Utility Functions --------------------------------------------------------

def filter_available_actions(actions: List[str], context: Dict[str, Any]) -> List[str]:
    """Filter action list to only those available in current game state."""
    available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
    if not available:
        return actions

    available_strs = {f'ACTION{a}' for a in available}
    filtered = [a for a in actions if a in available_strs]

    if not filtered:
        return [f'ACTION{a}' for a in available]

    return filtered


def get_random_available_action(context: Dict[str, Any]) -> str:
    """Get a random action from available actions in context."""
    available = context.get('available_actions', [1, 2, 3, 4])
    return f'ACTION{random.choice(available)}'


def get_available_action_weights(context: Dict[str, Any], default_weight: float = 1.0) -> Dict[str, float]:
    """Get a weights dict initialized to default_weight for all available actions only."""
    available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
    return {f'ACTION{a}': default_weight for a in available}


def get_available_actions_list(context: Dict[str, Any]) -> List[str]:
    """Get list of available action strings from context."""
    available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
    return [f'ACTION{a}' for a in available]


def is_action_available(action: Optional[str], context: Dict[str, Any]) -> bool:
    """Check if an action string is in the available actions for this game."""
    if action is None:
        return True
    if not isinstance(action, str) or not action.startswith('ACTION'):
        return False
    try:
        action_num = int(action.replace('ACTION', ''))
        available = context.get('available_actions', [])
        if not available:
            return True
        return action in available or action_num in available
    except (ValueError, TypeError):
        return False


def validate_action(action: Optional[str], context: Dict[str, Any]) -> Optional[str]:
    """Validate an action and return it if available, None otherwise."""
    if is_action_available(action, context):
        return action
    return None


# -- Action6 Coordinate System -----------------------------------------------

class Action6CoordinateProvider:
    """
    Centralized provider for ACTION6 coordinates.

    ACTION6 is a PARAMETERIZED action that requires explicit (x, y) coordinates.
    Every part of the system that can produce ACTION6 MUST provide coordinates.

    Coordinate System (64x64 grid):
        (0,0) ------------- (63,0)
          |                    |
          |   Y increases v    |
          |   X increases >    |
          |                    |
        (0,63) ----------- (63,63)
    """

    @staticmethod
    def get_coordinates(
        context: Dict[str, Any],
        engines: Optional[Any] = None,
        frame: Optional[Any] = None
    ) -> Dict[str, Any]:
        """Get coordinates for ACTION6 using best available strategy."""
        game_type = context.get('game_type', '')
        level = context.get('level', 1)

        # Extract actual 2D pixel data if game_state/obs was passed as frame.
        # The caller often passes the raw observation object, but Strategy 3
        # needs a List[List[int]] frame.
        if frame is not None and not isinstance(frame, list):
            raw = getattr(frame, 'frame', None) if not isinstance(frame, dict) else frame.get('frame')
            if raw is not None:
                try:
                    if hasattr(raw, 'tolist'):
                        raw = raw.tolist()
                    if isinstance(raw, list) and raw:
                        # FrameDataRaw.frame returns List[ndarray]
                        if hasattr(raw[0], 'tolist'):
                            raw = raw[0].tolist()
                        # Squeeze nested [[[pixel]]] -> [[pixel]]
                        while (isinstance(raw, list) and raw
                               and isinstance(raw[0], list) and raw[0]
                               and isinstance(raw[0][0], list)):
                            raw = raw[0]
                    frame = raw if isinstance(raw, list) else None
                except Exception:
                    frame = None
            else:
                frame = None

        # Strategy 1: Detected objects/pseudobuttons
        if engines:
            try:
                a6e = engines.action6_behavior
                if a6e and hasattr(a6e, 'get_untried_objects_for_frontier'):
                    tried_colors = context.get('tried_colors', [])
                    objects = a6e.get_untried_objects_for_frontier(
                        game_type=game_type, level=level,
                        frame=frame, tried_colors=tried_colors
                    )
                    if objects:
                        obj = objects[0]
                        return {
                            'x': obj.get('center_x', obj.get('x', 32)),
                            'y': obj.get('center_y', obj.get('y', 32)),
                            'source': 'detected_object',
                            'target_object': obj
                        }
            except Exception:
                pass

            # Strategy 2: Grid exploration targets
            try:
                va = engines.visual_analyzer
                if va and hasattr(va, 'get_grid_exploration_targets'):
                    targets = va.get_grid_exploration_targets()
                    if targets:
                        target = targets[0]
                        return {
                            'x': target.get('x', 32),
                            'y': target.get('y', 32),
                            'source': 'grid_exploration',
                            'grid_target': target
                        }
            except Exception:
                pass

        # Strategy 3: Frame analysis for interesting regions
        if frame is not None:
            try:
                game_key = f"{game_type}_L{level}"
                coords = Action6CoordinateProvider._find_interesting_region(frame, game_key)
                if coords:
                    return {**coords, 'source': 'frame_analysis'}
            except Exception:
                pass

        # Strategy 4: Random valid position
        return {
            'x': random.randint(4, 60),
            'y': random.randint(4, 60),
            'source': 'random_fallback'
        }

    # Per-game cycling state: game_key -> cycle index
    _cycle_indices: Dict[str, int] = {}

    @staticmethod
    def _find_interesting_region(frame: List[List[int]], game_key: str = "") -> Optional[Dict[str, int]]:
        """Find visually interesting regions in the frame, cycling through objects per-game."""
        if frame is None or len(frame) < 4:
            return None

        interesting_points: List[Tuple[int, int, int]] = []
        for y, row in enumerate(frame):
            for x, pixel in enumerate(row):
                val = int(pixel) if hasattr(pixel, '__int__') else pixel
                if val != 0:
                    interesting_points.append((x, y, val))

        if not interesting_points:
            return None

        color_groups: Dict[int, List[Tuple[int, int]]] = {}
        for x, y, color in interesting_points:
            if color not in color_groups:
                color_groups[color] = []
            color_groups[color].append((x, y))

        frame_area = len(frame) * (len(frame[0]) if len(frame) > 0 else 64)
        valid_groups = {
            c: pts for c, pts in color_groups.items()
            if 3 <= len(pts) <= frame_area * 0.4
        }
        if not valid_groups:
            valid_groups = color_groups

        sorted_colors = sorted(valid_groups.keys(), key=lambda c: len(valid_groups[c]), reverse=True)

        # Per-game cycling: each game tracks its own index through color groups
        current_idx = Action6CoordinateProvider._cycle_indices.get(game_key, 0)
        current_idx += 1
        Action6CoordinateProvider._cycle_indices[game_key] = current_idx

        idx = current_idx % len(sorted_colors)
        target_color = sorted_colors[idx]
        target_group = valid_groups[target_color]

        # Randomly sample from the group instead of using the center.
        # Center-targeting creates deterministic cycling that oscillates
        # puzzle state in Lights-Out games (clicking same cell twice
        # undoes the first click).
        point = random.choice(target_group)
        return {'x': point[0], 'y': point[1]}

    @staticmethod
    def enrich_result_with_coordinates(
        result: "RungResult",
        context: Dict[str, Any],
        engines: Optional[Any] = None,
        frame: Optional[Any] = None
    ) -> "RungResult":
        """Ensure a RungResult for ACTION6 has coordinates."""
        if result.action != 'ACTION6':
            return result

        metadata = result.metadata or {}
        has_coords = (
            ('x' in metadata and 'y' in metadata) or
            'pixel_position' in metadata or
            'target' in metadata or
            'grid_target' in metadata
        )

        if has_coords:
            return result

        coords = Action6CoordinateProvider.get_coordinates(context, engines, frame)
        new_metadata = {**metadata, **coords}
        return RungResult(
            action=result.action,
            confidence=result.confidence,
            reason=result.reason + f" [coords added: ({coords['x']},{coords['y']})]",
            weights=result.weights,
            metadata=new_metadata,
            provenance=result.provenance
        )

    @staticmethod
    def is_action6_game(context: Dict[str, Any]) -> bool:
        """Check if ACTION6 is the only available action (click-only game)."""
        available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
        return list(available) == [6]

    @staticmethod
    def action6_available(context: Dict[str, Any]) -> bool:
        """Check if ACTION6 is available."""
        available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
        return 6 in available


# -- DecisionRung ABC ---------------------------------------------------------

class DecisionRung(ABC):
    """
    Base class for all decision rungs.
    Each rung evaluates the current state and optionally suggests an action.

    PRIMITIVE INTEGRATION:
    Rungs can declare ``required_primitives`` to access seed primitives.
    The primitive registry is loaded lazily to avoid circular imports.
    """

    name: str = "base_rung"
    category: str = "unknown"
    default_priority: int = 50
    confidence_threshold: float = 0.3

    required_primitives: List[str] = []

    def __init__(
        self,
        core_gameplay_ref: Any = None,
        engine_registry: Optional["EngineRegistry"] = None
    ):
        self.core: Any = core_gameplay_ref
        self._engine_registry: Optional["EngineRegistry"] = engine_registry
        self.enabled = True
        self.priority_override: Optional[int] = None
        self.stats: Dict[str, Any] = {
            'calls': 0,
            'suggestions': 0,
            'accepted': 0,
            'avg_confidence': 0.0
        }
        self._primitives = None
        self._primitives_validated = False

    @property
    def engines(self):
        """Access modular engines via registry (stub in standalone mode)."""
        if self._engine_registry is not None:
            return self._engine_registry
        self._engine_registry = _STUB_REGISTRY
        return self._engine_registry

    def _ensure_primitives(self) -> bool:
        """Ensure primitives are loaded and validated."""
        if self._primitives_validated:
            return self._primitives is not None

        self._primitives = _load_primitives()
        self._primitives_validated = True

        if self.required_primitives and self._primitives:
            missing: List[str] = []
            for pname in self.required_primitives:
                if not self._primitives.get(pname):
                    missing.append(pname)
            if missing:
                logger.warning(f"[RUNG-{self.name}] Missing primitives: {missing}")

        return self._primitives is not None

    def call_primitive(self, name: str, *args: Any, **kwargs: Any) -> Any:
        """Call a seed primitive by name."""
        if not self._ensure_primitives() or self._primitives is None:
            return None

        try:
            primitive = self._primitives.get(name)
            if primitive and hasattr(primitive, 'execute'):
                return primitive.execute(*args, **kwargs)
            elif primitive and callable(getattr(primitive, 'func', None)):
                return primitive.func(*args, **kwargs)
        except Exception as e:
            logger.debug(f"[RUNG-{self.name}] Primitive {name} failed: {e}")

        return None

    def has_primitive(self, name: str) -> bool:
        """Check if a primitive is available."""
        if not self._ensure_primitives() or self._primitives is None:
            return False
        return self._primitives.get(name) is not None

    @abstractmethod
    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        """Evaluate this rung and return a result."""
        pass

    def get_priority(self) -> int:
        """Return current priority (considering override)"""
        return self.priority_override if self.priority_override is not None else self.default_priority

    def record_outcome(self, was_accepted: bool, outcome_score: float = 0.0):
        """Record whether this rung's suggestion was used and how it went"""
        self.stats['calls'] += 1
        if was_accepted:
            self.stats['accepted'] += 1
        n = self.stats['calls']
        self.stats['avg_confidence'] = (self.stats['avg_confidence'] * (n-1) + outcome_score) / n

## Emergency Rungs

Hard safety constraints: InfiniteLoopBreakerRung, CoordinateOscillationRung

In [ ]:
"""
Emergency Rungs - Hard safety constraints
=========================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


class InfiniteLoopBreakerRung(DecisionRung):
    """Emergency escape from stuck loops - EMERGENCY"""
    name = "infinite_loop_breaker"
    category = "emergency"
    default_priority = 1  # Highest priority when triggered
    confidence_threshold = 0.9

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            stuck_count = context.get('recent_stuck_count', 0)

            if stuck_count >= 15:
                # Emergency! Prefer movement actions (ACTION1-4) that haven't
                # already failed, since random ACTION6 clicks rarely unstick
                # the game and can create a self-reinforcing emergency loop.
                available = context.get('available_actions', [1, 2, 3, 4])
                failed = context.get('failed_actions', set())
                failed_nums = {int(a.replace('ACTION', '')) for a in failed if isinstance(a, str) and a.startswith('ACTION')}

                # Priority 1: untried movement actions
                movement = [a for a in available if a in (1, 2, 3, 4) and a not in failed_nums]
                if movement:
                    action = f'ACTION{random.choice(movement)}'
                # Priority 2: any untried action
                elif [a for a in available if a not in failed_nums]:
                    action = f'ACTION{random.choice([a for a in available if a not in failed_nums])}'
                # Priority 3: true last resort - anything available
                else:
                    action = get_random_available_action(context)

                return RungResult(
                    action=action,
                    confidence=0.95,
                    reason=f"EMERGENCY: Breaking infinite loop (stuck {stuck_count})",
                    metadata={'stuck_count': stuck_count, 'emergency': True}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Loop breaker failed: {e}")


class CoordinateOscillationRung(DecisionRung):
    """Detect bouncing between coordinates and break loop - EMERGENCY"""
    name = "coordinate_oscillation"
    category = "emergency"
    default_priority = 3
    confidence_threshold = 0.8

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ah = self.engines.action_handler
        if ah is None:
            return RungResult()

        try:
            if hasattr(ah, 'detect_oscillation'):
                oscillation = ah.detect_oscillation()
                if oscillation.get('oscillation_detected', False):
                    coords = oscillation.get('oscillating_coords', [])
                    if len(coords) >= 2:
                        available_list = get_available_actions_list(context)

                        # H33: For ACTION6-only games (click puzzles), pick a
                        # random unexplored coordinate instead of a different
                        # action type. The oscillation is in POSITION, not action.
                        if available_list == ['ACTION6'] or available_list == [6]:
                            frame = _get_frame(game_state)
                            if frame is not None:
                                osc_set = set()
                                for c in coords:
                                    if isinstance(c, (list, tuple)) and len(c) >= 2:
                                        osc_set.add((c[0], c[1]))
                                # Pick a random non-zero pixel NOT in oscillating set
                                candidates = []
                                try:
                                    raw = frame
                                    if hasattr(raw, 'tolist'):
                                        raw = raw.tolist()
                                    if isinstance(raw, list) and raw:
                                        while isinstance(raw[0], list) and isinstance(raw[0][0], list):
                                            raw = raw[0]
                                        for y, row in enumerate(raw):
                                            for x, val in enumerate(row):
                                                v = int(val) if hasattr(val, '__int__') else val
                                                if v > 0 and (x, y) not in osc_set:
                                                    candidates.append((x, y))
                                except Exception:
                                    pass
                                if candidates:
                                    cx, cy = random.choice(candidates)
                                    return RungResult(
                                        action='ACTION6',
                                        confidence=0.85,
                                        reason=f"H33: Breaking click oscillation, trying ({cx},{cy})",
                                        metadata={'x': cx, 'y': cy, 'oscillation': oscillation},
                                    )

                        # Original: try a different action type
                        current_action = context.get('last_action', 'ACTION1')
                        alternatives = [a for a in available_list if a != current_action]
                        if alternatives:
                            return RungResult(
                                action=random.choice(alternatives),
                                confidence=0.85,
                                reason=f"Breaking oscillation between {len(coords)} coords",
                                metadata={'oscillation': oscillation}
                            )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Coordinate oscillation failed: {e}")


# Registry of rungs in this module
RUNGS = {
    'infinite_loop_breaker': InfiniteLoopBreakerRung,
    'coordinate_oscillation': CoordinateOscillationRung,
}

## Orientation Rungs

15 rungs for understanding the environment: SurveyRung, PaletteDetectionRung, FrameInterpretationRung, etc.

In [ ]:
"""
Orientation Rungs - Understanding the world
===========================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


class SurveyRung(DecisionRung):
    """Survey the environment at level start - ORIENTATION

    Uses grid_analyzer engine to analyze frame and build survey context.
    Identifies objects, colors, and grid structure for the agent's world model.
    """
    name = "survey"
    category = "orientation"
    default_priority = 5
    confidence_threshold = 0.0  # Always runs, modifies context not action

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Check if survey already done for this level
        if context.get('survey_complete', False):
            return RungResult(confidence=0.0, reason="Survey already complete")

        # Build survey context using grid_analyzer
        try:
            survey: Dict[str, Any] = self._build_survey_context(game_state, context)
            context['survey'] = survey
            context['survey_complete'] = True

            # Genuine epistemic resolution: we now know what objects exist
            resolved = ['what_objects_exist']
            if survey.get('has_boundary'):
                resolved.append('has_boundary_structure')

            return RungResult(
                confidence=0.1,  # Low confidence - doesn't suggest action, just observes
                reason=f"Survey complete: {len(survey.get('detected_features', {}))} features detected",
                metadata={'survey': survey},
                resolved_questions=resolved,
            )
        except Exception as e:
            return RungResult(reason=f"Survey failed: {e}")

    def _build_survey_context(self, game_state: Any, context: Dict[str, Any]) -> Dict[str, Any]:
        """Build survey context from frame analysis using grid_analyzer."""
        survey: Dict[str, Any] = {
            'detected_features': {},
            'unique_colors': set(),
            'object_count': 0,
            'grid_size': (0, 0),
            'has_boundary': False
        }

        # Get frame from game_state
        frame = _get_frame(game_state)
        if frame is None:
            return survey

        # Convert numpy array to list if needed
        if hasattr(frame, 'tolist'):
            frame = frame.tolist()

        # Analyze grid structure
        if isinstance(frame, list) and len(frame) > 0:
            survey['grid_size'] = (len(frame), len(frame[0]) if len(frame) > 0 else 0)

            # Find unique colors and objects
            colors: set = set()
            object_positions: Dict[int, List[tuple]] = {}

            for y, row in enumerate(frame):
                for x, color in enumerate(row):
                    if color > 0:  # Non-background
                        colors.add(color)
                        if color not in object_positions:
                            object_positions[color] = []
                        object_positions[color].append((y, x))

            survey['unique_colors'] = colors
            survey['object_count'] = len(object_positions)

            # Detect features for each color
            for color, positions in object_positions.items():
                feature = {
                    'color': color,
                    'pixel_count': len(positions),
                    'positions': positions[:10],  # Sample for memory
                    'is_single': len(positions) <= 4,
                    'is_large': len(positions) > 20
                }

                # Check if on boundary (potential boundary marker)
                boundary_positions = [p for p in positions if p[0] == 0 or p[1] == 0
                                      or p[0] == len(frame) - 1 or p[1] == len(frame[0]) - 1]
                if len(boundary_positions) > len(positions) * 0.5:
                    feature['likely_boundary'] = True
                    survey['has_boundary'] = True

                survey['detected_features'][f'color_{color}'] = feature

        # Try to use grid_analyzer for more sophisticated analysis
        grid_analyzer = self.engines.grid_analyzer
        if grid_analyzer and frame:
            try:
                if hasattr(grid_analyzer, 'analyze_grid_structure'):
                    analysis = grid_analyzer.analyze_grid_structure(frame)
                    survey['grid_analysis'] = analysis
            except Exception:
                pass  # Grid analyzer enhancement is optional

        return survey


class QuestioningRung(DecisionRung):
    """Q1-Q9 questioning engine - can BLOCK actions - ORIENTATION"""
    name = "questioning_engine"
    category = "orientation"
    default_priority = 10
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sme = self.engines.scientific_method_engine
        if sme is None:
            return RungResult()

        try:
            if not hasattr(sme, 'questioning_engine'):
                return RungResult()

            qe = sme.questioning_engine
            blocking_questions: List[Any] = qe.get_blocking_questions() if hasattr(qe, 'get_blocking_questions') else []

            if blocking_questions:
                # Q4, Q9, or META is blocking - force specific action types
                allowed_actions = qe.get_allowed_actions(blocking_questions)
                return RungResult(
                    action=random.choice(allowed_actions) if allowed_actions else None,
                    confidence=0.8,
                    reason=f"Blocked by questions: {blocking_questions}",
                    metadata={'blocking_questions': blocking_questions, 'allowed': allowed_actions}
                )
            return RungResult(confidence=0.0)
        except Exception as e:
            return RungResult(reason=f"Questioning failed: {e}")


class ExplorationPhaseRung(DecisionRung):
    """Phase-based exploration forcing - ORIENTATION"""
    name = "exploration_phase"
    category = "orientation"
    default_priority = 22
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            budget_used = context.get('budget_used_percent', 0)
            coverage = context.get('coverage_percent', 0)

            # Discovery phase: 0-30% budget AND low coverage
            if budget_used < 0.3 and coverage < 0.3:
                exploration_actions = filter_available_actions(
                    ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION6'], context
                )
                chosen_action = random.choice(exploration_actions)

                # If ACTION6 (click), provide coordinates from visual_analyzer
                metadata: Dict[str, Any] = {
                    'phase': 'discovery',
                    'budget_used': budget_used,
                    'coverage': coverage
                }
                if chosen_action == 'ACTION6':
                    # Try to get grid exploration targets for coordinates
                    va = self.engines.visual_analyzer if self.engines else None
                    if va and hasattr(va, 'get_grid_exploration_targets'):
                        targets = va.get_grid_exploration_targets()
                        if targets:
                            target = targets[0]
                            metadata['x'] = target.get('x', 32)
                            metadata['y'] = target.get('y', 32)
                            metadata['grid_target'] = target
                        else:
                            # Fallback: random position in 64x64 grid
                            metadata['x'] = random.randint(4, 60)
                            metadata['y'] = random.randint(4, 60)
                    else:
                        # No visual analyzer - use random position
                        metadata['x'] = random.randint(4, 60)
                        metadata['y'] = random.randint(4, 60)

                # Dynamic confidence: starts at 0.55 (fresh game, unknown
                # territory) and decays as exploration exhausts itself.
                # Without this, the hardcoded 0.6 exceeds the router's 0.50
                # commit threshold every time, monopolising the first
                # iteration and preventing any other rung from executing.
                #
                # Decay factors:
                #   - budget_used: you've spent actions without advancing
                #   - coverage: grid has been explored (less to discover)
                #   - action_count penalty: even at 0% budget_used,
                #     repeated calls without progress should decay.
                action_count = context.get('action_count', 0)
                # How many actions have occurred since last level change?
                # Proxy: if score hasn't increased, exploration isn't working.
                actions_since_progress = action_count - context.get(
                    'last_progress_action', 0
                )
                # Decay: steep drop from budget consumption, mild from
                # action repetition (each action without progress = -0.005).
                staleness_penalty = min(0.3, actions_since_progress * 0.005)
                confidence = max(
                    0.15,  # Floor: never fully block, but yield to better rungs
                    0.55 - budget_used * 0.8 - coverage * 0.5 - staleness_penalty
                )

                return RungResult(
                    action=chosen_action,
                    confidence=confidence,
                    reason=f"Discovery phase: budget={budget_used:.0%}, coverage={coverage:.0%}, conf={confidence:.2f}",
                    metadata=metadata
                )
            return RungResult(metadata={'phase': 'intermediate' if budget_used < 0.7 else 'final'})
        except Exception as e:
            return RungResult(reason=f"Exploration phase failed: {e}")


class FrustrationDetectionRung(DecisionRung):
    """Detect stuck agents and trigger network signals - ORIENTATION"""
    name = "frustration_detection"
    category = "orientation"
    default_priority = 13
    confidence_threshold = 0.6

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        fd = self.engines.frustration_detector
        if fd is None:
            return RungResult()

        try:
            if hasattr(fd, 'is_frustrated'):
                frustration = fd.is_frustrated()
                if frustration.get('is_frustrated', False):
                    # Force exploration when frustrated
                    return RungResult(
                        action=get_random_available_action(context),
                        confidence=0.65,
                        reason=f"Frustration detected: {frustration.get('reason', 'unknown')}",
                        metadata={'frustration': frustration}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Frustration detection failed: {e}")


class PaletteDetectionRung(DecisionRung):
    """
    TWO-STAGE DECOMPOSITION: Stage 1 - Detect palette/legend blocks.

    Based on ARC-AGI-2 insights (76.11% success):
    "~70% of models cluster around wrong solutions where they use the 'palette'
    as top-to-bottom instead of inside-out."

    This rung runs EARLY to:
    1. Extract all objects from the frame (hollow frames, fills, irregular)
    2. Detect multi-colored palette/legend blocks
    3. Determine correct mapping direction (inside-out vs top-to-bottom)
    4. Populate context for downstream rungs

    Sets context fields:
    - detected_palette: PaletteInfo dict or None
    - extracted_objects: Dict with categorized objects
    - detected_transformations: List of detected transformation rules
    """
    name = "palette_detection"
    category = "orientation"
    default_priority = 3  # Very early - before frame_interpretation
    confidence_threshold = 0.0  # Context setter, not action suggester

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._palette_detector: Optional[Any] = None
        self._cached_analysis: Dict[str, Any] = {}  # Cache by frame hash

    def _get_palette_detector(self) -> Optional[Any]:
        """Lazy-load palette detector."""
        if self._palette_detector is None:
            try:
                from engines.perception.palette_detector import PaletteDetector
                self._palette_detector = PaletteDetector()
            except ImportError as e:
                logger.debug(f"[PALETTE-RUNG] PaletteDetector not available: {e}")
        return self._palette_detector

    def _frame_to_hash(self, frame: Any) -> str:
        """Generate hash for frame caching."""
        if frame is None:
            return "none"
        try:
            import hashlib
            if hasattr(frame, 'tobytes'):
                return hashlib.md5(frame.tobytes()).hexdigest()[:16]
            return hashlib.md5(str(frame).encode()).hexdigest()[:16]
        except Exception:
            return "unknown"

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Get frame from game_state
        frame = _get_frame(game_state)
        if frame is None:
            return RungResult(
                confidence=0.0,
                reason="No frame available for palette detection"
            )

        # Check cache
        frame_hash = self._frame_to_hash(frame)
        if frame_hash in self._cached_analysis:
            cached = self._cached_analysis[frame_hash]
            context['detected_palette'] = cached.get('detected_palette')
            context['extracted_objects'] = cached.get('extracted_objects')
            context['detected_transformations'] = cached.get('detected_transformations')
            return RungResult(
                confidence=0.0,  # Context-setter only: no action to suggest
                reason=f"Cached palette analysis: {'found' if cached.get('detected_palette') else 'none'}",
                metadata={'cached': True, **cached}
            )

        # Get detector
        detector = self._get_palette_detector()
        if detector is None:
            return RungResult(
                confidence=0.0,
                reason="Palette detector not available"
            )

        try:
            import numpy as np
            frame_arr = np.array(frame) if not isinstance(frame, np.ndarray) else frame
        except Exception as e:
            return RungResult(
                confidence=0.0,
                reason=f"Could not convert frame to array: {e}"
            )

        # Stage 1: Extract objects
        try:
            extracted = detector.extract_objects(frame_arr)
            extracted_dict = {
                'palettes': [o.to_dict() for o in extracted.get('palettes', [])],
                'hollow_frames': [o.to_dict() for o in extracted.get('hollow_frames', [])],
                'filled_shapes': [o.to_dict() for o in extracted.get('filled_shapes', [])],
                'irregular_shapes': [o.to_dict() for o in extracted.get('irregular_shapes', [])],
                'object_count': len(extracted.get('all_objects', [])),
            }
        except Exception as e:
            extracted_dict = {'error': str(e), 'object_count': 0}
            extracted = {}

        # Stage 1.5: Detect palette
        palette_dict = None
        try:
            palette = detector.detect_palette(frame_arr)
            if palette:
                palette_dict = palette.to_dict()
        except Exception as e:
            logger.debug(f"[PALETTE-RUNG] Palette detection failed: {e}")

        # Stage 2: Detect transformations
        transformations = []
        try:
            if palette and extracted:
                trans = detector.detect_transformations(extracted, palette, frame_arr)
                transformations = [t.to_dict() for t in trans]
        except Exception as e:
            logger.debug(f"[PALETTE-RUNG] Transformation detection failed: {e}")

        # Set context
        context['detected_palette'] = palette_dict
        context['extracted_objects'] = extracted_dict
        context['detected_transformations'] = transformations

        # Cache result
        analysis = {
            'detected_palette': palette_dict,
            'extracted_objects': extracted_dict,
            'detected_transformations': transformations,
        }
        self._cached_analysis[frame_hash] = analysis

        # Limit cache size
        if len(self._cached_analysis) > 100:
            oldest = list(self._cached_analysis.keys())[:50]
            for k in oldest:
                del self._cached_analysis[k]

        # Build reason
        parts = []
        if palette_dict:
            parts.append(f"palette={palette_dict.get('palette_type', 'unknown')}")
            parts.append(f"direction={palette_dict.get('mapping_direction', 'unknown')}")
            parts.append(f"conf={palette_dict.get('confidence', 0):.2f}")
        else:
            parts.append("no_palette")

        parts.append(f"objects={extracted_dict.get('object_count', 0)}")
        if transformations:
            parts.append(f"transforms={len(transformations)}")

        return RungResult(
            confidence=0.0,  # Context-setter only: enriches context, never suggests action
            reason=f"Two-stage analysis: {', '.join(parts)}",
            metadata={
                'palette_found': palette_dict is not None,
                'object_count': extracted_dict.get('object_count', 0),
                'transformation_count': len(transformations),
                'analysis': analysis,
            }
        )

    def clear_cache(self):
        """Clear analysis cache (call on game/level change)."""
        self._cached_analysis.clear()


class SparseGridRung(DecisionRung):
    """
    SPARSE GRID REPRESENTATION: Efficient frame analysis for pattern matching.

    Converts frames to sparse representation (only non-background cells) for:
    1. Efficient structural comparison between frames
    2. Position-invariant pattern hashing
    3. Color-invariant pattern matching
    4. Connected component extraction

    Sets context fields:
    - sparse_grid: SparseGrid object for current frame
    - sparse_hash: Structural hash of current frame
    - sparse_cell_count: Number of non-background cells
    - sparse_colors: Set of colors used (excluding background)
    - sparse_components: List of connected component bounding boxes
    """
    name = "sparse_grid"
    category = "orientation"
    default_priority = 3  # Very early - alongside palette_detection
    confidence_threshold = 0.0  # Context setter, not action suggester

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._cached_sparse: Dict[str, Any] = {}  # Cache by frame hash
        self._previous_sparse: Optional[Any] = None  # For diff calculation

    def _frame_to_hash(self, frame: Any) -> str:
        """Generate hash for frame caching."""
        if frame is None:
            return "none"
        try:
            import hashlib
            if hasattr(frame, 'tobytes'):
                return hashlib.md5(frame.tobytes()).hexdigest()[:16]
            return hashlib.md5(str(frame).encode()).hexdigest()[:16]
        except Exception:
            return "unknown"

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Get frame from game_state
        frame = _get_frame(game_state)
        if frame is None:
            return RungResult(
                confidence=0.0,
                reason="No frame available for sparse grid"
            )

        # Check cache
        frame_hash = self._frame_to_hash(frame)
        if frame_hash in self._cached_sparse:
            cached = self._cached_sparse[frame_hash]
            context['sparse_grid'] = cached.get('sparse_grid')
            context['sparse_hash'] = cached.get('sparse_hash')
            context['sparse_cell_count'] = cached.get('sparse_cell_count')
            context['sparse_colors'] = cached.get('sparse_colors')
            context['sparse_components'] = cached.get('sparse_components')
            context['sparse_diff'] = cached.get('sparse_diff')
            return RungResult(
                confidence=0.1,
                reason=f"Cached sparse: {cached.get('sparse_cell_count', 0)} cells",
                metadata={'cached': True, **cached}
            )

        # Import sparse grid module
        try:
            from engines.perception.sparse_grid import sparse_from_frame
        except ImportError as e:
            return RungResult(
                confidence=0.0,
                reason=f"Sparse grid module not available: {e}"
            )

        try:
            import numpy as np
            frame_arr = np.array(frame) if not isinstance(frame, np.ndarray) else frame
        except Exception as e:
            return RungResult(
                confidence=0.0,
                reason=f"Could not convert frame to array: {e}"
            )

        # Create sparse grid
        try:
            sparse = sparse_from_frame(frame_arr)
            sparse_hash = sparse.structural_hash()
            cell_count = len(sparse)
            _raw_colors = sparse.colors  # Property, not method
            colors = _raw_colors if isinstance(_raw_colors, (set, list, tuple, frozenset)) else set()

            # Extract connected components
            components = []
            try:
                comps = sparse.extract_connected_components()
                for comp in comps:
                    bbox = comp.bounding_box  # Property, not method
                    if bbox:
                        components.append({
                            'min_y': bbox[0], 'min_x': bbox[1],
                            'max_y': bbox[2], 'max_x': bbox[3],
                            'cell_count': len(comp),
                            'colors': list(comp.colors) if isinstance(comp.colors, (set, list, tuple, frozenset)) else [],
                        })
            except Exception as e:
                logger.debug(f"[SPARSE-RUNG] Component extraction failed: {e}")

            # Calculate diff from previous frame
            diff_info = None
            if self._previous_sparse is not None:
                try:
                    diff = self._previous_sparse.diff(sparse)  # Use method, not function
                    diff_info = {
                        'added_count': len(diff.added),
                        'removed_count': len(diff.removed),
                        'changed_count': len(diff.changed),
                        'total_changes': diff.total_changes,  # Correct property name
                    }
                except Exception as e:
                    logger.debug(f"[SPARSE-RUNG] Diff calculation failed: {e}")

            # Store for next diff
            self._previous_sparse = sparse

        except Exception as e:
            return RungResult(
                confidence=0.0,
                reason=f"Sparse grid creation failed: {e}"
            )

        # Set context
        context['sparse_grid'] = sparse
        context['sparse_hash'] = sparse_hash
        context['sparse_cell_count'] = cell_count
        context['sparse_colors'] = colors
        context['sparse_components'] = components
        context['sparse_diff'] = diff_info

        # Cache result
        sparse_data = {
            'sparse_grid': sparse,
            'sparse_hash': sparse_hash,
            'sparse_cell_count': cell_count,
            'sparse_colors': colors,
            'sparse_components': components,
            'sparse_diff': diff_info,
        }
        self._cached_sparse[frame_hash] = sparse_data

        # Limit cache size
        if len(self._cached_sparse) > 100:
            oldest = list(self._cached_sparse.keys())[:50]
            for k in oldest:
                del self._cached_sparse[k]

        # Build reason
        parts = [f"cells={cell_count}", f"colors={len(colors)}"]
        if components:
            parts.append(f"components={len(components)}")
        if diff_info:
            parts.append(f"delta={diff_info['total_changes']}")

        return RungResult(
            confidence=0.1,  # Low - context setter
            reason=f"Sparse grid: {', '.join(parts)}",
            metadata={
                'sparse_hash': sparse_hash,
                'cell_count': cell_count,
                'color_count': len(colors),
                'component_count': len(components),
                'diff_info': diff_info,
            }
        )

    def clear_cache(self):
        """Clear sparse cache (call on game/level change)."""
        self._cached_sparse.clear()
        self._previous_sparse = None


class FrameInterpretationRung(DecisionRung):
    """
    HIGH PRIORITY: Interpret dramatic frame changes and set context.

    This rung sets context flags for downstream rungs based on frame delta
    magnitude. Does NOT suppress actions - the ARC API is blocking so spam
    is impossible anyway.

    Sets context flags:
    - likely_physics_game: True if large frame delta
    - expect_large_deltas: True if physics signature detected
    - detected_process_type: The type of process observed
    """
    name = "frame_interpretation"
    category = "orientation"
    default_priority = 4  # Early, after emergency rungs
    confidence_threshold = 0.0  # Context setter, not action suggester

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._event_detector: Optional[Any] = None
        self._recent_process_types: List[str] = []

    def _get_event_detector(self) -> Optional[Any]:
        """Lazy-load event detector."""
        if self._event_detector is None:
            try:
                from engines.perception.event_detector import EventDetector
                self._event_detector = EventDetector()
            except ImportError:
                pass
        return self._event_detector

    def _has_physics_signature(self, events: List[Any]) -> bool:
        """Check if events show physics-like behavior."""
        if not events:
            return False

        movement_count = sum(
            1 for e in events
            if hasattr(e, 'event_type') and str(e.event_type) == 'MOVEMENT'
        )
        return movement_count >= 3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        delta_count = context.get('frame_delta_count', 0)
        recent_events = context.get('recent_events', [])

        # Interpret dramatic changes
        if delta_count > 500:  # Significant frame change
            # Annotate context for downstream rungs
            context['likely_physics_game'] = True
            context['expect_large_deltas'] = True

            # Check for physics signature
            if recent_events and self._has_physics_signature(recent_events):
                context['detected_process_type'] = 'PHYSICS_SIMULATION'
                self._recent_process_types.append('PHYSICS_SIMULATION')

            return RungResult(
                confidence=0.0,  # No action suggestion
                reason=f"Large frame delta ({delta_count}) - likely physics/animation game",
                metadata={
                    'delta_count': delta_count,
                    'physics_signature': bool(recent_events and self._has_physics_signature(recent_events)),
                    'recent_process_types': self._recent_process_types[-5:] if self._recent_process_types else [],
                }
            )

        # Small delta - probably direct control game
        if delta_count > 0 and delta_count < 100:
            context['likely_direct_control'] = True

        return RungResult()


class BreakthroughBudgetRung(DecisionRung):
    """Dynamic action allocation based on breakthrough potential - ORIENTATION"""
    name = "breakthrough_budget"
    category = "orientation"
    default_priority = 6
    confidence_threshold = 0.0  # Context modifier, not action suggester

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        allocator = self.engines.breakthrough_allocator
        if allocator is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')

            if hasattr(allocator, 'get_budget'):
                budget = allocator.get_budget(game_type)
                context['action_budget'] = budget.get('per_level', 400)
                context['total_budget'] = budget.get('total', 2000)
                context['budget_phase'] = budget.get('phase', 'DISCOVERY')

                return RungResult(
                    confidence=0.1,  # Low - doesn't suggest action
                    reason=f"Budget phase: {budget.get('phase', 'DISCOVERY')}, per_level={budget.get('per_level', 400)}",
                    metadata={'budget': budget}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Breakthrough budget failed: {e}")


class RegulatorySignalRung(DecisionRung):
    """Network homeostasis through distributed signals - ORIENTATION"""
    name = "regulatory_signal"
    category = "orientation"
    default_priority = 7
    confidence_threshold = 0.0  # Context modifier

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        re = self.engines.regulatory_engine
        if re is None:
            return RungResult()

        try:
            if hasattr(re, 'get_active_signals'):
                signals = re.get_active_signals()

                # Apply signal effects to context
                for signal in signals:
                    if signal.get('type') == 'diversity_stress':
                        context['knowledge_diversity_boost'] = context.get('knowledge_diversity_boost', 0) + 0.15
                    elif signal.get('type') == 'metabolism_stress':
                        context['action_budget_multiplier'] = context.get('action_budget_multiplier', 1.0) + 0.1
                    elif signal.get('type') == 'exploration_need':
                        context['mutation_rate'] = context.get('mutation_rate', 0) + 0.05

                if signals:
                    return RungResult(
                        confidence=0.1,
                        reason=f"Regulatory signals: {len(signals)} active",
                        metadata={'signals': signals}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Regulatory signal failed: {e}")


class GridExplorationRung(DecisionRung):
    """Systematic 8x8 grid walking when stuck - EXPLORATION"""
    name = "grid_exploration"
    category = "orientation"
    default_priority = 47
    confidence_threshold = 0.3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        va = self.engines.visual_analyzer
        if va is None:
            return RungResult()

        try:
            # ACTION6 is typically 'click' - only suggest if available
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            if 6 not in available:
                return RungResult()  # Click not available for this game

            if hasattr(va, 'get_grid_exploration_targets'):
                targets = va.get_grid_exploration_targets()
                if targets:
                    target = targets[0]
                    return RungResult(
                        action='ACTION6',
                        confidence=0.35,
                        reason=f"Grid exploration: ({target.get('x', 0)}, {target.get('y', 0)}) - systematic search",
                        metadata={'grid_target': target, 'grid_index': va.grid_walking_index if hasattr(va, 'grid_walking_index') else 0}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Grid exploration failed: {e}")


class AffordanceDetectionRung(DecisionRung):
    """Wire seed affordance primitives into gameplay decisions - ORIENTATION

    The AFFORDANCE category (is_reference, is_interactive, is_obstacle,
    is_container, is_tool, is_movable) exists in seed_primitives.py but
    was NEVER queried by any decision rung.

    This rung:
    1. Runs affordance primitives on objects detected in the frame
    2. Tags objects with affordance labels (reference, interactive, obstacle)
    3. Injects affordance data into context for downstream rungs
    4. Specifically detects reference objects (CRITICAL for FT09)

    ROOT CAUSE ADDRESSED: "No is_reference primitive active" - the entire
    affordance detection category was registered but disconnected.
    """
    name = "affordance_detection"
    category = "orientation"
    default_priority = 8
    confidence_threshold = 0.3
    required_primitives = [
        'is_reference', 'is_interactive', 'is_obstacle',
        'is_container', 'is_tool', 'is_movable'
    ]

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Cache affordances per game_key to avoid recomputing every frame
        self._affordance_cache: Dict[str, Dict[str, List[str]]] = {}
        # Track interaction history for is_interactive primitive
        self._interaction_history: List[Dict[str, Any]] = []
        # Track rule history for is_reference primitive
        self._rule_history: List[Dict[str, Any]] = []
        # Track effect history for is_tool primitive
        self._effect_history: List[Dict[str, Any]] = []
        # Discovered objects by game
        self._object_registry: Dict[str, List[Dict[str, Any]]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        """Detect affordances and inject into context. Does NOT suggest actions."""
        frame = _get_frame(game_state)
        if frame is None:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Detect distinct objects in frame
        objects = self._detect_objects(frame)
        if not objects:
            return RungResult()

        self._object_registry[game_key] = objects

        # Run affordance primitives on each object
        affordances: Dict[str, List[str]] = {
            'reference': [],
            'interactive': [],
            'obstacle': [],
            'container': [],
            'tool': [],
            'movable': [],
        }

        for obj in objects:
            obj_id = obj.get('object_id', '')

            # is_obstacle: checks frame for blocking behavior
            if self.call_primitive('is_obstacle', obj_id, frame):
                affordances['obstacle'].append(obj_id)

            # is_interactive: checks interaction history
            if self.call_primitive('is_interactive', obj_id, self._interaction_history):
                affordances['interactive'].append(obj_id)

            # is_reference: checks rule history
            if self.call_primitive('is_reference', obj_id, frame, self._rule_history):
                affordances['reference'].append(obj_id)

            # is_tool: checks effect history
            if self.call_primitive('is_tool', obj_id, self._effect_history):
                affordances['tool'].append(obj_id)

        # Inject affordance data into context for downstream rungs
        context['affordances'] = affordances
        context['detected_objects'] = objects
        context['reference_objects'] = affordances['reference']
        context['interactive_objects'] = affordances['interactive']
        context['obstacle_objects'] = affordances['obstacle']

        # Heuristic: if no interaction history yet but we see distinct colored
        # objects that don't move, they might be reference objects
        if not affordances['reference'] and len(objects) > 2:
            # Objects with unique colors that don't appear elsewhere could be references
            color_counts: Dict[int, int] = {}
            for obj in objects:
                c = obj.get('color', 0)
                color_counts[c] = color_counts.get(c, 0) + 1
            unique_color_objs = [
                o for o in objects
                if color_counts.get(o.get('color', 0), 0) == 1
            ]
            if unique_color_objs:
                context['potential_reference_objects'] = unique_color_objs

        # This rung is informational - enriches context, doesn't suggest action
        if affordances['reference']:
            return RungResult(
                reason=f"Affordances detected: {sum(len(v) for v in affordances.values())} tagged objects, {len(affordances['reference'])} reference objects",
                metadata={'affordances': affordances}
            )

        return RungResult(
            reason=f"Affordances: {sum(len(v) for v in affordances.values())} tagged",
            metadata={'affordances': affordances}
        )

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Update interaction and effect histories for affordance detection."""
        if frame_before is None or frame_after is None:
            return

        frame_changed = False
        try:
            if isinstance(frame_before, list) and isinstance(frame_after, list):
                frame_changed = frame_before != frame_after
            else:
                import numpy as np
                frame_changed = not np.array_equal(frame_before, frame_after)
        except Exception:
            pass

        # Update interaction history
        if action == 'ACTION6':
            x, y = action_data.get('x', 0), action_data.get('y', 0)
            # Find which object was at (x, y)
            game_type = context.get('game_type', '')
            level = context.get('level', 1)
            game_key = f"{game_type}_L{level}"
            objects = self._object_registry.get(game_key, [])
            for obj in objects:
                if self._point_in_object(x, y, obj):
                    self._interaction_history.append({
                        'object_id': obj['object_id'],
                        'action': action,
                        'responded': frame_changed,
                        'x': x, 'y': y,
                    })
                    if frame_changed:
                        self._effect_history.append({
                            'tool_object': obj['object_id'],
                            'caused_effect': True,
                        })
                    break

        elif frame_changed and action in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4']:
            # Directional movement that caused change - might have interacted with tile
            self._interaction_history.append({
                'object_id': f'tile_at_movement_{action}',
                'action': action,
                'responded': True,
            })

            # Check if significant state change happened (potential rule discovery)
            try:
                change_count = self._count_frame_changes(frame_before, frame_after)
                if change_count > 10:  # Significant state change
                    self._rule_history.append({
                        'reference_object': f'tile_at_movement_{action}',
                        'rule_confidence': 0.5,
                        'change_count': change_count,
                    })
            except Exception:
                pass

    def _detect_objects(self, frame: List[List[int]]) -> List[Dict[str, Any]]:
        """Detect distinct colored objects in the frame."""
        if frame is None or (isinstance(frame, list) and len(frame) == 0):
            return []

        color_pixels: Dict[int, List[Tuple[int, int]]] = {}
        try:
            for y, row in enumerate(frame):
                for x, pixel in enumerate(row):
                    val = int(pixel) if hasattr(pixel, '__int__') else pixel
                    if val != 0:
                        if val not in color_pixels:
                            color_pixels[val] = []
                        color_pixels[val].append((x, y))
        except Exception:
            return []

        objects = []
        frame_area = len(frame) * (len(frame[0]) if len(frame) > 0 else 64)
        for color, pixels in color_pixels.items():
            size = len(pixels)
            if size < 2 or size > frame_area * 0.5:
                continue
            avg_x = sum(p[0] for p in pixels) // size
            avg_y = sum(p[1] for p in pixels) // size
            min_x = min(p[0] for p in pixels)
            max_x = max(p[0] for p in pixels)
            min_y = min(p[1] for p in pixels)
            max_y = max(p[1] for p in pixels)
            objects.append({
                'object_id': f'color_{color}',
                'color': color,
                'center_x': avg_x,
                'center_y': avg_y,
                'size': size,
                'bbox': (min_x, min_y, max_x, max_y),
                'positions': pixels if size < 200 else [],  # Don't store huge position lists
            })

        return objects

    @staticmethod
    def _point_in_object(x: int, y: int, obj: Dict[str, Any]) -> bool:
        """Check if point is within object bounding box."""
        bbox = obj.get('bbox')
        if bbox:
            return bbox[0] <= x <= bbox[2] and bbox[1] <= y <= bbox[3]
        return False

    @staticmethod
    def _count_frame_changes(frame_before: Any, frame_after: Any) -> int:
        """Count number of pixel changes between frames."""
        count = 0
        try:
            if isinstance(frame_before, list):
                for y in range(min(len(frame_before), len(frame_after))):
                    for x in range(min(len(frame_before[y]), len(frame_after[y]))):
                        if frame_before[y][x] != frame_after[y][x]:
                            count += 1
            else:
                import numpy as np
                count = int(np.sum(np.array(frame_before) != np.array(frame_after)))
        except Exception:
            pass
        return count


class ControlTrackerRung(DecisionRung):
    """Track which objects the agent controls - ORIENTATION (self-model)

    Uses engines/self_model/control_tracker.py to:
    1. Track action-movement correlations to identify controlled objects
    2. Provide "I am this object" identity information
    3. Suggest actions that move the controlled object toward goals

    This is CRITICAL for agents to understand their embodiment in the game.
    """
    name = "control_tracker"
    category = "orientation"
    default_priority = 8  # Early - need to know what we control
    confidence_threshold = 0.3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ct = self.engines.control_tracker
        if ct is None:
            return RungResult()

        try:
            game_id = context.get('game_id', context.get('game_type', ''))
            level = context.get('level', context.get('level_number', 1))

            if not game_id:
                return RungResult()

            # Get controlled objects for this game/level
            if hasattr(ct, 'get_controlled_objects'):
                controlled = ct.get_controlled_objects(game_id, level)

                if controlled:
                    # We know what we control - add to context for other rungs
                    best = controlled[0]  # Highest confidence
                    action_map = best.action_map

                    # If we have a goal/target, suggest action to move toward it
                    target = context.get('target_position') or context.get('goal_position')
                    player_pos = context.get('player_position')

                    if target and player_pos:
                        dx = target[0] - player_pos[0]
                        dy = target[1] - player_pos[1]

                        # Find action that moves in correct direction
                        for action, direction in action_map.items():
                            if direction == 'right' and dx > 0:
                                return RungResult(
                                    action=action,
                                    confidence=0.6,
                                    reason=f"Move {best.object_id} right toward target",
                                    metadata={'controlled_object': best.to_dict(), 'direction': 'right'}
                                )
                            elif direction == 'left' and dx < 0:
                                return RungResult(
                                    action=action,
                                    confidence=0.6,
                                    reason=f"Move {best.object_id} left toward target",
                                    metadata={'controlled_object': best.to_dict(), 'direction': 'left'}
                                )
                            elif direction == 'down' and dy > 0:
                                return RungResult(
                                    action=action,
                                    confidence=0.6,
                                    reason=f"Move {best.object_id} down toward target",
                                    metadata={'controlled_object': best.to_dict(), 'direction': 'down'}
                                )
                            elif direction == 'up' and dy < 0:
                                return RungResult(
                                    action=action,
                                    confidence=0.6,
                                    reason=f"Move {best.object_id} up toward target",
                                    metadata={'controlled_object': best.to_dict(), 'direction': 'up'}
                                )

                    # No target - just return info about what we control
                    return RungResult(
                        confidence=0.3,
                        reason=f"Control tracker: {best.object_id} ({best.confidence.value})",
                        metadata={
                            'controlled_objects': [c.to_dict() for c in controlled],
                            'primary_control': best.to_dict()
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Control tracker failed: {e}")


class ImaginationBudgetRung(DecisionRung):
    """Allocate computational budget based on novelty - ORIENTATION"""
    name = "imagination_budget"
    category = "orientation"
    default_priority = 4
    confidence_threshold = 0.0  # Context modifier

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ib = self.engines.imagination_budget
        if ib is None:
            return RungResult()

        try:
            if hasattr(ib, 'calculate_budget'):
                budget = ib.calculate_budget(
                    is_novel=context.get('is_novel_game', False),
                    is_frontier=context.get('frontier_mode', False),
                    surprise_score=context.get('surprise_score', 0)
                )

                context['imagination_budget_remaining'] = budget.get('total', 0.5)
                context['question_tier'] = budget.get('tier', 'Q1')

                return RungResult(
                    confidence=0.1,
                    reason=f"Imagination budget: {budget.get('total', 0.5):.2f}, tier={budget.get('tier', 'Q1')}",
                    metadata={'budget': budget}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Imagination budget failed: {e}")


class NetworkExplorationStatsRung(DecisionRung):
    """Track coverage, identify coldspots/hotspots - ORIENTATION"""
    name = "network_exploration_stats"
    category = "orientation"
    default_priority = 9
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        net = self.engines.network_exploration_tracker
        if net is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            if hasattr(net, 'get_exploration_stats'):
                stats = net.get_exploration_stats(game_type, level)

                context['coverage_percent'] = stats.get('coverage_percent', 0)

                # If there are coldspots, bias toward them
                coldspots = stats.get('coldspots', [])
                if coldspots:
                    direction = stats.get('recommended_direction')
                    direction_map = {'north': 'ACTION1', 'south': 'ACTION2', 'west': 'ACTION3', 'east': 'ACTION4'}
                    if direction and direction in direction_map:
                        return RungResult(
                            action=direction_map[direction],
                            confidence=0.45,
                            reason=f"Exploring coldspot: {direction}, coverage={stats.get('coverage_percent', 0):.0%}",
                            metadata={'stats': stats, 'coldspots': len(coldspots)}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Network exploration stats failed: {e}")


# =============================================================================
# WIRED METACOGNITION RUNGS (Feb 2026)


class SelfTrustBoostRung(DecisionRung):
    """Manage wA (self-trust) based on context - ORIENTATION

    Wires: engines/consciousness/i_thread.py:
        - boost_self_trust()
        - restore_self_trust()

    On frontier levels (no winning sequences), boosts self-trust to encourage
    exploration over network following. When sequences become available,
    restores normal trust balance.

    This implements the Two Streams principle: trust yourself more when
    network wisdom doesn't apply.
    """
    name = "self_trust_boost"
    category = "orientation"
    default_priority = 3  # Very early - affects all downstream decisions
    confidence_threshold = 0.0  # Always runs, just adjusts weights

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._last_frontier_state: Dict[str, bool] = {}  # agent_id -> was_frontier

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        i_thread = self.engines.i_thread
        if i_thread is None:
            return RungResult()

        try:
            agent_id = context.get('agent_id', 'default')
            is_frontier = context.get('frontier_mode', False) or context.get('is_frontier', False)
            has_sequence = context.get('has_winning_sequence', False) or context.get('active_sequence')

            # Track state transition
            was_frontier = self._last_frontier_state.get(agent_id, False)
            self._last_frontier_state[agent_id] = is_frontier

            # Boost when entering frontier (no sequences available)
            if is_frontier and not has_sequence and not was_frontier:
                if hasattr(i_thread, 'boost_self_trust'):
                    original, boosted, new_wB = i_thread.boost_self_trust(
                        agent_id=agent_id,
                        boost_amount=0.25,
                        reason='frontier_exploration'
                    )
                    return RungResult(
                        confidence=0.1,  # Context modifier, not action suggestion
                        reason=f"Frontier boost: wA {original:.2f} -> {boosted:.2f}",
                        metadata={
                            'boost_applied': True,
                            'original_wA': original,
                            'boosted_wA': boosted,
                            'trigger': 'frontier_entry'
                        }
                    )

            # Restore when leaving frontier (sequences now available)
            if (not is_frontier or has_sequence) and was_frontier:
                if hasattr(i_thread, 'restore_self_trust'):
                    restored_wA, restored_wB = i_thread.restore_self_trust(agent_id=agent_id)
                    return RungResult(
                        confidence=0.1,
                        reason=f"Trust restored: wA -> {restored_wA:.2f}",
                        metadata={
                            'restore_applied': True,
                            'restored_wA': restored_wA,
                            'trigger': 'frontier_exit'
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Self trust boost failed: {e}")


# Registry of rungs in this module
RUNGS = {
    'survey': SurveyRung,
    'questioning_engine': QuestioningRung,
    'exploration_phase': ExplorationPhaseRung,
    'frustration_detection': FrustrationDetectionRung,
    'palette_detection': PaletteDetectionRung,
    'sparse_grid': SparseGridRung,
    'frame_interpretation': FrameInterpretationRung,
    'breakthrough_budget': BreakthroughBudgetRung,
    'regulatory_signal': RegulatorySignalRung,
    'grid_exploration': GridExplorationRung,
    'affordance_detection': AffordanceDetectionRung,
    'control_tracker': ControlTrackerRung,
    'imagination_budget': ImaginationBudgetRung,
    'network_exploration_stats': NetworkExplorationStatsRung,
    'self_trust_boost': SelfTrustBoostRung,
}

## Hypothesis Rungs

16 rungs for forming and testing theories: ScientificMethodRung, TwoStreamsRung, BeliefSystemRung, etc.

In [ ]:
"""
Hypothesis Rungs - Form and test theories
=========================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


class ScientificMethodRung(DecisionRung):
    """Theory formation and testing - HYPOTHESIS"""
    name = "scientific_method"
    category = "hypothesis"
    default_priority = 12
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sme = self.engines.scientific_method_engine
        if sme is None:
            return RungResult()

        try:
            theory_stage = sme.get_theory_stage() if hasattr(sme, 'get_theory_stage') else 'exploring'

            if theory_stage == 'contradicted':
                # Force exploration/revision using available movement actions
                exploration_actions = filter_available_actions(
                    ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'], context
                )
                return RungResult(
                    action=random.choice(exploration_actions),
                    confidence=0.7,
                    reason=f"Theory contradicted - forcing exploration",
                    metadata={'theory_stage': theory_stage},
                    resolved_questions=['does_hypothesis_hold'],
                )
            elif theory_stage == 'speculating':
                # Boost exploration using available movement actions
                movement_actions = filter_available_actions(
                    ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'], context
                )
                return RungResult(
                    confidence=0.3,
                    weights={a: 1.2 for a in movement_actions},  # Boost available movement
                    reason=f"Speculating - exploration boosted",
                    metadata={'theory_stage': theory_stage},
                    resolved_questions=['hypothesis_speculating'],
                )
            return RungResult(
                metadata={'theory_stage': theory_stage},
                resolved_questions=['theory_stage_observed'],
            )
        except Exception as e:
            return RungResult(reason=f"Scientific method failed: {e}")


class TwoStreamsRung(DecisionRung):
    """Stream A (private) vs Stream B (network) conflict detection - HYPOTHESIS

    Implements the two-stream consciousness model from the unified theory:
    - Stream A: Private memory (agent's personal experience)
    - Stream B: Collective wisdom (network knowledge)

    Uses i_thread engine for stream weights (wA, wB).
    """
    name = "two_streams"
    category = "hypothesis"
    default_priority = 30
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        i_thread = self.engines.i_thread
        agent_id = context.get('agent_id', '')

        # Default weights if i_thread not available
        wA, wB = 0.5, 0.5

        try:
            if i_thread and agent_id:
                state = i_thread.get_state(agent_id)
                if state:
                    wA = state.w_a
                    wB = state.w_b

            stream_a_actions = context.get('stream_a_proposals', set())
            stream_b_actions = context.get('stream_b_proposals', set())

            conflict = stream_a_actions and stream_b_actions and stream_a_actions != stream_b_actions

            if conflict:
                # Conflict = learning signal
                return RungResult(
                    confidence=0.5,
                    reason=f"Stream conflict: A={stream_a_actions}, B={stream_b_actions}, wA={wA:.2f}",
                    metadata={'conflict': True, 'wA': wA, 'wB': wB, 'stream_a': list(stream_a_actions), 'stream_b': list(stream_b_actions)}
                )
            return RungResult(metadata={'conflict': False, 'wA': wA, 'wB': wB})
        except Exception as e:
            return RungResult(reason=f"Two streams failed: {e}")


class MetacognitivePredictionRung(DecisionRung):
    """Make predictions, learn from errors - HYPOTHESIS"""
    name = "metacognitive_prediction"
    category = "hypothesis"
    default_priority = 18
    confidence_threshold = 0.3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Use metacognitive_engine which has get_current_prediction()
        me = self.engines.metacognitive_engine
        if me is None:
            return RungResult()

        try:
            prediction = me.get_current_prediction() if hasattr(me, 'get_current_prediction') else None

            if prediction:
                action = prediction.get('test_action')
                # CRITICAL: Validate action is available in this game
                if action and is_action_available(action, context):
                    return RungResult(
                        action=action,
                        confidence=prediction.get('confidence', 0.3),
                        reason=f"Testing prediction: {prediction.get('hypothesis', '?')}",
                        metadata={'prediction': prediction}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Metacognitive prediction failed: {e}")


class TheoryGateRung(DecisionRung):
    """Working theory must score proposals, contradicted = force exploration - FINALIZER

    Uses scientific_method_engine to check current theory status and force
    exploration when theory is contradicted.
    """
    name = "theory_gate"
    category = "hypothesis"
    default_priority = 32
    confidence_threshold = 0.6

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sme = self.engines.scientific_method_engine
        if sme is None:
            return RungResult()

        try:
            # Get game context for theory lookup
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            # get_working_theory requires game_type and level_number
            theory = None
            if hasattr(sme, 'get_working_theory') and game_type:
                theory = sme.get_working_theory(game_type, level)

            if theory and theory.get('stage') == 'contradicted':
                # Force exploration/revision
                exploration_actions = filter_available_actions(
                    ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4', 'ACTION6'], context
                )
                action = random.choice(exploration_actions)
                return RungResult(
                    action=action,
                    confidence=0.7,
                    reason=f"Theory contradicted: forcing exploration with {action}",
                    metadata={'theory': theory, 'forced_exploration': True}
                )
            return RungResult(metadata={'theory_stage': theory.get('stage') if theory else 'none'})
        except Exception as e:
            return RungResult(reason=f"Theory gate failed: {e}")


class SensationEngineRung(DecisionRung):
    """Emotional context for actions based on object feelings - HYPOTHESIS"""
    name = "sensation_engine"
    category = "hypothesis"
    default_priority = 33
    confidence_threshold = 0.35

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        se = self.engines.sensation_engine
        if se is None:
            return RungResult()

        try:
            if hasattr(se, 'get_tetrahedral_sensation'):
                sensation = se.get_tetrahedral_sensation(context)

                # Convert sensations to action biases (only for available movement actions)
                weights: Dict[str, float] = {}
                available_movement = filter_available_actions(
                    ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'], context
                )
                if sensation.get('approach_score', 0) > 0.5:
                    # Bias toward available movement actions
                    for action in available_movement:
                        weights[action] = 1.0 + sensation['approach_score'] * 0.3
                if sensation.get('threat_level', 0) > 0.5:
                    # Bias away from certain directions
                    threat_direction = sensation.get('threat_direction')
                    if threat_direction and threat_direction in weights:
                        weights[threat_direction] = max(0.1, 1.0 - sensation['threat_level'])

                if weights:
                    return RungResult(
                        confidence=0.4,
                        weights=weights,
                        reason=f"Sensation: approach={sensation.get('approach_score', 0):.2f}, threat={sensation.get('threat_level', 0):.2f}",
                        metadata={'sensation': sensation}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Sensation engine failed: {e}")


class IThreadRung(DecisionRung):
    """Maintain persistent identity, weave stream weights - HYPOTHESIS"""
    name = "i_thread"
    category = "hypothesis"
    default_priority = 31
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ithread = self.engines.i_thread
        if ithread is None:
            return RungResult()

        try:

            # Get stream weights
            wA = ithread.get_wA() if hasattr(ithread, 'get_wA') else 0.5
            wB = ithread.get_wB() if hasattr(ithread, 'get_wB') else 0.5

            # Check for death personas (near cull)
            cull_distance = context.get('cull_distance', 1.0)
            if cull_distance < 0.2 and hasattr(ithread, 'spawn_death_persona'):
                persona = ithread.spawn_death_persona(context.get('agent_role', 'generalist'))
                if persona and persona.get('suggested_action'):
                    action = persona['suggested_action']
                    # CRITICAL: Validate action is available in this game
                    if is_action_available(action, context):
                        return RungResult(
                            action=action,
                            confidence=0.7,
                            reason=f"Death persona ({persona.get('name', 'unknown')}): {persona.get('reason', '')}",
                            metadata={'persona': persona, 'cull_distance': cull_distance}
                        )

            return RungResult(metadata={'wA': wA, 'wB': wB, 'cull_distance': cull_distance})
        except Exception as e:
            return RungResult(reason=f"I-Thread failed: {e}")


class EventUnderstandingRung(DecisionRung):
    """
    Use causal world model to inform decisions.

    This rung builds understanding from frame-to-frame changes:
    - Tracks object movements and interactions
    - Detects collisions, fusions, and other events
    - Attributes causality to actions
    - Classifies the overall process type

    Uses this understanding to:
    - Set context flags for downstream rungs
    - Boost weights for actions that caused productive events
    - Predict continuation of causal chains
    """
    name = "event_understanding"
    category = "hypothesis"
    default_priority = 23  # After orientation, before exploitation
    confidence_threshold = 0.4

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._event_detector: Optional[Any] = None
        self._object_tracker: Optional[Any] = None
        self._event_history: List[Dict] = []  # Recent events for causal chain analysis
        self._db: Optional[Any] = None

    def _get_event_detector(self) -> Optional[Any]:
        """Lazy-load event detector."""
        if self._event_detector is None:
            try:
                from engines.perception.event_detector import EventDetector
                self._event_detector = EventDetector()
            except ImportError:
                pass
        return self._event_detector

    def _get_object_tracker(self) -> Optional[Any]:
        """Lazy-load object tracker."""
        if self._object_tracker is None:
            try:
                from engines.perception.object_tracker import ObjectTracker
                self._object_tracker = ObjectTracker()
            except ImportError:
                pass
        return self._object_tracker

    def _get_db(self) -> Optional[Any]:
        """Lazy-load database interface."""
        if self._db is None:
            try:
                self._db = DatabaseInterface()
            except ImportError:
                pass
        return self._db

    def _get_recent_events(self, context: Dict[str, Any]) -> List[Dict]:
        """Get recent events from context or database."""
        if 'recent_events' in context:
            return context['recent_events']

        db = self._get_db()
        if db is None:
            return self._event_history[-10:]

        try:
            game_type = context.get('game_type', '')
            result = db.execute_query("""
                SELECT event_type, objects_involved, positions, confidence
                FROM detected_events
                WHERE game_type = ?
                ORDER BY timestamp DESC
                LIMIT 10
            """, (game_type,))

            return [
                {'type': r['event_type'], 'objects': r['objects_involved'],
                 'positions': r['positions'], 'confidence': r['confidence']}
                for r in (result if result else [])
            ]
        except Exception:
            return self._event_history[-10:]

    def _last_action_caused_productive_event(self, events: List[Dict]) -> bool:
        """Check if the last action caused a productive event."""
        if not events:
            return False

        # Productive events: COLLECTION, TRANSFORMATION toward goal, FUSION
        productive_types = {'COLLECTION', 'TRANSFORMATION', 'FUSION'}

        for event in events[:3]:  # Check recent events
            event_type = event.get('type', event.get('event_type', ''))
            if isinstance(event_type, str) and event_type in productive_types:
                return True
            elif hasattr(event_type, 'value') and event_type.value in productive_types:
                return True

        return False

    def _detected_physics_process(self, events: List[Dict]) -> bool:
        """Check if physics simulation was detected."""
        if not events:
            return False

        movement_count = 0
        for event in events:
            event_type = event.get('type', event.get('event_type', ''))
            if 'MOVEMENT' in str(event_type):
                movement_count += 1

        return movement_count >= 3

    def _get_active_causal_chain(self, events: List[Dict]) -> List[Dict]:
        """Identify active causal chain from recent events."""
        if len(events) < 2:
            return []

        # Look for sequence of related events
        chain = []
        for event in events:
            event_type = str(event.get('type', event.get('event_type', '')))
            if event_type in {'MOVEMENT', 'COLLISION', 'FUSION', 'CHAIN_REACTION'}:
                chain.append(event)
            elif chain:
                break  # Chain broken

        return chain

    def _predict_chain_continuation(self, chain: List[Dict]) -> Optional[str]:
        """Predict what action would continue the causal chain."""
        if not chain:
            return None

        # If last event was a collision, maybe continue pushing
        last_event = chain[-1]
        last_type = str(last_event.get('type', last_event.get('event_type', '')))

        if 'COLLISION' in last_type or 'MOVEMENT' in last_type:
            # Continue in same direction if we have that info
            positions = last_event.get('positions', [])
            if len(positions) >= 2:
                # Calculate movement direction
                try:
                    dy = float(positions[1][0]) - float(positions[0][0])
                    dx = float(positions[1][1]) - float(positions[0][1])

                    if abs(dy) > abs(dx):
                        return 'ACTION2' if dy > 0 else 'ACTION1'  # Down or Up
                    elif abs(dx) > 0:
                        return 'ACTION4' if dx > 0 else 'ACTION3'  # Right or Left
                except (IndexError, TypeError, ValueError):
                    pass

        return None

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        recent_events = self._get_recent_events(context)

        if not recent_events:
            return RungResult()  # No understanding yet

        # Build weight modifiers based on understanding
        weights = get_available_action_weights(context, 1.0)

        # If last action caused productive event, boost similar actions
        if self._last_action_caused_productive_event(recent_events):
            last_action = context.get('last_action')
            if last_action and last_action in weights:
                weights[last_action] = 1.3  # Boost similar actions

        # If physics simulation detected, set context
        if self._detected_physics_process(recent_events):
            context['physics_game_confirmed'] = True

        # If we understand the causal chain, boost actions that extend it
        causal_chain = self._get_active_causal_chain(recent_events)
        if causal_chain:
            next_action = self._predict_chain_continuation(causal_chain)
            if next_action and is_action_available(next_action, context):
                return RungResult(
                    action=next_action,
                    confidence=0.55,
                    reason=f"Continuing causal chain of {len(causal_chain)} events",
                    weights=weights,
                    metadata={
                        'chain_length': len(causal_chain),
                        'last_event_type': str(causal_chain[-1].get('type', '')),
                    }
                )

        return RungResult(weights=weights)


class ResonanceDetectorRung(DecisionRung):
    """Cross-role pattern discovery for objective truth - HYPOTHESIS

    Implements the Resonance Discovery Principle from harmonies theory:
    When agents with radically different biases (Pioneers, Generalists, Exploiters)
    converge on the same pattern, that's evidence of OBJECTIVE TRUTH.

    Resonance detection is the bridge between "widely believed" and "actually true".
    High resonance + high role diversity = structural truth transcending individual bias.
    """
    name = "resonance_detector"
    category = "hypothesis"
    default_priority = 34
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        rd = self.engines.resonance_detector
        if rd is None:
            return RungResult()

        try:
            role = context.get('agent_role', 'generalist')

            # Role-specific query frequencies (from harmonies theory)
            query_probs = {'pioneer': 0.15, 'optimizer': 0.20, 'generalist': 0.30, 'exploiter': 0.10}
            if random.random() > query_probs.get(role, 0.15):
                return RungResult()  # Skip query this time

            if hasattr(rd, 'get_resonant_patterns'):
                # Query patterns with minimum resonance score
                patterns = rd.get_resonant_patterns(min_score=0.6, limit=10)
                if patterns:
                    best = patterns[0]
                    resonance_score = best.get('resonance_score', 0)

                    if resonance_score > 0.6:
                        suggested_action = best.get('suggested_action')
                        # CRITICAL: Validate action is available in this game
                        if not is_action_available(suggested_action, context):
                            return RungResult(reason=f"Resonance pattern suggested unavailable action: {suggested_action}")

                        # Build epistemological provenance
                        # Cross-role resonance is the gold standard for validation
                        role_diversity = best.get('role_diversity', 1)
                        game_types = best.get('game_types', [])

                        provenance = KnowledgeProvenance(
                            detection_source='resonance_patterns',
                            sample_size=role_diversity * len(game_types),
                            agent_diversity=role_diversity,  # Different ROLES = different cognitive biases
                            temporal_spread_generations=0.0,  # Not tracked for resonance
                            validation_type='cross_role_convergence',  # The gold standard
                            positive_outcomes=role_diversity,  # Each role validated
                            negative_outcomes=0,
                            crystallization_stage=4 if role_diversity >= 3 else 3,  # High: crystallized
                            resonance_games=len(game_types),  # How many games share this pattern
                            resonance_score=resonance_score
                        )

                        return RungResult(
                            action=suggested_action,
                            confidence=resonance_score,
                            reason=f"Resonant pattern ({role_diversity} roles, {len(game_types)} games): {best.get('theory_type', 'unknown')}",
                            metadata={
                                'pattern': best,
                                'pattern_hash': best.get('pattern_hash'),
                                'roles_found': best.get('roles_found', []),
                                'game_types': game_types,  # Store the actual list in metadata
                            },
                            provenance=provenance
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Resonance detector failed: {e}")


class InteractableTileDiscoveryRung(DecisionRung):
    """Learn that moving to specific tiles changes agent state - HYPOTHESIS

    For directional games like LS20 where:
    - Walking over tile X changes tool shape (gsu)
    - Walking over tile Y changes tool color (gic)
    - Walking over tile Z changes rotation (bgt)

    This rung:
    1. Tracks frame state BEFORE and AFTER each directional movement
    2. Detects when movement causes state changes BEYOND just position
    3. Maps tile positions to the state changes they cause
    4. Builds a "property modification map" of the level
    5. Suggests revisiting known modifier tiles when current state
       doesn't match the target

    ROOT CAUSE ADDRESSED: LS20 agents never discover that gsu/gic/bgt tiles
    modify the tool's properties. Without this, the 3-property matching
    system is invisible.
    """
    name = "interactable_tile_discovery"
    category = "hypothesis"
    default_priority = 28
    confidence_threshold = 0.4

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # game_key -> {(x, y) -> list of observed state changes}
        self._modifier_map: Dict[str, Dict[Tuple[int, int], List[Dict[str, Any]]]] = {}
        # Track agent position by correlating with frame changes
        self._estimated_position: Optional[Tuple[int, int]] = None
        # Track the "tool state" as a hash of non-position frame elements
        self._last_tool_state_hash: str = ""
        # Track state transitions: (old_state_hash, new_state_hash) -> position
        self._state_transitions: Dict[str, List[Dict[str, Any]]] = {}
        # Known modifier positions per game
        self._known_modifiers: Dict[str, List[Tuple[int, int]]] = {}
        # Count of property changes detected (to know when we've discovered modifiers)
        self._property_changes_detected: Dict[str, int] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        has_directional = any(
            (a in [1, 2, 3, 4] if isinstance(a, int) else a in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
            for a in available
        )
        if not has_directional:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Merge shared world model causal data (Part 7 unification, Task A3)
        shared_wm = context.get('world_model')
        if shared_wm and isinstance(shared_wm, dict):
            shared_causal = shared_wm.get('causal_map', {})
            if shared_causal:
                if game_key not in self._modifier_map:
                    self._modifier_map[game_key] = {}
                if game_key not in self._known_modifiers:
                    self._known_modifiers[game_key] = []
                for pos_key_str, entry in shared_causal.items():
                    # Parse "x,y" position key from shared causal map
                    try:
                        parts = pos_key_str.split(',')
                        if len(parts) == 2:
                            px, py = int(parts[0]), int(parts[1])
                            pos = (px, py)
                            # Check if this causal entry involves color changes
                            # (which would indicate a modifier tile)
                            obs_list = entry.get('observations', [])
                            has_color_change = False
                            for obs_item in obs_list:
                                for change in obs_item.get('changes', []):
                                    if change.get('old_color') != change.get('new_color'):
                                        has_color_change = True
                                        break
                                if has_color_change:
                                    break
                            if has_color_change and pos not in self._known_modifiers.get(game_key, []):
                                self._known_modifiers.setdefault(game_key, []).append(pos)
                                self._property_changes_detected[game_key] = \
                                    self._property_changes_detected.get(game_key, 0) + 1
                    except (ValueError, IndexError):
                        pass
            # Contribute own discoveries back to shared world model
            if game_key in self._modifier_map and self._modifier_map[game_key]:
                if 'causal_map' not in shared_wm:
                    shared_wm['causal_map'] = {}
                for pos, observations in self._modifier_map[game_key].items():
                    contrib_key = f"{pos[0]},{pos[1]}"
                    if contrib_key not in shared_wm['causal_map']:
                        shared_wm['causal_map'][contrib_key] = {
                            'action': 'movement',
                            'observations': [{'changes': obs.get('state_changes', [])} for obs in observations],
                            'total_observations': len(observations),
                        }

        # If we've discovered modifier tiles, suggest moving toward them
        modifiers = self._known_modifiers.get(game_key, [])
        if modifiers and self._estimated_position:
            # Find nearest unvisited-recently modifier
            ex, ey = self._estimated_position
            nearest = None
            nearest_dist = float('inf')
            for mx, my in modifiers:
                dist = abs(mx - ex) + abs(my - ey)  # Manhattan distance
                if 0 < dist < nearest_dist:
                    nearest_dist = dist
                    nearest = (mx, my)

            if nearest:
                # Suggest direction toward the modifier
                dx = nearest[0] - ex
                dy = nearest[1] - ey

                # Map displacement to action
                if abs(dx) > abs(dy):
                    action = 'ACTION3' if dx < 0 else 'ACTION4'  # left/right
                else:
                    action = 'ACTION1' if dy < 0 else 'ACTION2'  # up/down

                if is_action_available(action, context):
                    return RungResult(
                        action=action,
                        confidence=0.45,
                        reason=f"Moving toward modifier tile at {nearest} (dist={nearest_dist:.0f})",
                        metadata={
                            'target_modifier': nearest,
                            'source': 'interactable_tile_discovery',
                        }
                    )

        # If we haven't found modifiers yet, inject discovery metadata
        changes = self._property_changes_detected.get(game_key, 0)
        if changes > 0:
            return RungResult(
                reason=f"Discovered {changes} property-changing tiles in {game_key}",
                metadata={
                    'modifier_count': changes,
                    'known_modifiers': modifiers,
                }
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Detect when movement causes state changes beyond position."""
        if not action or action not in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4']:
            return
        if frame_before is None or frame_after is None:
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        try:
            # Compute frame diff
            changes = self._compute_change_regions(frame_before, frame_after)
            if not changes:
                return  # No change = wall hit

            # Separate position changes from state changes
            # Position change: a colored cluster moved (disappeared + reappeared)
            # State change: colors changed WITHOUT corresponding position change
            position_changes = []
            state_changes = []

            for change in changes:
                if change['type'] == 'moved':
                    position_changes.append(change)
                elif change['type'] == 'color_changed':
                    state_changes.append(change)

            # Update estimated position from position changes
            if position_changes:
                # Agent likely moved -- update position estimate
                for pc in position_changes:
                    if pc.get('new_pos'):
                        self._estimated_position = pc['new_pos']

            # KEY INSIGHT: If we see state_changes (color modifications) concurrent
            # with position_changes, the tile at the agent's new position likely
            # caused the state change
            if state_changes and self._estimated_position:
                if game_key not in self._modifier_map:
                    self._modifier_map[game_key] = {}
                if game_key not in self._known_modifiers:
                    self._known_modifiers[game_key] = []

                pos = self._estimated_position
                if pos not in self._modifier_map[game_key]:
                    self._modifier_map[game_key][pos] = []

                self._modifier_map[game_key][pos].append({
                    'state_changes': state_changes,
                    'action': action,
                })

                # After 2+ observations at same position, mark as confirmed modifier
                if len(self._modifier_map[game_key][pos]) >= 1:
                    if pos not in self._known_modifiers[game_key]:
                        self._known_modifiers[game_key].append(pos)
                        self._property_changes_detected[game_key] = \
                            self._property_changes_detected.get(game_key, 0) + 1

        except Exception:
            pass

    def _compute_change_regions(
        self, frame_before: Any, frame_after: Any
    ) -> List[Dict[str, Any]]:
        """Analyze frame diff to separate position changes from state changes."""
        changes: List[Dict[str, Any]] = []
        try:
            disappeared: Dict[int, List[Tuple[int, int]]] = {}  # color -> positions
            appeared: Dict[int, List[Tuple[int, int]]] = {}     # color -> positions
            color_changed: List[Tuple[int, int, int, int]] = []  # (x, y, old, new)

            if isinstance(frame_before, list):
                h = min(len(frame_before), len(frame_after))
                for y in range(h):
                    w = min(len(frame_before[y]), len(frame_after[y]))
                    for x in range(w):
                        old = int(frame_before[y][x]) if hasattr(frame_before[y][x], '__int__') else frame_before[y][x]
                        new = int(frame_after[y][x]) if hasattr(frame_after[y][x], '__int__') else frame_after[y][x]
                        if old != new:
                            if old != 0 and new == 0:
                                disappeared.setdefault(old, []).append((x, y))
                            elif old == 0 and new != 0:
                                appeared.setdefault(new, []).append((x, y))
                            elif old != 0 and new != 0:
                                color_changed.append((x, y, old, new))
            else:
                import numpy as np
                fb, fa = np.array(frame_before), np.array(frame_after)
                diff_mask = fb != fa
                ys, xs = np.where(diff_mask)
                for yi, xi in zip(ys, xs):
                    old, new = int(fb[yi, xi]), int(fa[yi, xi])
                    if old != 0 and new == 0:
                        disappeared.setdefault(old, []).append((int(xi), int(yi)))
                    elif old == 0 and new != 0:
                        appeared.setdefault(new, []).append((int(xi), int(yi)))
                    elif old != 0 and new != 0:
                        color_changed.append((int(xi), int(yi), old, new))

            # Classify: same-color disappeared+appeared = movement
            for color in set(disappeared.keys()) & set(appeared.keys()):
                d_pts = disappeared[color]
                a_pts = appeared[color]
                if d_pts and a_pts:
                    d_cx = sum(p[0] for p in d_pts) // len(d_pts)
                    d_cy = sum(p[1] for p in d_pts) // len(d_pts)
                    a_cx = sum(p[0] for p in a_pts) // len(a_pts)
                    a_cy = sum(p[1] for p in a_pts) // len(a_pts)
                    changes.append({
                        'type': 'moved',
                        'color': color,
                        'old_pos': (d_cx, d_cy),
                        'new_pos': (a_cx, a_cy),
                    })

            # Color changes at same position = state modification
            if color_changed:
                changes.append({
                    'type': 'color_changed',
                    'positions': [(x, y) for x, y, _, _ in color_changed],
                    'transitions': [(old, new) for _, _, old, new in color_changed],
                    'count': len(color_changed),
                })

        except Exception:
            pass
        return changes


class GoalRelationshipModelingRung(DecisionRung):
    """Model spatial relationships between game objects for win conditions - HYPOTHESIS

    For puzzle games like VC33 where the win condition involves spatial
    relationships between object types:
    - Passenger blocks (HQB) must be on correct conveyor tracks
    - Tracks are identified by color markers (fZK)
    - Colors must MATCH between passenger and marker

    This rung:
    1. Detect object "groups" by color in the frame
    2. Track which groups change position when actions are taken
    3. Infer goal relationships: "object A needs to be near object B"
    4. Suggest actions that move objects toward their goal positions
    5. Learn from score changes which relationships matter

    ROOT CAUSE ADDRESSED: VC33's win condition requires understanding
    spatial relationships between multiple sprite types simultaneously.
    """
    name = "goal_relationship_modeling"
    category = "hypothesis"
    default_priority = 29
    confidence_threshold = 0.4

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Track object positions over time: game_key -> color -> [(x, y, step)]
        self._object_trajectories: Dict[str, Dict[int, List[Tuple[int, int, int]]]] = {}
        # Track which objects moved in response to clicks
        self._click_object_response: Dict[str, Dict[Tuple[int, int], Set[int]]] = {}
        # Hypothesized goal pairs: (movable_color, target_color)
        self._goal_pairs: Dict[str, List[Tuple[int, int]]] = {}
        # Step counter
        self._step: Dict[str, int] = {}
        # Score at each step
        self._score_history: Dict[str, List[float]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        frame = _get_frame(game_state)
        if frame is None:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Need some click response data first
        responses = self._click_object_response.get(game_key, {})
        if len(responses) < 3:
            return RungResult()

        # Find which click positions move which objects
        # Strategy: click positions that move objects TOWARD color-matched targets
        goal_pairs = self._goal_pairs.get(game_key, [])
        if not goal_pairs:
            # Try to infer goal pairs from color matching
            objects = self._detect_color_groups(frame)
            if len(objects) >= 4:
                # Hypothesis: pairs of same-ish color groups are related
                colors = sorted(objects.keys())
                for i, c1 in enumerate(colors):
                    for c2 in colors[i+1:]:
                        # Colors within small range might be related
                        if abs(c1 - c2) <= 2 and c1 != c2:
                            if game_key not in self._goal_pairs:
                                self._goal_pairs[game_key] = []
                            self._goal_pairs[game_key].append((c1, c2))

        # Suggest clicking positions that historically moved objects
        movable_clicks = []
        for click_pos, moved_colors in responses.items():
            if moved_colors:  # This click moved something
                movable_clicks.append((click_pos, len(moved_colors)))

        if movable_clicks:
            # Pick click that moves the most objects
            movable_clicks.sort(key=lambda x: x[1], reverse=True)
            best_click = movable_clicks[0][0]
            return RungResult(
                action='ACTION6',
                confidence=0.40,
                reason=f"Goal modeling: click at {best_click} moves {movable_clicks[0][1]} object(s)",
                metadata={
                    'x': best_click[0],
                    'y': best_click[1],
                    'source': 'goal_relationship_modeling',
                    'moved_objects': movable_clicks[0][1],
                }
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Track object movement in response to actions."""
        if action != 'ACTION6':
            return
        if frame_before is None or frame_after is None:
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        click_x = action_data.get('x', 0)
        click_y = action_data.get('y', 0)
        click_pos = (click_x, click_y)

        self._step[game_key] = self._step.get(game_key, 0) + 1
        step = self._step[game_key]

        # Detect which objects moved
        objects_before = self._detect_color_groups(frame_before)
        objects_after = self._detect_color_groups(frame_after)

        moved_colors: Set[int] = set()
        for color in set(objects_before.keys()) & set(objects_after.keys()):
            before_center = objects_before[color]
            after_center = objects_after[color]
            # Check if center moved significantly (>2 pixels)
            dist = abs(before_center[0] - after_center[0]) + abs(before_center[1] - after_center[1])
            if dist > 2:
                moved_colors.add(color)

                # Record trajectory
                if game_key not in self._object_trajectories:
                    self._object_trajectories[game_key] = {}
                if color not in self._object_trajectories[game_key]:
                    self._object_trajectories[game_key][color] = []
                self._object_trajectories[game_key][color].append(
                    (after_center[0], after_center[1], step)
                )

        # Record which click moved which objects
        if game_key not in self._click_object_response:
            self._click_object_response[game_key] = {}
        self._click_object_response[game_key][click_pos] = moved_colors

    @staticmethod
    def _detect_color_groups(frame: Any) -> Dict[int, Tuple[int, int]]:
        """Detect color groups and their centroids."""
        groups: Dict[int, List[Tuple[int, int]]] = {}
        try:
            if isinstance(frame, list):
                for y, row in enumerate(frame):
                    for x, pixel in enumerate(row):
                        val = int(pixel) if hasattr(pixel, '__int__') else pixel
                        if val != 0:
                            groups.setdefault(val, []).append((x, y))
            else:
                import numpy as np
                arr = np.array(frame)
                for color in np.unique(arr):
                    if color == 0:
                        continue
                    ys, xs = np.where(arr == color)
                    groups[int(color)] = list(zip(xs.tolist(), ys.tolist()))
        except Exception:
            return {}

        # Filter to reasonable-sized groups and compute centroids
        centroids: Dict[int, Tuple[int, int]] = {}
        for color, positions in groups.items():
            size = len(positions)
            if 2 <= size <= 500:
                cx = sum(p[0] for p in positions) // size
                cy = sum(p[1] for p in positions) // size
                centroids[color] = (cx, cy)

        return centroids


class BeliefSystemRung(DecisionRung):
    """Track and use agent beliefs - HYPOTHESIS

    Uses engines/self_model/belief_system.py to:
    1. Query current beliefs about the game
    2. Use high-confidence beliefs to guide action selection
    3. Track belief invalidation cascades

    Beliefs provide persistent knowledge across actions.
    """
    name = "belief_system"
    category = "hypothesis"
    default_priority = 25
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        bs = self.engines.belief_system
        if bs is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            if not game_type:
                return RungResult()

            # Get active beliefs for this game
            if hasattr(bs, 'get_active_beliefs'):
                beliefs = bs.get_active_beliefs(game_type)

                if beliefs:
                    # Find high-confidence beliefs with action implications
                    for belief in beliefs:
                        if belief.get('confidence', 0) > 0.7:
                            # Check if belief suggests an action
                            statement = belief.get('statement', '')
                            if 'ACTION' in statement.upper():
                                # Extract action suggestion
                                for i in range(1, 8):
                                    if f'ACTION{i}' in statement.upper():
                                        return RungResult(
                                            action=f'ACTION{i}',
                                            confidence=belief.get('confidence', 0.5) * 0.7,
                                            reason=f"Belief: {statement[:50]}...",
                                            metadata={'belief': belief}
                                        )

                    # Return belief context for other rungs
                    return RungResult(
                        confidence=0.3,
                        reason=f"Belief system: {len(beliefs)} active beliefs",
                        metadata={'active_beliefs': beliefs[:5]}  # Top 5
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Belief system failed: {e}")


class HypothesisSystemRung(DecisionRung):
    """Manage agent hypotheses - HYPOTHESIS

    Uses engines/social/hypothesis_system.py to:
    1. Get untested hypotheses that need validation
    2. Suggest actions to test hypotheses
    3. Record test results for learning

    This enables agents to actively test their theories.
    """
    name = "hypothesis_system"
    category = "hypothesis"
    default_priority = 26
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        hs = self.engines.hypothesis_system
        if hs is None:
            return RungResult()

        try:
            agent_id = context.get('agent_id', '')
            game_type = context.get('game_type', '')

            if not agent_id or not game_type:
                return RungResult()

            # Get testable hypotheses for this game
            if hasattr(hs, 'get_agent_hypotheses'):
                hypotheses = hs.get_agent_hypotheses(agent_id, game_type, status='testing')

                if hypotheses:
                    # Find hypothesis with predicted action
                    for hyp in hypotheses:
                        predicted_action = hyp.get('predicted_action')
                        if predicted_action:
                            return RungResult(
                                action=predicted_action,
                                confidence=0.55,
                                reason=f"Testing hypothesis: {hyp.get('hypothesis_text', '')[:40]}...",
                                metadata={
                                    'hypothesis_id': hyp.get('hypothesis_id'),
                                    'hypothesis': hyp,
                                    'testing_mode': True
                                }
                            )

                        # Check for action sequence
                        sequence = hyp.get('action_sequence')
                        if sequence and isinstance(sequence, list) and len(sequence) > 0:
                            # Get position in sequence
                            seq_pos = context.get('hypothesis_sequence_position', 0)
                            if seq_pos < len(sequence):
                                return RungResult(
                                    action=sequence[seq_pos],
                                    confidence=0.5,
                                    reason=f"Hypothesis sequence step {seq_pos + 1}/{len(sequence)}",
                                    metadata={
                                        'hypothesis_id': hyp.get('hypothesis_id'),
                                        'sequence_position': seq_pos,
                                        'full_sequence': sequence
                                    }
                                )

            # Check for hypothesis suggestions from patterns
            if hasattr(hs, 'suggest_hypothesis_from_pattern'):
                observations = context.get('recent_observations', [])
                if observations:
                    suggestion = hs.suggest_hypothesis_from_pattern(agent_id, game_type, observations)
                    if suggestion:
                        return RungResult(
                            confidence=0.3,
                            reason=f"Hypothesis suggestion: {suggestion.get('suggested_hypothesis', '')[:40]}...",
                            metadata={'hypothesis_suggestion': suggestion}
                        )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Hypothesis system failed: {e}")


class SymbolicTrackerRung(DecisionRung):
    """Track symbolic state for transformation puzzles - HYPOTHESIS

    Uses engines/self_model/symbolic_tracker.py to:
    1. Identify key objects (controllable) vs lock objects (target)
    2. Track symbolic properties: shape, color, orientation
    3. Suggest actions to make key match lock

    Essential for transformation/matching puzzles.
    """
    name = "symbolic_tracker"
    category = "hypothesis"
    default_priority = 24
    confidence_threshold = 0.45

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Private causal map for symbolic observations: game_key -> {pos_key -> effects}
        self._causal_map: Dict[str, Dict[str, Any]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        st = self.engines.symbolic_tracker
        if st is None:
            return RungResult()

        try:
            frame = _get_frame(game_state)
            if frame is None:
                return RungResult()

            game_type = context.get('game_type', '')
            level = context.get('level', 1)
            game_key = f"{game_type}_L{level}"

            # Merge shared world model causal data (Part 7 unification, Task A3)
            shared_wm = context.get('world_model')
            if shared_wm and isinstance(shared_wm, dict):
                shared_causal = shared_wm.get('causal_map', {})
                if shared_causal and game_key not in self._causal_map:
                    self._causal_map[game_key] = {}
                for pos_key_str, entry in shared_causal.items():
                    if game_key in self._causal_map:
                        if pos_key_str not in self._causal_map[game_key]:
                            # Convert shared format to local format
                            obs_list = entry.get('observations', [])
                            effects = []
                            for obs_item in obs_list:
                                for change in obs_item.get('changes', []):
                                    effects.append(change)
                            if effects:
                                self._causal_map[game_key][pos_key_str] = effects
                # Contribute own observations back to shared world model
                if game_key in self._causal_map and self._causal_map[game_key]:
                    if 'causal_map' not in shared_wm:
                        shared_wm['causal_map'] = {}
                    for pos_key, effects in self._causal_map[game_key].items():
                        if pos_key not in shared_wm['causal_map']:
                            shared_wm['causal_map'][pos_key] = {
                                'action': 'symbolic_transform',
                                'observations': [{'changes': effects}],
                                'total_observations': len(effects),
                            }

            # Identify symbolic objects
            if hasattr(st, 'identify_symbolic_objects'):
                controlled_colors = context.get('controlled_colors', [])
                objects = st.identify_symbolic_objects(frame, controlled_colors)

                keys = objects.get('keys', {})
                locks = objects.get('locks', {})
                tools = objects.get('tools', {})

                if keys and locks:
                    # Check match score
                    if hasattr(st, 'calculate_match_score'):
                        match_score = st.calculate_match_score()

                        if match_score < 1.0:
                            # Not matching - try to identify transformation needed
                            if hasattr(st, 'suggest_transformation'):
                                suggestion = st.suggest_transformation()
                                if suggestion:
                                    action = suggestion.get('action')
                                    if action:
                                        return RungResult(
                                            action=action,
                                            confidence=0.55,
                                            reason=f"Symbolic match {match_score:.0%} - {suggestion.get('reason', 'transform')}",
                                            metadata={
                                                'match_score': match_score,
                                                'keys': keys,
                                                'locks': locks,
                                                'suggestion': suggestion
                                            }
                                        )

                        # Near match - return status
                        return RungResult(
                            confidence=0.3 + match_score * 0.3,
                            reason=f"Symbolic tracking: {len(keys)} keys, {len(locks)} locks, match={match_score:.0%}",
                            metadata={'keys': keys, 'locks': locks, 'tools': tools, 'match_score': match_score}
                        )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Symbolic tracker failed: {e}")


class DeliberationSystemRung(DecisionRung):
    """TRM-inspired iterative refinement - HYPOTHESIS

    Uses scientific_method_engine to get theory hints and deliberation results
    for multi-agent reasoning convergence.
    """
    name = "deliberation_system"
    category = "hypothesis"
    default_priority = 29
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sme = self.engines.scientific_method_engine
        if sme is None:
            return RungResult()

        try:
            # Use scientific method engine's theory hint for deliberation guidance
            if hasattr(sme, 'get_active_theory_hint'):
                hint = sme.get_active_theory_hint()

                if hint and hint.get('prediction'):
                    # Theory has a prediction - this is deliberation output
                    action = hint.get('prediction', {}).get('action')
                    confidence = hint.get('weight', 0.5) + 0.3  # Boost confidence

                    # CRITICAL: Validate action is available in this game
                    if action and is_action_available(action, context):
                        return RungResult(
                            action=action,
                            confidence=min(0.9, confidence),
                            reason=f"Deliberation hint: {hint.get('reason', 'theory-guided')}",
                            metadata={'deliberation_hint': hint}
                        )

            # Fallback: Check context for deliberation results from other systems
            deliberation = context.get('deliberation_result')
            if deliberation and deliberation.get('convergence_achieved', False):
                action = deliberation.get('consensus_action')
                # CRITICAL: Validate action is available in this game
                if action and is_action_available(action, context):
                    return RungResult(
                        action=action,
                        confidence=deliberation.get('refinement_confidence', 0.6),
                        reason=f"Deliberation converged: {deliberation.get('refinement_passes', 0)} passes",
                        metadata={'deliberation': deliberation}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Deliberation system failed: {e}")


class HypothesisTestingRung(DecisionRung):
    """Test untested assumptions to validate or disprove them - HYPOTHESIS

    Wires: engines/cognition/metacognition.py:
        - get_untested_assumptions()
        - register_assumption()
        - challenge_assumption()

    Prioritizes actions that would test currently-held assumptions.
    E.g., if agent assumes "ACTION1 moves me up", this rung will
    suggest ACTION1 when testing is needed.
    """
    name = "hypothesis_testing"
    category = "hypothesis"
    default_priority = 19  # After metacognitive_prediction (18)
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        me = self.engines.metacognitive_engine
        if me is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            # Get untested assumptions for this game/level
            if not hasattr(me, 'get_untested_assumptions'):
                return RungResult()

            untested = me.get_untested_assumptions(game_type, level)

            if not untested:
                return RungResult()

            # Find an assumption we can test
            for assumption in untested:
                assumption_text = assumption.get('assumption_text', '')
                assumption_type = assumption.get('assumption_type', '')
                assumption_id = assumption.get('assumption_id', '')

                # Parse assumption to find testable action
                # E.g., "ACTION1 moves me up" -> suggest ACTION1
                import re
                action_match = re.search(r'ACTION(\d+)', assumption_text.upper())

                if action_match:
                    action = f"ACTION{action_match.group(1)}"
                    # Store assumption_id for challenge_assumption callback
                    context['_testing_assumption_id'] = assumption_id
                    context['_testing_assumption_text'] = assumption_text

                    return RungResult(
                        action=action,
                        confidence=0.55,
                        reason=f"Testing: {assumption_text[:50]}",
                        metadata={
                            'assumption_id': assumption_id,
                            'assumption_type': assumption_type,
                            'assumption_text': assumption_text,
                        }
                    )

                # For non-action assumptions (e.g., "blue is goal"), no direct action
                # but we can store for later validation

            return RungResult(
                confidence=0.1,
                reason=f"{len(untested)} untested assumptions, none directly testable",
                metadata={'untested_count': len(untested)}
            )
        except Exception as e:
            return RungResult(reason=f"Hypothesis testing failed: {e}")


class AssumptionFormationRung(DecisionRung):
    """Form and register assumptions based on observed patterns - HYPOTHESIS

    Wires: engines/cognition/metacognition.py:
        - register_assumption()
        - challenge_assumption()

    Monitors gameplay to detect correlations and form testable assumptions.
    E.g., "When I press ACTION1, the blue object moves up" -> registers assumption.

    This is the WRITE side of the assumption system (HypothesisTestingRung is READ).
    Together they form a complete hypothesis testing loop:
    1. AssumptionFormationRung detects correlation -> register_assumption()
    2. HypothesisTestingRung suggests action to test -> get_untested_assumptions()
    3. After outcome -> challenge_assumption() to validate/invalidate
    """
    name = "assumption_formation"
    category = "hypothesis"
    default_priority = 16  # Before hypothesis_testing
    confidence_threshold = 0.3

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Track recent observations for pattern detection
        self._recent_observations: List[Dict[str, Any]] = []
        self._max_observations = 20

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        me = self.engines.metacognitive_engine
        if me is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)
            agent_id = context.get('agent_id', 'default')
            last_action = context.get('last_action')
            frame_change = context.get('frame_change', {})

            # Skip if no action taken yet
            if not last_action:
                return RungResult()

            # Record observation
            observation = {
                'action': last_action,
                'frame_change': frame_change,
                'score_delta': context.get('score_delta', 0),
                'game_type': game_type,
                'level': level,
            }
            self._recent_observations.append(observation)
            if len(self._recent_observations) > self._max_observations:
                self._recent_observations.pop(0)

            # Need at least 3 observations to detect patterns
            if len(self._recent_observations) < 3:
                return RungResult()

            # Look for consistent action->outcome correlations
            assumptions_formed = []
            action_outcomes: Dict[str, List[Dict]] = {}

            for obs in self._recent_observations:
                action = obs.get('action', '')
                if action:
                    if action not in action_outcomes:
                        action_outcomes[action] = []
                    action_outcomes[action].append(obs)

            # Check each action for consistent outcomes
            for action, outcomes in action_outcomes.items():
                if len(outcomes) >= 2:
                    # Check for consistent positive score
                    positive_scores = [o for o in outcomes if o.get('score_delta', 0) > 0]
                    if len(positive_scores) >= 2:
                        assumption_text = f"{action} consistently gives positive score"
                        if hasattr(me, 'register_assumption'):
                            assumption_id = me.register_assumption(
                                agent_id=agent_id,
                                game_type=game_type,
                                level_number=level,
                                assumption=assumption_text,
                                assumption_type='rule'
                            )
                            assumptions_formed.append(assumption_text)

                    # Check for consistent negative score (form avoidance assumption)
                    negative_scores = [o for o in outcomes if o.get('score_delta', 0) < 0]
                    if len(negative_scores) >= 2:
                        assumption_text = f"{action} consistently causes penalty"
                        if hasattr(me, 'register_assumption'):
                            me.register_assumption(
                                agent_id=agent_id,
                                game_type=game_type,
                                level_number=level,
                                assumption=assumption_text,
                                assumption_type='rule'
                            )
                            assumptions_formed.append(assumption_text)

            if assumptions_formed:
                return RungResult(
                    confidence=0.3,
                    reason=f"Formed {len(assumptions_formed)} assumptions",
                    metadata={'assumptions': assumptions_formed}
                )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Assumption formation failed: {e}")


# Registry of rungs in this module
RUNGS = {
    'scientific_method': ScientificMethodRung,
    'two_streams': TwoStreamsRung,
    'metacognitive_prediction': MetacognitivePredictionRung,
    'theory_gate': TheoryGateRung,
    'sensation_engine': SensationEngineRung,
    'i_thread': IThreadRung,
    'event_understanding': EventUnderstandingRung,
    'resonance_detector': ResonanceDetectorRung,
    'interactable_tile_discovery': InteractableTileDiscoveryRung,
    'goal_relationship_modeling': GoalRelationshipModelingRung,
    'belief_system': BeliefSystemRung,
    'hypothesis_system': HypothesisSystemRung,
    'symbolic_tracker': SymbolicTrackerRung,
    'deliberation_system': DeliberationSystemRung,
    'hypothesis_testing': HypothesisTestingRung,
    'assumption_formation': AssumptionFormationRung,
}

## Exploitation Rungs

39 rungs for using known knowledge: DiscoveryExploitationRung, ReplayLearningRung, SpatialMapRung, etc.

In [ ]:
"""
Exploitation Rungs - Use known knowledge
========================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


def _get_frame_2d(game_state: Any) -> Optional[List[List[int]]]:
    """Extract frame as a 2D list-of-lists of ints.

    Handles FrameDataRaw.frame (List[ndarray]), raw ndarray, or nested lists.
    Returns None if frame can't be extracted.
    """
    raw = _get_frame(game_state)
    if raw is None:
        return None
    try:
        # Convert numpy to list
        if hasattr(raw, 'tolist'):
            raw = raw.tolist()
        if not isinstance(raw, list) or not raw:
            return None
        # Squeeze: List[ndarray] or [[[pixel]]] -> [[pixel]]
        # FrameDataRaw.frame returns List[ndarray]; .tolist() on that gives [[[int]]]
        while raw and isinstance(raw[0], list) and raw[0] and isinstance(raw[0][0], list):
            raw = raw[0]
        # Handle List[ndarray] directly (no .tolist on the outer list)
        if hasattr(raw[0], 'tolist'):
            raw = raw[0].tolist()
        # Validate 2D structure
        if not isinstance(raw, list) or not raw or not isinstance(raw[0], list):
            return None
        return raw
    except Exception:
        return None


class DiscoveryExploitationRung(DecisionRung):
    """Exploit recent discoveries immediately - EXPLOITATION

    Uses discovery_engine to get current discovery state and suggest actions
    that exploit what the agent has learned about object behaviors.
    """
    name = "discovery_exploitation"
    category = "exploitation"
    default_priority = 20
    confidence_threshold = 0.3

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        discovery_engine = self.engines.discovery_engine
        if discovery_engine is None:
            return RungResult()

        try:
            # Get current discovery state from the engine
            discovery_state = discovery_engine.get_state() if hasattr(discovery_engine, 'get_state') else None

            if not discovery_state:
                # Also check context for discovery info from evolution runner
                discovery = context.get('last_discovery')
                if not discovery:
                    return RungResult()
            else:
                # Extract discovery info from state
                discoveries = discovery_state.discoveries if hasattr(discovery_state, 'discoveries') else {}
                if not discoveries:
                    return RungResult()

                # Find the most recently discovered controllable object
                discovery = None
                for obj_id, behavior in discoveries.items():
                    if str(behavior) == 'ObjectBehavior.PLAYER_CONTROLLED':
                        discovery = {
                            'controlled_object': obj_id,
                            'behavior': str(behavior),
                            'reliability_score': 0.8,
                            'validated': True,
                            'action': 'ACTION1'  # Movement discovery suggests movement
                        }
                        break

                if not discovery:
                    return RungResult()

            action = discovery.get('action', '')
            reliability = discovery.get('reliability_score', 0.0)
            validated = discovery.get('validated', False)

            if not action.startswith('ACTION'):
                return RungResult()

            # CRITICAL: Validate action is available in this game
            if not is_action_available(action, context):
                return RungResult(reason=f"Discovery action {action} not available")

            decay = self._consecutive_no_change * 0.08
            if reliability >= 0.6 or validated:
                obj_info = discovery.get('controlled_object', discovery.get('controlled_color', '?'))
                conf = max(0.10, min(0.55, 0.9 - decay))
                return RungResult(
                    action=action,
                    confidence=conf,
                    reason=f"Exploiting discovery: {obj_info} (rel={reliability:.2f})",
                    metadata={'discovery': discovery}
                )
            elif reliability >= 0.3:
                obj_info = discovery.get('controlled_object', discovery.get('controlled_color', '?'))
                conf = max(0.10, min(0.55, 0.5 - decay))
                return RungResult(
                    action=action,
                    confidence=conf,
                    reason=f"Testing hypothesis: {obj_info} (rel={reliability:.2f})",
                    metadata={'discovery': discovery}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Discovery exploitation failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1


class EmbeddingSuggestionRung(DecisionRung):
    """Cross-game neural similarity matching - EXPLOITATION"""
    name = "embedding_suggestion"
    category = "exploitation"
    default_priority = 25
    confidence_threshold = 0.7

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sm = self.engines.self_model
        if sm is None:
            return RungResult()

        try:
            suggestion = sm.get_embedding_suggested_action(
                game_type=None,  # Search all games
                level=None,
                current_frame=_get_frame(game_state),
                top_k=10
            )

            if suggestion and suggestion.get('confidence', 0) >= self.confidence_threshold:
                action = suggestion.get('action')
                # CRITICAL: Validate action is available in this game
                if is_action_available(action, context):
                    return RungResult(
                        action=action,
                        confidence=suggestion.get('confidence', 0),
                        reason=f"Embedding match: {suggestion.get('similar_count', 0)} similar frames",
                        metadata={'suggestion': suggestion}
                    )

            # Even below threshold, return as weighted boost
            if suggestion and suggestion.get('confidence', 0) >= 0.4:
                suggested_action = suggestion.get('action')
                # CRITICAL: Validate action is available in this game
                if suggested_action and is_action_available(suggested_action, context):
                        return RungResult(
                            confidence=suggestion.get('confidence', 0),
                            weights={suggested_action: 1.0 + suggestion.get('confidence', 0) * 0.5},
                            reason=f"Embedding boost (below threshold): conf={suggestion.get('confidence', 0):.2f}",
                            metadata={'suggestion': suggestion}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Embedding suggestion failed: {e}")


class NetworkWisdomRung(DecisionRung):
    """Query network-wide action wisdom from action traces - EXPLOITATION

    This is Stream B (collective wisdom) in the two-stream model.
    Queries action_traces and winning_sequences to find what worked across all agents.

    Uses engines.memory.episodic_memory.EpisodicMemory for database queries.
    """
    name = "network_wisdom"
    category = "exploitation"
    default_priority = 35
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Use episodic_memory engine to query network wisdom
        episodic = self.engines.episodic_memory
        if episodic is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])

            # Query network wisdom for each available action
            best_action = None
            best_confidence = 0.0
            best_reason = ""

            for action_num in available:
                action_name = f"ACTION{action_num}"
                if hasattr(episodic, '_get_network_action_wisdom'):
                    wisdom = episodic._get_network_action_wisdom(game_type, action_name)
                    if wisdom.get('recommendation') == 'use':
                        conf = wisdom.get('confidence', 0)
                        if conf > best_confidence:
                            best_confidence = conf
                            best_action = action_name
                            best_reason = wisdom.get('reasoning', '')

            if best_action and best_confidence >= self.confidence_threshold:
                return RungResult(
                    action=best_action,
                    confidence=best_confidence,
                    reason=f"Network wisdom: {best_action} ({best_reason})",
                    metadata={'source': 'episodic_memory'}
                )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Network wisdom failed: {e}")


class FrontierTopologyRung(DecisionRung):
    """Network-level topology aggregation for frontier levels - EXPLOITATION

    Queries action_traces from ALL agents to build a collective map:
    - "From this frame_hash, what actions have been tried?"
    - "What were the outcomes (score_change, game_over)?"
    - Boost actions that led to positive outcomes
    - Penalize actions that led to death
    - Heavily boost UNTRIED actions (exploration bonus)

    This is the "whole point" - combining all agents' partial explorations
    into shared knowledge, even if no single agent has explored much.
    """
    name = "frontier_topology"
    category = "exploitation"
    default_priority = 28
    confidence_threshold = 0.3  # Can help even with sparse data

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            is_frontier = context.get('frontier_mode', False)
            if not is_frontier:
                return RungResult()

            # Get current frame hash from context
            frame_hash = context.get('frame_hash', '')
            if not frame_hash:
                return RungResult(reason="No frame_hash in context")

            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            # Query network's collective knowledge for this exact frame
            db = None
            if self.engines:
                try:
                    db = self.engines._get_db_interface()
                except Exception:
                    pass
            if db is None:
                return RungResult(reason="No database connection")

            # Network-level topology: What has ANYONE tried from this frame?
            results = db.execute_query("""
                SELECT
                    action_number,
                    COUNT(*) as attempts,
                    SUM(CASE WHEN score_change > 0 THEN 1 ELSE 0 END) as positive_outcomes,
                    SUM(CASE WHEN resulted_in_game_over = 1 THEN 1 ELSE 0 END) as deaths,
                    AVG(score_change) as avg_score_change,
                    SUM(CASE WHEN frame_changed = 1 THEN 1 ELSE 0 END) as frame_changes,
                    COUNT(DISTINCT session_id) as unique_sessions,
                    MIN(created_at) as first_seen,
                    MAX(created_at) as last_seen
                FROM action_traces
                WHERE frame_hash = ?
                  AND game_id LIKE ?
                  AND level_number = ?
                  AND action_number BETWEEN 1 AND 7
                GROUP BY action_number
            """, (frame_hash, f"{game_type}%", level))

            # Initialize weights with EXPLORATION BONUS for untried actions (available only)
            weights = get_available_action_weights(context, 1.5)  # Start high - untried = bonus!
            tried_actions: Set[str] = set()
            total_data_points = 0
            best_action = None
            best_score = -999

            # Provenance tracking
            total_positive = 0
            total_negative = 0
            unique_sessions: Set[int] = set()
            first_seen = None
            last_seen = None

            if results:
                for row in results:
                    action = f"ACTION{row['action_number']}"
                    tried_actions.add(action)
                    attempts = row['attempts'] or 1
                    positive = row['positive_outcomes'] or 0
                    deaths = row['deaths'] or 0
                    avg_change = row['avg_score_change'] or 0
                    frame_changes = row['frame_changes'] or 0
                    total_data_points += attempts

                    # CRITICAL: Skip actions that aren't available in this game
                    if not is_action_available(action, context):
                        continue

                    # Track provenance data
                    total_positive += positive
                    total_negative += deaths
                    sessions = row.get('unique_sessions', 1) or 1
                    unique_sessions.add(sessions)  # Approximate - actual is per-action
                    if row.get('first_seen'):
                        if first_seen is None or row['first_seen'] < first_seen:
                            first_seen = row['first_seen']
                    if row.get('last_seen'):
                        if last_seen is None or row['last_seen'] > last_seen:
                            last_seen = row['last_seen']

                    # Calculate action quality
                    success_rate = positive / attempts if attempts > 0 else 0
                    death_rate = deaths / attempts if attempts > 0 else 0
                    movement_rate = frame_changes / attempts if attempts > 0 else 0

                    # Score: positive outcomes good, deaths bad, movement good
                    action_score = (
                        success_rate * 2.0 +      # Big bonus for score increases
                        avg_change * 0.5 +         # Bonus for positive score change
                        movement_rate * 0.3 -      # Small bonus for causing frame change
                        death_rate * 1.5           # Penalty for deaths
                    )

                    # Convert score to weight (0.1 to 1.3 range for tried actions)
                    # Tried actions lose the exploration bonus but gain knowledge bonus
                    weight = max(0.1, min(1.3, 0.7 + action_score))
                    weights[action] = weight

                    if action_score > best_score:
                        best_score = action_score
                        best_action = action

            # Count untried actions (still have exploration bonus of 1.5)
            untried_actions = [a for a in weights if a not in tried_actions]
            untried_count = len(untried_actions)

            # Calculate confidence based on data coverage
            coverage = len(tried_actions) / 7.0
            sample_confidence = min(1.0, total_data_points / 20.0)
            confidence = coverage * 0.6 + sample_confidence * 0.4

            # Build provenance - track HOW this knowledge became knowable
            # Generation spread: if we have generation data, use it; else estimate from timestamps
            temporal_generations = 0.0

            # Try to get generation spread from action_traces if available
            try:
                gen_results = db.execute_query("""
                    SELECT MIN(generation) as min_gen, MAX(generation) as max_gen
                    FROM action_traces
                    WHERE frame_hash = ?
                      AND game_id LIKE ?
                      AND level_number = ?
                      AND generation IS NOT NULL
                """, (frame_hash, f"{game_type}%", level))

                if gen_results and gen_results[0].get('min_gen') is not None:
                    min_gen = gen_results[0]['min_gen']
                    max_gen = gen_results[0]['max_gen']
                    temporal_generations = float(max_gen - min_gen)
            except Exception:
                pass  # Generation column may not exist yet

            provenance = KnowledgeProvenance(
                detection_source='action_traces',
                sample_size=total_data_points,
                agent_diversity=len(unique_sessions),  # Unique sessions as proxy for diversity
                temporal_spread_generations=temporal_generations,
                validation_type='outcome_based' if total_positive > 0 else 'frequency',
                positive_outcomes=total_positive,
                negative_outcomes=total_negative,
                crystallization_stage=2 if total_data_points > 10 else 1,  # Detected -> Classified
            )

            # Build reason string
            if untried_count == 7:
                reason = f"Frontier topology: No data for this frame - all actions have exploration bonus"
            elif untried_count > 0:
                reason = f"Frontier topology: {7-untried_count}/7 actions mapped, {untried_count} untried (boosted)"
            else:
                reason = f"Frontier topology: All actions mapped, best={best_action} (score={best_score:.2f})"

            return RungResult(
                action=best_action if best_action and confidence >= 0.5 else None,
                confidence=confidence,
                reason=reason,
                weights=weights,
                metadata={
                    'tried_actions': list(tried_actions),
                    'untried_actions': untried_actions,
                    'total_data_points': total_data_points,
                    'best_action': best_action,
                    'best_score': best_score,
                    'coverage': coverage
                },
                provenance=provenance,
            )
        except Exception as e:
            return RungResult(reason=f"Frontier topology failed: {e}")


class MapIntelCollisionRung(DecisionRung):
    """Obstacle avoidance when last action caused no frame change - EXPLOITATION

    Uses context 'frame_changed' flag to detect collisions and suggest
    perpendicular movement alternatives.
    """
    name = "map_intel_collision"
    category = "exploitation"
    default_priority = 24
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            # Check if last action caused no frame change (collision detection)
            # This is set by evolution_runner when it detects no change between frames
            frame_changed = context.get('frame_changed', True)
            last_action = context.get('last_action', '')
            available = context.get('available_actions', [1, 2, 3, 4])

            # Only applies to movement actions (1-4) that are available
            movement_available = [f'ACTION{a}' for a in available if a in [1, 2, 3, 4]]

            # If frame changed or last action wasn't movement, no collision recovery needed
            if frame_changed or last_action not in movement_available:
                return RungResult()

            # Get perpendicular alternatives (filtered by available)
            perpendicular_map = {
                'ACTION1': ['ACTION3', 'ACTION4'],
                'ACTION2': ['ACTION3', 'ACTION4'],
                'ACTION3': ['ACTION1', 'ACTION2'],
                'ACTION4': ['ACTION1', 'ACTION2'],
            }

            alternatives = perpendicular_map.get(last_action, [])
            # Filter to only available alternatives
            alternatives = [a for a in alternatives if a in movement_available]
            if alternatives:
                action = random.choice(alternatives)
                return RungResult(
                    action=action,
                    confidence=0.6,
                    reason=f"Collision recovery: {last_action} blocked, trying {action}",
                    metadata={'blocked_action': last_action, 'alternatives': alternatives}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Map intel collision failed: {e}")


class AbstractionTemplatesRung(DecisionRung):
    """Use pattern templates from winning sequences - EXPLOITATION"""
    name = "abstraction_templates"
    category = "exploitation"
    default_priority = 45
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        engine = self.engines.abstraction_engine
        if engine is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            if hasattr(engine, 'should_use_template') and engine.should_use_template(game_type, level):
                template = engine.get_template_for_replay(game_type, level)
                if template:
                    action_idx = context.get('template_position', 0)
                    if action_idx < len(template):
                        action = template[action_idx]
                        # CRITICAL: Validate action is available in this game
                        if is_action_available(action, context):
                            return RungResult(
                                action=action,
                                confidence=0.6,
                                reason=f"Following template: step {action_idx + 1}/{len(template)}",
                                metadata={'template': template, 'position': action_idx}
                            )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Abstraction templates failed: {e}")


class FewShotInvariantsRung(DecisionRung):
    """Relational bias from few-shot control relations - EXPLOITATION"""
    name = "few_shot_invariants"
    category = "exploitation"
    default_priority = 46
    confidence_threshold = 0.35

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sm = self.engines.self_model
        if sm is None:
            return RungResult()

        try:
            if hasattr(sm, 'get_few_shot_control_relations'):
                invariants = sm.get_few_shot_control_relations()
                if invariants and invariants.get('sample_size', 0) >= 2:
                    action = invariants.get('suggested_action')
                    # CRITICAL: Validate action is available in this game
                    if action and is_action_available(action, context):
                        return RungResult(
                            action=action,
                            confidence=0.5,
                            reason=f"Few-shot invariant: sample_size={invariants.get('sample_size')}",
                            metadata={'invariants': invariants}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Few-shot invariants failed: {e}")


class FrontierCheckpointRung(DecisionRung):
    """Replay best frontier checkpoint on unbeaten levels - EXPLOITATION

    On frontier (unbeaten) levels, queries the frontier_checkpoints table for
    the best known partial progress. Replays that checkpoint sequence to skip
    already-explored territory, then lets exploration take over.

    This implements constructive pathfinding - building winning sequences
    incrementally by remembering the best partial progress across all agents.

    See: architecture/frontier_checkpoint_system.md
    """
    name = "frontier_checkpoint"
    category = "exploitation"
    default_priority = 6  # Very early - before three_try_sequence (8)
    confidence_threshold = 0.85

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._cached_checkpoint: Optional[Dict[str, Any]] = None
        self._cache_key: Optional[Tuple[str, int]] = None
        self._db: Any = None  # Lazy-loaded database interface
        self._consecutive_no_change = 0

    def _get_db(self) -> Any:
        """Get database interface, lazy-loading if needed."""
        if self._db is None and self.engines:
            try:
                self._db = self.engines._get_db_interface()
            except Exception:
                pass
        return self._db

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            # Only fire on frontier levels (no winning sequence exists)
            is_frontier = context.get('is_frontier', False) or context.get('frontier_mode', False)
            if not is_frontier:
                return RungResult()

            # Check if we're already replaying a checkpoint
            checkpoint_position = context.get('checkpoint_position', 0)
            checkpoint_sequence = context.get('checkpoint_sequence')

            # If checkpoint sequence is active and we have more actions to replay
            if checkpoint_sequence and checkpoint_position < len(checkpoint_sequence):
                action = checkpoint_sequence[checkpoint_position]
                # CRITICAL: Validate action is available in this game
                if is_action_available(action, context):
                    decay = self._consecutive_no_change * 0.08
                    conf = max(0.10, min(0.55, 0.85 - decay))
                    return RungResult(
                        action=action,
                        confidence=conf,
                        reason=f"Frontier checkpoint replay: step {checkpoint_position + 1}/{len(checkpoint_sequence)}",
                        metadata={
                            'checkpoint_replay': True,
                            'checkpoint_position': checkpoint_position,
                            'checkpoint_length': len(checkpoint_sequence),
                        }
                    )
                # Action not available - skip checkpoint replay
                return RungResult(reason=f"Checkpoint action {action} not available")

            # If no active checkpoint, try to load one from database
            db = self._get_db()
            if checkpoint_sequence is None and db is not None:
                game_type = context.get('game_type', '')
                level = context.get('level', 1)

                # Check cache first
                cache_key = (game_type, level)
                if self._cache_key != cache_key:
                    self._cached_checkpoint = self._query_best_checkpoint(game_type, level)
                    self._cache_key = cache_key

                if self._cached_checkpoint:
                    sequence = self._cached_checkpoint.get('action_sequence', [])
                    if sequence:
                        first_action = sequence[0]
                        # CRITICAL: Validate first action is available
                        if is_action_available(first_action, context):
                            decay = self._consecutive_no_change * 0.08
                            conf = max(0.10, min(0.55, 0.85 - decay))
                            return RungResult(
                                action=first_action,
                                confidence=conf,
                                reason=f"Starting frontier checkpoint: {len(sequence)} actions from best progress",
                                metadata={
                                    'checkpoint_replay': True,
                                    'checkpoint_position': 0,
                                    'checkpoint_length': len(sequence),
                                    'checkpoint_loaded': True,
                                    'checkpoint_data': self._cached_checkpoint,
                                }
                            )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Frontier checkpoint failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1

    def _query_best_checkpoint(self, game_type: str, level: int) -> Optional[Dict[str, Any]]:
        """Query database for best checkpoint for this game_type/level."""
        db = self._get_db()
        if db is None:
            return None

        try:
            result: List[Dict[str, Any]] = db.execute_query("""
                SELECT action_sequence, actions_count, survival_score,
                       unique_frames_seen, terminal_frame_hash
                FROM frontier_checkpoints
                WHERE game_type = ? AND level_number = ?
                ORDER BY survival_score DESC, times_extended DESC
                LIMIT 1
            """, (game_type, level))

            if result and len(result) > 0:
                row = result[0]
                action_sequence: Any = json.loads(row.get('action_sequence', '[]')) if isinstance(row.get('action_sequence'), str) else row.get('action_sequence', [])
                return {
                    'action_sequence': action_sequence,
                    'actions_count': row.get('actions_count', 0),
                    'survival_score': row.get('survival_score', 0),
                    'unique_frames_seen': row.get('unique_frames_seen', 0),
                    'terminal_frame_hash': row.get('terminal_frame_hash'),
                }
        except Exception as e:
            logger.debug(f"[FRONTIER-CHECKPOINT] Query failed: {e}")

        return None

    def clear_cache(self) -> None:
        """Clear cached checkpoint (call on level transition)."""
        self._cached_checkpoint = None
        self._cache_key = None


class ThreeTrySequenceRung(DecisionRung):
    """Try up to 3 ranked sequences before exploration - GAME-LEVEL"""
    name = "three_try_sequence"
    category = "exploitation"
    default_priority = 8  # Early - before most decisions
    confidence_threshold = 0.7

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            # Check if we have an active sequence
            active_sequence = context.get('active_sequence')
            sequence_position = context.get('sequence_position', 0)

            if active_sequence and sequence_position < len(active_sequence):
                action = active_sequence[sequence_position]
                # CRITICAL: Validate action is available in this game
                if is_action_available(action, context):
                    decay = self._consecutive_no_change * 0.08
                    conf = max(0.10, min(0.55, 0.8 - decay))
                    return RungResult(
                        action=action,
                        confidence=conf,
                        reason=f"Following sequence: step {sequence_position + 1}/{len(active_sequence)}",
                        metadata={'sequence_length': len(active_sequence), 'position': sequence_position}
                    )
                # Action not available - skip this sequence step
                return RungResult(reason=f"Sequence action {action} not available")
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Three-try sequence failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1


class MultiStageMatchingRung(DecisionRung):
    """Cascading sequence matching with 5 fallback strategies - EXPLOITATION"""
    name = "multi_stage_matching"
    category = "exploitation"
    default_priority = 42
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        pipeline = self.engines.multi_stage_pipeline
        if pipeline is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            if hasattr(pipeline, 'get_sequence_with_fallback'):
                # Returns Tuple[Optional[List[int]], str, Dict[str, Any]]
                seq_actions, stage_used, match_meta = pipeline.get_sequence_with_fallback(game_type, level)
                if seq_actions:
                    first_action = seq_actions[0] if seq_actions else None
                    # CRITICAL: Validate action is available in this game
                    if first_action and is_action_available(first_action, context):
                        return RungResult(
                            action=first_action,
                            confidence=match_meta.get('confidence', 0.5),
                            reason=f"Multi-stage match: {stage_used}",
                            metadata={'match_result': {'sequence': seq_actions, 'stage': stage_used, **match_meta}}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Multi-stage matching failed: {e}")


class NearMissAnalyzerRung(DecisionRung):
    """Learn from high-score failures - POST-HOC"""
    name = "near_miss_analyzer"
    category = "exploitation"
    default_priority = 48
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        nma = self.engines.near_miss_analyzer
        if nma is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            if hasattr(nma, 'get_insights'):
                insights = nma.get_insights(game_type, level)
                if insights:
                    # Use insights to suggest action
                    suggested = insights.get('suggested_action')
                    # CRITICAL: Validate action is available in this game
                    if suggested and is_action_available(suggested, context):
                        return RungResult(
                            action=suggested,
                            confidence=insights.get('confidence', 0.4),
                            reason=f"Near-miss insight: {insights.get('category', 'unknown')}",
                            metadata={'insights': insights}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Near-miss analyzer failed: {e}")


class StateMatchingRung(DecisionRung):
    """
    Compare current player properties to learned goal requirements - EXPLOITATION

    Part of Symbolic Reasoning Implementation (Phase 4).

    This rung uses LEARNED data (not hard-coded game knowledge) to:
    1. Check if current player properties match goal requirements
    2. If mismatch, find a transformer that can fix it
    3. Suggest navigation toward transformer or goal

    Data Sources:
    - player_state_history: Current player properties (from PlayerLocalizer/PropertyExtractor)
    - goal_requirements: Learned requirements from successes/failures
    - property_transformations: Known transformers that change properties

    Graceful degradation: Returns empty result when no learned data exists.
    """
    name = "state_matching"
    category = "exploitation"
    default_priority = 26  # After rule_transfer (25), before frontier_topology (28)
    confidence_threshold = 0.5

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._db: Optional[Any] = None
        # Cache for current game session
        self._cached_requirements: Dict[str, Any] = {}
        self._cached_transformers: Dict[str, Any] = {}

    def _get_db(self) -> Optional[Any]:
        """Lazy-load database interface."""
        if self._db is None:
            try:
                self._db = DatabaseInterface()
            except ImportError as e:
                logger.debug(f"[STATE-MATCHING] Failed to load DatabaseInterface: {e}")
        return self._db

    def _get_goal_requirements(self, game_type: str, level: int) -> Optional[Dict[str, Any]]:
        """Query learned goal requirements for this game/level."""
        cache_key = f"{game_type}:{level}"
        if cache_key in self._cached_requirements:
            return self._cached_requirements[cache_key]

        db = self._get_db()
        if db is None:
            return None

        try:
            result = db.execute_query("""
                SELECT required_dominant_color, required_shape_phash, required_orientation,
                       times_succeeded, times_failed, confidence
                FROM goal_requirements
                WHERE game_id LIKE ? AND level_number = ?
                  AND confidence >= 0.5
                ORDER BY confidence DESC
                LIMIT 1
            """, (f"{game_type}%", level))

            row = result[0] if result else None
            if row:
                req = {
                    'dominant_color': row['required_dominant_color'],
                    'shape_signature': row['required_shape_phash'],
                    'orientation': row['required_orientation'],
                    'times_succeeded': row['times_succeeded'],
                    'times_failed': row['times_failed'],
                    'confidence': row['confidence'],
                }
                self._cached_requirements[cache_key] = req
                return req

            self._cached_requirements[cache_key] = None
            return None
        except Exception as e:
            logger.debug(f"[STATE-MATCHING] Goal query failed: {e}")
            return None

    def _get_transformers(self, game_type: str, level: int, property_needed: str) -> List[Dict[str, Any]]:
        """Query known transformers that can change the specified property."""
        cache_key = f"{game_type}:{level}:{property_needed}"
        if cache_key in self._cached_transformers:
            return self._cached_transformers[cache_key]

        db = self._get_db()
        if db is None:
            return []

        try:
            result = db.execute_query("""
                SELECT object_position_x, object_position_y,
                       value_before, value_after,
                       times_observed, confidence
                FROM property_transformations
                WHERE game_id LIKE ? AND level_number = ?
                  AND property_changed = ?
                  AND confidence >= 0.5
                ORDER BY confidence DESC
                LIMIT 5
            """, (f"{game_type}%", level, property_needed))

            transformers = []
            for row in result if result else []:
                transformers.append({
                    'position': (row['object_position_x'], row['object_position_y']),
                    'value_before': row['value_before'],
                    'value_after': row['value_after'],
                    'times_observed': row['times_observed'],
                    'confidence': row['confidence'],
                })

            self._cached_transformers[cache_key] = transformers
            return transformers
        except Exception as e:
            logger.debug(f"[STATE-MATCHING] Transformer query failed: {e}")
            return []

    def _get_current_properties(self, context: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        """Get current player properties from context or recent state history."""
        # First check if properties are in context (injected by evolution_runner)
        if 'player_properties' in context:
            return context['player_properties']

        # Otherwise query recent player_state_history
        db = self._get_db()
        if db is None:
            return None

        try:
            session_id = context.get('session_id')
            if not session_id:
                return None

            result = db.execute_query("""
                SELECT dominant_color, shape_phash, orientation, properties_json
                FROM player_state_history
                WHERE session_id = ?
                  AND dominant_color IS NOT NULL
                ORDER BY id DESC
                LIMIT 1
            """, (session_id,))

            row = result[0] if result else None
            if row:
                return {
                    'dominant_color': row['dominant_color'],
                    'shape_signature': row['shape_phash'],
                    'orientation': row['orientation'],
                }
            return None
        except Exception as e:
            logger.debug(f"[STATE-MATCHING] Properties query failed: {e}")
            return None

    def _find_mismatches(
        self,
        current: Dict[str, Any],
        required: Dict[str, Any]
    ) -> List[str]:
        """Find which properties don't match requirements."""
        mismatches = []

        # Check dominant_color
        if required.get('dominant_color') and current.get('dominant_color'):
            req_color = str(required['dominant_color'])
            cur_color = str(current['dominant_color'])
            if req_color != cur_color:
                mismatches.append('dominant_color')

        # Check orientation
        if required.get('orientation') is not None and current.get('orientation') is not None:
            if int(required['orientation']) != int(current['orientation']):
                mismatches.append('orientation')

        # Check shape_signature (allow some hamming distance)
        if required.get('shape_signature') and current.get('shape_signature'):
            req_sig = required['shape_signature']
            cur_sig = current['shape_signature']
            if len(req_sig) == len(cur_sig):
                hamming = sum(c1 != c2 for c1, c2 in zip(req_sig, cur_sig))
                if hamming > 8:  # Allow up to 8-bit difference in 64-bit signature
                    mismatches.append('shape_signature')

        return mismatches

    def _suggest_direction_to_position(
        self,
        target_pos: Tuple[int, int],
        context: Dict[str, Any]
    ) -> Optional[str]:
        """Suggest action to move toward a target position."""
        # Get current player position from context
        player_pos = context.get('player_position')
        if not player_pos:
            return None

        curr_row, curr_col = player_pos
        target_row, target_col = target_pos

        # Calculate direction
        row_diff = target_row - curr_row
        col_diff = target_col - curr_col

        # Prioritize larger difference
        if abs(row_diff) > abs(col_diff):
            if row_diff < 0:
                return 'ACTION1'  # Up
            else:
                return 'ACTION2'  # Down
        else:
            if col_diff < 0:
                return 'ACTION3'  # Left
            elif col_diff > 0:
                return 'ACTION4'  # Right

        return None

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        """
        Evaluate state matching and suggest action if applicable.

        Returns empty result if:
        - No learned goal requirements exist
        - Current properties not available
        - Properties already match (no action needed from this rung)
        """
        game_type = context.get('game_type', '')
        level = context.get('level', 1)

        # Step 1: Get learned goal requirements
        requirements = self._get_goal_requirements(game_type, level)
        if not requirements:
            # No learned requirements yet - graceful degradation
            return RungResult()

        # Step 2: Get current player properties
        current_props = self._get_current_properties(context)
        if not current_props:
            # Can't determine current state
            return RungResult(reason="No current properties available")

        # Step 3: Find mismatches
        mismatches = self._find_mismatches(current_props, requirements)

        if not mismatches:
            # Properties match! Suggest moving toward goal
            # (but we don't know goal position, so just report match)
            return RungResult(
                confidence=0.3,  # Low confidence - just informational
                reason=f"Properties match goal requirements (conf={requirements['confidence']:.2f})",
                metadata={
                    'state': 'properties_match',
                    'requirements': requirements,
                    'current': current_props,
                }
            )

        # Step 4: Find transformer to fix first mismatch
        first_mismatch = mismatches[0]
        transformers = self._get_transformers(game_type, level, first_mismatch)

        if not transformers:
            # We know there's a mismatch but don't know any transformers
            return RungResult(
                confidence=0.2,  # Very low - we identified a problem but can't solve it
                reason=f"Property mismatch ({first_mismatch}) but no known transformers",
                metadata={
                    'state': 'mismatch_no_transformer',
                    'mismatches': mismatches,
                    'current': current_props,
                    'required': requirements,
                }
            )

        # Step 5: Suggest navigation toward transformer
        best_transformer = transformers[0]
        target_pos = best_transformer['position']

        if target_pos[0] is None or target_pos[1] is None:
            # Transformer position unknown
            return RungResult(
                confidence=0.3,
                reason=f"Found transformer for {first_mismatch} but position unknown",
                metadata={
                    'state': 'transformer_unknown_position',
                    'transformer': best_transformer,
                }
            )

        suggested_action = self._suggest_direction_to_position(target_pos, context)

        if suggested_action:
            # Validate action is available
            if not is_action_available(suggested_action, context):
                return RungResult(
                    reason=f"State matching suggested unavailable action: {suggested_action}"
                )

            return RungResult(
                action=suggested_action,
                confidence=min(0.6, best_transformer['confidence']),
                reason=f"Navigate to transformer for {first_mismatch} ({best_transformer['value_before']}->{best_transformer['value_after']})",
                metadata={
                    'state': 'navigating_to_transformer',
                    'target_position': target_pos,
                    'mismatch': first_mismatch,
                    'transformer': best_transformer,
                    'current': current_props,
                    'required': requirements,
                }
            )

        # Can't determine direction (maybe at transformer already?)
        return RungResult(
            confidence=0.3,
            reason=f"Need {first_mismatch} change, transformer at {target_pos}",
            metadata={
                'state': 'at_or_near_transformer',
                'transformer': best_transformer,
                'mismatches': mismatches,
            }
        )

    def clear_cache(self):
        """Clear cached data (call on game/level change)."""
        self._cached_requirements.clear()
        self._cached_transformers.clear()


class SpatialRelationshipRung(DecisionRung):
    """
    Learn and exploit spatial relationships in click puzzles.

    Tracks: "clicking position A affects positions B, C, D"
    Uses: "to change position X, I should click position Y"

    This rung is particularly useful for:
    - Tile puzzles where clicking cycles colors (self-only or multi-cell)
    - Games with spatial dependencies discovered from frame diffs
    - Any game where clicks have predictable spatial effects

    NOTE: Effect patterns are learned from CausalMap observations,
    not assumed. The rung discovers what each click actually does.
    """
    name = "spatial_relationship"
    category = "exploitation"
    default_priority = 44  # After state_matching (42), before frontier_topology
    confidence_threshold = 0.5

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._effect_learner: Optional[Any] = None
        self._goal_tracker: Optional[Any] = None
        self._property_extractor: Optional[Any] = None
        self._db: Optional[Any] = None
        # Cache
        self._effect_pattern_cache: Dict[str, List[Tuple[int, int]]] = {}
        self._goal_cache: Dict[Tuple[str, int], Dict] = {}

    def _get_db(self) -> Optional[Any]:
        """Lazy-load database interface."""
        if self._db is None:
            try:
                self._db = DatabaseInterface()
            except ImportError:
                pass
        return self._db

    def _get_effect_learner(self) -> Optional[Any]:
        """Lazy-load spatial effect learner."""
        if self._effect_learner is None:
            try:
                from engines.perception.spatial_learning import SpatialEffectLearner
                db = self._get_db()
                if db:
                    self._effect_learner = SpatialEffectLearner(db)
            except ImportError:
                pass
        return self._effect_learner

    def _get_goal_tracker(self) -> Optional[Any]:
        """Lazy-load goal tracker."""
        if self._goal_tracker is None:
            try:
                from engines.perception.spatial_learning import MultiObjectGoalTracker
                db = self._get_db()
                if db:
                    self._goal_tracker = MultiObjectGoalTracker(db)
            except ImportError:
                pass
        return self._goal_tracker

    def _get_property_extractor(self) -> Optional[Any]:
        """Lazy-load property extractor."""
        if self._property_extractor is None:
            try:
                from engines.perception.spatial_learning import PropertyExtractor
                self._property_extractor = PropertyExtractor()
            except ImportError:
                pass
        return self._property_extractor

    def _extract_grid_state(self, frame: Any) -> Dict[Tuple[int, int], int]:
        """Extract current grid state from frame."""
        extractor = self._get_property_extractor()
        if extractor is None or frame is None:
            return {}

        try:
            import numpy as np
            if isinstance(frame, np.ndarray):
                return extractor.extract_grid_state(frame)
        except Exception:
            pass
        return {}

    def _get_effect_pattern(self, game_type: str) -> List[Tuple[int, int]]:
        """Get learned effect pattern for this game."""
        if game_type in self._effect_pattern_cache:
            return self._effect_pattern_cache[game_type]

        learner = self._get_effect_learner()
        if learner is None:
            return []

        try:
            pattern = learner.get_effect_pattern(game_type)
            self._effect_pattern_cache[game_type] = pattern
            return pattern
        except Exception:
            return []

    def _get_target_configuration(
        self,
        game_type: str,
        level: int
    ) -> Optional[Dict[Tuple[int, int], int]]:
        """Get known winning configuration for this level."""
        cache_key = (game_type, level)
        if cache_key in self._goal_cache:
            return self._goal_cache[cache_key]

        tracker = self._get_goal_tracker()
        if tracker is None:
            return None

        try:
            target = tracker.get_target_configuration(game_type, level)
            if target:
                self._goal_cache[cache_key] = target
            return target
        except Exception:
            return None

    def _find_differences(
        self,
        current: Dict[Tuple[int, int], int],
        target: Dict[Tuple[int, int], int]
    ) -> List[Tuple[int, int]]:
        """Find positions where current differs from target."""
        differences = []
        all_positions = set(current.keys()) | set(target.keys())

        for pos in all_positions:
            if current.get(pos, 0) != target.get(pos, 0):
                differences.append(pos)

        return differences

    def _find_best_click(
        self,
        current: Dict[Tuple[int, int], int],
        target: Dict[Tuple[int, int], int],
        differences: List[Tuple[int, int]],
        effect_pattern: List[Tuple[int, int]],
        grid_size: int = 8
    ) -> Optional[Tuple[int, int]]:
        """Find click position that reduces difference from goal."""
        best_click = None
        best_improvement = 0

        # Try clicking each grid position
        for gx in range(grid_size):
            for gy in range(grid_size):
                improvement = 0

                # Simulate effect
                for (rel_x, rel_y) in effect_pattern:
                    affected_x = gx + rel_x
                    affected_y = gy + rel_y

                    if (affected_x, affected_y) in differences:
                        improvement += 1

                if improvement > best_improvement:
                    best_improvement = improvement
                    best_click = (gx, gy)

        return best_click if best_improvement > 0 else None

    def _grid_to_pixel(
        self,
        grid_pos: Tuple[int, int],
        frame_size: int = 64,
        grid_size: int = 8
    ) -> Tuple[int, int]:
        """Convert grid position to pixel coordinates (center of cell)."""
        cell_size = frame_size // grid_size
        pixel_x = grid_pos[0] * cell_size + cell_size // 2
        pixel_y = grid_pos[1] * cell_size + cell_size // 2
        return (pixel_x, pixel_y)

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Learn from click effects (called by outcome processor)."""
        if action != 'ACTION6':
            return

        learner = self._get_effect_learner()
        extractor = self._get_property_extractor()

        if learner is None or extractor is None:
            return

        try:
            import numpy as np
            if not isinstance(frame_before, np.ndarray) or not isinstance(frame_after, np.ndarray):
                return

            game_type = context.get('game_type', '')
            click_x = action_data.get('x', 0)
            click_y = action_data.get('y', 0)

            # Detect grid size and convert pixel to grid
            grid_size = extractor._detect_grid_size(frame_before)
            click_grid_pos = extractor.pixel_to_grid(
                click_x, click_y,
                frame_before.shape[1], frame_before.shape[0],
                grid_size
            )

            # Detect position changes
            changes = extractor.detect_position_changes(frame_before, frame_after, grid_size)

            if changes:
                learner.record_click_effect(game_type, click_grid_pos, changes)
                # Invalidate cache
                self._effect_pattern_cache.pop(game_type, None)

        except Exception:
            pass

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # ACTION6 only
        available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
        if 6 not in available:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)

        # Get current grid state
        frame = _get_frame(game_state)
        current_state = self._extract_grid_state(frame)

        if not current_state:
            return RungResult()

        # Get target configuration (if known)
        target = self._get_target_configuration(game_type, level)

        if not target:
            return RungResult()  # Don't know goal yet

        # Find differences
        differences = self._find_differences(current_state, target)

        if not differences:
            return RungResult(
                confidence=0.3,
                reason="Grid state matches known goal configuration",
                metadata={'state': 'at_goal'}
            )

        # Get effect pattern
        effect_pattern = self._get_effect_pattern(game_type)

        if not effect_pattern:
            return RungResult()  # Don't know effects yet

        # Find click that moves us toward goal
        best_click = self._find_best_click(
            current_state, target, differences, effect_pattern
        )

        if best_click:
            click_x, click_y = self._grid_to_pixel(best_click)
            return RungResult(
                action='ACTION6',
                confidence=0.6,
                reason=f"Click ({best_click[0]}, {best_click[1]}) to change {len(differences)} tiles toward goal",
                metadata={
                    'grid_position': best_click,
                    'pixel_position': (click_x, click_y),
                    'differences_count': len(differences),
                    'effect_pattern_size': len(effect_pattern),
                }
            )

        return RungResult()

    def clear_cache(self):
        """Clear cached data."""
        self._effect_pattern_cache.clear()
        self._goal_cache.clear()


class SubgoalPlanningRung(DecisionRung):
    """Decompose complex levels into subgoals - EXPLOITATION"""
    name = "subgoal_planning"
    category = "exploitation"
    default_priority = 38
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        planner = self.engines.subgoal_planner
        if planner is None:
            return RungResult()

        try:
            if hasattr(planner, 'get_current_subgoal'):
                subgoal = planner.get_current_subgoal()
                if subgoal and subgoal.get('next_action'):
                    action = subgoal['next_action']
                    # CRITICAL: Validate action is available in this game
                    if not is_action_available(action, context):
                        return RungResult(reason=f"Subgoal action {action} not available")
                    return RungResult(
                        action=action,
                        confidence=subgoal.get('confidence', 0.5),
                        reason=f"Subgoal {subgoal.get('index', '?')}/{subgoal.get('total', '?')}: {subgoal.get('description', '')}",
                        metadata={'subgoal': subgoal}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Subgoal planning failed: {e}")


class VisualAnalyzerRung(DecisionRung):
    """Identify priority targets for ACTION6 clicks - EXPLOITATION"""
    name = "visual_analyzer"
    category = "exploitation"
    default_priority = 36
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        va = self.engines.visual_analyzer
        if va is None:
            return RungResult()

        try:
            # ACTION6 is typically 'click' - only suggest if available
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            if 6 not in available:
                return RungResult()  # Click not available for this game

            frame = _get_frame(game_state)

            if hasattr(va, 'get_priority_targets'):
                targets = va.get_priority_targets(frame)
                if targets:
                    best = targets[0]
                    return RungResult(
                        action='ACTION6',
                        confidence=best.get('confidence', 0.5),
                        reason=f"Visual target: {best.get('reason', 'unknown')} at ({best.get('x', 0)}, {best.get('y', 0)})",
                        metadata={'target': best, 'all_targets': len(targets)}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Visual analyzer failed: {e}")


class NetworkObjectInventoryRung(DecisionRung):
    """Query network knowledge about interactable objects - EXPLOITATION"""
    name = "network_object_inventory"
    category = "exploitation"
    default_priority = 37
    confidence_threshold = 0.45

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        sm = self.engines.self_model
        if sm is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            # ACTION6 is typically 'click' - only suggest if available
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            if 6 not in available:
                return RungResult()  # Click not available for this game

            if hasattr(sm, 'get_network_object_inventory'):
                inventory = sm.get_network_object_inventory(game_type, level)
                if inventory.get('total_unique', 0) > 0:
                    # Bias toward interacting with known objects
                    interactable = inventory.get('interactable', [])
                    if interactable:
                        # Extract coordinates from first interactable object
                        first_obj = interactable[0]
                        x = first_obj.get('x', first_obj.get('center_x', 32))
                        y = first_obj.get('y', first_obj.get('center_y', 32))
                        return RungResult(
                            action='ACTION6',
                            confidence=0.5,
                            reason=f"Network inventory: {len(interactable)} interactable at ({x},{y})",
                            metadata={
                                'x': x,
                                'y': y,
                                'inventory': inventory,
                                'target_object': first_obj
                            }
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Network object inventory failed: {e}")


class ClickBehaviorLearningRung(DecisionRung):
    """Learn and predict click behaviors using ClickBehaviorClassifier - EXPLOITATION

    Uses engines/self_model/click_behavior.py to:
    1. Predict what clicking an object will do (collect, toggle, trigger, etc.)
    2. Suggest clicks on objects with positive behaviors (score+)
    3. Avoid clicks on objects with negative behaviors (score-)

    This enables intelligent clicking based on learned object behaviors.
    """
    name = "click_behavior_learning"
    category = "exploitation"
    default_priority = 36  # Just above NetworkObjectInventoryRung
    confidence_threshold = 0.4

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        cb = self.engines.click_behavior
        if cb is None:
            return RungResult()

        try:
            # ACTION6 is typically 'click' - only suggest if available
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            if 6 not in available:
                return RungResult()  # Click not available for this game

            game_type = context.get('game_type', '')
            if not game_type:
                return RungResult()

            # Get current frame for locating objects by color
            frame = _get_frame(game_state)

            # First: Check for collectible objects (positive score impact)
            if hasattr(cb, 'get_collectible_objects'):
                collectible_ids = cb.get_collectible_objects(game_type)
                if collectible_ids and hasattr(cb, 'get_all_profiles'):
                    # Find profiles for collectible objects to get their colors
                    profiles = cb.get_all_profiles(game_type, min_clicks=2)
                    profile_map = {p.object_id: p for p in profiles}
                    for obj_id in collectible_ids:
                        profile = profile_map.get(obj_id)
                        if profile and profile.color > 0 and frame is not None:
                            # Locate this color in the current frame
                            pos = self._find_color_in_frame(frame, profile.color)
                            if pos:
                                x, y = pos
                                decay = self._consecutive_no_change * 0.08
                                raw = 0.6 + profile.avg_score_impact * 0.2
                                conf = max(0.10, min(0.55, raw - decay))
                                return RungResult(
                                    action='ACTION6',
                                    confidence=conf,
                                    reason=f"Collectible color={profile.color} at ({x},{y}) - avg impact: {profile.avg_score_impact:.2f}",
                                    metadata={
                                        'x': x,
                                        'y': y,
                                        'object_id': obj_id,
                                        'color': profile.color,
                                        'behavior_type': 'collect'
                                    }
                                )

            # Second: Check for trigger objects (chain reactions)
            if hasattr(cb, 'get_trigger_objects'):
                trigger_ids = cb.get_trigger_objects(game_type)
                if trigger_ids and hasattr(cb, 'get_all_profiles'):
                    profiles = cb.get_all_profiles(game_type, min_clicks=2)
                    profile_map = {p.object_id: p for p in profiles}
                    for obj_id in trigger_ids:
                        profile = profile_map.get(obj_id)
                        if profile and profile.color > 0 and frame is not None:
                            pos = self._find_color_in_frame(frame, profile.color)
                            if pos:
                                x, y = pos
                                decay = self._consecutive_no_change * 0.08
                                conf = max(0.10, min(0.55, 0.5 - decay))
                                return RungResult(
                                    action='ACTION6',
                                    confidence=conf,
                                    reason=f"Trigger color={profile.color} at ({x},{y}) - may cause chain reaction",
                                    metadata={
                                        'x': x,
                                        'y': y,
                                        'object_id': obj_id,
                                        'color': profile.color,
                                        'behavior_type': 'trigger'
                                    }
                                )

            # Third: Predict behavior for objects in current frame
            if frame is not None and hasattr(cb, 'predict_click_behavior'):
                # Try center region first as likely interactive area
                prediction = cb.predict_click_behavior(
                    object_id='center_region',
                    color=-1,  # Unknown
                    game_type=game_type
                )
                if prediction and prediction.behavior.value not in ('unknown', 'no_effect', 'destroy'):
                    return RungResult(
                        action='ACTION6',
                        confidence=prediction.confidence * 0.8,
                        reason=f"Predicted behavior: {prediction.behavior.value} at center",
                        metadata={
                            'x': 32,
                            'y': 32,
                            'prediction': prediction.behavior.value,
                            'behavior_type': 'predicted'
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Click behavior learning failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1

    @staticmethod
    def _find_color_in_frame(
        frame: List[List[int]], target_color: int
    ) -> Optional[Tuple[int, int]]:
        """Find centroid of a specific color in the frame."""
        try:
            pixels: List[Tuple[int, int]] = []
            for y, row in enumerate(frame):
                for x, pixel in enumerate(row):
                    val = int(pixel) if hasattr(pixel, '__int__') else pixel
                    if val == target_color:
                        pixels.append((x, y))
            if not pixels:
                return None
            avg_x = sum(p[0] for p in pixels) // len(pixels)
            avg_y = sum(p[1] for p in pixels) // len(pixels)
            return (avg_x, avg_y)
        except Exception:
            return None


# =============================================================================
# NEW RUNGS: Long-term solutions for LS20, FT09, VC33 (Feb 2026)


class WallAwareNavigationRung(DecisionRung):
    """Track which directional actions produce movement vs hit walls - EXPLOITATION

    For directional-action games (like LS20 with [1,2,3,4]):
    1. Track which actions produced frame changes (= valid movement)
    2. Track which actions produced no frame change (= wall/obstacle)
    3. Build a movement feasibility map per-frame
    4. Bias toward actions that historically produce movement
    5. Avoid actions that repeatedly hit walls

    ROOT CAUSE ADDRESSED: LS20 agents spend 96% of actions hitting walls
    because they don't learn which directions are blocked.
    """
    name = "wall_aware_navigation"
    category = "exploitation"
    default_priority = 35
    confidence_threshold = 0.4

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Track per-game-type, per-level: action -> (successes, failures)
        self._movement_history: Dict[str, Dict[str, List[int]]] = {}
        # Track recent action->outcome pairs for current game session
        self._recent_outcomes: List[Tuple[str, bool]] = []  # (action, frame_changed)
        self._consecutive_wall_hits: Dict[str, int] = {}  # action -> consecutive wall count

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])

        # Only for directional games (actions 1-4 present, not click-only)
        has_directional = any(
            (a in [1, 2, 3, 4] if isinstance(a, int) else a in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
            for a in available
        )
        if not has_directional:
            return RungResult()

        # H5: Detect stuck state from game_player's stuck_count
        stuck_count = context.get('recent_stuck_count', 0)
        is_stuck = stuck_count >= 3

        # Build weights: penalize actions that consistently hit walls
        weights: Dict[str, float] = {}
        directional_actions = []
        for a in available:
            action_name = f'ACTION{a}' if isinstance(a, int) else a
            if not action_name.startswith('ACTION'):
                continue
            directional_actions.append(action_name)
            consecutive_walls = self._consecutive_wall_hits.get(action_name, 0)
            if consecutive_walls >= 3:
                weights[action_name] = max(0.05, 1.0 / (1 + consecutive_walls))
            elif consecutive_walls >= 1:
                weights[action_name] = 0.5
            else:
                weights[action_name] = 1.0

        # H5: When stuck, systematically rotate through UNTRIED directions
        if is_stuck and directional_actions:
            # Sort by wall hits ascending; among equal, rotate via stuck_count
            sorted_actions = sorted(
                directional_actions,
                key=lambda a: self._consecutive_wall_hits.get(a, 0)
            )
            # Use stuck_count to rotate through actions with equal wall counts
            choice_idx = stuck_count % len(sorted_actions)
            chosen = sorted_actions[choice_idx]
            if is_action_available(chosen, context):
                return RungResult(
                    action=chosen,
                    confidence=0.75,
                    reason=f"STUCK-ESCAPE: {chosen} (rotation idx={choice_idx}, stuck={stuck_count})",
                    weights=weights,
                    metadata={'stuck_recovery': True, 'stuck_count': stuck_count}
                )

        # Even when not stuck, suggest direction away from known walls
        if directional_actions and self._consecutive_wall_hits:
            # Pick the direction with fewest consecutive wall hits
            best_dir = min(
                directional_actions,
                key=lambda a: self._consecutive_wall_hits.get(a, 0)
            )
            best_walls = self._consecutive_wall_hits.get(best_dir, 0)
            worst_walls = max(self._consecutive_wall_hits.values())
            if worst_walls >= 2 and best_walls == 0 and is_action_available(best_dir, context):
                return RungResult(
                    action=best_dir,
                    confidence=0.45,
                    reason=f"Wall-aware: {best_dir} untried (others hit {worst_walls}x walls)",
                    weights=weights,
                    metadata={'wall_avoidance': True}
                )

        if self._recent_outcomes:
            # Count recent successes per action
            action_success: Dict[str, int] = {}
            action_total: Dict[str, int] = {}
            for action, changed in self._recent_outcomes[-20:]:
                action_total[action] = action_total.get(action, 0) + 1
                if changed:
                    action_success[action] = action_success.get(action, 0) + 1

            # Find action with best recent success rate
            best_action = None
            best_rate = 0.0
            for action, total in action_total.items():
                if total >= 2:
                    rate = action_success.get(action, 0) / total
                    if rate > best_rate and is_action_available(action, context):
                        best_rate = rate
                        best_action = action

            if best_action and best_rate > 0.5:
                # Suggest the most productive direction
                return RungResult(
                    action=best_action,
                    confidence=0.35 + best_rate * 0.25,
                    reason=f"Wall-aware: {best_action} has {best_rate:.0%} movement rate (last 20 actions)",
                    weights=weights,
                    metadata={'movement_rate': best_rate}
                )

        # Return as weights filter only
        if any(w < 0.5 for w in weights.values()):
            return RungResult(
                weights=weights,
                reason=f"Wall penalties: {[(k, f'{v:.2f}') for k, v in weights.items() if v < 1.0]}"
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Track whether directional actions produce movement or hit walls."""
        if not action or not action.startswith('ACTION'):
            return

        # Only track directional actions
        try:
            action_num = int(action.replace('ACTION', ''))
            if action_num not in [1, 2, 3, 4]:
                return
        except ValueError:
            return

        # H6: Use meaningful_frame_changed from context (pixel-count threshold)
        # instead of raw frame comparison. Raw comparison is fooled by game
        # animations/timers that change pixels every frame even on wall-hits.
        frame_changed = context.get('meaningful_frame_changed', False)
        if frame_changed is None:
            frame_changed = False

        # Update consecutive wall hit tracking
        if frame_changed:
            self._consecutive_wall_hits[action] = 0
        else:
            self._consecutive_wall_hits[action] = self._consecutive_wall_hits.get(action, 0) + 1

        # Track recent outcomes (ring buffer)
        self._recent_outcomes.append((action, frame_changed))
        if len(self._recent_outcomes) > 100:
            self._recent_outcomes = self._recent_outcomes[-100:]


class ObjectColorTargetingRung(DecisionRung):
    """Detect and systematically target distinct colored objects - EXPLOITATION

    For click-based games (FT09, VC33):
    1. Scan frame for distinct non-background colored object clusters
    2. Find centroid of each cluster
    3. Cycle through untried colors/positions systematically
    4. Use click_behavior profiles to prefer productive colors

    ROOT CAUSE ADDRESSED: FT09 clicks only at center (36,36), VC33 scatters
    randomly. Neither systematically targets actual game objects.
    """
    name = "object_color_targeting"
    category = "exploitation"
    default_priority = 34
    confidence_threshold = 0.35

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._tried_positions: Dict[str, Set[Tuple[int, int]]] = {}  # game_key -> set of (x,y)
        self._color_centroids_cache: Dict[str, List[Dict[str, Any]]] = {}
        self._last_frame_hash: str = ""
        self._consecutive_no_change = 0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        # Get frame
        frame = _get_frame(game_state)
        if frame is None:
            return RungResult()
        # Convert numpy to list for safe truthiness and iteration
        if hasattr(frame, 'tolist'):
            frame = frame.tolist()
        if not isinstance(frame, list) or len(frame) == 0:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Find colored object centroids in frame
        centroids = self._find_colored_object_centroids(frame)
        if not centroids:
            return RungResult()

        # Check click_behavior profiles to prefer productive colors
        cb = self.engines.click_behavior if self.engines else None
        productive_colors: Set[int] = set()
        dangerous_colors: Set[int] = set()
        if cb and hasattr(cb, 'get_all_profiles'):
            try:
                profiles = cb.get_all_profiles(game_type, min_clicks=2)
                for p in profiles:
                    if p.avg_score_impact > 0 or p.dominant_behavior.value in ('toggle', 'trigger', 'activate', 'move'):
                        productive_colors.add(p.color)
                    elif p.dominant_behavior.value in ('destroy', 'no_effect'):
                        dangerous_colors.add(p.color)
            except Exception:
                pass

        # Get tried positions for this game session
        tried = self._tried_positions.get(game_key, set())

        # ----- AFFORDANCE INTEGRATION -----
        # AffordanceDetectionRung injects classified object IDs into context.
        # interactive_objects = objects that responded to clicks (is_interactive)
        # obstacle_objects   = objects that block movement (is_obstacle)
        # reference_objects   = template/legend objects (is_reference)
        # If affordance data exists, use it to dramatically narrow the search
        # space instead of targeting every colored cluster.
        interactive_ids: List[str] = context.get('interactive_objects', [])
        obstacle_ids: List[str] = context.get('obstacle_objects', [])
        reference_ids: List[str] = context.get('reference_objects', [])
        has_affordance_data = bool(interactive_ids or obstacle_ids or reference_ids)

        # Score centroids by: interactive > productive > untried > tried, avoid obstacles
        best_target = None
        best_score = -999.0
        for c in centroids:
            cx, cy, color, size = c['x'], c['y'], c['color'], c['size']
            pos_key = (cx // 4, cy // 4)  # Quantize to 4px grid to avoid exact-match issues
            obj_id = f'color_{color}'

            score = 0.0

            # Affordance-based scoring (highest priority when available)
            if has_affordance_data:
                if obj_id in interactive_ids:
                    score += 5.0  # STRONG boost: proven interactive
                elif obj_id in reference_ids:
                    score += 1.0  # Reference objects are informative but not click targets
                elif obj_id in obstacle_ids:
                    score -= 4.0  # Obstacles rarely respond to clicks
                elif interactive_ids:
                    # We know WHICH objects are interactive; penalize unknowns
                    score -= 1.0

            # Click behavior profile scoring
            if color in productive_colors:
                score += 3.0
            if color in dangerous_colors:
                score -= 5.0
            if pos_key not in tried:
                score += 2.0
            # Prefer larger objects (more likely interactive)
            score += min(1.0, size / 50.0)
            # Small random tiebreak
            score += random.random() * 0.1

            if score > best_score:
                best_score = score
                best_target = c

        if best_target is None:
            return RungResult()

        # Mark as tried
        tx, ty = best_target['x'], best_target['y']
        pos_key = (tx // 4, ty // 4)
        if game_key not in self._tried_positions:
            self._tried_positions[game_key] = set()
        self._tried_positions[game_key].add(pos_key)

        # Clean up tried positions (reset after full cycle)
        if len(self._tried_positions[game_key]) > len(centroids) * 2:
            self._tried_positions[game_key] = set()

        confidence = 0.40
        if best_target['color'] in productive_colors:
            confidence = 0.60
        # Anti-monopoly: decay + cap
        decay = self._consecutive_no_change * 0.08
        confidence = max(0.10, min(0.55, confidence - decay))

        return RungResult(
            action='ACTION6',
            confidence=confidence,
            reason=f"Object targeting: color={best_target['color']} at ({tx},{ty}) size={best_target['size']} score={best_score:.1f}",
            metadata={
                'x': tx,
                'y': ty,
                'target_color': best_target['color'],
                'target_size': best_target['size'],
                'source': 'object_color_targeting',
            }
        )

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1

    def _find_colored_object_centroids(
        self, frame: List[List[int]]
    ) -> List[Dict[str, Any]]:
        """Find centroids of colored object clusters in frame."""
        if frame is None or (isinstance(frame, list) and len(frame) == 0):
            return []

        # Collect all non-background pixels by color
        color_pixels: Dict[int, List[Tuple[int, int]]] = {}
        try:
            for y, row in enumerate(frame):
                for x, pixel in enumerate(row):
                    val = int(pixel) if hasattr(pixel, '__int__') else pixel
                    if val != 0:  # Non-background
                        if val not in color_pixels:
                            color_pixels[val] = []
                        color_pixels[val].append((x, y))
        except Exception:
            return []

        # Filter out very large clusters (likely background/borders) and very small (noise)
        centroids = []
        frame_area = len(frame) * (len(frame[0]) if len(frame) > 0 else 64)
        for color, pixels in color_pixels.items():
            size = len(pixels)
            if size < 4 or size > frame_area * 0.4:
                continue  # Skip noise and background-like colors
            avg_x = sum(p[0] for p in pixels) // size
            avg_y = sum(p[1] for p in pixels) // size
            centroids.append({
                'x': avg_x,
                'y': avg_y,
                'color': color,
                'size': size,
            })

        # Sort by size descending (larger objects first)
        centroids.sort(key=lambda c: c['size'], reverse=True)
        return centroids


class CausalClickMappingRung(DecisionRung):
    """Build and use click->effect causal maps from frame diffs - EXPLOITATION

    For click-based puzzle games (FT09, VC33):
    1. After each click, compare frame_before and frame_after
    2. Record "click at (x,y) caused changes at [(x1,y1), (x2,y2), ...]"
    3. Build a per-game causal map: position -> effect pattern
    4. Use the causal map to plan clicks that move toward goals
    5. Detect toggle patterns (click X cycles through color palette)

    ROOT CAUSE ADDRESSED: System has 8-12% frame change rates but never
    builds causal models of what changes cause what effects. The data
    exists in frame_before/frame_after but is never analyzed for causality.

    Integrates with SpatialEffectLearner (already wired via on_action_complete)
    but adds an active planning layer on top.
    """
    name = "causal_click_mapping"
    category = "exploitation"
    default_priority = 33
    confidence_threshold = 0.45

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Per-game causal maps: game_key -> {(click_x, click_y) -> [(change_x, change_y, old_color, new_color)]}
        self._causal_map: Dict[str, Dict[Tuple[int, int], List[Tuple[int, int, int, int]]]] = {}
        # Track last click position for correlating with frame changes
        self._last_click_pos: Optional[Tuple[int, int]] = None
        # Toggle cycle detection: game_key -> {position -> [color_sequence]}
        self._color_cycles: Dict[str, Dict[Tuple[int, int], List[int]]] = {}
        # Repetition tracking: all positions clicked (including no-effect)
        self._all_clicked: Dict[str, Dict[Tuple[int, int], int]] = {}  # game_key -> {pos -> click_count}
        # Recent position history for decay calculation
        self._position_history: List[Tuple[int, int]] = []
        self._HISTORY_WINDOW = 10

    def _get_repetition_decay(self, pos: Tuple[int, int], game_key: str, proximity: int = 4) -> float:
        """Confidence decay based on how many times we've clicked near this position."""
        clicked = self._all_clicked.get(game_key, {})
        click_count = clicked.get(pos, 0)
        # Also count nearby positions in recent history
        recent = self._position_history[-6:]
        near_count = sum(
            1 for px, py in recent
            if abs(px - pos[0]) <= proximity and abs(py - pos[1]) <= proximity
        )
        # Decay: 0.12 per repeat click + 0.08 per recent near-hit
        return min(0.55, click_count * 0.12 + near_count * 0.08)

    def _record_click(self, pos: Tuple[int, int], game_key: str) -> None:
        """Record that we clicked this position."""
        if game_key not in self._all_clicked:
            self._all_clicked[game_key] = {}
        self._all_clicked[game_key][pos] = self._all_clicked[game_key].get(pos, 0) + 1
        self._position_history.append(pos)
        if len(self._position_history) > self._HISTORY_WINDOW:
            self._position_history = self._position_history[-self._HISTORY_WINDOW:]

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Merge shared world model causal data (Part 7 unification)
        # Context builder stores causal_map with string keys like "30,32"
        # and changes as dicts {x, y, from_color, to_color}.
        # This rung uses tuple keys (30, 32) and changes as tuples (x, y, old, new).
        shared_wm = context.get('world_model')
        if shared_wm and isinstance(shared_wm, dict):
            shared_causal = shared_wm.get('causal_map', {})
            if shared_causal and game_key not in self._causal_map:
                self._causal_map[game_key] = {}
            for pos_key_str, entry in shared_causal.items():
                if game_key in self._causal_map:
                    # Convert string key "x,y" to tuple (x, y)
                    try:
                        parts = pos_key_str.split(',')
                        pos_tuple = (int(parts[0].strip()), int(parts[1].strip()))
                    except (ValueError, IndexError):
                        continue
                    if pos_tuple not in self._causal_map[game_key]:
                        # Convert shared format (list of dicts) to rung format (list of tuples)
                        obs_list = entry.get('observations', [])
                        effects: List[Tuple[int, int, int, int]] = []
                        for obs_item in obs_list:
                            for change in obs_item.get('changes', []):
                                if isinstance(change, dict):
                                    fc = change.get('from_color', 0)
                                    tc = change.get('to_color', 0)
                                    if isinstance(fc, list) or isinstance(tc, list):
                                        continue
                                    effects.append((
                                        int(change.get('x', 0)),
                                        int(change.get('y', 0)),
                                        int(fc),
                                        int(tc),
                                    ))
                                elif isinstance(change, (list, tuple)) and len(change) >= 4:
                                    effects.append((int(change[0]), int(change[1]), int(change[2]), int(change[3])))
                        if effects:
                            self._causal_map[game_key][pos_tuple] = effects

        # Do we have a causal map for this game?
        causal = self._causal_map.get(game_key, {})
        if not causal:
            # No causal data yet - defer to other rungs for exploration
            return RungResult()

        # Get current frame
        frame = _get_frame(game_state)

        if frame is None:
            return RungResult()

        # Strategy 1: Find clicks that produce the MOST changes
        # (useful for exploration phase - find the most "active" positions)
        # BUT: decay confidence when clicking the same position repeatedly
        if len(causal) >= 1:
            # Find position with most effect range
            best_pos = None
            best_effect_count = 0
            for pos, effects in causal.items():
                if len(effects) > best_effect_count:
                    best_effect_count = len(effects)
                    best_pos = pos

            if best_pos and best_effect_count >= 2:
                decay = self._get_repetition_decay(best_pos, game_key)
                base_conf = 0.45 + min(0.2, best_effect_count * 0.05)
                confidence = max(0.10, base_conf - decay)

                # If decayed below threshold, fall through to exploration
                if confidence > 0.25:
                    self._record_click(best_pos, game_key)
                    return RungResult(
                        action='ACTION6',
                        confidence=confidence,
                        reason=f"Causal map: clicking ({best_pos[0]},{best_pos[1]}) affects {best_effect_count} cells [decay={decay:.2f}]",
                        metadata={
                            'x': best_pos[0],
                            'y': best_pos[1],
                            'effect_count': best_effect_count,
                            'source': 'causal_click_mapping',
                            'repetition_decay': decay,
                        }
                    )

        # Strategy 2: Affordance-guided exploration.
        # AffordanceDetectionRung classifies objects as is_reference, is_interactive, etc.
        # For FT09-like puzzles: reference objects (bsT sprites) encode the rules.
        # Prioritize clicking interactive > reference > any non-bg pixel.
        interactive_ids: List[str] = context.get('interactive_objects', [])
        reference_ids: List[str] = context.get('reference_objects', [])
        detected_objects: List[Dict[str, Any]] = context.get('detected_objects', [])
        # Track ALL clicked positions (causal + no-effect) for proper exploration
        tried_positions = set(causal.keys()) | set(self._all_clicked.get(game_key, {}).keys())

        # If we have affordance-classified objects, target them first
        if detected_objects:
            # Priority 1: Interactive objects (proven to respond to clicks)
            for obj in detected_objects:
                obj_id = obj.get('object_id', '')
                cx, cy = obj.get('center_x', 0), obj.get('center_y', 0)
                if (cx, cy) in tried_positions:
                    continue
                if obj_id in interactive_ids:
                    self._record_click((cx, cy), game_key)
                    return RungResult(
                        action='ACTION6',
                        confidence=0.50,
                        reason=f"Causal exploration: interactive object '{obj_id}' at ({cx},{cy})",
                        metadata={
                            'x': cx, 'y': cy,
                            'source': 'causal_affordance_interactive',
                        }
                    )

            # Priority 2: Reference objects (templates/legends that encode rules)
            for obj in detected_objects:
                obj_id = obj.get('object_id', '')
                cx, cy = obj.get('center_x', 0), obj.get('center_y', 0)
                if (cx, cy) in tried_positions:
                    continue
                if obj_id in reference_ids:
                    self._record_click((cx, cy), game_key)
                    return RungResult(
                        action='ACTION6',
                        confidence=0.45,
                        reason=f"Causal exploration: reference object '{obj_id}' at ({cx},{cy})",
                        metadata={
                            'x': cx, 'y': cy,
                            'source': 'causal_affordance_reference',
                        }
                    )

        # Fallback: explore untried non-background positions systematically.
        # No gate on effect_pattern -- we always want to explore the grid.
        frame_h = len(frame)
        frame_w = len(frame[0]) if len(frame) > 0 else 64

        for y in range(0, frame_h, 4):  # Sample every 4 pixels
            for x in range(0, frame_w, 4):
                if (x, y) not in tried_positions:
                    try:
                        pixel = frame[y][x]
                        # Coerce to plain int (handles numpy scalars & 3D arrays)
                        if hasattr(pixel, '__len__'):
                            pixel_val = int(pixel[0])
                        else:
                            pixel_val = int(pixel)
                        if pixel_val != 0:  # Non-background
                            self._record_click((x, y), game_key)
                            return RungResult(
                                action='ACTION6',
                                confidence=0.40,
                                reason=f"Causal exploration: untried non-bg at ({x},{y})",
                                metadata={
                                    'x': x, 'y': y,
                                    'source': 'causal_exploration',
                                }
                            )
                    except (IndexError, TypeError, ValueError):
                        pass

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Record causal mapping from click position to frame changes."""
        if action != 'ACTION6':
            return

        click_x = action_data.get('x', 0)
        click_y = action_data.get('y', 0)

        if frame_before is None or frame_after is None:
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # H39c: Find changed pixels using numpy (was: nested Python loops)
        changes: List[Tuple[int, int, int, int]] = []
        try:
            arr_b = np.asarray(frame_before)
            arr_a = np.asarray(frame_after)
            # Squeeze 3D to 2D: (C,H,W) -> (H,W). Take first channel.
            if arr_b.ndim == 3:
                arr_b = arr_b[0]
            if arr_a.ndim == 3:
                arr_a = arr_a[0]
            if arr_b.shape != arr_a.shape:
                return
            diff_mask = arr_b != arr_a
            ys, xs = np.where(diff_mask)
            for y_idx, x_idx in zip(ys, xs):
                changes.append((
                    int(x_idx), int(y_idx),
                    int(arr_b[y_idx][x_idx]),
                    int(arr_a[y_idx][x_idx])
                ))
        except Exception:
            return

        if not changes:
            # H38: Record no-effect click in shared world_model
            shared_wm = context.get('world_model')
            if shared_wm and isinstance(shared_wm, dict):
                no_effect = shared_wm.setdefault('no_effect_positions', {})
                gk_no_effect = no_effect.setdefault(game_key, {})
                pos_str = f"{click_x},{click_y}"
                gk_no_effect[pos_str] = gk_no_effect.get(pos_str, 0) + 1
            return

        # Store in causal map
        if game_key not in self._causal_map:
            self._causal_map[game_key] = {}
        self._causal_map[game_key][(click_x, click_y)] = changes

        # Track color cycles at each changed position
        if game_key not in self._color_cycles:
            self._color_cycles[game_key] = {}
        for cx, cy, old_color, new_color in changes:
            pos = (cx, cy)
            if pos not in self._color_cycles[game_key]:
                self._color_cycles[game_key][pos] = [old_color]
            self._color_cycles[game_key][pos].append(new_color)

        # H38: Write-back confirmed observations to shared world_model
        # Shallow copy propagation: mutations here flow back through
        # context_builder -> result_recorder -> DB automatically
        shared_wm = context.get('world_model')
        if shared_wm and isinstance(shared_wm, dict):
            shared_causal = shared_wm.setdefault('causal_map', {})
            pos_str = f"{click_x},{click_y}"
            if pos_str not in shared_causal:
                shared_causal[pos_str] = {'observations': []}
            obs_list = shared_causal[pos_str]['observations']
            obs_list.append({
                'changes': [
                    {'x': cx, 'y': cy, 'from_color': oc, 'to_color': nc}
                    for cx, cy, oc, nc in changes
                ],
                'game_key': game_key,
            })
            # H39c: Cap observation growth to prevent unbounded DB bloat
            if len(obs_list) > 10:
                shared_causal[pos_str]['observations'] = obs_list[-10:]


class ControlledMovementPlanningRung(DecisionRung):
    """Plan movement by tracking action-to-direction correlations - EXPLOITATION

    For directional-action games (like LS20):
    1. Correlate ACTION1-4 with actual pixel displacement
    2. Build "ACTION1=up, ACTION2=down, ACTION3=left, ACTION4=right" mapping
    3. Detect "stuck" positions (all directions blocked = trapped)
    4. When stuck, suggest unexplored directions or backtracking

    ROOT CAUSE ADDRESSED: LS20 agents never learn that ACTION1=up because they
    don't correlate actions with frame changes. This rung builds the mapping.
    """
    name = "controlled_movement_planning"
    category = "exploitation"
    default_priority = 32
    confidence_threshold = 0.4

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # action -> (total_dx, total_dy, success_count)
        self._action_displacement: Dict[str, Tuple[float, float, int]] = {}
        # Track visited frame hashes to detect loops
        self._visited_frame_hashes: List[str] = []
        self._max_frame_history = 50

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])

        # Only for directional games
        has_directional = any(
            (a in [1, 2, 3, 4] if isinstance(a, int) else a in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
            for a in available
        )
        if not has_directional:
            return RungResult()

        # Need enough data to have learned direction mappings
        total_observations = sum(d[2] for d in self._action_displacement.values())
        if total_observations < 8:
            return RungResult()  # Not enough data yet

        # Build direction map: which action goes which way?
        direction_map: Dict[str, Tuple[float, float]] = {}
        for action, (total_dx, total_dy, count) in self._action_displacement.items():
            if count >= 2:
                avg_dx = total_dx / count
                avg_dy = total_dy / count
                direction_map[action] = (avg_dx, avg_dy)

        if not direction_map:
            return RungResult()

        # Check current frame hash for loop detection
        frame = _get_frame(game_state)
        if frame is not None:
            try:
                frame_hash = str(hash(str(frame)))[:16]
                if frame_hash in self._visited_frame_hashes[-20:]:
                    # We've been here recently - try a less-used direction
                    action_counts: Dict[str, int] = {}
                    for a, (_, _, c) in self._action_displacement.items():
                        action_counts[a] = c
                    if action_counts:
                        least_used = min(action_counts, key=action_counts.get)  # type: ignore[arg-type]
                        if is_action_available(least_used, context):
                            return RungResult(
                                action=least_used,
                                confidence=0.45,
                                reason=f"Loop detected - trying least-used direction: {least_used}",
                                metadata={'reason': 'loop_escape'}
                            )
                self._visited_frame_hashes.append(frame_hash)
                if len(self._visited_frame_hashes) > self._max_frame_history:
                    self._visited_frame_hashes = self._visited_frame_hashes[-self._max_frame_history:]
            except Exception:
                pass

        # Find actions with the strongest directional signal
        # Suggest the one that moves us in a direction we haven't explored much
        best_action = None
        best_magnitude = 0.0
        for action, (avg_dx, avg_dy) in direction_map.items():
            magnitude = (avg_dx ** 2 + avg_dy ** 2) ** 0.5
            if magnitude > best_magnitude and is_action_available(action, context):
                best_magnitude = magnitude
                best_action = action

        if best_action and best_magnitude > 0.5:
            dx, dy = direction_map[best_action]
            return RungResult(
                action=best_action,
                confidence=0.40,
                reason=f"Movement planning: {best_action} moves avg ({dx:.1f},{dy:.1f})",
                metadata={
                    'direction': direction_map,
                    'source': 'movement_planning',
                }
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Track movement displacement per action to learn direction mapping."""
        if not action or not action.startswith('ACTION'):
            return
        try:
            action_num = int(action.replace('ACTION', ''))
            if action_num not in [1, 2, 3, 4]:
                return
        except ValueError:
            return

        if frame_before is None or frame_after is None:
            return

        # Find displacement by comparing object positions
        try:
            displacement = self._compute_displacement(frame_before, frame_after)
            if displacement is not None:
                dx, dy = displacement
                if action not in self._action_displacement:
                    self._action_displacement[action] = (0.0, 0.0, 0)
                prev_dx, prev_dy, prev_count = self._action_displacement[action]
                self._action_displacement[action] = (prev_dx + dx, prev_dy + dy, prev_count + 1)
        except Exception:
            pass

    def _compute_displacement(
        self,
        frame_before: Any,
        frame_after: Any
    ) -> Optional[Tuple[float, float]]:
        """Compute the net displacement of moved objects between frames."""
        try:
            # Find pixels that disappeared and appeared
            disappeared: List[Tuple[int, int, int]] = []
            appeared: List[Tuple[int, int, int]] = []

            if isinstance(frame_before, list):
                height = min(len(frame_before), len(frame_after))
                for y in range(height):
                    width = min(len(frame_before[y]), len(frame_after[y]))
                    for x in range(width):
                        old = int(frame_before[y][x]) if hasattr(frame_before[y][x], '__int__') else frame_before[y][x]
                        new = int(frame_after[y][x]) if hasattr(frame_after[y][x], '__int__') else frame_after[y][x]
                        if old != new:
                            if old != 0 and new == 0:
                                disappeared.append((x, y, old))
                            elif old == 0 and new != 0:
                                appeared.append((x, y, new))
            else:
                import numpy as np
                fb = np.array(frame_before)
                fa = np.array(frame_after)
                diff_mask = fb != fa
                ys, xs = np.where(diff_mask)
                for yi, xi in zip(ys, xs):
                    old, new = int(fb[yi, xi]), int(fa[yi, xi])
                    if old != 0 and new == 0:
                        disappeared.append((int(xi), int(yi), old))
                    elif old == 0 and new != 0:
                        appeared.append((int(xi), int(yi), new))

            if not disappeared or not appeared:
                return None

            # Compute centroid displacement
            d_cx = sum(p[0] for p in disappeared) / len(disappeared)
            d_cy = sum(p[1] for p in disappeared) / len(disappeared)
            a_cx = sum(p[0] for p in appeared) / len(appeared)
            a_cy = sum(p[1] for p in appeared) / len(appeared)

            return (a_cx - d_cx, a_cy - d_cy)
        except Exception:
            return None


# =============================================================================
# DEEPER SOLUTION RUNGS: Algorithmic fixes for LS20, FT09, VC33 (Feb 2026)
# Addresses: maze solving, constraint satisfaction, multi-step planning,
#            affordance detection, budget awareness, irreversibility detection,
#            and win-condition modeling.


class SpatialMapRung(DecisionRung):
    """Build and use a spatial map for pathfinding - EXPLOITATION

    For maze/navigation games like LS20:
    1. Build a map of explored positions (open/wall/unknown)
    2. Track which positions have been visited
    3. Use BFS pathfinding to route toward unexplored regions or targets
    4. Avoid known walls and dead-ends

    ROOT CAUSE ADDRESSED: LS20 agents random-walk through mazes with 96%
    wall-hit rate. A spatial map enables efficient navigation.
    """
    name = "spatial_map"
    category = "exploitation"
    default_priority = 31
    confidence_threshold = 0.4

    # Action to direction mapping (standard ARC grid, step=1)
    ACTION_DELTAS = {
        'ACTION1': (0, -1),   # Up
        'ACTION2': (0, 1),    # Down
        'ACTION3': (-1, 0),   # Left
        'ACTION4': (1, 0),    # Right
    }

    @staticmethod
    def _make_deltas(step: int = 1) -> Dict[str, Tuple[int, int]]:
        """Build action->delta mapping for a given step size."""
        return {
            'ACTION1': (0, -step),
            'ACTION2': (0, step),
            'ACTION3': (-step, 0),
            'ACTION4': (step, 0),
        }

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # game_key -> {(x, y) -> 'open'|'wall'|'visited'|'modifier'}
        self._maps: Dict[str, Dict[Tuple[int, int], str]] = {}
        # Estimated agent position
        self._position: Dict[str, Tuple[int, int]] = {}  # game_key -> (x, y)
        # Path queue: game_key -> list of actions to follow
        self._planned_path: Dict[str, List[str]] = {}
        # Frontier: unexplored positions adjacent to visited positions
        self._frontier: Dict[str, Set[Tuple[int, int]]] = {}
        # Track consecutive no-change actions to detect walls
        self._last_action_moved: Dict[str, bool] = {}
        # H19: Track session_id to detect new games (for position reset)
        self._last_session: Dict[str, str] = {}  # game_key -> session_id
        # H40: Solver-seeded wall/target data for LS20
        self._solver_seeded: Dict[str, bool] = {}  # game_key -> seeded flag
        self._unvisited_targets: Dict[str, Set[Tuple[int, int]]] = {}
        # H44: Fuel-aware BFS -- item positions and fuel budget
        self._item_positions: Dict[str, Set[Tuple[int, int]]] = {}
        self._max_fuel: Dict[str, int] = {}
        self._last_nav_target: Dict[str, Optional[Tuple[int, int]]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        has_directional = any(
            (a in [1, 2, 3, 4] if isinstance(a, int) else a in ['ACTION1', 'ACTION2', 'ACTION3', 'ACTION4'])
            for a in available
        )
        if not has_directional:
            return RungResult()

        # H51b: Use full game_id (includes variant hash) to prevent
        # cross-variant map poisoning. Two LS20 variants have different
        # maze layouts but shared game_type='ls20'.
        game_id = context.get('game_id', context.get('game_type', ''))
        level = context.get('level', 1)
        game_key = f"{game_id}_L{level}"

        # H5: Detect stuck state for confidence escalation
        stuck_count = context.get('recent_stuck_count', 0)
        is_stuck = stuck_count >= 3
        # H18: Boost base confidence so spatial_map wins more decisions.
        path_conf = 0.75 if is_stuck else 0.65
        explore_conf = 0.70 if is_stuck else 0.50

        # Initialize map if needed
        if game_key not in self._maps:
            self._maps[game_key] = {}
            self._frontier[game_key] = set()

        # === H40: Solver-aware spatial navigation ===
        # When solver provides walls/targets + agent position, switch to
        # precise navigation using solver data. Game-agnostic: step size
        # comes from context['spatial_step_size'] (default 1).
        solver_walls = context.get('solver_walls', [])
        solver_targets = context.get('solver_targets', [])
        agent_pos = context.get('agent_position')

        if solver_walls and agent_pos:
            return self._evaluate_solver_spatial(
                game_key, agent_pos, solver_walls, solver_targets,
                context, path_conf, explore_conf, is_stuck,
            )

        # === Default: autonomous navigation (no solver data) ===
        # Uses frame-detected agent_position and step_size when available;
        # falls back to virtual-grid tracking from (32,32) with step=1.
        step_size = context.get('spatial_step_size', 1)
        deltas = self._make_deltas(step_size)

        # H19: Detect new game -- reset position but KEEP wall map.
        session_id = context.get('session_id', '')
        if session_id and session_id != self._last_session.get(game_key):
            self._last_session[game_key] = session_id
            self._planned_path[game_key] = []
            spatial_map = self._maps[game_key]
            # Use detected position or default center
            init_pos = agent_pos or (32, 32)
            self._position[game_key] = init_pos
            spatial_map[init_pos] = 'visited'
            self._frontier[game_key] = set()
            for adx, ady in [(0, -step_size), (0, step_size),
                             (-step_size, 0), (step_size, 0)]:
                neighbor = (init_pos[0] + adx, init_pos[1] + ady)
                if neighbor not in spatial_map:
                    self._frontier[game_key].add(neighbor)

        # H51e: Don't override dead-reckoned position from evaluate().
        # on_action_complete() handles position syncing with proper
        # dead-reckoning fallback. Syncing here would overwrite accurate
        # dead-reckoned positions with a stuck frame-detection default.

        # If we have a planned path, follow it
        planned = self._planned_path.get(game_key, [])
        if planned:
            next_action = planned[0]
            if is_action_available(next_action, context):
                self._planned_path[game_key] = planned[1:]
                return RungResult(
                    action=next_action,
                    confidence=path_conf,
                    reason=f"Following planned path ({len(planned)} steps remaining)",
                    metadata={'source': 'spatial_map_path', 'stuck_boost': is_stuck}
                )

        pos = self._position.get(game_key)
        if pos is None:
            pos = agent_pos or (32, 32)
            self._position[game_key] = pos
            self._maps[game_key][pos] = 'visited'
            for adx, ady in [(0, -step_size), (0, step_size),
                             (-step_size, 0), (step_size, 0)]:
                self._frontier[game_key].add(
                    (pos[0] + adx, pos[1] + ady)
                )

        spatial_map = self._maps[game_key]

        # === H50: Prioritize discovered scoring positions ===
        # If we know positions where scoring happened before,
        # route to those before generic frontier exploration.
        scoring_positions = context.get('discovered_scoring_positions', [])
        if scoring_positions:
            scoring_set = {
                p for p in scoring_positions
                if p not in spatial_map or spatial_map[p] != 'wall'
            }
            if scoring_set:
                path = self._bfs_to_nearest(
                    pos, scoring_set, spatial_map, deltas,
                )
                if path and 0 < len(path) <= 30:
                    self._planned_path[game_key] = path[1:]
                    first_action = path[0]
                    if is_action_available(first_action, context):
                        return RungResult(
                            action=first_action,
                            confidence=path_conf,
                            reason=f"H50: Routing to scoring position ({len(path)} steps)",
                            metadata={'source': 'spatial_map_h50'}
                        )

        # Strategy: BFS to nearest frontier (unexplored) position
        frontier = self._frontier.get(game_key, set())
        if frontier:
            path = self._bfs_to_nearest(
                pos, frontier, spatial_map, deltas,
            )
            if path and len(path) > 0:
                self._planned_path[game_key] = path[1:]
                first_action = path[0]
                if is_action_available(first_action, context):
                    return RungResult(
                        action=first_action,
                        confidence=path_conf,
                        reason=f"Pathfinding to frontier (path length {len(path)})",
                        metadata={
                            'source': 'spatial_map_bfs',
                            'path_length': len(path),
                            'map_size': len(spatial_map),
                            'frontier_size': len(frontier),
                            'stuck_boost': is_stuck,
                        }
                    )

        # No frontier or path found -- explore randomly among non-wall directions
        safe_actions = []
        for action_str, (dx, dy) in deltas.items():
            if not is_action_available(action_str, context):
                continue
            neighbor = (pos[0] + dx, pos[1] + dy)
            if spatial_map.get(neighbor) != 'wall':
                safe_actions.append(action_str)

        if safe_actions:
            chosen = random.choice(safe_actions)
            return RungResult(
                action=chosen,
                confidence=explore_conf,
                reason=f"Exploring non-wall direction (map has {len(spatial_map)} cells)",
                metadata={'source': 'spatial_map_explore', 'stuck_boost': is_stuck}
            )

        return RungResult()

    def _evaluate_solver_spatial(
        self,
        game_key: str,
        agent_pos: Tuple[int, int],
        solver_walls: List,
        solver_targets: List,
        context: Dict[str, Any],
        path_conf: float,
        explore_conf: float,
        is_stuck: bool,
    ) -> RungResult:
        """H40: Solver-aware spatial navigation with fuel tracking.

        Uses solver-provided walls and targets for precise navigation.
        Step size comes from context['spatial_step_size'] (game-agnostic).
        """
        spatial_map = self._maps[game_key]
        remaining = context.get('remaining_actions', 999)
        step_size = context.get('spatial_step_size', 1)
        deltas = self._make_deltas(step_size)

        # H44: Reset solver state on new session (targets/items/planned path)
        session_id = context.get('session_id', '')
        if session_id and session_id != self._last_session.get(game_key):
            self._last_session[game_key] = session_id
            self._solver_seeded[game_key] = False
            self._planned_path[game_key] = []
            logger.info(f"[SPATIAL-NAV] Session reset for {game_key}, pos={agent_pos}")

        # Seed solver walls on first encounter for this level
        if not self._solver_seeded.get(game_key):
            for wall in solver_walls:
                wx, wy = int(wall[0]), int(wall[1])
                spatial_map[(wx, wy)] = 'wall'
            # Mark target positions as navigation goals
            target_positions = set()
            for t in solver_targets:
                if len(t) >= 2:
                    target_positions.add((int(t[0]), int(t[1])))
            self._unvisited_targets[game_key] = target_positions
            # H44: Seed item positions (fuel pickups) and max fuel.
            # Items use 5x5 region check: agent collects an item when
            # on any grid pos (gx, gy) where gx<=ix<gx+5 and gy<=iy<gy+5.
            # Map grid positions to item indices, matching solver logic.
            solver_items = context.get('solver_items', [])
            item_positions = set()
            if solver_items and agent_pos:
                x_off = agent_pos[0] % step_size
                y_off = agent_pos[1] % step_size
                for gx in range(x_off, 64, step_size):
                    for gy in range(y_off, 64, step_size):
                        for item in solver_items:
                            if len(item) >= 2:
                                ix, iy = int(item[0]), int(item[1])
                                if gx <= ix < gx + step_size and gy <= iy < gy + step_size:
                                    item_positions.add((gx, gy))
            self._item_positions[game_key] = item_positions
            self._max_fuel[game_key] = context.get('solver_max_fuel', 999)
            self._solver_seeded[game_key] = True
            logger.info(
                f"[SPATIAL-NAV] Seeded {game_key}: {len(solver_walls)} walls, "
                f"{len(target_positions)} targets, {len(item_positions)} items, "
                f"fuel={self._max_fuel[game_key]}, step={step_size}"
            )

        # Use actual agent position from cognitive_loop
        pos = agent_pos
        self._position[game_key] = pos
        spatial_map[pos] = 'visited'

        # Update unvisited targets: remove if agent is on/near a target
        targets_left = self._unvisited_targets.get(game_key, set())
        # Targets match at exact position
        targets_left.discard(pos)

        # H44: Invalidate planned path if nav target changed (changer visited)
        nav_override = context.get('solver_nav_target')
        cur_nav = tuple(nav_override) if nav_override else None
        prev_nav = self._last_nav_target.get(game_key)
        if cur_nav != prev_nav:
            self._last_nav_target[game_key] = cur_nav
            self._planned_path[game_key] = []

        # Follow existing planned path if valid
        planned = self._planned_path.get(game_key, [])
        if planned:
            next_action = planned[0]
            if is_action_available(next_action, context):
                self._planned_path[game_key] = planned[1:]
                # H44: Keep high confidence for fuel-aware paths
                follow_conf = 0.82 if self._max_fuel.get(game_key, 999) < 999 else path_conf
                return RungResult(
                    action=next_action,
                    confidence=follow_conf,
                    reason=f"H40: Following path to target ({len(planned)} steps left)",
                    metadata={'source': 'h40_path', 'remaining_fuel': remaining}
                )
            # Planned path action unavailable -- replan
            self._planned_path[game_key] = []

        # H44: Fuel-aware navigation with item collection
        max_fuel = self._max_fuel.get(game_key, 999)
        items_left = self._item_positions.get(game_key, set())
        # Fix: remaining_actions is game-wide budget; max_fuel is per-level.
        # Use actions_this_level to compute actual remaining fuel in this level.
        actions_this_level = context.get('actions_this_level', 0)
        effective_fuel = min(remaining, max(0, max_fuel - actions_this_level))

        # Priority 1: If fuel is critically low, route to nearest item
        if effective_fuel < max_fuel * 0.4 and items_left:
            item_path = self._bfs_to_nearest(pos, items_left, spatial_map, deltas)
            if item_path and len(item_path) < effective_fuel - 2:
                self._planned_path[game_key] = item_path[1:] if len(item_path) > 1 else []
                first_action = item_path[0]
                if is_action_available(first_action, context):
                    return RungResult(
                        action=first_action,
                        confidence=0.85,
                        reason=f"H44: Refueling ({effective_fuel} fuel, "
                               f"item {len(item_path)} steps away)",
                        metadata={
                            'source': 'h44_refuel',
                            'path_length': len(item_path),
                            'remaining_fuel': effective_fuel,
                        }
                    )

        # Priority 2: Fuel-aware BFS to config-aware nav target or nearest target
        # H44: SPEED 1d passes solver_nav_target (changer or target position)
        # based on config matching. Use it as BFS destination if present.
        nav_override = context.get('solver_nav_target')
        if nav_override:
            bfs_destinations = {tuple(nav_override)}
        else:
            bfs_destinations = targets_left
        if bfs_destinations:
            # Use fuel-aware BFS when fuel > 0, plain BFS when exhausted.
            # LS20 fuel exhaustion costs a life but doesn't end the level --
            # the agent must keep navigating even at fuel=0.
            if effective_fuel > 0:
                path = self._fuel_aware_bfs(
                    pos, bfs_destinations, items_left,
                    spatial_map, deltas, effective_fuel, max_fuel,
                )
            else:
                path = self._bfs_to_nearest(
                    pos, bfs_destinations, spatial_map, deltas,
                )
            dest_label = f"nav({nav_override})" if nav_override else f"{len(targets_left)} targets"
            logger.debug(
                f"[SPATIAL-NAV] H44 BFS {game_key}: pos={pos} fuel={effective_fuel} "
                f"items={len(items_left)} dest={dest_label} "
                f"path={'FOUND '+str(len(path)) if path else 'NONE'}"
            )
            if path:
                self._planned_path[game_key] = path[1:] if len(path) > 1 else []
                first_action = path[0]
                if is_action_available(first_action, context):
                    return RungResult(
                        action=first_action,
                        confidence=0.80,
                        reason=f"H44: Fuel-aware BFS to {dest_label} ({len(path)} steps, "
                               f"{effective_fuel} fuel)",
                        metadata={
                            'source': 'h44_fuel_bfs',
                            'path_length': len(path),
                            'targets_remaining': len(targets_left),
                            'remaining_fuel': effective_fuel,
                        }
                    )

        # Priority 3: Explore non-wall directions (all targets visited or
        # unreachable -- may need to find changers)
        safe_actions = []
        for action, (dx, dy) in deltas.items():
            if not is_action_available(action, context):
                continue
            neighbor = (pos[0] + dx, pos[1] + dy)
            if spatial_map.get(neighbor) != 'wall':
                # Prefer unvisited positions
                if spatial_map.get(neighbor) != 'visited':
                    safe_actions.insert(0, action)
                else:
                    safe_actions.append(action)

        if safe_actions:
            chosen = safe_actions[0] if safe_actions else random.choice(safe_actions)
            return RungResult(
                action=chosen,
                confidence=explore_conf,
                reason=f"H40: Exploring (all targets visited/unreachable, {remaining} fuel)",
                metadata={'source': 'h40_explore', 'remaining_fuel': remaining}
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Update spatial map based on movement results."""
        if action not in self.ACTION_DELTAS:
            return

        # H51b: Use full game_id to prevent cross-variant map poisoning
        game_id = context.get('game_id', context.get('game_type', ''))
        level = context.get('level', 1)
        game_key = f"{game_id}_L{level}"

        # H40: In solver-spatial mode, cognitive_loop tracks position
        # precisely. Just update visited positions from agent_position.
        agent_pos = context.get('agent_position')
        if agent_pos and self._solver_seeded.get(game_key):
            if game_key not in self._maps:
                self._maps[game_key] = {}
            self._maps[game_key][agent_pos] = 'visited'
            self._position[game_key] = agent_pos
            # Remove from targets if we're standing on one
            targets = self._unvisited_targets.get(game_key)
            if targets:
                targets.discard(agent_pos)
            # H44: Remove collected items
            items = self._item_positions.get(game_key)
            if items:
                items.discard(agent_pos)
            # Invalidate path if we deviated; recompute for fuel-limited levels
            planned = self._planned_path.get(game_key, [])
            if planned and planned[0] != action:
                self._planned_path[game_key] = []
                # H44: Proactively recompute path from new position when
                # fuel is limited, so path is ready when rung is next selected
                max_fuel = self._max_fuel.get(game_key, 999)
                if max_fuel < 999:
                    targets_left = self._unvisited_targets.get(game_key, set())
                    items_left = self._item_positions.get(game_key, set())
                    if targets_left:
                        step_size = context.get('spatial_step_size', 5)
                        deltas = self._make_deltas(step_size)
                        actions_this_level = context.get('actions_this_level', 0)
                        remaining = context.get('remaining_actions', 999)
                        eff_fuel = min(remaining, max(0, max_fuel - actions_this_level))
                        path = self._fuel_aware_bfs(
                            agent_pos, targets_left, items_left,
                            self._maps[game_key], deltas, eff_fuel, max_fuel,
                        )
                        if path:
                            self._planned_path[game_key] = path
            return

        # === Autonomous position tracking (no solver data) ===
        if game_key not in self._maps:
            self._maps[game_key] = {}
        if game_key not in self._frontier:
            self._frontier[game_key] = set()

        step_size = context.get('spatial_step_size', 1)
        deltas = self._make_deltas(step_size)

        # H19: Detect new game -- use detected position when available
        session_id = context.get('session_id', '')
        if session_id and session_id != self._last_session.get(game_key):
            self._last_session[game_key] = session_id
            det_pos = context.get('agent_position')
            init_pos = det_pos or (32, 32)
            self._position[game_key] = init_pos
            self._planned_path[game_key] = []
            self._maps[game_key][init_pos] = 'visited'
            self._frontier[game_key] = {
                (init_pos[0] + dx2, init_pos[1] + dy2)
                for dx2, dy2 in [(0, -step_size), (0, step_size),
                                 (-step_size, 0), (step_size, 0)]
                if (init_pos[0] + dx2, init_pos[1] + dy2) not in self._maps[game_key]
            }

        # === H51/H51e: Position tracking + wall detection ===
        # Two strategies for determining movement:
        #   1. Position detection: if det_pos != old_pos -> definitively moved
        #   2. Frame-changed fallback: when position detection is stuck (same
        #      position returned every time, e.g. default (32,32)), use the
        #      API's frame_changed signal which IS reliable for wall detection.
        # When movement is confirmed but position detection is stuck, use
        # dead reckoning (apply action delta to current position).
        old_pos = self._position.get(game_key)

        det_pos = context.get('agent_position')
        dx, dy = deltas.get(action, (0, 0))

        if old_pos is None:
            old_pos = det_pos or (32, 32)
            self._position[game_key] = old_pos
            self._maps[game_key][old_pos] = 'visited'

        # Determine if agent actually moved
        if det_pos and old_pos and det_pos != old_pos:
            # Position detection confirms movement -- most reliable
            actually_moved = True
        else:
            # Position detection unavailable or stuck at same coords.
            # Fall back to frame_changed signal from the game API.
            frame_changed = context.get('meaningful_frame_changed',
                                        context.get('frame_changed'))
            if frame_changed is None:
                if frame_before is None or frame_after is None:
                    return
                frame_changed = False
                try:
                    if isinstance(frame_before, list) and isinstance(frame_after, list):
                        frame_changed = frame_before != frame_after
                    else:
                        import numpy as np
                        frame_changed = not np.array_equal(frame_before, frame_after)
                except Exception:
                    pass
            actually_moved = frame_changed

        wall_pos = (old_pos[0] + dx, old_pos[1] + dy)

        if actually_moved:
            # Use position detection if it found a NEW position;
            # otherwise dead-reckon from action delta.
            if det_pos and det_pos != old_pos:
                actual_pos = det_pos
            else:
                actual_pos = wall_pos  # H51e: dead reckoning
            self._maps[game_key][actual_pos] = 'visited'
            self._position[game_key] = actual_pos
            self._last_action_moved[game_key] = True
            self._frontier[game_key].discard(actual_pos)
            for adx, ady in [(0, -step_size), (0, step_size),
                             (-step_size, 0), (step_size, 0)]:
                neighbor = (actual_pos[0] + adx, actual_pos[1] + ady)
                if neighbor not in self._maps[game_key]:
                    self._frontier[game_key].add(neighbor)
            planned = self._planned_path.get(game_key, [])
            if planned and planned[0] != action:
                self._planned_path[game_key] = []
        else:
            self._maps[game_key][wall_pos] = 'wall'
            self._last_action_moved[game_key] = False
            self._frontier[game_key].discard(wall_pos)

    def _bfs_to_nearest(
        self,
        start: Tuple[int, int],
        targets: Set[Tuple[int, int]],
        spatial_map: Dict[Tuple[int, int], str],
        deltas: Optional[Dict[str, Tuple[int, int]]] = None,
    ) -> List[str]:
        """BFS pathfinding from start to nearest target, avoiding walls.

        Args:
            deltas: Action-to-delta mapping. Defaults to ACTION_DELTAS (step=1).
                    Pass _make_deltas(step) for games with larger step sizes.
        """
        from collections import deque

        if deltas is None:
            deltas = self.ACTION_DELTAS

        queue: deque = deque()
        queue.append((start, []))
        visited: Set[Tuple[int, int]] = {start}

        # Limit search to prevent hanging
        max_iterations = 2000

        for _ in range(max_iterations):
            if not queue:
                break

            pos, path = queue.popleft()

            if pos in targets:
                return path

            for action, (dx, dy) in deltas.items():
                neighbor = (pos[0] + dx, pos[1] + dy)
                if neighbor in visited:
                    continue
                if spatial_map.get(neighbor) == 'wall':
                    continue
                # H40: Bounds check for pixel coordinate mode
                if neighbor[0] < 0 or neighbor[0] >= 64 or neighbor[1] < 0 or neighbor[1] >= 64:
                    continue

                visited.add(neighbor)
                queue.append((neighbor, path + [action]))

        return []  # No path found

    def _fuel_aware_bfs(
        self,
        start: Tuple[int, int],
        targets: Set[Tuple[int, int]],
        items: Set[Tuple[int, int]],
        spatial_map: Dict[Tuple[int, int], str],
        deltas: Dict[str, Tuple[int, int]],
        fuel: int,
        max_fuel: int,
    ) -> List[str]:
        """H44: BFS that tracks fuel and routes through items for refueling.

        State: (x, y, fuel, collected_bitmask)
        Walking onto an item refuels to max_fuel.
        States with fuel <= 0 are pruned (agent would die).

        Falls back to regular _bfs_to_nearest when no items exist.
        """
        from collections import deque as dq

        if not items:
            # No items -> regular BFS with fuel cap
            path = self._bfs_to_nearest(start, targets, spatial_map, deltas)
            if path and len(path) > fuel - 5:
                path = path[:max(1, fuel - 10)]
            return path

        item_list = sorted(items)
        item_idx = {pos: i for i, pos in enumerate(item_list)}

        queue = dq()
        queue.append((start, [], fuel, 0))  # pos, path, fuel, collected_mask
        # Prune: best_fuel[(x, y, collected)] = max fuel seen at this state
        best_fuel: Dict[Tuple[int, int, int], int] = {
            (start[0], start[1], 0): fuel
        }

        for _ in range(8000):
            if not queue:
                break

            pos, path, cur_fuel, collected = queue.popleft()

            if pos in targets:
                return path

            for action, (dx, dy) in deltas.items():
                nx, ny = pos[0] + dx, pos[1] + dy

                if spatial_map.get((nx, ny)) == 'wall':
                    continue
                if nx < 0 or nx >= 64 or ny < 0 or ny >= 64:
                    continue

                new_fuel = cur_fuel - 1
                new_collected = collected

                # Item collection -> refuel to max
                if (nx, ny) in item_idx:
                    idx = item_idx[(nx, ny)]
                    if not (collected & (1 << idx)):
                        new_collected |= (1 << idx)
                        new_fuel = max_fuel

                if new_fuel <= 0:
                    continue  # Dead -- prune

                core = (nx, ny, new_collected)
                if best_fuel.get(core, -1) >= new_fuel:
                    continue  # Seen with more fuel -- skip
                best_fuel[core] = new_fuel

                queue.append(((nx, ny), path + [action], new_fuel, new_collected))

        return []  # No fuel-feasible path found


class ConstraintSatisfactionRung(DecisionRung):
    """Solve constraint satisfaction puzzles using learned causal maps - EXPLOITATION

    For tile-cycling click games like FT09:
    1. Read the causal map built by CausalClickMappingRung
    2. Read the current grid state and target state
    3. Solve: find a set of clicks that transforms current -> target
    4. Use greedy algorithm: each click should reduce total error

    The key insight: each click cycles the clicked tile through its color
    palette (learned from CausalMap). The puzzle is solvable by computing
    how many clicks each tile needs to reach its target color.

    NOTE: Effect patterns come from CausalMap observations. The rung does
    NOT assume neighbor toggling -- it uses whatever effects were observed.
    Standard FT09 tiles affect ONLY the clicked tile.

    ROOT CAUSE ADDRESSED: FT09 requires simultaneous constraint satisfaction.
    CausalClickMappingRung learns individual click effects, but doesn't
    combine them into a solution plan. This rung does the planning.
    """
    name = "constraint_satisfaction"
    category = "exploitation"
    default_priority = 30
    confidence_threshold = 0.5

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # game_key -> planned click sequence
        self._planned_clicks: Dict[str, List[Tuple[int, int]]] = {}
        # game_key -> learned effect kernels {(click_x, click_y) -> [(cx, cy, old, new)]}
        self._effect_kernels: Dict[str, Dict[Tuple[int, int], List[Tuple[int, int, int, int]]]] = {}
        # Track: for this game, how many colors cycle? (e.g., [9, 8] = 2-cycle)
        self._color_cycles: Dict[str, Dict[int, List[int]]] = {}
        # Attempts at solving
        self._solve_attempts: Dict[str, int] = {}
        # H33: Positions that caused goal regression -- avoid re-clicking
        self._regression_positions: Dict[str, Set[Tuple[int, int]]] = {}
        # H33: Track last click + progress for regression detection
        self._last_click: Dict[str, Optional[Tuple[int, int]]] = {}
        self._last_goal_matches: Dict[str, int] = {}
        # H33: Board state hash history for cycle detection
        self._state_hashes: Dict[str, List[int]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Check if we have a planned click sequence -- high confidence to prevent
        # exploration interrupting a multi-click solution in progress
        planned = self._planned_clicks.get(game_key, [])
        if planned:
            # H33: Abort plan if goal progress is declining (oscillation signal)
            shared_wm = context.get('world_model')
            if shared_wm and isinstance(shared_wm, dict):
                gp = shared_wm.get('goal_progress', {})
                if gp.get('declining'):
                    self._planned_clicks[game_key] = []
                    planned = []
                    logger.debug(
                        "[CSAT-H33] Aborted plan for %s: goal progress declining",
                        game_key,
                    )
        if planned:
            next_click = planned[0]
            # H33: Skip positions that previously caused regression
            avoid = self._regression_positions.get(game_key, set())
            if next_click in avoid:
                self._planned_clicks[game_key] = planned[1:]
                # Try next in plan, or fall through to re-solve
                remaining = [p for p in planned[1:] if p not in avoid]
                if remaining:
                    next_click = remaining[0]
                    self._planned_clicks[game_key] = remaining[1:]
                else:
                    self._planned_clicks[game_key] = []
                    planned = []
            if planned:
                self._planned_clicks[game_key] = planned[1:]
                return RungResult(
                    action='ACTION6',
                    confidence=0.80,
                    reason=f"Constraint solver: planned click at {next_click} ({len(planned)-1} remaining)",
                    metadata={
                        'x': next_click[0],
                        'y': next_click[1],
                        'source': 'constraint_satisfaction',
                    }
                )

        # Merge shared world model causal data (Part 7 unification)
        # Context builder stores causal_map with string keys like "30,32"
        # and changes as dicts {x, y, from_color, to_color}.
        # This rung uses tuple keys (30, 32) and changes as tuples (x, y, old, new).
        shared_wm = context.get('world_model')
        if shared_wm and isinstance(shared_wm, dict):
            shared_causal = shared_wm.get('causal_map', {})
            if shared_causal and game_key not in self._effect_kernels:
                self._effect_kernels[game_key] = {}
            for pos_key_str, entry in shared_causal.items():
                if game_key in self._effect_kernels:
                    # Convert string key "x,y" to tuple (x, y)
                    try:
                        parts = pos_key_str.split(',')
                        pos_tuple = (int(parts[0].strip()), int(parts[1].strip()))
                    except (ValueError, IndexError):
                        continue
                    if pos_tuple not in self._effect_kernels[game_key]:
                        # Convert shared format (list of dicts) to rung format (list of tuples)
                        # H39a: Filter to only current level's observations
                        # Observations without game_key are solver-seeded
                        # (applicable to all levels)
                        all_obs = entry.get('observations', [])
                        obs_list = [
                            o for o in all_obs
                            if not o.get('game_key')
                            or o.get('game_key') == game_key
                        ]
                        if not obs_list:
                            continue
                        effects: List[Tuple[int, int, int, int]] = []
                        for obs_item in obs_list:
                            for change in obs_item.get('changes', []):
                                if isinstance(change, dict):
                                    fc = change.get('from_color', 0)
                                    tc = change.get('to_color', 0)
                                    if isinstance(fc, list) or isinstance(tc, list):
                                        continue
                                    effects.append((
                                        int(change.get('x', 0)),
                                        int(change.get('y', 0)),
                                        int(fc),
                                        int(tc),
                                    ))
                                elif isinstance(change, (list, tuple)) and len(change) >= 4:
                                    effects.append((int(change[0]), int(change[1]), int(change[2]), int(change[3])))
                        if effects:
                            self._effect_kernels[game_key][pos_tuple] = effects
                            # Bootstrap color cycles from shared data
                            if game_key not in self._color_cycles:
                                self._color_cycles[game_key] = {}
                            for _, _, old_c, new_c in effects:
                                if old_c not in self._color_cycles[game_key]:
                                    self._color_cycles[game_key][old_c] = []
                                cycle = self._color_cycles[game_key][old_c]
                                if new_c not in cycle:
                                    cycle.append(new_c)

        # Need causal map to plan -- even 1 observation enables greedy planning
        raw_kernels = self._effect_kernels.get(game_key, {})
        if len(raw_kernels) < 1:
            return RungResult()

        # Deduplicate pixel-level kernels to tile-level kernels.
        # Multiple exploratory clicks on the same game tile create separate
        # kernel entries (e.g. 94 raw entries for a 9-tile grid).
        # This clusters them so the solver sees one entry per actual tile.
        kernels = self._build_tile_kernels(raw_kernels)

        # Get current frame as 2D list-of-lists
        frame = _get_frame_2d(game_state)
        if frame is None:
            logger.debug(f"[CSAT] {game_key}: frame is None after _get_frame_2d")
            return RungResult()

        # Read goal state from context (Part 7.4 -- reference panel based)
        # Priority: constraint_decoder goal > context goal_state > heuristic
        goal_state = None
        shared_wm_for_goal = context.get('world_model')
        if isinstance(shared_wm_for_goal, dict):
            constraint_goal = shared_wm_for_goal.get('constraint_goal_state')
            if isinstance(constraint_goal, dict) and constraint_goal:
                goal_state = constraint_goal
        if goal_state is None:
            goal_state = context.get('goal_state')
            if not isinstance(goal_state, dict) or not goal_state:
                goal_state = None

        # Log solver inputs on first attempt per game_key
        if game_key not in self._solve_attempts:
            tile_positions = sorted(kernels.keys())
            logger.info(f"[CSAT] {game_key}: raw_kernels={len(raw_kernels)} "
                        f"tiles={len(kernels)} "
                        f"goal={'decoded' if goal_state else 'heuristic'} "
                        f"cycles={len(self._color_cycles.get(game_key, {}))}")
            logger.info(f"[CSAT] tile_positions={tile_positions}")

        # H45: Try analytical solver first (matches solver's exact approach),
        # then lookahead (forward simulation), then greedy (hill-climbing)
        solution = self._analytical_solve(frame, kernels, game_key, goal_state=goal_state)
        solver_name = 'analytical'
        if not solution:
            solution = self._lookahead_solve(
                frame, kernels, game_key, depth=5,
                goal_state=goal_state,
                causal_map=context.get('causal_map'),
            )
            solver_name = 'lookahead'
        if not solution:
            solution = self._greedy_solve(frame, kernels, game_key, goal_state=goal_state)
            solver_name = 'greedy'

        if solution:
            self._planned_clicks[game_key] = solution[1:]
            first = solution[0]
            self._solve_attempts[game_key] = self._solve_attempts.get(game_key, 0) + 1
            return RungResult(
                action='ACTION6',
                confidence=0.80,
                reason=f"Constraint solver ({solver_name}): {len(solution)}-click "
                       f"solution, starting at {first}",
                metadata={
                    'x': first[0],
                    'y': first[1],
                    'solution_length': len(solution),
                    'source': f'constraint_{solver_name}',
                }
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Learn effect kernels from click outcomes (shared with CausalClickMappingRung)."""
        if action != 'ACTION6':
            return
        if frame_before is None or frame_after is None:
            return

        click_x = action_data.get('x', 0)
        click_y = action_data.get('y', 0)

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        changes: List[Tuple[int, int, int, int]] = []
        try:
            if isinstance(frame_before, list) and isinstance(frame_after, list):
                for y in range(min(len(frame_before), len(frame_after))):
                    for x in range(min(len(frame_before[y]), len(frame_after[y]))):
                        old = int(frame_before[y][x]) if hasattr(frame_before[y][x], '__int__') else frame_before[y][x]
                        new = int(frame_after[y][x]) if hasattr(frame_after[y][x], '__int__') else frame_after[y][x]
                        if old != new:
                            changes.append((x, y, old, new))
            else:
                import numpy as np
                diff = np.array(frame_before) != np.array(frame_after)
                ys, xs = np.where(diff)
                for yi, xi in zip(ys, xs):
                    changes.append((
                        int(xi), int(yi),
                        int(frame_before[yi][xi]),
                        int(frame_after[yi][xi])
                    ))
        except Exception:
            return

        # H33: Track board state hash for cycle detection (runs even without changes)
        click_pos = (click_x, click_y)
        try:
            if hasattr(frame_after, 'tobytes'):
                h = hash(frame_after.tobytes())
            else:
                h = hash(str(frame_after))
            history = self._state_hashes.setdefault(game_key, [])
            if h in history[-10:]:
                # Board state revisited -- we're oscillating
                self._planned_clicks[game_key] = []
                logger.debug("[CSAT-H33] State revisited for %s, clearing plan", game_key)
            history.append(h)
            if len(history) > 20:
                self._state_hashes[game_key] = history[-20:]
        except Exception:
            pass

        # H33: Track goal progress regression per click position
        self._last_click[game_key] = click_pos
        world_model = context.get('world_model', {})
        if isinstance(world_model, dict):
            gp = world_model.get('goal_progress', {})
            current_matches = gp.get('matches', -1)
            prev_matches = self._last_goal_matches.get(game_key, -1)
            if prev_matches >= 0 and current_matches < prev_matches:
                # Progress declined after this click -- mark position as regressive
                avoid = self._regression_positions.setdefault(game_key, set())
                avoid.add(click_pos)
                self._planned_clicks[game_key] = []
                logger.debug(
                    "[CSAT-H33] Position %s regressed %s: %d->%d matches",
                    click_pos, game_key, prev_matches, current_matches,
                )
            elif current_matches > prev_matches and click_pos in self._regression_positions.get(game_key, set()):
                # Position improved progress -- rehabilitate it
                self._regression_positions.get(game_key, set()).discard(click_pos)
            if current_matches >= 0:
                self._last_goal_matches[game_key] = current_matches

        if not changes:
            return

        if game_key not in self._effect_kernels:
            self._effect_kernels[game_key] = {}
        self._effect_kernels[game_key][(click_x, click_y)] = changes

        # Learn color cycles (ensure int keys/values for hashability)
        if game_key not in self._color_cycles:
            self._color_cycles[game_key] = {}
        for _, _, old_color, new_color in changes:
            try:
                oc, nc = int(old_color), int(new_color)
            except (TypeError, ValueError):
                continue
            if oc not in self._color_cycles[game_key]:
                self._color_cycles[game_key][oc] = []
            cycle = self._color_cycles[game_key][oc]
            if nc not in cycle:
                cycle.append(nc)

    def _predict_next_color(self, current_color: int, game_key: str,
                             fallback_new: int) -> int:
        """Predict what color a cell will become after a click, using learned cycles.

        FT09 example: color cycle {9: [8], 8: [9]} means 9->8->9->8...
        If current=9, predict 8. If current=8, predict 9.
        Falls back to the recorded new color if no cycle data for current color.
        """
        cycle_map = self._color_cycles.get(game_key, {})
        next_colors = cycle_map.get(current_color, [])
        if next_colors:
            return next_colors[0]
        return fallback_new

    def _get_affected_positions(
        self, kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]],
        click_pos: Tuple[int, int]
    ) -> List[Tuple[int, int, int]]:
        """Get positions affected by a click with their fallback new-color.

        Returns list of (cx, cy, fallback_new) for each affected position.
        """
        effects = kernels.get(click_pos, [])
        seen: Dict[Tuple[int, int], int] = {}
        for cx, cy, _old, new in effects:
            seen[(cx, cy)] = new
        return [(cx, cy, n) for (cx, cy), n in seen.items()]

    def _analytical_solve(
        self,
        frame: List[List[int]],
        kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]],
        game_key: str,
        goal_state: Optional[Dict] = None,
    ) -> Optional[List[Tuple[int, int]]]:
        """H45: Solve tile puzzle analytically -- clicks = (target - current) mod cycle.

        For independent-tile puzzles: each tile has a color cycle
        (e.g. 9->8->9 is a 2-cycle). The number of clicks needed is:
            clicks = index_of(target, cycle) - index_of(current, cycle)  mod len(cycle)

        This matches the solver's analytical approach -- no search needed.
        Game-agnostic: works for any puzzle where each click position
        independently cycles through a known color sequence.
        """
        if not goal_state:
            return None

        cycles = self._color_cycles.get(game_key, {})
        if not cycles:
            return None

        solution: List[Tuple[int, int]] = []

        for tile_pos, effects in kernels.items():
            # Find the most representative effect pixel for this tile
            # (the one closest to the tile center)
            cx, cy = tile_pos
            if cy < 0 or cy >= len(frame) or cx < 0 or cx >= len(frame[0]):
                continue

            current = int(frame[cy][cx])

            # Goal state uses string keys "x,y"
            pos_key = f"{cx},{cy}"
            target = goal_state.get(pos_key)
            if target is None:
                # Try tuple key format
                target = goal_state.get((cx, cy))
            if target is None or int(target) == current:
                continue

            target = int(target)

            # Walk the color cycle from current to target
            cycle_chain = cycles.get(current)
            if not cycle_chain:
                continue

            # Build full cycle: follow transitions until we loop or find target
            c = current
            clicks_to_target = 0
            found = False
            for _ in range(10):  # max cycle length safety
                next_colors = cycles.get(c, [])
                if not next_colors:
                    break
                c = int(next_colors[0])  # ensure hashable int
                clicks_to_target += 1
                if c == target:
                    found = True
                    break
                if c == current:
                    break  # Full cycle without finding target

            if found and clicks_to_target > 0:
                for _ in range(clicks_to_target):
                    solution.append(tile_pos)

        if solution:
            logger.info(
                "[CSAT-H45] Analytical solution for %s: %d clicks "
                "across %d tiles",
                game_key, len(solution),
                len(set(solution)),
            )

        return solution if solution else None

    def _build_tile_kernels(
        self,
        kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]]
    ) -> Dict[Tuple[int, int], List[Tuple[int, int, int, int]]]:
        """Deduplicate pixel-level kernels into tile-level kernels.

        Raw causal observations record effects at individual click coordinates.
        Multiple click positions within the same game tile create separate
        kernel entries (e.g. 94 entries for a 9-tile grid). Each kernel also
        includes noise from timer/counter pixels that change every action.

        This method:
        1. Filters each kernel's effects to those NEAR the click position
           (removes timer/counter noise at distant frame locations)
        2. Computes the centroid of filtered effects (= tile center)
        3. Clusters kernels with centroids within 5 pixels (same tile)
        4. Keeps the kernel with the most local effects per tile cluster
        5. RE-KEYS the output to the rounded centroid (= tile center)
           so the solver clicks actual tile positions, not between-tile noise

        Result: ~9 tile-level entries keyed to actual tile centers.
        """
        if len(kernels) <= 12:
            return kernels  # Already small enough

        # Step 1: For each kernel, compute centroid of NEARBY effects only.
        # This filters out timer/counter noise at distant frame positions.
        # Radius 10: tiles are ~4-8px wide, so 10px captures tile + close
        # neighbors while excluding distant timer/counter pixels.
        NEARBY_RADIUS = 10
        centroids: Dict[Tuple[int, int], Tuple[float, float]] = {}
        nearby_counts: Dict[Tuple[int, int], int] = {}
        for click_pos, effects in kernels.items():
            if not effects:
                continue
            nearby = [(cx, cy) for cx, cy, _, _ in effects
                      if abs(cx - click_pos[0]) <= NEARBY_RADIUS
                      and abs(cy - click_pos[1]) <= NEARBY_RADIUS]
            if len(nearby) < 2:
                continue  # Too few local effects -- noise click between tiles
            cx_avg = sum(p[0] for p in nearby) / len(nearby)
            cy_avg = sum(p[1] for p in nearby) / len(nearby)
            centroids[click_pos] = (cx_avg, cy_avg)
            nearby_counts[click_pos] = len(nearby)

        if not centroids:
            return kernels  # Fallback: no valid centroids found

        # Step 2: Cluster by centroid proximity (L-inf <= 5 = same tile).
        # Tile centroids are ~8 pixels apart, so threshold 5 safely separates.
        # Process kernels with most LOCAL effects first (best representatives).
        sorted_items = sorted(
            centroids.items(),
            key=lambda kv: nearby_counts[kv[0]],
            reverse=True
        )

        tile_kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]] = {}
        used: Set[Tuple[int, int]] = set()

        for click_pos, centroid in sorted_items:
            if click_pos in used:
                continue
            used.add(click_pos)
            # Filter effects to tile-local only (remove timer/counter noise)
            filtered = [
                (cx, cy, old_c, new_c)
                for cx, cy, old_c, new_c in kernels[click_pos]
                if abs(cx - click_pos[0]) <= NEARBY_RADIUS
                and abs(cy - click_pos[1]) <= NEARBY_RADIUS
            ]
            if not filtered:
                continue
            # RE-KEY to rounded centroid so solver clicks tile center,
            # not the original exploration position (which may be off-tile).
            tile_center = (round(centroid[0]), round(centroid[1]))
            tile_kernels[tile_center] = filtered
            # Mark all other kernels with nearby centroids as duplicates
            for other_pos, other_centroid in centroids.items():
                if other_pos in used:
                    continue
                if (abs(centroid[0] - other_centroid[0]) <= 5.0
                        and abs(centroid[1] - other_centroid[1]) <= 5.0):
                    used.add(other_pos)

        return tile_kernels

    def _lookahead_solve(
        self,
        frame: List[List[int]],
        kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]],
        game_key: str,
        depth: int = 3,
        goal_state: Optional[Dict] = None,
        causal_map: Any = None,
    ) -> List[Tuple[int, int]]:
        """Lookahead solver: simulate N-step click sequences via DFS.

        Instead of greedily picking the single best next click, this explores
        all possible sequences up to `depth` clicks and returns the one that
        minimizes total error. This is the 'lookahead planning' architectural
        primitive - forward-simulating future states before committing.

        H49: When causal_map is provided, uses CausalMap.simulate_action()
        for game-agnostic forward simulation.  Falls back to kernel-based
        color cycling when causal_map has no knowledge.

        Complexity: O(K^depth) where K = number of click positions.
        Capped at depth=3 with max 15 click positions to stay under 3375 evals.
        """
        try:
            # Extract current state
            current: Dict[Tuple[int, int], int] = {}
            for y, row in enumerate(frame):
                for x, pixel in enumerate(row):
                    val = int(pixel) if hasattr(pixel, '__int__') else pixel
                    if val != 0:
                        current[(x, y)] = val

            if not current:
                return []

            # Determine target - prefer goal_state from reference panel (Part 7.4)
            target_map: Optional[Dict[Tuple[int, int], int]] = None
            target_color: Optional[int] = None
            if goal_state and isinstance(goal_state, dict) and len(goal_state) > 0:
                target_map = goal_state  # {(x,y): target_color}
            else:
                # Fallback: most common color among GRID CELLS ONLY
                # (positions affected by known click effects, not entire frame)
                grid_positions: Set[Tuple[int, int]] = set()
                for effects in kernels.values():
                    for cx, cy, _, _ in effects:
                        grid_positions.add((cx, cy))
                color_counts: Dict[int, int] = {}
                for pos in grid_positions:
                    if pos in current:
                        color_counts[current[pos]] = color_counts.get(current[pos], 0) + 1
                if not color_counts:
                    return []
                target_color = max(color_counts, key=color_counts.get)  # type: ignore[arg-type]

            # Restrict error computation to grid cells (known interactive positions)
            grid_cells_set: Set[Tuple[int, int]] = set()
            for effects in kernels.values():
                for cx, cy, _, _ in effects:
                    grid_cells_set.add((cx, cy))

            def _compute_error(state: Dict[Tuple[int, int], int]) -> int:
                if target_map is not None:
                    return sum(1 for pos, c in state.items()
                               if pos in target_map and c != target_map[pos])
                # Only count errors at grid cell positions
                return sum(1 for pos in grid_cells_set
                           if pos in state and state[pos] != target_color)

            initial_error = _compute_error(current)
            if initial_error == 0:
                return []

            # Limit click positions to most impactful (sort by effect count)
            sorted_clicks = sorted(
                kernels.items(), key=lambda kv: len(kv[1]), reverse=True
            )[:15]
            click_positions = [pos for pos, _ in sorted_clicks]

            # DFS with pruning
            best_sequence: List[Tuple[int, int]] = []
            best_remaining_error = initial_error

            def _simulate_click(
                state: Dict[Tuple[int, int], int],
                click_pos: Tuple[int, int]
            ) -> Dict[Tuple[int, int], int]:
                """Apply click effects -- H49: prefer CausalMap, fallback to kernels."""
                # H49: Try CausalMap simulation first (game-agnostic)
                if causal_map is not None:
                    try:
                        new_state, conf = causal_map.simulate_action(
                            state, click_pos,
                        )
                        if conf > 0.3:
                            return new_state
                    except Exception:
                        pass
                # Fallback: kernel-based color cycling
                new_state = dict(state)
                affected = self._get_affected_positions(kernels, click_pos)
                for cx, cy, fallback_new in affected:
                    if (cx, cy) in new_state:
                        current = new_state[(cx, cy)]
                        new_state[(cx, cy)] = self._predict_next_color(
                            current, game_key, fallback_new
                        )
                return new_state

            def _dfs(
                state: Dict[Tuple[int, int], int],
                sequence: List[Tuple[int, int]],
                current_depth: int
            ) -> None:
                nonlocal best_sequence, best_remaining_error

                error = _compute_error(state)

                if error < best_remaining_error:
                    best_remaining_error = error
                    best_sequence = list(sequence)

                if error == 0 or current_depth >= depth:
                    return

                last_click = sequence[-1] if sequence else None
                for click_pos in click_positions:
                    # Skip same position as last click -- in toggle games this is a no-op
                    if click_pos == last_click:
                        continue
                    new_state = _simulate_click(state, click_pos)
                    new_error = _compute_error(new_state)
                    # Skip clicks that produce no state change (zero-effect positions)
                    if new_state == state:
                        continue
                    # Require strict improvement to avoid wasting depth on lateral moves
                    if new_error < error:
                        _dfs(new_state, sequence + [click_pos], current_depth + 1)

            _dfs(current, [], 0)

            if best_sequence and best_remaining_error < initial_error:
                return best_sequence

        except Exception:
            pass

        return []

    def _greedy_solve(
        self,
        frame: List[List[int]],
        kernels: Dict[Tuple[int, int], List[Tuple[int, int, int, int]]],
        game_key: str,
        goal_state: Optional[Dict] = None
    ) -> List[Tuple[int, int]]:
        """Greedy solver: pick clicks that reduce error most.

        The goal is matching each tile to its target color (from constraint
        sprites or reference panel). We greedily pick the click that
        reduces the most differences from the target.
        """
        try:
            # Extract current state as flat dict: (x,y) -> color
            current: Dict[Tuple[int, int], int] = {}
            for y, row in enumerate(frame):
                for x, pixel in enumerate(row):
                    val = int(pixel) if hasattr(pixel, '__int__') else pixel
                    if val != 0:
                        current[(x, y)] = val

            if not current:
                return []

            # Determine target - prefer goal_state from reference panel (Part 7.4)
            target_map: Optional[Dict[Tuple[int, int], int]] = None
            target_color: Optional[int] = None
            if goal_state and isinstance(goal_state, dict) and len(goal_state) > 0:
                target_map = goal_state  # {(x,y): target_color}
            else:
                # Fallback: most common color among GRID CELLS ONLY
                grid_positions: Set[Tuple[int, int]] = set()
                for effects in kernels.values():
                    for cx, cy, _, _ in effects:
                        grid_positions.add((cx, cy))
                color_counts: Dict[int, int] = {}
                for pos in grid_positions:
                    if pos in current:
                        color_counts[current[pos]] = color_counts.get(current[pos], 0) + 1
                if not color_counts:
                    return []
                target_color = max(color_counts, key=color_counts.get)  # type: ignore[arg-type]

            def _compute_error_greedy(state: Dict[Tuple[int, int], int]) -> int:
                if target_map is not None:
                    return sum(1 for pos, c in state.items()
                               if pos in target_map and c != target_map[pos])
                return sum(1 for pos in grid_positions
                           if pos in state and state[pos] != target_color)

            # Count how many cells differ from target
            initial_errors = _compute_error_greedy(current)
            if initial_errors == 0:
                return []  # Already solved

            # Build relative kernel: for each click position, what's the relative
            # offset pattern of affected cells?
            solution: List[Tuple[int, int]] = []
            max_clicks = min(20, len(kernels) * 3)  # Safety limit

            # Copy current state for simulation
            sim_state = dict(current)

            for _ in range(max_clicks):
                # Find click that resolves the most errors
                best_click = None
                best_improvement = 0

                for click_pos in kernels:
                    improvement = 0
                    affected = self._get_affected_positions(kernels, click_pos)
                    for cx, cy, fallback_new in affected:
                        if (cx, cy) in sim_state:
                            cell_target = (target_map.get((cx, cy), sim_state[(cx, cy)])
                                           if target_map is not None else target_color)
                            was_wrong = sim_state[(cx, cy)] != cell_target
                            would_be = self._predict_next_color(
                                sim_state[(cx, cy)], game_key, fallback_new
                            )
                            would_be_right = would_be == cell_target
                            if was_wrong and would_be_right:
                                improvement += 1
                            elif not was_wrong and not would_be_right:
                                improvement -= 1

                    if improvement > best_improvement:
                        best_improvement = improvement
                        best_click = click_pos

                if best_click is None or best_improvement <= 0:
                    break

                # Apply click to simulated state using color cycling
                affected = self._get_affected_positions(kernels, best_click)
                for cx, cy, fallback_new in affected:
                    if (cx, cy) in sim_state:
                        sim_state[(cx, cy)] = self._predict_next_color(
                            sim_state[(cx, cy)], game_key, fallback_new
                        )

                solution.append(best_click)

                # Check if solved
                remaining_errors = _compute_error_greedy(sim_state)
                if remaining_errors == 0:
                    break

            return solution if solution else []

        except Exception:
            return []


class TriggerSequencesRung(DecisionRung):
    """Learn and use trigger chains - EXPLOITATION

    Uses engines/self_model/trigger_sequences.py to:
    1. Record trigger chains (X causes Y which causes Z)
    2. Replay proven action sequences for levels
    3. Predict trigger effects

    Critical for puzzle games with cause-effect chains.
    """
    name = "trigger_sequences"
    category = "exploitation"
    default_priority = 43
    confidence_threshold = 0.5

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ts = self.engines.trigger_sequences
        if ts is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', context.get('level_number', 1))

            if not game_type:
                return RungResult()

            # Check for proven trigger sequence for this level
            if hasattr(ts, 'get_proven_sequence'):
                proven = ts.get_proven_sequence(game_type, level)
                if proven:
                    # Get current position in sequence
                    action_count = context.get('action_count', 0)
                    if action_count < len(proven):
                        step = proven[action_count]
                        decay = self._consecutive_no_change * 0.08
                        conf = max(0.10, min(0.55, 0.75 - decay))
                        return RungResult(
                            action=step.get('action', f'ACTION{step.get("step_number", 1)}'),
                            confidence=conf,
                            reason=f"Proven trigger sequence step {action_count + 1}/{len(proven)}",
                            metadata={
                                'trigger_step': step,
                                'full_sequence': proven,
                                'sequence_position': action_count
                            }
                        )

            # Check for known trigger effects
            if hasattr(ts, 'predict_trigger_effect'):
                frame = _get_frame(game_state)
                if frame is not None:
                    # Get available actions
                    available = context.get('available_actions', list(range(1, 8)))
                    for action_num in available:
                        action = f'ACTION{action_num}'
                        prediction = ts.predict_trigger_effect(game_type, level, action)
                        if prediction and prediction.get('effect') == 'score_change':
                            if prediction.get('score_delta', 0) > 0:
                                decay = self._consecutive_no_change * 0.08
                                conf = max(0.10, min(0.55, 0.6 - decay))
                                return RungResult(
                                    action=action,
                                    confidence=conf,
                                    reason=f"Trigger predicts +score: {prediction.get('effect_target')}",
                                    metadata={'trigger_prediction': prediction}
                                )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Trigger sequences failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track frame changes for confidence decay."""
        try:
            frame_changed = False
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_changed = not (frame_before == frame_after).all()
                elif isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
            if frame_changed:
                self._consecutive_no_change = 0
            else:
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1


class EmbeddingMatcherRung(DecisionRung):
    """Use frame embeddings to find similar past situations - EXPLOITATION

    Uses engines/self_model/embedding_matcher.py to:
    1. Find similar past game frames using neural embeddings
    2. Return what action worked best in those situations
    3. Apply recency weighting (recent experiences weighted higher)

    This enables implicit generalization through learned representations.
    """
    name = "embedding_matcher"
    category = "exploitation"
    default_priority = 44
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        em = self.engines.embedding_matcher
        if em is None:
            return RungResult()

        try:
            frame = _get_frame(game_state)
            if frame is None:
                return RungResult()

            game_type = context.get('game_type', '')
            level = context.get('level', context.get('level_number', 1))

            if not game_type:
                return RungResult()

            # Get embedding-based action suggestion
            if hasattr(em, 'get_embedding_suggested_action'):
                suggestion = em.get_embedding_suggested_action(
                    game_type=game_type,
                    level=level,
                    current_frame=frame,
                    top_k=5
                )

                if suggestion and suggestion.get('suggested_action'):
                    action = suggestion['suggested_action']
                    confidence = suggestion.get('confidence', 0.5)
                    similar_count = len(suggestion.get('similar_situations', []))

                    return RungResult(
                        action=f'ACTION{action}' if isinstance(action, int) else action,
                        confidence=confidence * 0.8,
                        reason=f"Embedding match: {similar_count} similar situations suggest action {action}",
                        metadata={
                            'similar_situations': suggestion.get('similar_situations', [])[:3],
                            'embedding_confidence': confidence
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Embedding matcher failed: {e}")


class FewShotRelationsRung(DecisionRung):
    """Use few-shot invariants for quick control bootstrapping - EXPLOITATION

    Uses engines/self_model/few_shot_relations.py to:
    1. Get control invariants (what ALWAYS works for this action)
    2. Get control variants (what CHANGES based on context)
    3. Suggest actions based on proven invariants

    Enables quick learning from small numbers of examples.
    """
    name = "few_shot_relations"
    category = "exploitation"
    default_priority = 52
    confidence_threshold = 0.45

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        fsr = self.engines.few_shot_relations
        if fsr is None:
            return RungResult()

        try:
            game_id = context.get('game_id', context.get('game_type', ''))
            level = context.get('level', context.get('level_number', 1))

            if not game_id:
                return RungResult()

            # Get few-shot control relations
            if hasattr(fsr, 'get_few_shot_control_relations'):
                relations = fsr.get_few_shot_control_relations(game_id, level)

                if relations and relations.get('confidence', 0) > 0.5:
                    invariants = relations.get('invariants', {})

                    # Find an action with strong invariants
                    for action, props in invariants.items():
                        if props and props.get('success_rate', 0) > 0.7:
                            return RungResult(
                                action=action,
                                confidence=props.get('success_rate', 0.6) * 0.7,
                                reason=f"Few-shot invariant: {action} has {props.get('success_rate', 0):.0%} success",
                                metadata={
                                    'invariants': invariants,
                                    'variants': relations.get('variants', {}),
                                    'relation_confidence': relations.get('confidence')
                                }
                            )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Few-shot relations failed: {e}")


class NetworkSharingRung(DecisionRung):
    """Query network for shared control hypotheses - EXPLOITATION

    Uses engines/self_model/network_sharing.py to:
    1. Get validated control hypotheses from other agents
    2. Learn from network "I am this object" discoveries
    3. Use high-reliability patterns from the network

    Implements the thought process colony for self-model knowledge.

    ANTI-MONOPOLY: Confidence decays via two mechanisms:
    1. Per-game evaluation counter: decays 0.03 per evaluate() call for same game.
       This prevents constant high confidence even when other rungs' actions
       change the frame (which resets the no-change counter).
    2. No-change streak: existing frame-change tracking (secondary).
    Max confidence capped at 0.48 to stay below exploration rungs (~0.50).
    """
    name = "network_sharing"
    category = "exploitation"
    default_priority = 50
    confidence_threshold = 0.45

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0
        self._last_hypothesis_id = None
        # Per-game evaluation counter: tracks how many times this rung
        # has returned a suggestion for the same game+level without progress.
        # Key: "game_type_level", Value: count of evaluate() returns.
        self._game_eval_count: Dict[str, int] = {}
        self._last_game_level: Optional[str] = None

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ns = self.engines.network_sharing
        if ns is None:
            return RungResult()

        try:
            game_id = context.get('game_id', context.get('game_type', ''))
            level = context.get('level', context.get('level_number', 1))
            game_type = game_id.split('-')[0] if '-' in game_id else game_id

            if not game_type:
                return RungResult()

            # Get network control hypotheses
            if hasattr(ns, 'get_network_control_hypotheses'):
                hypotheses = ns.get_network_control_hypotheses(game_type, level)

                if hypotheses:
                    # Find highest-reliability hypothesis
                    # Key is 'reliability' (from get_network_control_hypotheses),
                    # NOT 'reliability_score' (that's the DB column name).
                    best = max(hypotheses, key=lambda h: h.get('reliability', 0))

                    if best.get('reliability', 0) > 0.6:
                        action_map = best.get('action_response_map', {})

                        # Suggest first action from the map
                        if action_map:
                            raw_key = list(action_map.keys())[0]
                            # Normalize: DB stores keys as "3" (bare number)
                            # or "ACTION6" depending on discovery path.
                            # The rung MUST return proper ACTION format.
                            if raw_key.startswith('ACTION'):
                                action = raw_key
                            elif raw_key.isdigit():
                                action = f'ACTION{raw_key}'
                            else:
                                return RungResult()  # Unrecognisable key

                            # ANTI-MONOPOLY: Dual decay mechanism.
                            # 1. Per-game eval count: decays even when other
                            #    rungs' actions change the frame.
                            # 2. No-change streak: secondary frame-based decay.
                            # Cap at 0.48 to stay below exploration (~0.50).
                            game_key = f"{game_type}_L{level}"
                            eval_count = self._game_eval_count.get(game_key, 0)
                            self._game_eval_count[game_key] = eval_count + 1

                            # Reset counter on level change
                            if self._last_game_level and self._last_game_level != game_key:
                                old_type = self._last_game_level.split('_L')[0]
                                if old_type == game_type:
                                    # Same game, new level = progress, reset
                                    self._game_eval_count[game_key] = 0
                                    eval_count = 0
                            self._last_game_level = game_key

                            raw_conf = best.get('reliability', 0.5) * 0.7
                            no_change_decay = self._consecutive_no_change * 0.08
                            eval_decay = eval_count * 0.03
                            confidence = max(0.10, min(0.48, raw_conf - no_change_decay - eval_decay))

                            hyp_id = best.get('hypothesis_id', 'unknown')
                            self._last_hypothesis_id = hyp_id

                            return RungResult(
                                action=action,
                                confidence=confidence,
                                reason=f"Network hypothesis: {hyp_id[:20]}",
                                metadata={
                                    'hypothesis': best,
                                    'action_map': action_map,
                                    'validation_count': best.get('validation_attempts', 0),
                                    'no_change_streak': self._consecutive_no_change
                                }
                            )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Network sharing failed: {e}")

    def on_action_complete(self, action: str, action_data: Any = None,
                           frame_before: Any = None, frame_after: Any = None,
                           context: Any = None, **kwargs: Any) -> None:
        """Track whether actions produce frame changes.

        Increments no-change streak when the frame is unchanged,
        resets on any change. This drives confidence decay in evaluate().
        """
        try:
            if frame_before is not None and frame_after is not None:
                if hasattr(frame_before, 'shape'):
                    frame_same = (frame_before == frame_after).all()
                else:
                    frame_same = frame_before == frame_after
                if frame_same:
                    self._consecutive_no_change += 1
                else:
                    self._consecutive_no_change = 0
            else:
                # Can't compare -- assume no change to be conservative
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1


class PrimitiveSuggesterRung(DecisionRung):
    """Use primitives directly for action suggestions - EXPLOITATION

    Uses engines/social/primitive_suggester.py to:
    1. Apply seed primitives to the current frame
    2. Map primitive outputs to action suggestions
    3. Use learned effectiveness (RLVR feedback)

    Simple, direct primitive-to-action mapping without CODS complexity.
    """
    name = "primitive_suggester"
    category = "exploitation"
    default_priority = 48
    confidence_threshold = 0.35

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        ps = self.engines.primitive_suggester
        if ps is None:
            return RungResult()

        try:
            frame = _get_frame(game_state)
            if frame is None:
                return RungResult()

            game_type = context.get('game_type', '')
            recent_actions = context.get('last_actions', [])

            # Get primitive-based suggestion
            if hasattr(ps, 'suggest_action'):
                result = ps.suggest_action(
                    frame=frame,
                    game_type=game_type,
                    recent_actions=recent_actions
                )

                if result and result.action:
                    # Convert to dict if it's a dataclass
                    result_dict = result.to_dict() if hasattr(result, 'to_dict') else {}

                    return RungResult(
                        action=f'ACTION{result.action}',
                        confidence=result.confidence * 0.8,
                        reason=f"Primitive {result.primitive}: {result.reasoning[:50]}",
                        metadata={
                            'primitive': result.primitive,
                            'primitives_applied': result_dict.get('primitives_applied', []),
                            'candidates': result_dict.get('candidates', [])[:3]
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Primitive suggester failed: {e}")


class ValenceGoalsRung(DecisionRung):
    """Use valence associations and inferred goals - FILTER/EXPLOITATION

    Uses engines/self_model/valence_goals.py to:
    1. Check valence of nearby objects (good/bad/neutral)
    2. Use inferred goals to guide action selection
    3. Avoid negative-valence objects, approach positive ones

    Provides emotional coloring to guide exploration.
    """
    name = "valence_goals"
    category = "exploitation"
    default_priority = 35
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        vg = self.engines.valence_goals
        if vg is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', context.get('level_number', 1))
            player_pos = context.get('player_position')

            if not game_type:
                return RungResult()

            # Check for inferred goals
            if hasattr(vg, 'get_inferred_goal'):
                goal = vg.get_inferred_goal(game_type, level)

                if goal and goal.get('confidence', 0) > 0.5:
                    goal_type = goal.get('goal_type')
                    targets = goal.get('target_regions', [])

                    if targets and player_pos:
                        # Move toward goal target
                        target = targets[0]
                        dx = target[1] - player_pos[0] if len(target) > 1 else 0
                        dy = target[0] - player_pos[1] if len(target) > 0 else 0

                        if abs(dx) > abs(dy):
                            action = 'ACTION3' if dx > 0 else 'ACTION4'
                            direction = 'right' if dx > 0 else 'left'
                        else:
                            action = 'ACTION2' if dy > 0 else 'ACTION1'
                            direction = 'down' if dy > 0 else 'up'

                        return RungResult(
                            action=action,
                            confidence=goal.get('confidence', 0.5) * 0.6,
                            reason=f"Goal {goal_type}: move {direction} toward target",
                            metadata={'goal': goal, 'target': target}
                        )

            # Check valence of nearby objects for avoidance
            if hasattr(vg, 'get_negative_valence_objects'):
                dangers = vg.get_negative_valence_objects(game_type)

                if dangers:
                    # Return as filter info rather than action
                    return RungResult(
                        confidence=0.3,
                        reason=f"Valence: {len(dangers)} negative objects to avoid",
                        metadata={
                            'negative_objects': dangers[:5],
                            'filter_mode': True
                        }
                    )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Valence goals failed: {e}")


class ReplayLearningRung(DecisionRung):
    """Prediction-based learning during sequence replay - EXPLOITATION"""
    name = "replay_learning"
    category = "exploitation"
    default_priority = 43
    confidence_threshold = 0.5

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        rle = self.engines.replay_learning_engine
        if rle is None:
            return RungResult()

        try:
            if hasattr(rle, 'get_current_prediction'):
                prediction = rle.get_current_prediction()
                if prediction and context.get('is_replay', False):
                    action = prediction.get('action')
                    # CRITICAL: Validate action is available in this game
                    if action and is_action_available(action, context):
                        return RungResult(
                            action=action,
                            confidence=prediction.get('confidence', 0.5),
                            reason=f"Replay prediction: {prediction.get('hypothesis', 'unknown')}",
                            metadata={'prediction': prediction, 'is_replay': True}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Replay learning failed: {e}")


class CompletionPredictionRung(DecisionRung):
    """Estimate steps to completion - EXPLOITATION"""
    name = "completion_prediction"
    category = "exploitation"
    default_priority = 39
    confidence_threshold = 0.4

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            prediction = context.get('completion_prediction', {})

            if prediction:
                match_progress = prediction.get('match_progress', 0)
                remaining = prediction.get('remaining_steps', 100)

                # If close to completion, stay on sequence
                if match_progress > 0.8 and remaining < 10:
                    sequence_action = context.get('next_sequence_action')
                    # CRITICAL: Validate action is available in this game
                    if sequence_action and is_action_available(sequence_action, context):
                        return RungResult(
                            action=sequence_action,
                            confidence=0.7,
                            reason=f"Near completion: {match_progress:.0%}, {remaining} steps left",
                            metadata={'prediction': prediction}
                        )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Completion prediction failed: {e}")


class RuleTransferRung(DecisionRung):
    """Apply learned rules from other games - EXPLOITATION

    Wires: engines/cognition/rule_induction.py:
        - get_applicable_rules()
        - update_rule_success()  # Feedback loop

    Queries the learned_rules table for rules that match the current game frame.
    Enables cross-game knowledge transfer where patterns discovered in one game
    can guide action selection in structurally similar games.

    Feedback Loop: After action outcome, calls update_rule_success() to adjust
    rule confidence. Successful rules gain confidence (+0.05), failed rules lose (-0.1).
    """
    name = "rule_transfer"
    category = "exploitation"
    default_priority = 25  # After hypothesis, before network wisdom
    confidence_threshold = 0.5

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._rule_engine: Optional[Any] = None
        # Track active rule for feedback loop
        self._active_rule_id: Optional[str] = None
        self._active_rule_game: Optional[str] = None

    def _get_rule_engine(self) -> Optional[Any]:
        """Lazy-load rule induction engine."""
        if self._rule_engine is None:
            try:
                from engines.cognition.rule_induction import RuleInductionEngine
                db = DatabaseInterface()
                self._rule_engine = RuleInductionEngine(db)
            except ImportError as e:
                logger.debug(f"[RULE-TRANSFER] Failed to load RuleInductionEngine: {e}")
        return self._rule_engine

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        engine = self._get_rule_engine()
        if engine is None:
            return RungResult()

        try:
            frame = _get_frame(game_state)
            if frame is None:
                return RungResult()

            agent_id = context.get('agent_id')
            min_confidence = self.confidence_threshold

            # Get rules that match current game frame
            applicable_rules = engine.get_applicable_rules(
                current_frame=frame,
                agent_id=agent_id,
                min_confidence=min_confidence
            )

            if not applicable_rules:
                return RungResult()

            # Use highest-confidence rule
            best_rule, confidence = applicable_rules[0]
            action_template = best_rule.get('action_template', {})
            suggested_action = action_template.get('action')

            if not suggested_action:
                return RungResult()

            # CRITICAL: Validate action is available in this game
            if not is_action_available(suggested_action, context):
                return RungResult(reason=f"Rule suggested unavailable action: {suggested_action}")

            # Track rule for feedback loop (instance-level, not context-level)
            self._active_rule_id = best_rule.get('rule_id')
            self._active_rule_game = context.get('game_type', '')

            return RungResult(
                action=suggested_action,
                confidence=confidence,
                reason=f"Rule transfer: {best_rule.get('rule_type', 'unknown')} from {best_rule.get('source_game', 'network')}",
                metadata={
                    'rule_id': best_rule.get('rule_id'),
                    'rule_type': best_rule.get('rule_type'),
                    'source_game': best_rule.get('source_game'),
                    'match_confidence': confidence,
                    'success_count': best_rule.get('success_count', 0),
                }
            )
        except Exception as e:
            return RungResult(reason=f"Rule transfer failed: {e}")

    def record_outcome(self, was_accepted: bool, success: bool = False) -> None:
        """Update rule success/failure after action outcome.

        Wires: engines/cognition/rule_induction.py:update_rule_success()
        This closes the feedback loop - rules that work gain confidence,
        rules that fail lose confidence.
        """
        super().record_outcome(was_accepted)

        # If rule was used and we have outcome, update the rule
        if was_accepted and self._active_rule_id and self._rule_engine is not None:
            try:
                self._rule_engine.update_rule_success(
                    rule_id=self._active_rule_id,
                    success=success,
                    target_game_id=self._active_rule_game or 'unknown'
                )
            except Exception:
                pass  # Don't fail silently, but don't break gameplay
            finally:
                # Clear active rule after feedback
                self._active_rule_id = None
                self._active_rule_game = None


class ConstraintDecoderRung(DecisionRung):
    """Decode constraint sprites and feed formal constraints to the solver - EXPLOITATION

    For constraint-satisfaction click games (e.g. FT09):
    1. Identify constraint/key sprites adjacent to interactive tiles
    2. Decode what each constraint demands:
       - Which tile must become a specific target color
       - Whether the pattern requires per-tile or grouped matching
    3. Store decoded constraints in context for ConstraintSatisfactionRung
    4. Suggest targeted clicks when a single click satisfies a constraint

    FT09 specifics:
    - Grid cells cycle through their color palette when clicked
    - Each tile affects ONLY itself (standard tiles)
    - Key/constraint sprites sit adjacent to tiles and encode target colors
    - The center pixel of each key sprite = the target color for that tile
    - The puzzle is solved when ALL tiles match their key sprite targets

    ROOT CAUSE ADDRESSED: ConstraintSatisfactionRung has a greedy/lookahead
    solver but no way to understand WHAT the constraints are. Without decoded
    constraints, the solver falls back to "most common color" heuristic which
    is wrong for FT09 where each tile has its own target color.
    """
    name = "constraint_decoder"
    category = "exploitation"
    default_priority = 31  # Just above constraint_satisfaction (30)
    confidence_threshold = 0.0  # Context-setter, not action-proposer

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Per-game decoded constraints: game_key -> list of constraint dicts
        self._decoded_constraints: Dict[str, List[Dict[str, Any]]] = {}
        # Track whether we've analyzed the current level's layout
        self._analyzed_levels: Set[str] = set()
        # Retry counter: cap at 5 attempts to prevent infinite re-analysis
        self._analyze_attempts: Dict[str, int] = {}
        # Detected grid cells and their positions
        self._grid_cells: Dict[str, List[Tuple[int, int]]] = {}
        # Detected color cycle length per game
        self._color_cycle_len: Dict[str, int] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Get frame as 2D list for pixel analysis
        frame = _get_frame_2d(game_state)
        if frame is None:
            return RungResult()

        # Analyze constraints for this level. Retry up to 5 times if first
        # frames were ambiguous (grid might not be visible on frame 1).
        max_attempts = 5
        if game_key not in self._analyzed_levels:
            attempts = self._analyze_attempts.get(game_key, 0)
            if attempts < max_attempts:
                self._analyze_attempts[game_key] = attempts + 1
                self._analyze_constraints(frame, game_key, context)
            if self._decoded_constraints.get(game_key) or attempts >= max_attempts:
                self._analyzed_levels.add(game_key)

        constraints = self._decoded_constraints.get(game_key, [])
        if not constraints:
            return RungResult()

        # Detect color cycle from causal map observations
        world_model = context.get('world_model', {})
        causal_map = world_model.get('causal_map', {}) if isinstance(world_model, dict) else {}
        if game_key not in self._color_cycle_len and causal_map:
            self._detect_color_cycle(causal_map, game_key)

        # Write decoded constraints to a format ConstraintSatisfactionRung
        # can consume. We return a RungResult with metadata containing the
        # constraints. The constraint_satisfaction rung reads shared context.
        # Also inject into world_model for cross-rung sharing.
        if isinstance(world_model, dict):
            world_model['decoded_constraints'] = constraints
            world_model['grid_cells'] = self._grid_cells.get(game_key, [])
            if game_key in self._color_cycle_len:
                world_model['color_cycle_len'] = self._color_cycle_len[game_key]

        # Build goal state from decoded constraints
        goal_state = self._build_goal_from_constraints(frame, constraints, game_key)
        if goal_state and isinstance(world_model, dict):
            # Write to constraint_goal_state for ConstraintSatisfactionRung
            world_model['constraint_goal_state'] = goal_state
            # Also write to goal_state so CausalMap.import_from_world_model()
            # picks it up and enables SPEED 1 (MAPPED) planning
            world_model['goal_state'] = goal_state

        # Write current cell colors so CausalMap can compute deltas
        grid_positions = self._grid_cells.get(game_key, [])
        if grid_positions and isinstance(world_model, dict):
            cell_states = {}
            for cx, cy in grid_positions:
                if 0 <= cy < len(frame) and 0 <= cx < len(frame[0]):
                    color = frame[cy][cx]
                    if hasattr(color, '__int__'):
                        color = int(color)
                    cell_states[(cx, cy)] = color
            if cell_states:
                world_model['cell_states'] = cell_states

        # This rung is primarily a context-setter (confidence=0)
        # It enriches the world_model so constraint_satisfaction can plan
        return RungResult(
            confidence=0.0,
            reason=f"Decoded {len(constraints)} constraints for {game_key}",
            metadata={
                'constraint_count': len(constraints),
                'grid_cells': len(self._grid_cells.get(game_key, [])),
                'source': 'constraint_decoder',
            }
        )

    def _analyze_constraints(self, frame: Any, game_key: str,
                             context: Dict[str, Any]) -> None:
        """Analyze frame to find constraint sprites and grid cells.

        Strategy:
        1. Find the interactive grid region (cells that can be clicked)
        2. Find constraint sprites around/beside the grid
        3. Decode each constraint's spatial mask

        For FT09-like games, the layout is typically:
        - A small NxN grid of colored cells (the puzzle)
        - Constraint sprites around or overlaid on the grid
        - A reference/goal region showing target state
        """
        try:
            frame_list = frame
            if hasattr(frame, 'tolist'):
                frame_list = frame.tolist()
            if not isinstance(frame_list, list) or not frame_list:
                return

            h = len(frame_list)
            w = len(frame_list[0]) if h > 0 else 0
            if h == 0 or w == 0:
                return

            # Step 1: Find distinct colored regions (potential grid cells)
            # Grid cells are typically small, uniform-color rectangles
            color_regions = self._find_uniform_regions(frame_list, h, w)

            # Step 2: Identify grid structure
            # Grid cells form a regular pattern (equal size, equal spacing)
            grid_cells = self._identify_grid_pattern(color_regions)
            if grid_cells:
                self._grid_cells[game_key] = [
                    (cell['cx'], cell['cy']) for cell in grid_cells
                ]

            # Step 3: Find constraint sprites
            # Constraints are small colored sprites near/around the grid
            # that don't belong to the interactive grid itself
            visual_scene = context.get('visual_scene', {})
            constraints = self._extract_constraints_from_layout(
                frame_list, grid_cells, color_regions, visual_scene, h, w
            )

            if constraints:
                self._decoded_constraints[game_key] = constraints

        except Exception:
            pass

    def _find_uniform_regions(self, frame: List[List[int]],
                              h: int, w: int) -> List[Dict[str, Any]]:
        """Find contiguous uniform-color rectangular regions in the frame."""
        visited = [[False] * w for _ in range(h)]
        regions: List[Dict[str, Any]] = []

        for y in range(h):
            for x in range(w):
                if visited[y][x]:
                    continue
                color = frame[y][x]
                if hasattr(color, '__int__'):
                    color = int(color)
                if color == 0:  # Skip background
                    visited[y][x] = True
                    continue

                # Flood-fill to find connected region of same color
                pixels: List[Tuple[int, int]] = []
                stack = [(x, y)]
                while stack:
                    cx, cy = stack.pop()
                    if cx < 0 or cx >= w or cy < 0 or cy >= h:
                        continue
                    if visited[cy][cx]:
                        continue
                    cell_color = frame[cy][cx]
                    if hasattr(cell_color, '__int__'):
                        cell_color = int(cell_color)
                    if cell_color != color:
                        continue
                    visited[cy][cx] = True
                    pixels.append((cx, cy))
                    stack.extend([(cx+1, cy), (cx-1, cy), (cx, cy+1), (cx, cy-1)])

                if len(pixels) < 4:
                    continue  # Skip tiny noise regions

                min_x = min(p[0] for p in pixels)
                max_x = max(p[0] for p in pixels)
                min_y = min(p[1] for p in pixels)
                max_y = max(p[1] for p in pixels)
                cx_r = (min_x + max_x) // 2
                cy_r = (min_y + max_y) // 2
                width_r = max_x - min_x + 1
                height_r = max_y - min_y + 1

                regions.append({
                    'color': color,
                    'cx': cx_r, 'cy': cy_r,
                    'min_x': min_x, 'min_y': min_y,
                    'max_x': max_x, 'max_y': max_y,
                    'width': width_r, 'height': height_r,
                    'pixel_count': len(pixels),
                    'is_rect': len(pixels) >= width_r * height_r * 0.8,
                })

        return regions

    def _identify_grid_pattern(
        self, regions: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Identify which regions form a regular grid pattern.

        Grid cells have similar sizes and regular spacing.
        Returns the grid cells sorted by position.
        """
        if len(regions) < 4:
            return []

        # Group regions by approximate size (within 20%)
        size_groups: Dict[Tuple[int, int], List[Dict[str, Any]]] = {}
        for r in regions:
            if not r['is_rect']:
                continue
            # Quantize size to 4px bins
            sw = (r['width'] // 4) * 4
            sh = (r['height'] // 4) * 4
            key = (max(sw, 4), max(sh, 4))
            if key not in size_groups:
                size_groups[key] = []
            size_groups[key].append(r)

        # Find the largest size group (most regions of similar size = likely grid)
        if not size_groups:
            return []

        best_group = max(size_groups.values(), key=len)
        if len(best_group) < 4:
            return []

        # Verify regular spacing: check if centers form a grid
        # Sort by y then x
        sorted_cells = sorted(best_group, key=lambda c: (c['cy'], c['cx']))

        # Check for at least 2 rows and 2 columns
        ys = sorted(set(c['cy'] // 4 for c in sorted_cells))
        xs = sorted(set(c['cx'] // 4 for c in sorted_cells))

        if len(ys) >= 2 and len(xs) >= 2:
            return sorted_cells

        return []

    def _extract_constraints_from_layout(
        self,
        frame: List[List[int]],
        grid_cells: List[Dict[str, Any]],
        all_regions: List[Dict[str, Any]],
        visual_scene: Any,
        h: int, w: int
    ) -> List[Dict[str, Any]]:
        """Extract constraint information from the frame layout.

        Constraints are inferred from:
        1. Non-grid colored regions near the grid (constraint sprites)
        2. Reference/legend panels showing target states
        3. Small indicator sprites that encode 3x3 masks
        """
        constraints: List[Dict[str, Any]] = []
        grid_positions = set()
        if grid_cells:
            for cell in grid_cells:
                grid_positions.add((cell['cx'] // 4, cell['cy'] // 4))

        # Look for reference panel in visual scene
        ref_panel = None
        if isinstance(visual_scene, dict):
            panels = visual_scene.get('panels', [])
            for panel in panels:
                if isinstance(panel, dict) and panel.get('role') == 'reference':
                    ref_panel = panel
                    break

        # Strategy A: If we have a reference panel, extract target colors
        # from it directly as the constraint goal
        if ref_panel and grid_cells:
            ref_bounds = ref_panel.get('bounds', {})
            rx1 = ref_bounds.get('x1', 0)
            ry1 = ref_bounds.get('y1', 0)
            rx2 = ref_bounds.get('x2', w)
            ry2 = ref_bounds.get('y2', h)

            # The reference panel should show the goal state for the grid
            for i, cell in enumerate(grid_cells):
                # Map grid cell position to corresponding position in ref panel
                # This is approximate - assumes ref panel has same spatial layout
                rel_x = cell['cx'] - grid_cells[0]['min_x']
                rel_y = cell['cy'] - grid_cells[0]['min_y']
                ref_x = rx1 + rel_x
                ref_y = ry1 + rel_y

                if 0 <= ref_y < h and 0 <= ref_x < w:
                    target_color = frame[ref_y][ref_x]
                    if hasattr(target_color, '__int__'):
                        target_color = int(target_color)
                    constraints.append({
                        'type': 'cell_color',
                        'cell_idx': i,
                        'cell_pos': (cell['cx'], cell['cy']),
                        'target_color': target_color,
                        'source': 'reference_panel',
                    })

        # Strategy B: Detect target-color indicator sprites near grid cells.
        # Small sprites adjacent to interactive tiles encode the target color
        # for that tile. The center pixel of the sprite = the target color.
        non_grid_regions = [
            r for r in all_regions
            if (r['cx'] // 4, r['cy'] // 4) not in grid_positions
            and r['pixel_count'] < 100  # Small sprites only
            and r['is_rect']
        ]

        for sprite in non_grid_regions:
            # Find which grid cell this sprite is closest to
            if not grid_cells:
                break
            closest_cell = min(
                grid_cells,
                key=lambda c: abs(c['cx'] - sprite['cx']) + abs(c['cy'] - sprite['cy'])
            )
            distance = abs(closest_cell['cx'] - sprite['cx']) + abs(closest_cell['cy'] - sprite['cy'])

            # If sprite is adjacent to a grid cell, it might be a constraint indicator
            cell_size = max(closest_cell['width'], closest_cell['height'], 4)
            if distance < cell_size * 3:
                constraints.append({
                    'type': 'target_color_indicator',
                    'sprite_pos': (sprite['cx'], sprite['cy']),
                    'sprite_color': sprite['color'],
                    'target_cell_pos': (closest_cell['cx'], closest_cell['cy']),
                    'target_color': sprite['color'],  # Center pixel = target
                    'distance': distance,
                    'source': 'spatial_proximity',
                })

        return constraints

    def _detect_color_cycle(self, causal_map: Dict[str, Any],
                            game_key: str) -> None:
        """Detect how many colors cells cycle through from causal observations."""
        colors_seen: Set[int] = set()
        for _pos_key, entry in causal_map.items():
            if not isinstance(entry, dict):
                continue
            for obs in entry.get('observations', []):
                for change in obs.get('changes', []):
                    if isinstance(change, dict):
                        fc = change.get('from_color', 0)
                        tc = change.get('to_color', 0)
                        # Skip corrupted entries (list instead of int)
                        if isinstance(fc, (int, float, str)):
                            colors_seen.add(int(fc))
                        if isinstance(tc, (int, float, str)):
                            colors_seen.add(int(tc))

        # Cycle length = number of distinct non-zero colors observed
        non_zero = colors_seen - {0}
        if len(non_zero) >= 2:
            self._color_cycle_len[game_key] = len(non_zero)

    def _build_goal_from_constraints(
        self,
        frame: Any,
        constraints: List[Dict[str, Any]],
        game_key: str
    ) -> Optional[Dict[Tuple[int, int], int]]:
        """Build a goal_state dict from decoded constraints.

        Returns {(x,y): target_color} for ConstraintSatisfactionRung.
        """
        goal: Dict[Tuple[int, int], int] = {}
        for c in constraints:
            if c.get('type') == 'cell_color' and 'target_color' in c:
                pos = c.get('cell_pos')
                if pos:
                    goal[pos] = c['target_color']
        return goal if goal else None

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Re-analyze constraints if the frame changed significantly.

        Level transitions reset constraint analysis so the new level's
        constraints are decoded fresh.
        """
        if action != 'ACTION6':
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # If level changed, force re-analysis for the new level
        if not hasattr(self, '_last_level_seen'):
            self._last_level_seen: Dict[str, int] = {}
        prev = self._last_level_seen.get(game_type, level)
        self._last_level_seen[game_type] = level
        if level != prev:
            self._analyzed_levels.discard(game_key)
            self._analyze_attempts.pop(game_key, None)


# ---------------------------------------------------------------------------
# H35 NEW RUNGS: CLASSIFY, EXTRACT_GOAL, VERIFY
# ---------------------------------------------------------------------------


class GameClassifierRung(DecisionRung):
    """CLASSIFY: Set problem_class, click_semantic, and goal_template_hint.

    Fires very early (priority 5) to inform all downstream rungs about the
    game type. Context-setter only (confidence=0.0, never proposes actions).

    H38: For unknown games, analyzes first frame to detect grid patterns
    and color distribution, setting goal_template_hint for CSAT/CausalMapping.
    """
    name = "game_classifier"
    category = "exploitation"
    default_priority = 5
    confidence_threshold = 0.0

    def _detect_grid_pattern(self, frame: List[List[int]]) -> bool:
        """Check if non-background pixels form a regular grid."""
        # Collect non-background pixel positions
        positions = []
        for y, row in enumerate(frame):
            for x, val in enumerate(row):
                v = int(val) if hasattr(val, '__int__') else val
                if v > 0:
                    positions.append((x, y))

        if len(positions) < 9:
            return False

        # Check regularity: find most common x and y gaps
        xs = sorted(set(p[0] for p in positions))
        ys = sorted(set(p[1] for p in positions))

        if len(xs) < 3 or len(ys) < 3:
            return False

        # Compute gaps between consecutive x and y coords
        x_gaps = [xs[i + 1] - xs[i] for i in range(len(xs) - 1)]
        y_gaps = [ys[i + 1] - ys[i] for i in range(len(ys) - 1)]

        if not x_gaps or not y_gaps:
            return False

        # Mode gap should account for >60% of gaps (regular spacing)
        from collections import Counter
        x_mode = Counter(x_gaps).most_common(1)[0]
        y_mode = Counter(y_gaps).most_common(1)[0]

        x_regular = x_mode[1] / len(x_gaps) > 0.6
        y_regular = y_mode[1] / len(y_gaps) > 0.6

        return x_regular and y_regular

    def _infer_goal_template(self, frame: List[List[int]],
                             available: list) -> str:
        """Infer goal template from frame structure for unknown games."""
        click_only = (available == [6])
        nav_only = (set(available) <= {1, 2, 3, 4})

        if nav_only:
            return 'navigation_collection'

        if click_only:
            if self._detect_grid_pattern(frame):
                return 'uniform_color'
            return 'alignment_matching'

        return 'unknown'

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        game_type = context.get('game_type', '')
        available = context.get('available_actions', [])
        world_model = context.get('world_model', {})

        if not isinstance(world_model, dict):
            return RungResult()

        if game_type == 'ft09':
            world_model['problem_class'] = 'constraint_satisfaction'
            world_model['click_semantic'] = 'toggle'
            world_model['goal_template_hint'] = 'uniform_color'
        elif game_type == 'vc33':
            world_model['problem_class'] = 'sequential_switching'
            world_model['click_semantic'] = 'cycle'
            world_model['goal_template_hint'] = 'alignment_matching'
        elif game_type == 'ls20':
            world_model['problem_class'] = 'maze_navigation'
            world_model['click_semantic'] = 'none'
            world_model['goal_template_hint'] = 'navigation_collection'
        else:
            if available == [6]:
                world_model['problem_class'] = 'click_puzzle'
            elif set(available) <= {1, 2, 3, 4}:
                world_model['problem_class'] = 'navigation'
            else:
                world_model['problem_class'] = 'mixed'
            world_model['click_semantic'] = 'unknown'

            # H38: Infer goal template from frame analysis
            frame = _get_frame_2d(game_state)
            if frame is not None:
                world_model['goal_template_hint'] = self._infer_goal_template(
                    frame, available)
            else:
                world_model['goal_template_hint'] = 'unknown'

        return RungResult(
            confidence=0.0,
            reason=f"Classified: {world_model.get('problem_class', '?')}",
            metadata={'problem_class': world_model.get('problem_class'),
                      'goal_template_hint': world_model.get('goal_template_hint'),
                      'source': 'game_classifier'},
        )


class SolverGoalExtractionRung(DecisionRung):
    """EXTRACT_GOAL: Load solver-seeded goal states into constraint_goal_state.

    Reads solver_goal_states from world_model (seeded by context_builder from
    world_model_states DB) and writes constraint_goal_state for CSAT to consume.
    Context-setter only (confidence=0.0).

    Fires before constraint_decoder so solver goals take priority. If no solver
    goals exist, constraint_decoder's frame-analysis fallback still works.
    """
    name = "solver_goal_extraction"
    category = "exploitation"
    default_priority = 27
    confidence_threshold = 0.0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        world_model = context.get('world_model', {})
        if not isinstance(world_model, dict):
            return RungResult()

        level = context.get('level', 1)
        solver_goals = world_model.get('solver_goal_states', {})
        level_goal = solver_goals.get(str(level), {})

        if not level_goal:
            return RungResult()

        # Convert string keys "x,y" to tuple keys (x, y)
        goal_state: Dict[Tuple[int, int], int] = {}
        for k, v in level_goal.items():
            if isinstance(k, str):
                try:
                    parts = k.strip('()').split(',')
                    goal_state[(int(parts[0].strip()), int(parts[1].strip()))] = int(v)
                except (ValueError, IndexError):
                    continue
            else:
                goal_state[k] = v

        if goal_state:
            world_model['constraint_goal_state'] = goal_state
            world_model['goal_state'] = goal_state
            return RungResult(
                confidence=0.0,
                reason=f"Loaded solver goal: {len(goal_state)} cells for L{level}",
                metadata={'goal_cells': len(goal_state),
                          'source': 'solver_goal_extraction'},
            )

        return RungResult()


class GoalProgressRung(DecisionRung):
    """VERIFY: Compare current frame to goal state and track progress.

    After CSAT plans and executes clicks, this rung checks how many cells
    match the goal. Writes goal_progress to world_model so other rungs can
    detect when progress is declining (agent undoing work via toggles).
    Context-setter only (confidence=0.0).
    """
    name = "goal_progress"
    category = "exploitation"
    default_priority = 48
    confidence_threshold = 0.0

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self._progress_history: Dict[str, List[int]] = {}
        self._best_progress: Dict[str, int] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        world_model = context.get('world_model', {})
        if not isinstance(world_model, dict):
            return RungResult()

        goal_state = world_model.get('constraint_goal_state', {})
        if not goal_state:
            return RungResult()

        frame = _get_frame_2d(game_state)
        if frame is None:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        # Count matching cells
        matches = 0
        total = len(goal_state)
        for (gx, gy), target_color in goal_state.items():
            if 0 <= gy < len(frame) and 0 <= gx < len(frame[0]):
                current = frame[gy][gx]
                if hasattr(current, '__int__'):
                    current = int(current)
                if current == target_color:
                    matches += 1

        # Track progress
        history = self._progress_history.setdefault(game_key, [])
        history.append(matches)
        if len(history) > 20:
            self._progress_history[game_key] = history[-20:]

        best = self._best_progress.get(game_key, 0)
        if matches > best:
            self._best_progress[game_key] = matches

        declining = (len(history) >= 3
                     and matches < history[-2] < history[-3])

        world_model['goal_progress'] = {
            'matches': matches,
            'total': total,
            'ratio': matches / max(total, 1),
            'best': max(best, matches),
            'declining': declining,
        }

        return RungResult(
            confidence=0.0,
            reason=f"Goal progress: {matches}/{total} "
                   f"({matches / max(total, 1):.0%})",
            metadata={'matches': matches, 'total': total,
                      'best': max(best, matches),
                      'source': 'goal_progress'},
        )

    def on_action_complete(
        self,
        action: str,
        outcome_context: Optional[Dict[str, Any]] = None,
    ) -> None:
        """Reset progress tracking on level change."""
        if outcome_context:
            game_type = outcome_context.get('game_type', '')
            level = outcome_context.get('level', 1)
            game_key = f"{game_type}_L{level}"
            if not hasattr(self, '_last_level_seen'):
                self._last_level_seen: Dict[str, int] = {}
            prev = self._last_level_seen.get(game_type, level)
            self._last_level_seen[game_type] = level
            if level != prev:
                self._progress_history.pop(game_key, None)
                self._best_progress.pop(game_key, None)


# ---------------------------------------------------------------------------
# H41b NEW RUNGS: MAP_EFFECTS forward model + VERIFY prediction error
# ---------------------------------------------------------------------------


class EffectPredictionRung(DecisionRung):
    """MAP_EFFECTS: Forward model -- predict frame changes BEFORE acting.

    Reads the causal_map from world_model (learned by CausalClickMapping or
    seeded by solver) and predicts what will happen if we click a given cell.
    Stores the prediction in world_model['effect_prediction'] for
    ActionOutcomeVerifierRung to check after the action executes.

    For click games: uses causal_map entry for the proposed click position.
    For nav games: uses wall map to predict whether movement succeeds.
    Context-setter only (confidence=0.0) -- never proposes actions.
    """
    name = "effect_prediction"
    category = "exploitation"
    default_priority = 26  # Before EXTRACT_GOAL (27), after CLASSIFY (5)
    confidence_threshold = 0.0

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        world_model = context.get('world_model', {})
        if not isinstance(world_model, dict):
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        frame = _get_frame_2d(game_state)
        if frame is None:
            return RungResult()

        # Build prediction from causal map (click games)
        causal_map = world_model.get('causal_map', {})
        if causal_map:
            predictions: Dict[str, Any] = {}
            for pos_key, entry in causal_map.items():
                obs_list = entry.get('observations', [])
                if not obs_list:
                    continue
                # Most recent observation's changes are our prediction
                latest = obs_list[-1] if obs_list else {}
                changes = latest.get('changes', [])
                if changes:
                    predictions[pos_key] = {
                        'expected_changes': changes,
                        'confidence': min(len(obs_list) / 5.0, 1.0),
                    }

            if predictions:
                world_model['effect_prediction'] = {
                    'game_key': game_key,
                    'predictions': predictions,
                    'source': 'causal_map',
                }
                return RungResult(
                    confidence=0.0,
                    reason=f"Forward model: {len(predictions)} positions mapped",
                    metadata={'positions_mapped': len(predictions),
                              'source': 'effect_prediction'},
                )

        # Nav games: predict from wall map
        wall_map = world_model.get('wall_map', {})
        if wall_map:
            world_model['effect_prediction'] = {
                'game_key': game_key,
                'wall_positions': len(wall_map),
                'source': 'wall_map',
            }
            return RungResult(
                confidence=0.0,
                reason=f"Nav model: {len(wall_map)} walls known",
                metadata={'walls_known': len(wall_map),
                          'source': 'effect_prediction'},
            )

        return RungResult()


class ActionOutcomeVerifierRung(DecisionRung):
    """VERIFY: Compare predicted vs actual frame changes after each action.

    Tracks prediction accuracy over time. When the forward model is wrong
    (high prediction error), signals MAP_EFFECTS rungs to re-learn.

    Writes to world_model:
      - outcome_stats.prediction_accuracy: ratio of correct predictions
      - outcome_stats.actions_without_change: consecutive no-effect count
      - outcome_stats.stagnation_detected: bool
      - outcome_stats.model_needs_update: bool

    Context-setter only (confidence=0.0) -- never proposes actions.
    """
    name = "action_outcome_verifier"
    category = "exploitation"
    default_priority = 49  # After goal_progress (48)
    confidence_threshold = 0.0

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Per-game tracking
        self._correct_predictions: Dict[str, int] = {}
        self._total_predictions: Dict[str, int] = {}
        self._no_change_streak: Dict[str, int] = {}
        self._total_actions: Dict[str, int] = {}
        # Frame hash history for cycle detection
        self._frame_hashes: Dict[str, List[int]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        world_model = context.get('world_model', {})
        if not isinstance(world_model, dict):
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        total = self._total_predictions.get(game_key, 0)
        correct = self._correct_predictions.get(game_key, 0)
        no_change = self._no_change_streak.get(game_key, 0)
        total_actions = self._total_actions.get(game_key, 0)

        accuracy = correct / max(total, 1)
        stagnation = no_change >= 5

        world_model['outcome_stats'] = {
            'prediction_accuracy': accuracy,
            'total_predictions': total,
            'actions_without_change': no_change,
            'total_actions': total_actions,
            'stagnation_detected': stagnation,
            'model_needs_update': total >= 5 and accuracy < 0.3,
        }

        reason_parts = []
        if total > 0:
            reason_parts.append(f"model {accuracy:.0%} accurate ({correct}/{total})")
        if stagnation:
            reason_parts.append(f"STAGNANT ({no_change} no-change)")
        if not reason_parts:
            return RungResult()

        return RungResult(
            confidence=0.0,
            reason=f"Outcome: {', '.join(reason_parts)}",
            metadata={'accuracy': accuracy, 'stagnation': stagnation,
                      'source': 'action_outcome_verifier'},
        )

    def on_action_complete(
        self,
        action: str,
        action_data: Any = None,
        frame_before: Any = None,
        frame_after: Any = None,
        context: Any = None,
        **kwargs: Any,
    ) -> None:
        """Compare prediction with actual outcome."""
        if not isinstance(context, dict):
            context = kwargs.get('outcome_context', {}) or {}

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        self._total_actions[game_key] = self._total_actions.get(game_key, 0) + 1

        # Detect frame change
        frame_changed = False
        if frame_before is not None and frame_after is not None:
            try:
                fb = frame_before
                fa = frame_after
                if hasattr(fb, 'tolist'):
                    fb = fb.tolist()
                if hasattr(fa, 'tolist'):
                    fa = fa.tolist()
                frame_changed = (fb != fa)
            except Exception:
                pass

        # Track no-change streaks
        if frame_changed:
            self._no_change_streak[game_key] = 0
        else:
            self._no_change_streak[game_key] = (
                self._no_change_streak.get(game_key, 0) + 1
            )

        # Track frame hash for cycle detection
        if frame_after is not None:
            try:
                if hasattr(frame_after, 'tobytes'):
                    h = hash(frame_after.tobytes())
                else:
                    h = hash(str(frame_after))
                history = self._frame_hashes.setdefault(game_key, [])
                history.append(h)
                if len(history) > 30:
                    self._frame_hashes[game_key] = history[-30:]
            except Exception:
                pass

        # Check prediction accuracy (if effect_prediction was set)
        world_model = context.get('world_model', {})
        if not isinstance(world_model, dict):
            return

        prediction = world_model.get('effect_prediction', {})
        if not prediction or prediction.get('game_key') != game_key:
            return

        # For click actions: check if predicted changes match actual
        if 'ACTION6' in action.upper() and prediction.get('source') == 'causal_map':
            click_x = None
            click_y = None
            if isinstance(action_data, dict):
                click_x = action_data.get('x')
                click_y = action_data.get('y')

            if click_x is not None and click_y is not None:
                pos_key = f"{click_x},{click_y}"
                pos_pred = prediction.get('predictions', {}).get(pos_key)
                if pos_pred:
                    self._total_predictions[game_key] = (
                        self._total_predictions.get(game_key, 0) + 1
                    )
                    # Prediction correct if frame changed and expected changes exist
                    expected = pos_pred.get('expected_changes', [])
                    if expected and frame_changed:
                        self._correct_predictions[game_key] = (
                            self._correct_predictions.get(game_key, 0) + 1
                        )

        # For nav actions: check if movement succeeded (frame changed)
        elif prediction.get('source') == 'wall_map':
            self._total_predictions[game_key] = (
                self._total_predictions.get(game_key, 0) + 1
            )
            if frame_changed:
                self._correct_predictions[game_key] = (
                    self._correct_predictions.get(game_key, 0) + 1
                )


class DistanceGuidedClickRung(DecisionRung):
    """H46: Click selection via color-group distance tracking -- EXPLOITATION.

    For click-action games: detect colored object groups, compute
    their distance to matching groups in the goal state, and select
    clicks that historically reduced total distance. Greedy hill-climbing
    using the solver's distance heuristic.

    Game-agnostic: works for any click game where the goal state has
    colored objects that need to match the current state's arrangement.
    """
    name = "distance_guided_click"
    category = "exploitation"
    default_priority = 29
    confidence_threshold = 0.5

    # Background color threshold -- colors with >30% of pixels are background
    BG_RATIO = 0.30

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # game_key -> {click_pos -> [{'reduction': float}, ...]}
        self._click_history: Dict[str, Dict[Tuple[int, int], List[Dict]]] = {}
        # game_key -> last distance before click
        self._last_distance: Dict[str, float] = {}
        # game_key -> last click position
        self._last_click_pos: Dict[str, Optional[Tuple[int, int]]] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()
        # Skip if directional actions also available (maze game, not click game)
        if any(a in [1, 2, 3, 4] for a in available if isinstance(a, int)):
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        frame = _get_frame_2d(game_state)
        if frame is None:
            return RungResult()

        # Get goal state from solver knowledge or context
        goal_state = self._get_goal(context)
        if not goal_state:
            return RungResult()

        # Compute current distance to goal
        current_distance = self._compute_pixel_distance(frame, goal_state)
        self._last_distance[game_key] = current_distance

        if current_distance == 0:
            return RungResult()  # Already at goal

        # Select click based on learned reduction history
        history = self._click_history.get(game_key, {})
        best_click = None
        best_avg_reduction = 0.0

        for pos, records in history.items():
            if not records:
                continue
            avg_reduction = sum(r['reduction'] for r in records) / len(records)
            if avg_reduction > best_avg_reduction:
                best_click = pos
                best_avg_reduction = avg_reduction

        if best_click and best_avg_reduction > 0:
            return RungResult(
                action='ACTION6',
                confidence=0.65,
                reason=f"H46: Distance-guided click at {best_click} "
                       f"(avg reduction {best_avg_reduction:.1f}, "
                       f"dist={current_distance:.0f})",
                metadata={
                    'x': best_click[0],
                    'y': best_click[1],
                    'source': 'distance_guided_click',
                    'distance': current_distance,
                }
            )

        # Fallback: try known click positions from causal_map
        causal = self._get_causal_positions(context)
        if causal:
            # Pick an untried position
            untried = [p for p in causal if p not in history]
            if untried:
                pick = untried[0]
                return RungResult(
                    action='ACTION6',
                    confidence=0.50,
                    reason=f"H46: Exploring causal click at {pick} "
                           f"(dist={current_distance:.0f})",
                    metadata={
                        'x': pick[0],
                        'y': pick[1],
                        'source': 'distance_guided_explore',
                        'distance': current_distance,
                    }
                )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any],
    ) -> None:
        """Track distance changes after each click."""
        if action != 'ACTION6':
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        goal_state = self._get_goal(context)
        if not goal_state:
            return

        click_pos = (action_data.get('x', 0), action_data.get('y', 0))

        # Compute distance after click
        frame_after_2d = None
        if frame_after is not None:
            try:
                import numpy as np
                arr = np.array(frame_after)
                if arr.ndim == 3:
                    arr = arr.squeeze()
                frame_after_2d = arr.tolist()
            except Exception:
                if isinstance(frame_after, list):
                    frame_after_2d = frame_after

        if frame_after_2d is None:
            return

        dist_after = self._compute_pixel_distance(frame_after_2d, goal_state)
        dist_before = self._last_distance.get(game_key, dist_after)

        reduction = dist_before - dist_after

        if game_key not in self._click_history:
            self._click_history[game_key] = {}
        if click_pos not in self._click_history[game_key]:
            self._click_history[game_key][click_pos] = []
        self._click_history[game_key][click_pos].append({
            'reduction': reduction,
        })

        # Keep only last 10 records per position
        records = self._click_history[game_key][click_pos]
        if len(records) > 10:
            self._click_history[game_key][click_pos] = records[-10:]

        self._last_distance[game_key] = dist_after

    @staticmethod
    def _get_goal(context: Dict[str, Any]) -> Optional[Dict]:
        """Get goal state from solver knowledge or context."""
        wm = context.get('world_model', {})
        if isinstance(wm, dict):
            goal = wm.get('constraint_goal_state')
            if goal:
                return goal
            goal = wm.get('goal_state')
            if goal:
                return goal
        return context.get('goal_state')

    @staticmethod
    def _get_causal_positions(context: Dict[str, Any]) -> List[Tuple[int, int]]:
        """Extract known click positions from causal_map."""
        wm = context.get('world_model', {})
        if not isinstance(wm, dict):
            return []
        causal = wm.get('causal_map', {})
        positions = []
        for key in causal:
            if isinstance(key, str) and ',' in key:
                try:
                    parts = key.split(',')
                    positions.append((int(parts[0].strip()), int(parts[1].strip())))
                except (ValueError, IndexError):
                    pass
        return positions

    @staticmethod
    def _compute_pixel_distance(
        frame: List[List[int]],
        goal_state: Dict,
    ) -> float:
        """Compute sum of mismatching pixels between frame and goal state.

        Game-agnostic: counts how many goal-state positions differ from
        the current frame. Lower = closer to goal.
        """
        mismatches = 0
        for key, target_val in goal_state.items():
            # Parse position from string key "x,y" or tuple (x, y)
            if isinstance(key, str):
                try:
                    parts = key.strip('()').split(',')
                    x, y = int(parts[0].strip()), int(parts[1].strip())
                except (ValueError, IndexError):
                    continue
            elif isinstance(key, tuple) and len(key) == 2:
                x, y = int(key[0]), int(key[1])
            else:
                continue

            if 0 <= y < len(frame) and 0 <= x < len(frame[0]):
                current = int(frame[y][x])
                if current != int(target_val):
                    mismatches += 1

        return float(mismatches)


# Registry of rungs in this module
RUNGS = {
    'discovery_exploitation': DiscoveryExploitationRung,
    'embedding_suggestion': EmbeddingSuggestionRung,
    'network_wisdom': NetworkWisdomRung,
    'frontier_topology': FrontierTopologyRung,
    'map_intel_collision': MapIntelCollisionRung,
    'abstraction_templates': AbstractionTemplatesRung,
    'few_shot_invariants': FewShotInvariantsRung,
    'frontier_checkpoint': FrontierCheckpointRung,
    'three_try_sequence': ThreeTrySequenceRung,
    'multi_stage_matching': MultiStageMatchingRung,
    'near_miss_analyzer': NearMissAnalyzerRung,
    'state_matching': StateMatchingRung,
    'spatial_relationship': SpatialRelationshipRung,
    'subgoal_planning': SubgoalPlanningRung,
    'visual_analyzer': VisualAnalyzerRung,
    'network_object_inventory': NetworkObjectInventoryRung,
    'click_behavior_learning': ClickBehaviorLearningRung,
    'wall_aware_navigation': WallAwareNavigationRung,
    'object_color_targeting': ObjectColorTargetingRung,
    'causal_click_mapping': CausalClickMappingRung,
    'controlled_movement_planning': ControlledMovementPlanningRung,
    'spatial_map': SpatialMapRung,
    'constraint_satisfaction': ConstraintSatisfactionRung,
    'constraint_decoder': ConstraintDecoderRung,
    'trigger_sequences': TriggerSequencesRung,
    'embedding_matcher': EmbeddingMatcherRung,
    'few_shot_relations': FewShotRelationsRung,
    'network_sharing': NetworkSharingRung,
    'primitive_suggester': PrimitiveSuggesterRung,
    'valence_goals': ValenceGoalsRung,
    'replay_learning': ReplayLearningRung,
    'completion_prediction': CompletionPredictionRung,
    'rule_transfer': RuleTransferRung,
    'game_classifier': GameClassifierRung,
    'solver_goal_extraction': SolverGoalExtractionRung,
    'goal_progress': GoalProgressRung,
    'effect_prediction': EffectPredictionRung,
    'action_outcome_verifier': ActionOutcomeVerifierRung,
    'distance_guided_click': DistanceGuidedClickRung,
}

## Filter Rungs

11 rungs for safety gates and weight modulation: DeathAvoidanceRung, BudgetAwarePlanningRung, etc.

In [ ]:
"""
Filter Rungs - Modify action weights / safety gates
===================================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


class DeathAvoidanceRung(DecisionRung):
    """Position-bucket death pattern avoidance - FILTER (modifies weights)"""
    name = "death_avoidance"
    category = "filter"
    default_priority = 15
    confidence_threshold = 0.6

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        detector = self.engines.terminal_pattern_detector
        if detector is None:
            return RungResult()

        try:
            # Get graduated weights from terminal pattern detector
            if hasattr(detector, 'get_graduated_action_weights'):
                game_type = context.get('game_type', '')
                level = context.get('level', 1)
                position = context.get('position', (0, 0))
                frontier_mode = context.get('frontier_mode', False)

                weights = detector.get_graduated_action_weights(
                    game_type=game_type,
                    level=level,
                    position=position,
                    frontier_mode=frontier_mode
                )

                # Find most dangerous action
                min_weight = min(weights.values()) if weights else 1.0
                dangerous_actions = [a for a, w in weights.items() if w < 0.3]

                return RungResult(
                    confidence=0.7 if dangerous_actions else 0.1,
                    reason=f"Danger weights calculated, {len(dangerous_actions)} risky actions",
                    weights=weights,
                    metadata={'dangerous_actions': dangerous_actions, 'min_weight': min_weight}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Death avoidance failed: {e}")


class PriorLessonsRung(DecisionRung):
    """Apply prior game lessons as graduated action weights - FILTER

    Converts lessons from game_lessons_learned table into safety weights.
    Lessons with caused_death=True heavily penalize their key_action.
    Lessons from wins boost their key_action.

    This closes the "last mile" gap where lessons were collected but never
    used in action selection.
    """
    name = "prior_lessons"
    category = "filter"
    default_priority = 16  # Right after death_avoidance (15)
    confidence_threshold = 0.3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            # Get prior lessons from context (loaded by evolution runner)
            prior_lessons = context.get('prior_lessons', [])
            if not prior_lessons:
                return RungResult()

            # Initialize weights at 1.0 for AVAILABLE actions only
            weights = get_available_action_weights(context, 1.0)
            lessons_applied = 0

            for idx, lesson in enumerate(prior_lessons[:10]):  # Max 10 lessons
                key_action = lesson.get('key_action', '')
                if not key_action or not key_action.startswith('ACTION'):
                    continue

                confidence = lesson.get('confidence', 0.5)
                caused_death = lesson.get('caused_death', False)
                from_win = lesson.get('from_win', False)
                severity = lesson.get('severity', 1)
                occurrence = lesson.get('occurrence_count', 1)

                # Recency factor: earlier lessons in list are more recent/salient
                recency_factor = 1.0 - (idx * 0.05)

                if caused_death:
                    # Death lessons reduce weight significantly
                    # Formula: severity (1-3) * confidence * recency
                    penalty = min(0.9, severity * 0.25 * confidence * recency_factor)
                    weights[key_action] *= max(0.05, 1.0 - penalty)
                    lessons_applied += 1
                elif from_win:
                    # Win lessons boost the action
                    boost = min(0.5, 0.15 * confidence * recency_factor * min(occurrence, 5))
                    weights[key_action] = min(1.5, weights[key_action] * (1.0 + boost))
                    lessons_applied += 1
                else:
                    # Neutral lessons: slight penalty for failures
                    penalty = min(0.3, 0.1 * confidence * recency_factor)
                    weights[key_action] *= max(0.7, 1.0 - penalty)
                    lessons_applied += 1

            if lessons_applied == 0:
                return RungResult()

            # Find penalized actions
            penalized = [a for a, w in weights.items() if w < 0.7]
            boosted = [a for a, w in weights.items() if w > 1.0]

            # Genuine epistemic resolution: we know if we've encountered this before
            resolved = ['have_we_seen_this_before']
            death_lessons = [l for l in prior_lessons[:10] if l.get('caused_death')]
            if death_lessons:
                resolved.append('known_death_patterns')
            win_lessons = [l for l in prior_lessons[:10] if l.get('from_win')]
            if win_lessons:
                resolved.append('known_win_patterns')

            return RungResult(
                confidence=min(0.8, 0.3 + lessons_applied * 0.05),
                reason=f"Prior lessons: {lessons_applied} applied, {len(penalized)} penalized, {len(boosted)} boosted",
                weights=weights,
                metadata={
                    'lessons_applied': lessons_applied,
                    'penalized_actions': penalized,
                    'boosted_actions': boosted
                },
                resolved_questions=resolved,
            )
        except Exception as e:
            return RungResult(reason=f"Prior lessons failed: {e}")


class ThreeLayerFilterRung(DecisionRung):
    """Meta-learning filter preventing wasted actions - FILTER

    Implements three filtering layers using context and prior lessons:
    Layer 1: Failed action cache - penalize recently failed actions
    Layer 2: Object prefilter - penalize click actions with no valid target
    Layer 3: Pattern prediction - penalize actions with low success history
    """
    name = "three_layer_filter"
    category = "filter"
    default_priority = 55
    confidence_threshold = 0.0  # Modifies weights, doesn't suggest

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            weights: Dict[str, float] = {}
            frame = _get_frame(game_state)
            if frame is not None and hasattr(frame, 'tolist'):
                frame = frame.tolist()
            position = context.get('position', (0, 0))
            # Ensure position is a tuple of ints
            if hasattr(position, '__iter__') and not isinstance(position, str):
                position = tuple(int(p) for p in position[:2]) if len(list(position)) >= 2 else (0, 0)
            else:
                position = (0, 0)
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])

            # Layer 1: Cache check - penalize recently failed actions
            # Uses 'failed_actions' from context (set by evolution_runner)
            failed_actions = context.get('failed_actions', set())
            recent_actions = context.get('recent_actions', [])

            for i in available:
                action = f'ACTION{i}'
                if action in failed_actions:
                    weights[action] = 0.1  # Heavily penalize failed actions
                else:
                    weights[action] = 1.0

            # Layer 2: Object prefilter for click actions
            # Penalize ACTION5/6/7 if there's no non-background pixel at position.
            # Skip for games where click coordinates are set by
            # Action6CoordinateProvider (independent of agent position).
            game_id = context.get('game_id', '')
            game_type = game_id[:4] if len(game_id) >= 4 else ''
            position_independent_click = game_type in ('ft09', 'vc33')

            if frame is not None and isinstance(frame, list) and not position_independent_click:
                for action_num in [5, 6, 7]:
                    if action_num in available:
                        action = f'ACTION{action_num}'
                        # Check if there's something to click at current position
                        y, x = position if len(position) >= 2 else (0, 0)
                        has_object = False

                        # Check 3x3 region around position
                        for dy in range(-1, 2):
                            for dx in range(-1, 2):
                                check_y, check_x = y + dy, x + dx
                                if (0 <= check_y < len(frame) and
                                    0 <= check_x < len(frame[0]) and
                                    frame[check_y][check_x] > 0):
                                    has_object = True
                                    break
                            if has_object:
                                break

                        if not has_object:
                            weights[action] = weights.get(action, 1.0) * 0.3

            # Layer 3: Pattern prediction using prior lessons
            prior_lessons = context.get('prior_lessons', [])
            for lesson in prior_lessons[:5]:
                key_action = lesson.get('key_action', '')
                if key_action in weights and lesson.get('caused_death', False):
                    severity = lesson.get('severity', 1)
                    weights[key_action] = weights.get(key_action, 1.0) * max(0.2, 1.0 - severity * 0.2)

            penalized_count = sum(1 for w in weights.values() if w < 1.0)
            if penalized_count > 0:
                return RungResult(
                    confidence=0.3,
                    weights=weights,
                    reason=f"3-layer filter applied: {penalized_count} actions penalized",
                    metadata={'filter_weights': weights}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Three-layer filter failed: {e}")


class PariahAvoidanceRung(DecisionRung):
    """Avoid actions that historically led to failures - FILTER"""
    name = "pariah_avoidance"
    category = "filter"
    default_priority = 17
    confidence_threshold = 0.0  # Modifies weights

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        vpe = self.engines.viral_package_engine
        if vpe is None:
            return RungResult()

        try:
            agent_id = context.get('agent_id', '')
            game_id = context.get('game_id', '')
            level = context.get('level', 1)
            role = context.get('agent_role', 'generalist')

            if not agent_id:
                return RungResult()

            # Role-adjusted penalty multipliers
            role_multipliers = {
                'pioneer': 0.3,
                'optimizer': 1.0,
                'generalist': 0.7,
                'exploiter': 0.5
            }
            multiplier = role_multipliers.get(role, 0.7)

            # Use the correct API: get_role_adjusted_pariah_penalties or get_pariah_action_penalties
            if hasattr(vpe, 'get_role_adjusted_pariah_penalties'):
                penalties = vpe.get_role_adjusted_pariah_penalties(
                    agent_id=agent_id,
                    agent_role=role,
                    game_id=game_id,
                    current_level=level
                )
            elif hasattr(vpe, 'get_pariah_action_penalties'):
                penalties = vpe.get_pariah_action_penalties(
                    agent_id=agent_id,
                    game_id=game_id,
                    current_level=level
                )
            else:
                return RungResult()

            if not penalties:
                return RungResult()

            # Convert penalties to weights (penalty -> weight inversion)
            weights: Dict[str, float] = {}
            for action_num, penalty in penalties.items():
                action = f'ACTION{action_num}'
                # Apply role multiplier to penalty, then convert to weight
                adjusted_penalty = penalty * multiplier
                weights[action] = max(0.05, 1.0 - min(0.95, adjusted_penalty))

            if weights:
                return RungResult(
                    confidence=0.4,
                    weights=weights,
                    reason=f"Pariah avoidance: {len(penalties)} actions penalized, role={role}",
                    metadata={'penalties': len(penalties), 'role_multiplier': multiplier}
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Pariah avoidance failed: {e}")


class TerminalPatternRung(DecisionRung):
    """Recognize approaching terminal states and avoid fatal action - FILTER"""
    name = "terminal_pattern"
    category = "filter"
    default_priority = 14
    confidence_threshold = 0.7

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        tpd = self.engines.terminal_pattern_detector
        if tpd is None:
            return RungResult()

        try:
            frame = _get_frame(game_state)

            if hasattr(tpd, 'detect_terminal_approach'):
                terminal = tpd.detect_terminal_approach(frame, context.get('last_actions', []))
                if terminal.get('approaching_terminal', False):
                    fatal_action = terminal.get('fatal_action')
                    weights = get_available_action_weights(context, 1.0)
                    if fatal_action and fatal_action in weights:
                        weights[fatal_action] = 0.05  # Near-block the fatal action
                    return RungResult(
                        confidence=0.75,
                        weights=weights,
                        reason=f"Terminal approach detected: avoid {fatal_action}",
                        metadata={'terminal': terminal}
                    )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Terminal pattern failed: {e}")


class DestructiveActionDetectionRung(DecisionRung):
    """Detect and penalize irreversible/destructive actions - FILTER

    For games like VC33 where:
    - Platform shrinking is one-directional (irreversible)
    - Random clicking gradually destroys the level state
    - Some actions reduce the number of interactive objects

    This rung:
    1. Track "entropy" of the game state over time
    2. Detect when actions DECREASE the number of objects or increase uniformity
    3. Penalize actions that historically produce entropy increase
    4. Suggest the system be more conservative with clicks

    ROOT CAUSE ADDRESSED: VC33 random clicking gradually destroys level
    state without recovery. The agent doesn't know which clicks are
    destructive vs productive.
    """
    name = "destructive_action_detection"
    category = "filter"
    default_priority = 16
    confidence_threshold = 0.3

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Track state complexity over time: game_key -> [complexity_score, ...]
        self._complexity_history: Dict[str, List[float]] = {}
        # Track which click positions caused complexity decrease
        self._destructive_positions: Dict[str, Set[Tuple[int, int]]] = {}
        # Count of destructive actions detected
        self._destruction_count: Dict[str, int] = {}
        # Total clicks tracked
        self._total_clicks: Dict[str, int] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        available = context.get('available_actions', [])
        if 6 not in available and 'ACTION6' not in available:
            return RungResult()

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        total = self._total_clicks.get(game_key, 0)
        destructive = self._destruction_count.get(game_key, 0)

        if total < 5:
            return RungResult()  # Not enough data

        destruction_rate = destructive / max(total, 1)
        destructive_positions = self._destructive_positions.get(game_key, set())

        # If destruction rate is high, apply penalty weights
        if destruction_rate > 0.3:
            # Reduce confidence in all click actions
            weights: Dict[str, float] = {}
            for a in available:
                action_name = f'ACTION{a}' if isinstance(a, int) else a
                if action_name == 'ACTION6':
                    # Penalize clicks based on destruction rate
                    weights[action_name] = max(0.2, 1.0 - destruction_rate)
                else:
                    weights[action_name] = 1.0

            return RungResult(
                weights=weights,
                reason=f"Destructive action detection: {destruction_rate:.0%} of clicks are destructive ({destructive}/{total})",
                metadata={
                    'destruction_rate': destruction_rate,
                    'destructive_positions': len(destructive_positions),
                }
            )

        return RungResult()

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Track whether clicks increase or decrease state complexity."""
        if action != 'ACTION6':
            return
        if frame_before is None or frame_after is None:
            return

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        click_x = action_data.get('x', 0)
        click_y = action_data.get('y', 0)

        self._total_clicks[game_key] = self._total_clicks.get(game_key, 0) + 1

        # Measure complexity before and after
        complexity_before = self._measure_complexity(frame_before)
        complexity_after = self._measure_complexity(frame_after)

        if game_key not in self._complexity_history:
            self._complexity_history[game_key] = []
        self._complexity_history[game_key].append(complexity_after)

        # Detect destruction: complexity decreased significantly
        if complexity_before > 0 and complexity_after < complexity_before * 0.9:
            if game_key not in self._destructive_positions:
                self._destructive_positions[game_key] = set()
            self._destructive_positions[game_key].add((click_x // 4, click_y // 4))
            self._destruction_count[game_key] = self._destruction_count.get(game_key, 0) + 1

    @staticmethod
    def _measure_complexity(frame: Any) -> float:
        """Measure frame complexity as number of distinct non-zero colored regions."""
        try:
            colors: Set[int] = set()
            non_zero = 0
            if isinstance(frame, list):
                for row in frame:
                    for pixel in row:
                        val = int(pixel) if hasattr(pixel, '__int__') else pixel
                        if val != 0:
                            colors.add(val)
                            non_zero += 1
            else:
                import numpy as np
                arr = np.array(frame)
                non_zero_mask = arr != 0
                colors = set(int(v) for v in np.unique(arr[non_zero_mask]))
                non_zero = int(np.sum(non_zero_mask))

            return len(colors) * 10 + non_zero * 0.01
        except Exception:
            return 0.0


class BudgetAwarePlanningRung(DecisionRung):
    """Adjust behavior based on remaining action budget - FILTER

    Cross-cutting fix for all games but especially LS20 (42 moves/level):
    1. Early game (0-30% budget): Encourage exploration, tolerate failures
    2. Mid game (30-70% budget): Balance explore/exploit
    3. Late game (70-100% budget): Maximize exploitation, minimize waste
    4. Critical (>90% budget): Emergency mode - only proven actions

    Also tracks "progress per action" efficiency to detect when the agent
    is wasting its budget on unproductive actions.

    ROOT CAUSE ADDRESSED: LS20's timer pressure kills exploration.
    ~42 moves per level means every action must count. The system needs
    to shift from exploration to exploitation as budget depletes.
    """
    name = "budget_aware_planning"
    category = "filter"
    default_priority = 6
    confidence_threshold = 0.2  # Low threshold - mostly provides weights

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # Track productive vs wasted actions
        self._productive_actions: Dict[str, int] = {}
        self._total_actions: Dict[str, int] = {}

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        budget_used = context.get('budget_used_percent', 0)
        action_count = context.get('action_count', 0)
        action_budget = context.get('action_budget', 400)

        if action_budget <= 0:
            return RungResult()

        remaining_pct = 1.0 - budget_used

        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        puzzle_type = context.get('puzzle_type', 'unknown')
        game_key = f"{game_type}_L{level}"

        # Part 7.2: Deliberate experimentation mode
        # Levels 1-2: LEARNING phase - maximize information gain
        # Level 3: TRANSITIONING - start applying learned rules
        # Levels 4+: APPLYING - exploit knowledge to complete levels
        level_phase = context.get('level_phase', 'learning')
        if not level_phase or level_phase == 'learning':
            # Re-derive from level in case context didn't populate it
            if level <= 2:
                level_phase = 'learning'
            elif level == 3:
                level_phase = 'transitioning'
            else:
                level_phase = 'applying'

        # Calculate efficiency
        productive = self._productive_actions.get(game_key, 0)
        total = self._total_actions.get(game_key, 0)
        efficiency = productive / max(total, 1)

        # Phase-based behavior modification
        if remaining_pct < 0.10:
            # CRITICAL: Less than 10% budget remaining
            # Only allow actions with known positive outcomes
            # In learning phase, still don't penalize exploration
            exploration_penalty = 0.0 if level_phase == 'learning' else 0.7
            return RungResult(
                confidence=0.3,
                reason=f"CRITICAL budget: {remaining_pct:.0%} remaining, {action_count}/{action_budget} used. Efficiency: {efficiency:.0%}",
                metadata={
                    'budget_phase': 'critical',
                    'budget_remaining_pct': remaining_pct,
                    'efficiency': efficiency,
                    'confidence_boost': 0.3,  # Boost confidence of exploitation rungs
                    'level_phase': level_phase,
                    'puzzle_type': puzzle_type,
                    'exploration_penalty': exploration_penalty,
                }
            )
        elif remaining_pct < 0.30:
            # LATE: Shift strongly toward exploitation
            # In learning phase, never penalize exploration
            exploration_penalty = 0.0 if level_phase == 'learning' else 0.5
            return RungResult(
                reason=f"Late budget: {remaining_pct:.0%} remaining. Efficiency: {efficiency:.0%}",
                metadata={
                    'budget_phase': 'late',
                    'budget_remaining_pct': remaining_pct,
                    'efficiency': efficiency,
                    'exploration_penalty': exploration_penalty,
                    'level_phase': level_phase,
                    'puzzle_type': puzzle_type,
                }
            )
        elif remaining_pct < 0.70:
            # MID: Balanced
            exploration_penalty = 0.0 if level_phase == 'learning' else 0.0
            return RungResult(
                reason=f"Mid budget: {remaining_pct:.0%} remaining. Efficiency: {efficiency:.0%}",
                metadata={
                    'budget_phase': 'mid',
                    'budget_remaining_pct': remaining_pct,
                    'efficiency': efficiency,
                    'level_phase': level_phase,
                    'puzzle_type': puzzle_type,
                    'exploration_penalty': exploration_penalty,
                }
            )

        # EARLY: Full exploration allowed
        return RungResult(
            metadata={
                'budget_phase': 'early',
                'budget_remaining_pct': remaining_pct,
                'level_phase': level_phase,
                'puzzle_type': puzzle_type,
                'exploration_penalty': 0.0,
            }
        )

    def on_action_complete(
        self,
        action: str,
        action_data: Dict[str, Any],
        frame_before: Any,
        frame_after: Any,
        context: Dict[str, Any]
    ) -> None:
        """Track action productivity."""
        game_type = context.get('game_type', '')
        level = context.get('level', 1)
        game_key = f"{game_type}_L{level}"

        self._total_actions[game_key] = self._total_actions.get(game_key, 0) + 1

        # Count as productive if frame changed
        frame_changed = False
        try:
            if frame_before is not None and frame_after is not None:
                if isinstance(frame_before, list) and isinstance(frame_after, list):
                    frame_changed = frame_before != frame_after
                else:
                    import numpy as np
                    frame_changed = not np.array_equal(frame_before, frame_after)
        except Exception:
            pass

        if frame_changed:
            self._productive_actions[game_key] = self._productive_actions.get(game_key, 0) + 1


class TheoryContradictionRung(DecisionRung):
    """Filter actions that contradict current working theory - FILTER

    Wires: engines/cognition/metacognition.py:get_contradicted_actions()

    When metacognition's theory revision marks actions as contradicted
    (failed prediction -> action disproven), this rung applies negative
    weights to those actions, preventing repeated mistakes.
    """
    name = "theory_contradiction"
    category = "filter"
    default_priority = 17  # After death_avoidance (15), before hypothesis
    confidence_threshold = 0.3

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        me = self.engines.metacognitive_engine
        if me is None:
            return RungResult()

        try:
            # Get actions contradicted by failed theories
            if not hasattr(me, 'get_contradicted_actions'):
                return RungResult()

            contradicted = me.get_contradicted_actions()

            if not contradicted:
                return RungResult()

            # Build penalty weights for contradicted actions (available only)
            weights = get_available_action_weights(context, 1.0)
            penalized = []

            for action_str, contradiction_count in contradicted.items():
                if action_str in weights:
                    # More contradictions = stronger penalty
                    # 1 contradiction = 0.7, 2 = 0.5, 3+ = 0.3
                    penalty = min(0.7, 0.2 * contradiction_count)
                    weights[action_str] = max(0.3, 1.0 - penalty)
                    penalized.append(f"{action_str}({contradiction_count})")

            if not penalized:
                return RungResult()

            return RungResult(
                confidence=0.5,
                reason=f"Theory contradictions: {', '.join(penalized)}",
                weights=weights,
                metadata={'contradicted_actions': contradicted}
            )
        except Exception as e:
            return RungResult(reason=f"Theory contradiction failed: {e}")


class ViralPackageWeightsRung(DecisionRung):
    """Apply action weights from viral information packages - FILTER

    Wires: engines/social/viral_package_engine.py:get_package_action_weights()

    Viral packages carry knowledge from winning strategies that spread
    across the agent population. Each package contains action sequences
    weighted by success rate, infection strength, and emotional compatibility.

    This rung reads those packages and converts them to action weights,
    closing the loop: winning sequences -> viral packages -> action decisions.

    Data flow:
        winning_sequences -> viral_information_packages (post-game)
        -> agent_viral_infections (horizontal transfer)
        -> get_package_action_weights() (this rung reads)
        -> action weights applied to decision
    """
    name = "viral_package_weights"
    category = "filter"
    default_priority = 20  # After pariah_avoidance (19), before hypothesis
    confidence_threshold = 0.0  # Modifies weights, not a direct action

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        vpe = self.engines.viral_package_engine
        if vpe is None:
            return RungResult()

        try:
            agent_id = context.get('agent_id', '')
            if not agent_id:
                return RungResult()

            if not hasattr(vpe, 'get_package_action_weights'):
                return RungResult()

            generation = context.get('generation', 0)
            raw_weights = vpe.get_package_action_weights(
                agent_id=agent_id,
                generation=generation,
                track_retrieval=True,
            )

            if not raw_weights:
                return RungResult()

            # Convert int action keys to ACTION strings and normalize
            # get_package_action_weights returns {action_int: weight_float}
            weights: Dict[str, float] = {}
            max_weight = max(raw_weights.values()) if raw_weights else 1.0
            boosted_actions = []

            for action_num, weight in raw_weights.items():
                action = f'ACTION{action_num}'
                # Validate action is available in this game
                if not is_action_available(action, context):
                    continue
                # Normalize to 0.5-1.5 range (boost, not replace)
                normalized = 0.5 + (weight / max(max_weight, 0.001))
                weights[action] = min(1.5, normalized)
                if normalized > 1.0:
                    boosted_actions.append(f"{action}({normalized:.2f})")

            if weights:
                return RungResult(
                    confidence=0.35,
                    weights=weights,
                    reason=f"Viral packages: {len(raw_weights)} actions weighted, {len(boosted_actions)} boosted",
                    metadata={
                        'package_count': len(raw_weights),
                        'boosted': boosted_actions[:5],
                    }
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Viral package weights failed: {e}")


class MetacognitiveEliminationRung(DecisionRung):
    """Penalize actions that metacognition has systematically eliminated - FILTER

    Wires: engines/cognition/metacognition.py:get_eliminated_actions()

    The metacognitive engine tracks actions that consistently fail for a
    given game/level (e.g., ACTION3 always leads to death on level 2 of ft09).
    These eliminations are stored in metacognitive_eliminations table with
    confidence scores.

    This rung reads those eliminations and applies heavy weight penalties,
    effectively steering the agent away from provably bad actions without
    hard-blocking them (in case context has changed).

    Unlike contextual_failure (position-aware), this is GLOBAL elimination
    per game_type + level.

    Data flow:
        gameplay failures -> metacognitive_eliminations (recorded by MetacognitiveReasoningEngine)
        -> get_eliminated_actions() (this rung reads)
        -> heavy weight penalties on eliminated actions
    """
    name = "metacognitive_elimination"
    category = "filter"
    default_priority = 16  # Right with death_avoidance
    confidence_threshold = 0.0  # Modifies weights

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        me = self.engines.metacognitive_engine
        if me is None:
            return RungResult()

        try:
            if not hasattr(me, 'get_eliminated_actions'):
                return RungResult()

            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            if not game_type:
                return RungResult()

            eliminated = me.get_eliminated_actions(
                game_type=game_type,
                level_number=level,
                min_confidence=0.6,
            )

            if not eliminated:
                return RungResult()

            # Build penalty weights for eliminated actions (available only)
            weights = get_available_action_weights(context, 1.0)
            penalized = []

            for action_str in eliminated:
                # Normalize action format
                if not action_str.startswith('ACTION'):
                    action_str = f'ACTION{action_str}'
                if action_str in weights:
                    # Heavy penalty: 0.1 weight (nearly eliminated but not hard-blocked)
                    weights[action_str] = 0.1
                    penalized.append(action_str)

            if penalized:
                return RungResult(
                    confidence=0.6,
                    weights=weights,
                    reason=f"Metacognitive eliminations: {', '.join(penalized)}",
                    metadata={
                        'eliminated_count': len(penalized),
                        'eliminated_actions': penalized,
                    }
                )
            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Metacognitive elimination failed: {e}")


class ContextualFailureRung(DecisionRung):
    """Track contextual failures - position/direction/object-aware - FILTER

    Unlike global action elimination, tracks CONTEXTUAL failure signatures:
    - Position region where failure occurred
    - Direction of movement (toward/away from object)
    - Nearby object types at time of failure

    This allows "ACTION3 toward wall at (3,4)" to be avoided without
    eliminating ACTION3 globally (which could deadlock all movement).

    Failure signatures decay over time (things can change).
    """
    name = "contextual_failure"
    category = "filter"
    default_priority = 14  # Before death_avoidance
    confidence_threshold = 0.3

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        # In-memory failure signatures (per game_type, per level)
        # Structure: {(game_type, level): [FailureSignature, ...]}
        self._failure_signatures: Dict[Tuple[str, int], List[Dict[str, Any]]] = {}
        self._max_signatures_per_level = 50
        self._signature_decay_rate = 0.1  # Per evaluation cycle

    def _compute_position_region(self, position: Tuple[int, int]) -> Tuple[int, int]:
        """Bucket position into regions for fuzzy matching."""
        # 3x3 region bucketing - positions (0,0), (0,1), (0,2) all -> region (0, 0)
        return (position[0] // 3, position[1] // 3)

    def _compute_movement_direction(self, action: str, nearby_objects: List[Dict]) -> str:
        """Determine if action moves toward/away/parallel to nearest object."""
        # Simplified: Map actions to directions
        action_directions = {
            'ACTION1': 'up', 'ACTION2': 'down',
            'ACTION3': 'left', 'ACTION4': 'right',
            'ACTION5': 'stay', 'ACTION7': 'special',
        }
        direction = action_directions.get(action, 'unknown')

        if not nearby_objects:
            return 'no_nearby_object'

        # For simplicity, just return the direction - full implementation would
        # compute vector from agent to nearest object and compare
        return f"{direction}_near_object"

    def record_failure(
        self,
        game_type: str,
        level: int,
        position: Tuple[int, int],
        action: str,
        nearby_objects: List[Dict[str, Any]],
        outcome: str = 'death'
    ) -> None:
        """Record a contextual failure signature."""
        key = (game_type, level)

        if key not in self._failure_signatures:
            self._failure_signatures[key] = []

        signature = {
            'game_type': game_type,
            'level': level,
            'position_region': self._compute_position_region(position),
            'action': action,
            'movement_context': self._compute_movement_direction(action, nearby_objects),
            'nearby_colors': [o.get('color') for o in nearby_objects[:3]],
            'outcome': outcome,
            'confidence': 0.8,  # Initial confidence
            'created_at': datetime.now().isoformat(),
        }

        self._failure_signatures[key].append(signature)

        # Prune old signatures
        if len(self._failure_signatures[key]) > self._max_signatures_per_level:
            # Remove lowest confidence signatures
            self._failure_signatures[key].sort(key=lambda s: s['confidence'], reverse=True)
            self._failure_signatures[key] = self._failure_signatures[key][:self._max_signatures_per_level]

    def _decay_signatures(self, key: Tuple[str, int]) -> None:
        """Apply decay to signatures, removing those below threshold."""
        if key not in self._failure_signatures:
            return

        decayed = []
        for sig in self._failure_signatures[key]:
            sig['confidence'] -= self._signature_decay_rate
            if sig['confidence'] > 0.2:  # Keep if still confident
                decayed.append(sig)

        self._failure_signatures[key] = decayed

    def _match_signature(
        self,
        action: str,
        position: Tuple[int, int],
        nearby_objects: List[Dict],
        signatures: List[Dict[str, Any]]
    ) -> Optional[Dict[str, Any]]:
        """Check if current context matches any failure signature."""
        current_region = self._compute_position_region(position)
        current_context = self._compute_movement_direction(action, nearby_objects)
        current_colors = set(o.get('color') for o in nearby_objects[:3])

        for sig in signatures:
            # Must match action
            if sig['action'] != action:
                continue

            # Must match position region
            if sig['position_region'] != current_region:
                continue

            # Check color overlap (at least one matching nearby color)
            sig_colors = set(sig.get('nearby_colors', []))
            if sig_colors and current_colors and not sig_colors.intersection(current_colors):
                continue

            # Match found
            return sig

        return None

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)
            position = context.get('agent_position', (0, 0))
            nearby_objects = context.get('nearby_objects', [])

            key = (game_type, level)

            # Apply decay each evaluation
            self._decay_signatures(key)

            signatures = self._failure_signatures.get(key, [])
            if not signatures:
                return RungResult()

            # Record failure if last action caused death/penalty
            if context.get('last_outcome') == 'death' or context.get('score_delta', 0) < 0:
                last_action = context.get('last_action')
                last_position = context.get('last_position', position)
                if last_action:
                    self.record_failure(
                        game_type=game_type,
                        level=level,
                        position=last_position,
                        action=last_action,
                        nearby_objects=nearby_objects,
                        outcome='death' if context.get('last_outcome') == 'death' else 'penalty'
                    )

            # Check each action for matching failure signatures (available only)
            weights = get_available_action_weights(context, 1.0)
            penalized_actions = []
            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])

            for action_num in available:
                action = f'ACTION{action_num}'
                match = self._match_signature(action, position, nearby_objects, signatures)

                if match:
                    # Penalty proportional to signature confidence
                    penalty = match['confidence'] * 0.5  # Max 50% weight reduction
                    weights[action] = max(0.3, 1.0 - penalty)
                    penalized_actions.append(
                        f"{action}@{match['position_region']}({match['confidence']:.1f})"
                    )

            if penalized_actions:
                return RungResult(
                    confidence=0.4,
                    reason=f"Contextual failures: {', '.join(penalized_actions[:3])}",
                    weights=weights,
                    metadata={
                        'matched_signatures': len(penalized_actions),
                        'total_signatures': len(signatures),
                    }
                )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Contextual failure check failed: {e}")


# =============================================================================
# ORDERING PRESETS (DEPRECATED - Phase 6)
# =============================================================================
#
# DEPRECATION NOTICE: Static ordering presets are deprecated as of Phase 6.
# Use DecisionStrategy.COGNITIVE with CognitiveRouter for dynamic selection.
# These presets will be removed in v3.0.
#
# Migration path:
#   1. Switch strategy to COGNITIVE: DecisionRungSystem(strategy='cognitive')
#   2. CognitiveRouter handles graph-based rung selection automatically
#   3. Edge weights evolve based on observed outcomes
#
# For details see: architecture/cognitive_routing_implementation_plan.md
#


# Registry of rungs in this module
RUNGS = {
    'death_avoidance': DeathAvoidanceRung,
    'prior_lessons': PriorLessonsRung,
    'three_layer_filter': ThreeLayerFilterRung,
    'pariah_avoidance': PariahAvoidanceRung,
    'terminal_pattern': TerminalPatternRung,
    'destructive_action_detection': DestructiveActionDetectionRung,
    'budget_aware_planning': BudgetAwarePlanningRung,
    'theory_contradiction': TheoryContradictionRung,
    'viral_package_weights': ViralPackageWeightsRung,
    'metacognitive_elimination': MetacognitiveEliminationRung,
    'contextual_failure': ContextualFailureRung,
}

## Exploration & Fallback Rungs

Systematic search and smart random fallback: SmartActionSelectionRung

In [ ]:
"""
Exploration & Fallback Rungs - Systematic search and defaults
=============================================================
Extracted from decision_rung_system.py Phase 4.2.
"""


logger = logging.getLogger(__name__)


def _get_frame(game_state: Any) -> Any:
    """Extract frame from game_state whether it's a dict or object."""
    if isinstance(game_state, dict):
        return game_state.get('frame')
    return getattr(game_state, 'frame', None)


class SmartActionSelectionRung(DecisionRung):
    """Fallback: strategy-based random selection - FALLBACK"""
    name = "smart_action_selection"
    category = "fallback"
    default_priority = 99  # Always last
    confidence_threshold = 0.0  # Always provides answer

    def _get_action6_coordinates(self) -> Dict[str, int]:
        """Get coordinates for ACTION6 from visual_analyzer or random fallback."""
        va = self.engines.visual_analyzer if self.engines else None
        if va and hasattr(va, 'get_grid_exploration_targets'):
            targets = va.get_grid_exploration_targets()
            if targets:
                target = targets[0]
                return {'x': target.get('x', 32), 'y': target.get('y', 32), 'grid_target': target}
        # Fallback: random position
        return {'x': random.randint(4, 60), 'y': random.randint(4, 60)}

    def _avoid_tried_positions(
        self, coords: Dict[str, int], context: Dict[str, Any], max_attempts: int = 10
    ) -> Dict[str, int]:
        """Part 7.2: During learning phase, avoid re-clicking previously tried positions.

        Reads the world_model action_history to find positions already clicked,
        then picks a different random position if the proposed one was already tried.
        Falls back to the original coords if no novel position can be found.
        """
        wm = context.get('world_model') or {}
        history = wm.get('action_history', [])
        tried_positions: set = set()
        for entry in history[-30:]:
            if isinstance(entry, dict) and 'x' in entry and 'y' in entry:
                tried_positions.add((entry['x'], entry['y']))

        if not tried_positions:
            return coords

        # If proposed position already tried, try to find a novel one
        proposed = (coords.get('x', 32), coords.get('y', 32))
        if proposed not in tried_positions:
            return coords

        for _ in range(max_attempts):
            x = random.randint(4, 60)
            y = random.randint(4, 60)
            if (x, y) not in tried_positions:
                return {'x': x, 'y': y}

        # Couldn't find novel position; return original
        return coords

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        try:
            # Part 7.2: Level-aware strategy - learning phases use exploration,
            # applying phases use exploitation, otherwise use fallback_strategy
            level_phase = context.get('level_phase', '')
            if level_phase == 'learning':
                strategy = 'exploration'
            elif level_phase == 'applying':
                strategy = 'exploitation'
            else:
                strategy = context.get('fallback_strategy', 'balanced')

            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            available_strs = {f'ACTION{a}' for a in available}

            if strategy == 'exploration':
                all_weights = {'ACTION1': 1.2, 'ACTION2': 1.2, 'ACTION3': 1.2, 'ACTION4': 1.2,
                          'ACTION5': 0.5, 'ACTION6': 1.0, 'ACTION7': 0.3}
            elif strategy == 'exploitation':
                all_weights = {'ACTION1': 0.8, 'ACTION2': 0.8, 'ACTION3': 0.8, 'ACTION4': 0.8,
                          'ACTION5': 1.5, 'ACTION6': 1.2, 'ACTION7': 1.0}
            else:  # balanced
                all_weights = get_available_action_weights(context, 1.0)

            # Filter to only available actions
            weights = {k: v for k, v in all_weights.items() if k in available_strs}
            if not weights:
                weights = get_available_action_weights(context, 1.0)

            # Weighted random choice
            total = sum(weights.values())
            r = random.random() * total
            cumulative = 0
            for action, weight in weights.items():
                cumulative += weight
                if r <= cumulative:
                    # Add coordinates if ACTION6
                    metadata: Dict[str, Any] = {'strategy': strategy, 'level_phase': level_phase}
                    if action == 'ACTION6':
                        coords = self._get_action6_coordinates()
                        # Part 7.2: During learning phase, avoid re-clicking same positions
                        if level_phase == 'learning':
                            coords = self._avoid_tried_positions(coords, context)
                        metadata.update(coords)
                    return RungResult(
                        action=action,
                        confidence=0.1,
                        reason=f"Fallback ({strategy}): {action}",
                        weights=weights,
                        metadata=metadata
                    )

            fallback_action = get_random_available_action(context)
            # Add coordinates if ACTION6
            metadata = {'strategy': 'ultimate_fallback', 'level_phase': level_phase}
            if fallback_action == 'ACTION6':
                metadata.update(self._get_action6_coordinates())
            return RungResult(action=fallback_action, confidence=0.1, reason="Ultimate fallback", metadata=metadata)
        except Exception as e:
            fallback_action = get_random_available_action(context)
            metadata = {'error': str(e)}
            if fallback_action == 'ACTION6':
                metadata.update({'x': random.randint(4, 60), 'y': random.randint(4, 60)})
            return RungResult(action=fallback_action, confidence=0.1, reason=f"Fallback error: {e}", metadata=metadata)


class Action6ObjectExplorationRung(DecisionRung):
    """
    Use Action6BehaviorEngine to find clickable objects - EXPLORATION

    This rung uses the sophisticated pseudobutton/object selection system to:
    1. Find objects in the current frame that match known selectable shapes
    2. Prioritize unexplored objects for frontier exploration
    3. Return specific click coordinates for ACTION6

    This is critical for ACTION6-only games like vc33.

    CONFIDENCE DECAY (2025-01-13):
    Tracks recent (x,y) positions per game. When the same coordinates are
    produced repeatedly, confidence decays so the cognitive router yields
    to other rungs instead of committing to stale exploration.

    ANTI-MONOPOLY (2026-02-10):
    Per-game evaluation counter decays confidence by 0.02 per evaluate() call,
    regardless of coordinate diversity. This prevents the rung from holding
    98% of all actions by cycling through objects -- after ~15 evaluations,
    confidence drops low enough for exploitation rungs to compete.
    Resets on level change (progress = fresh exploration budget).
    """
    name = "action6_object_exploration"
    category = "exploration"
    default_priority = 38  # Higher priority than GridExplorationRung (47)
    confidence_threshold = 0.35

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self._consecutive_no_change = 0
        self._position_history: List[Tuple[int, int]] = []
        self._HISTORY_WINDOW = 8
        # Per-game evaluation counter: decays confidence even when coordinates
        # are diverse (prevents monopoly through object cycling).
        self._game_eval_count: Dict[str, int] = {}
        self._last_game_level: Optional[str] = None

    def _get_repetition_decay(self, x: int, y: int, proximity: int = 4) -> float:
        """
        Calculate confidence decay based on positional repetition.

        Returns a decay value (0.0 = no decay, up to ~0.50 for heavy repetition).
        Positions within ``proximity`` pixels of each other count as repeats.
        """
        if not self._position_history:
            return 0.0

        recent = self._position_history[-6:]
        repeat_count = sum(
            1 for px, py in recent
            if abs(px - x) <= proximity and abs(py - y) <= proximity
        )

        return min(0.50, repeat_count * 0.10)

    def _record_position(self, x: int, y: int) -> None:
        """Record a position in the history."""
        self._position_history.append((x, y))
        if len(self._position_history) > self._HISTORY_WINDOW:
            self._position_history = self._position_history[-self._HISTORY_WINDOW:]

    def _should_abstain(self, x: int, y: int, proximity: int = 4) -> bool:
        """Abstain if last 3+ recorded positions are all near proposed coords.

        This uses position history only -- no dependency on on_action_complete.
        After clicking the same spot 3 times, the rung yields entirely so the
        router's coordinate fallback can try different positions.
        """
        if len(self._position_history) < 3:
            return False
        recent = self._position_history[-3:]
        return all(
            abs(px - x) <= proximity and abs(py - y) <= proximity
            for px, py in recent
        )

    def evaluate(self, game_state: Any, context: Dict[str, Any]) -> RungResult:
        # Check if ACTION6 is available
        available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
        if 6 not in available:
            return RungResult()

        # Get the action6_behavior engine
        a6e = self.engines.action6_behavior
        if a6e is None:
            return RungResult()

        try:
            game_type = context.get('game_type', '')
            level = context.get('level', 1)

            # Get current frame from game_state
            frame = _get_frame(game_state)

            if frame is None:
                return RungResult()
            # Convert numpy to list for safe iteration
            if hasattr(frame, 'tolist'):
                frame = frame.tolist()
            if not isinstance(frame, list) or len(frame) == 0:
                return RungResult()

            # ANTI-MONOPOLY: Per-game eval counter -- decays confidence
            # even when coordinates are diverse (object cycling).
            game_key = f"{game_type}_L{level}"
            eval_count = self._game_eval_count.get(game_key, 0)
            self._game_eval_count[game_key] = eval_count + 1

            # Reset counter on level change (same game, new level = progress)
            if self._last_game_level and self._last_game_level != game_key:
                old_type = self._last_game_level.split('_L')[0]
                if old_type == game_type:
                    self._game_eval_count[game_key] = 0
                    eval_count = 0
            self._last_game_level = game_key

            # Cold-start protection for ACTION6-only games: suppress eval
            # decay during first 20 evaluations. For click-only games there
            # is no alternative to clicking, so decaying click confidence to
            # let non-click rungs win is counterproductive.
            available = context.get('available_actions', [])
            is_click_only = (list(available) == [6]) if available else False
            cold_start_window = 20
            if is_click_only and eval_count < cold_start_window:
                eval_decay = 0.0
            else:
                eval_decay = eval_count * 0.02

            # First try: Get objects matching known selectable shapes for this game
            if hasattr(a6e, 'get_untried_objects_for_frontier'):
                tried_colors = context.get('tried_colors', [])
                objects = a6e.get_untried_objects_for_frontier(
                    game_type=game_type,
                    level=level,
                    frame=frame,
                    tried_colors=tried_colors
                )
                if objects:
                    obj = objects[0]  # Highest confidence match
                    # Get center coordinates of the object
                    x = obj.get('center_x', obj.get('x', 32))
                    y = obj.get('center_y', obj.get('y', 32))
                    if self._should_abstain(x, y):
                        return RungResult(reason="Abstaining: repeated no-change at same position")
                    base_conf = min(0.50, 0.45 + obj.get('shape_confidence', 0) * 0.15)
                    decay = self._get_repetition_decay(x, y)
                    no_change_decay = self._consecutive_no_change * 0.08
                    confidence = max(0.10, base_conf - decay - no_change_decay - eval_decay)
                    self._record_position(x, y)
                    decay_note = f" [decay={decay:.2f}+nc={no_change_decay:.2f}+ev={eval_decay:.2f}]" if (decay > 0 or no_change_decay > 0 or eval_decay > 0) else ""
                    return RungResult(
                        action='ACTION6',
                        confidence=confidence,
                        reason=f"Object exploration: color={obj.get('color')} shape={obj.get('shape_signature')} at ({x},{y}){decay_note}",
                        metadata={
                            'x': x,
                            'y': y,
                            'target_object': obj,
                            'source': 'shape_matching',
                            'repetition_decay': decay
                        }
                    )

            # Second try: Get known pseudo-buttons for this game/level
            if hasattr(a6e, 'get_all_pseudo_buttons'):
                buttons = a6e.get_all_pseudo_buttons(game_type, level)
                if buttons:
                    # Find highest-confidence button that produces useful action
                    for button in buttons:
                        if button.get('confidence', 0) >= 0.5:
                            # Region coords are 0-7, convert to pixel coords (center of 8x8 region)
                            region_x = button.get('region_x', 4)
                            region_y = button.get('region_y', 4)
                            x = region_x * 8 + 4  # Center of region
                            y = region_y * 8 + 4
                            if self._should_abstain(x, y):
                                continue  # Skip this button, try next
                            base_conf = min(0.50, 0.40 + button.get('confidence', 0) * 0.20)
                            decay = self._get_repetition_decay(x, y)
                            no_change_decay = self._consecutive_no_change * 0.08
                            confidence = max(0.10, base_conf - decay - no_change_decay - eval_decay)
                            self._record_position(x, y)
                            decay_note = f" [decay={decay:.2f}+nc={no_change_decay:.2f}+ev={eval_decay:.2f}]" if (decay > 0 or no_change_decay > 0 or eval_decay > 0) else ""
                            return RungResult(
                                action='ACTION6',
                                confidence=confidence,
                                reason=f"Pseudo-button at region ({region_x},{region_y}) -> ({x},{y}){decay_note}",
                                metadata={
                                    'x': x,
                                    'y': y,
                                    'pseudo_button': button,
                                    'source': 'pseudo_button',
                                    'repetition_decay': decay
                                }
                            )

            # Third try: Get selectable objects from network knowledge
            if hasattr(a6e, 'get_selectable_objects'):
                objects = a6e.get_selectable_objects(game_type, level, min_confidence=0.4)
                if objects:
                    obj = objects[0]
                    coords = obj.get('coordinates', '')
                    # Parse coordinates like "(32,45)"
                    if coords:
                        import re
                        match = re.match(r'\((\d+),(\d+)\)', coords)
                        if match:
                            x, y = int(match.group(1)), int(match.group(2))
                            if self._should_abstain(x, y):
                                return RungResult(reason="Abstaining: repeated no-change at same position")
                            base_conf = min(0.50, 0.35 + obj.get('confidence', 0) * 0.25)
                            decay = self._get_repetition_decay(x, y)
                            no_change_decay = self._consecutive_no_change * 0.08
                            confidence = max(0.10, base_conf - decay - no_change_decay - eval_decay)
                            self._record_position(x, y)
                            decay_note = f" [decay={decay:.2f}+nc={no_change_decay:.2f}+ev={eval_decay:.2f}]" if (decay > 0 or no_change_decay > 0 or eval_decay > 0) else ""
                            return RungResult(
                                action='ACTION6',
                                confidence=confidence,
                                reason=f"Selectable object color={obj.get('object_color')} at ({x},{y}){decay_note}",
                                metadata={
                                    'x': x,
                                    'y': y,
                                    'selectable_object': obj,
                                    'source': 'network_knowledge',
                                    'repetition_decay': decay
                                }
                            )

            return RungResult()
        except Exception as e:
            return RungResult(reason=f"Action6 object exploration failed: {e}")

    def on_action_complete(self, action: str,
                           action_data: Any = None,
                           frame_before: Any = None,
                           frame_after: Any = None,
                           context: Any = None,
                           **kwargs: Any) -> None:
        """Track whether actions produce frame changes.

        Signature must match DecisionRungSystem.notify_action_complete caller:
            action, action_data, frame_before, frame_after, context
        """
        try:
            if frame_before is not None and frame_after is not None:
                import numpy as np
                pre = np.array(frame_before) if not isinstance(frame_before, np.ndarray) else frame_before
                post = np.array(frame_after) if not isinstance(frame_after, np.ndarray) else frame_after
                if pre.shape == post.shape and np.array_equal(pre, post):
                    self._consecutive_no_change += 1
                else:
                    self._consecutive_no_change = 0
            else:
                # No frame data -- assume no change (conservative)
                self._consecutive_no_change += 1
        except Exception:
            self._consecutive_no_change += 1


# Registry of rungs in this module
RUNGS = {
    'smart_action_selection': SmartActionSelectionRung,
    'action6_object_exploration': Action6ObjectExplorationRung,
}

## Rung Registry

Automatically discovers all DecisionRung subclasses defined above and builds the flat registry.

In [ ]:
# -- Build Rung Registry ------------------------------------------------------
# Collect all DecisionRung subclasses defined above into a flat registry.

RUNG_REGISTRY = {}
for _cls in list(globals().values()):
    if (isinstance(_cls, type) and issubclass(_cls, DecisionRung)
            and _cls is not DecisionRung and hasattr(_cls, 'name')):
        key = _cls.name
        if key != 'base_rung' and key not in RUNG_REGISTRY:
            RUNG_REGISTRY[key] = _cls

print(f"[OK] Built RUNG_REGISTRY with {len(RUNG_REGISTRY)} rungs")

# Category breakdown
_cats = defaultdict(list)
for _name, _cls in sorted(RUNG_REGISTRY.items()):
    _cats[_cls.category].append((_name, _cls))

for _cat in ['emergency', 'orientation', 'hypothesis', 'exploitation', 'filter', 'exploration', 'fallback']:
    _rungs = _cats.get(_cat, [])
    if _rungs:
        print(f"  {_cat:15s}: {len(_rungs):2d} rungs")

## Decision Rung System -- The Orchestrator

The `DecisionRungSystem` manages rung evaluation ordering, strategy dispatch, and action selection.
Includes **13+ ordering presets** that define which rungs run and in what priority order.

### Decision Strategies

| Strategy | Description |
|----------|-------------|
| **ladder** | First confident rung wins (fast, deterministic) |
| **weighted** | All rungs vote, weighted sum decides (richer) |
| **phased** | Different orderings based on budget consumption |
| **parallel** | All rungs run, highest confidence wins |
| **cognitive** | CognitiveRouter-driven dynamic selection |
| **context_adaptive** | Auto-selects ordering based on game type + agent role |

In [ ]:
from pathlib import Path
import traceback

# Stubs for cognitive router (not available in standalone mode)
def _load_cognitive_router():
    return None
_cognitive_router_loaded = True
_CognitiveRouter = None
_RungResult_Cognitive = None

# Stubs for lazy-loaded engines (not available in standalone notebook mode)
def get_temporal_integrator(db=None):
    return None

def get_deliberation_auditor(db=None):
    return None

def get_available_action_weights(context, default_weight=0.0):
    actions = context.get('available_actions', ['ACTION0'])
    return {a: default_weight for a in actions}

def get_random_available_action(context):
    import random
    actions = context.get('available_actions', ['ACTION0'])
    return random.choice(actions) if actions else 'ACTION0'

class EdgeInferenceEngine:
    def __init__(self, *a, **kw): pass
    def infer(self, *a, **kw): return None

class ShadowTester:
    def __init__(self, *a, **kw): pass
    def test(self, *a, **kw): return None

CognitiveRouterClass = None

ORDERING_PRESETS = {
    # Current behavior - efficiency-optimized (15 rungs - core only)
    'efficiency': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 3),
        ('palette_detection', 3),
        ('frontier_checkpoint', 4),
        ('three_try_sequence', 5),
        ('discovery_exploitation', 10),
        ('death_avoidance', 15),
        ('prior_lessons', 16),
        ('terminal_pattern', 17),
        ('embedding_suggestion', 20),
        ('frontier_topology', 25),
        ('exploration_phase', 30),
        ('two_streams', 35),
        ('primitive_suggester', 40),
        ('network_wisdom', 45),
        ('smart_action_selection', 99),
    ],

    # LLM-optimal - understanding first (all rungs including wired metacognition)
    'llm_optimal': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('self_trust_boost', 4),
        ('control_tracker', 5),
        ('budget_aware_planning', 5),
        ('affordance_detection', 6),
        ('imagination_budget', 6),
        ('breakthrough_budget', 6),
        ('regulatory_signal', 7),
        ('survey', 8),
        ('network_exploration_stats', 8),
        ('scientific_method', 10),
        ('questioning_engine', 12),
        ('frustration_detection', 14),
        ('two_streams', 16),
        ('i_thread', 18),
        ('metacognitive_prediction', 20),
        ('deliberation_system', 22),
        ('symbolic_tracker', 23),
        ('theory_gate', 24),
        ('belief_system', 25),
        ('hypothesis_system', 26),
        ('sensation_engine', 27),
        ('resonance_detector', 28),
        ('valence_goals', 29),
        ('network_wisdom', 30),
        ('death_avoidance', 32),
        ('terminal_pattern', 34),
        ('theory_contradiction', 35),
        ('metacognitive_elimination', 35),
        ('pariah_avoidance', 36),
        ('viral_package_weights', 37),
        ('three_layer_filter', 38),
        ('hypothesis_testing', 39),
        ('three_try_sequence', 40),
        ('rule_transfer', 41),
        ('discovery_exploitation', 42),
        ('trigger_sequences', 43),
        ('embedding_suggestion', 44),
        ('embedding_matcher', 45),
        ('multi_stage_matching', 46),
        ('primitive_suggester', 48),
        ('network_sharing', 49),
        ('replay_learning', 50),
        ('network_wisdom', 51),
        ('abstraction_templates', 54),
        ('few_shot_relations', 55),
        ('few_shot_invariants', 56),
        ('subgoal_planning', 58),
        ('visual_analyzer', 60),
        ('network_object_inventory', 62),
        ('action6_object_exploration', 64),
        ('click_behavior_learning', 65),
        ('causal_click_mapping', 66),
        ('constraint_satisfaction', 66),
        ('destructive_action_detection', 66),
        ('goal_relationship_modeling', 67),
        ('interactable_tile_discovery', 67),
        ('object_color_targeting', 66),
        ('wall_aware_navigation', 67),
        ('controlled_movement_planning', 67),
        ('spatial_map', 68),
        ('near_miss_analyzer', 68),
        ('completion_prediction', 68),
        ('frontier_topology', 70),
        ('map_intel_collision', 72),
        ('exploration_phase', 74),
        ('grid_exploration', 76),
        ('smart_action_selection', 99),
    ],

    # Human brain - parallel attention + fear interrupt
    'human_brain': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('death_avoidance', 4),
        ('terminal_pattern', 5),
        ('survey', 6),
        ('embedding_suggestion', 8),
        ('network_wisdom', 10),
        ('pariah_avoidance', 12),
        ('metacognitive_elimination', 12),
        ('viral_package_weights', 13),
        ('exploration_phase', 14),
        ('scientific_method', 20),
        ('theory_gate', 22),
        ('metacognitive_prediction', 24),
        ('i_thread', 26),
        ('two_streams', 28),
        ('sensation_engine', 30),
        ('primitive_suggester', 35),
        ('discovery_exploitation', 40),
        ('smart_action_selection', 99),
    ],

    # Full comprehensive ordering
    'comprehensive': [
        # CLASSIFY (Priority 1)
        ('game_classifier', 1),
        # EMERGENCY (Priority 1-5)
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('sparse_grid', 3),
        ('self_trust_boost', 4),
        ('frame_interpretation', 5),
        # ORIENTATION (Priority 5-20)
        ('budget_aware_planning', 6),
        ('imagination_budget', 6),
        ('breakthrough_budget', 7),
        ('affordance_detection', 8),
        ('control_tracker', 8),
        ('wall_aware_navigation', 9),
        ('regulatory_signal', 10),
        ('survey', 11),
        ('network_exploration_stats', 12),
        ('questioning_engine', 13),
        ('exploration_phase', 13),
        ('frustration_detection', 14),
        # FILTER (Priority 15-25)
        ('contextual_failure', 15),
        ('metacognitive_elimination', 15),
        ('destructive_action_detection', 16),
        ('death_avoidance', 16),
        ('terminal_pattern', 17),
        ('theory_contradiction', 18),
        ('pariah_avoidance', 19),
        ('viral_package_weights', 20),
        ('three_layer_filter', 21),
        # HYPOTHESIS (Priority 25-40)
        ('event_understanding', 23),
        ('symbolic_tracker', 24),
        ('assumption_formation', 25),
        ('interactable_tile_discovery', 25),
        ('goal_relationship_modeling', 26),
        ('belief_system', 26),
        ('hypothesis_system', 27),
        ('scientific_method', 28),
        ('theory_gate', 29),
        ('metacognitive_prediction', 30),
        ('hypothesis_testing', 31),
        ('deliberation_system', 32),
        ('two_streams', 33),
        ('i_thread', 34),
        ('valence_goals', 35),
        ('sensation_engine', 36),
        ('resonance_detector', 37),
        # EXPLOITATION (Priority 40-80)
        ('three_try_sequence', 40),
        ('rule_transfer', 41),
        ('state_matching', 42),
        ('trigger_sequences', 43),
        ('discovery_exploitation', 44),
        ('embedding_matcher', 45),
        ('effect_prediction', 42),
        ('solver_goal_extraction', 43),
        ('constraint_decoder', 44),
        ('causal_click_mapping', 45),
        ('spatial_relationship', 46),
        ('constraint_satisfaction', 46),
        ('object_color_targeting', 47),
        ('goal_progress', 47),
        ('action_outcome_verifier', 47),
        ('controlled_movement_planning', 47),
        ('spatial_map', 47),
        ('embedding_suggestion', 48),
        ('multi_stage_matching', 48),
        ('primitive_suggester', 49),
        ('network_sharing', 50),
        ('replay_learning', 51),
        ('network_wisdom', 52),
        ('abstraction_templates', 53),
        ('few_shot_relations', 54),
        ('few_shot_invariants', 55),
        ('subgoal_planning', 56),
        ('visual_analyzer', 57),
        ('network_object_inventory', 58),
        ('action6_object_exploration', 59),
        ('click_behavior_learning', 60),
        ('near_miss_analyzer', 61),
        ('completion_prediction', 62),
        ('frontier_topology', 63),
        ('map_intel_collision', 64),
        ('grid_exploration', 65),
        # FALLBACK
        ('smart_action_selection', 99),
    ],

    # Fix 1.4: ACTION6-only ordering for click-based puzzle games
    # (FT09, VC33). Prioritises visual analysis, click-effect learning,
    # and constraint satisfaction over movement-oriented rungs.
    'action6_only': [
        # CLASSIFY (Priority 1)
        ('game_classifier', 1),
        # EMERGENCY (Priority 1-5)
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('sparse_grid', 3),
        # VISUAL UNDERSTANDING (Priority 5-15) -- find what's clickable
        ('frame_interpretation', 5),
        ('affordance_detection', 6),
        ('visual_analyzer', 7),
        ('network_object_inventory', 8),
        ('action6_object_exploration', 9),
        ('survey', 10),
        # CLICK BEHAVIOUR (Priority 15-30) -- apply learned patterns
        ('click_behavior_learning', 15),
        ('object_color_targeting', 17),
        ('interactable_tile_discovery', 18),
        # MAP_EFFECTS forward model (Priority 26)
        ('effect_prediction', 26),
        # EXTRACT_GOAL + MAP_EFFECTS + PLAN (Priority 27-35)
        ('solver_goal_extraction', 27),
        ('constraint_decoder', 28),
        ('causal_click_mapping', 29),
        ('constraint_satisfaction', 30),
        ('goal_progress', 31),
        ('action_outcome_verifier', 32),
        ('goal_relationship_modeling', 32),
        ('near_miss_analyzer', 33),
        ('completion_prediction', 34),
        ('state_matching', 35),
        # HYPOTHESIS & REASONING (Priority 45-60)
        ('hypothesis_system', 45),
        ('scientific_method', 46),
        ('theory_gate', 47),
        ('hypothesis_testing', 48),
        ('belief_system', 49),
        ('two_streams', 50),
        ('discovery_exploitation', 51),
        # NETWORK KNOWLEDGE (Priority 60-70)
        ('network_sharing', 60),
        ('network_wisdom', 61),
        ('embedding_suggestion', 62),
        ('primitive_suggester', 63),
        # EXPLORATION FALLBACK (Priority 70-99)
        ('grid_exploration', 75),
        ('exploration_phase', 80),
        ('smart_action_selection', 99),
    ],

    # Phased approach - different order by budget phase
    'phased_orientation': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('imagination_budget', 3),
        ('survey', 5),
        ('questioning_engine', 10),
        ('exploration_phase', 15),
        ('scientific_method', 20),
        ('network_exploration_stats', 25),
        ('death_avoidance', 35),
        ('grid_exploration', 40),
        ('smart_action_selection', 99),
    ],
    'phased_hypothesis': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('scientific_method', 5),
        ('metacognitive_prediction', 10),
        ('theory_gate', 15),
        ('deliberation_system', 20),
        ('two_streams', 25),
        ('death_avoidance', 30),
        ('network_wisdom', 35),
        ('exploration_phase', 40),
        ('smart_action_selection', 99),
    ],
    'phased_exploitation': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('death_avoidance', 5),
        ('terminal_pattern', 7),
        ('three_try_sequence', 10),
        ('discovery_exploitation', 15),
        ('embedding_suggestion', 20),
        ('multi_stage_matching', 25),
        ('network_wisdom', 30),
        ('primitive_suggester', 35),
        ('completion_prediction', 40),
        ('frontier_topology', 45),
        ('smart_action_selection', 99),
    ],

    # Minimal - only essential rungs for fast execution
    'minimal': [
        ('infinite_loop_breaker', 1),
        ('death_avoidance', 5),
        ('discovery_exploitation', 10),
        ('network_wisdom', 20),
        ('exploration_phase', 30),
        ('smart_action_selection', 99),
    ],

    # ACTION6 WORLD - For games where ACTION6 is available/primary
    'action6_world': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('sparse_grid', 4),
        ('visual_analyzer', 5),
        ('affordance_detection', 5),
        ('budget_aware_planning', 5),
        ('control_tracker', 6),
        ('frame_interpretation', 7),
        ('event_understanding', 8),
        ('causal_click_mapping', 9),
        ('constraint_decoder', 10),
        ('trigger_sequences', 10),
        ('click_behavior_learning', 11),
        ('object_color_targeting', 12),
        ('constraint_satisfaction', 12),
        ('destructive_action_detection', 12),
        ('goal_relationship_modeling', 13),
        ('belief_system', 13),
        ('symbolic_tracker', 14),
        ('action6_object_exploration', 15),
        ('network_object_inventory', 16),
        ('primitive_suggester', 15),
        ('hypothesis_system', 16),
        ('scientific_method', 17),
        ('theory_gate', 18),
        ('assumption_formation', 19),
        ('network_wisdom', 20),
        ('network_sharing', 21),
        ('few_shot_relations', 22),
        ('resonance_detector', 23),
        ('valence_goals', 24),
        ('death_avoidance', 25),
        ('metacognitive_elimination', 25),
        ('pariah_avoidance', 26),
        ('viral_package_weights', 27),
        ('grid_exploration', 30),
        ('exploration_phase', 35),
        ('frustration_detection', 40),
        ('metacognitive_prediction', 41),
        ('smart_action_selection', 99),
    ],

    # ACTION6-only game (like vc33)
    'action6_only': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('sparse_grid', 4),
        ('visual_analyzer', 5),
        ('affordance_detection', 5),
        ('budget_aware_planning', 5),
        ('action6_object_exploration', 6),
        ('click_behavior_learning', 7),
        ('object_color_targeting', 7),
        ('causal_click_mapping', 8),
        ('constraint_decoder', 8),
        ('constraint_satisfaction', 8),
        ('destructive_action_detection', 9),
        ('goal_relationship_modeling', 9),
        ('trigger_sequences', 9),
        ('network_object_inventory', 10),
        ('event_understanding', 11),
        ('belief_system', 11),
        ('symbolic_tracker', 12),
        ('control_tracker', 13),
        ('hypothesis_system', 14),
        ('scientific_method', 15),
        ('theory_gate', 16),
        ('network_wisdom', 17),
        ('network_sharing', 18),
        ('primitive_suggester', 19),
        ('valence_goals', 20),
        ('death_avoidance', 25),
        ('metacognitive_elimination', 25),
        ('pariah_avoidance', 26),
        ('viral_package_weights', 27),
        ('grid_exploration', 30),
        ('exploration_phase', 35),
        ('smart_action_selection', 99),
    ],

    # Exploration-heavy for frontier games
    'frontier_exploration': [
        ('infinite_loop_breaker', 1),
        ('coordinate_oscillation', 2),
        ('palette_detection', 3),
        ('self_trust_boost', 4),
        ('budget_aware_planning', 4),
        ('affordance_detection', 5),
        ('frontier_checkpoint', 5),
        ('survey', 6),
        ('network_exploration_stats', 8),
        ('exploration_phase', 10),
        ('wall_aware_navigation', 11),
        ('controlled_movement_planning', 12),
        ('assumption_formation', 13),
        ('hypothesis_testing', 14),
        ('contextual_failure', 15),
        ('questioning_engine', 15),
        ('scientific_method', 20),
        ('rule_transfer', 22),
        ('action6_object_exploration', 24),
        ('click_behavior_learning', 25),
        ('causal_click_mapping', 26),
        ('constraint_decoder', 26),
        ('constraint_satisfaction', 26),
        ('interactable_tile_discovery', 27),
        ('object_color_targeting', 27),
        ('destructive_action_detection', 27),
        ('goal_relationship_modeling', 28),
        ('spatial_map', 28),
        ('grid_exploration', 28),
        ('metacognitive_elimination', 35),
        ('viral_package_weights', 36),
        ('death_avoidance', 40),
        ('discovery_exploitation', 45),
        ('smart_action_selection', 99),
    ],
}


class DecisionRungSystem:
    """
    Modular action decision system with swappable rung orderings.

    All rung classes are imported from the ``rungs`` package.
    """

    # Use the unified registry from rungs/__init__.py
    RUNG_REGISTRY = RUNG_REGISTRY  # From registry cell above

    def __init__(self,
                 strategy: str = 'context_adaptive',
                 core_gameplay_ref: Any = None,
                 config_path: Optional[str] = None,
                 engine_registry: Optional[Any] = None,
                 cognitive_router: Optional[Any] = None,
                 routing_trace_store: Optional[Any] = None):
        """
        Args:
            strategy: 'ladder', 'weighted', 'phased', 'parallel', 'cognitive', or 'context_adaptive'
            core_gameplay_ref: Reference to CoreGameplay instance (legacy)
            config_path: Optional path to custom ordering config
            engine_registry: EngineRegistry for modular engine access (preferred)
            cognitive_router: Pre-configured CognitiveRouter instance
            routing_trace_store: RoutingTraceStore for recording decision traces
        """
        self.strategy = DecisionStrategy(strategy)
        self.core: Any = core_gameplay_ref
        self._engine_registry: Optional[Any] = engine_registry
        self._routing_trace_store: Optional[Any] = routing_trace_store
        self.rungs: List[DecisionRung] = []
        self.ordering_name = 'default'
        self.config_path = config_path or ''  # No filesystem config in notebook mode

        # Stats
        self.total_decisions = 0
        self.rung_wins: Dict[str, int] = {}

        # Last decision metadata (for checkpoint handoff to context builder)
        self.last_decision_metadata: Dict[str, Any] = {}

        # Track winning rung for feedback loop
        self._last_winning_rung: Optional[DecisionRung] = None
        self._last_outcome_context: Dict[str, Any] = {}

        # Temporal integration
        self._temporal_integrator = None
        self._category_modulation_map = {
            'hypothesis': 'exploration',
            'orientation': 'exploration',
            'exploitation': 'exploitation',
            'filter': 'safety',
            'emergency': 'safety',
            'fallback': 'neutral',
            'metacognition': 'neutral',
            'unknown': 'neutral',
        }
        self._current_generation: int = 0
        self._current_action_in_generation: int = 0

        # Deliberation audit
        self._deliberation_auditor = None
        self._current_deliberation: Optional[Any] = None

        # Cognitive router
        self._cognitive_router: Optional[Any] = cognitive_router
        self._cognitive_router_initialized: bool = (cognitive_router is not None)
        if cognitive_router is not None:
            _load_cognitive_router()

        # H41: Rung affinity model for imitation learning
        self._rung_affinity_model: Optional[Any] = None

        # Load default ordering
        self._suppress_ordering_deprecation = (self.strategy == DecisionStrategy.COGNITIVE)
        self.load_ordering('comprehensive')

    @property
    def engines(self) :
        """Access modular engines via registry."""
        if self._engine_registry is not None:
            return self._engine_registry
        if self.core is not None:
            _STUB_REGISTRY
        else:
            _STUB_REGISTRY
        return self._engine_registry

    @property
    def temporal_integrator(self):
        """Lazy-load temporal integrator for multi-scale experience integration."""
        if self._temporal_integrator is None:
            try:
                from engines.memory.temporal_integrator import get_temporal_integrator
                db = None
                if self._engine_registry is not None:
                    try:
                        db = self._engine_registry._get_db_interface()
                    except Exception:
                        pass
                self._temporal_integrator = get_temporal_integrator(db)
            except ImportError:
                logger.debug("[RUNG-SYSTEM] TemporalIntegrator not available")
                self._temporal_integrator = None
        return self._temporal_integrator

    @property
    def deliberation_auditor(self):
        """Lazy-load deliberation auditor."""
        if self._deliberation_auditor is None:
            try:
                from engines.reasoning.deliberation_audit import (
                    get_deliberation_auditor,
                )
                db = None
                if self._engine_registry is not None:
                    try:
                        db = self._engine_registry._get_db_interface()
                    except Exception:
                        pass
                self._deliberation_auditor = get_deliberation_auditor(db)
            except ImportError:
                logger.debug("[RUNG-SYSTEM] DeliberationAuditor not available")
                self._deliberation_auditor = None
        return self._deliberation_auditor

    @property
    def cognitive_router(self):
        """Lazy-load cognitive router."""
        if self._cognitive_router is None and not self._cognitive_router_initialized:
            CognitiveRouterClass = _load_cognitive_router()
            if CognitiveRouterClass is not None:
                try:
                    self._cognitive_router = CognitiveRouterClass()
                    logger.info("[RUNG-SYSTEM] CognitiveRouter initialized")
                except Exception as e:
                    logger.warning(f"[RUNG-SYSTEM] Failed to initialize CognitiveRouter: {e}")
            self._cognitive_router_initialized = True
        return self._cognitive_router

    # -- Temporal context -----------------------------------------------------

    def set_temporal_context(self, generation: int, action_in_generation: int) -> None:
        """Set current temporal context for decay calculations."""
        self._current_generation = generation
        self._current_action_in_generation = action_in_generation

    def record_outcome(self, agent_id: str, game_type: str, outcome_value: float) -> None:
        """Record an action outcome for temporal integration."""
        if self.temporal_integrator is not None:
            self.temporal_integrator.record_outcome(
                agent_id=agent_id, game_type=game_type,
                generation=self._current_generation,
                action_in_generation=self._current_action_in_generation,
                outcome_value=outcome_value
            )

        if self.deliberation_auditor is not None and self._current_deliberation is not None:
            try:
                if outcome_value > 0.0:
                    outcome_type_str = "positive"
                elif outcome_value < 0.0:
                    outcome_type_str = "negative"
                else:
                    outcome_type_str = "neutral"
                self.deliberation_auditor.record_outcome(
                    outcome_type=outcome_type_str, score_change=outcome_value,
                )
                self.deliberation_auditor.finalize()
                self._current_deliberation = None
            except Exception as e:
                logger.debug(f"[RUNG-SYSTEM] Deliberation outcome recording failed: {e}")

    def _get_category_modulation(self, agent_id: str, game_type: str) -> Dict[str, float]:
        """Get rung category priority modulation from temporal integration."""
        if self.temporal_integrator is None:
            return {}
        return self.temporal_integrator.get_rung_modulation(
            agent_id=agent_id, game_type=game_type,
            current_generation=self._current_generation,
            current_action=self._current_action_in_generation
        )

    def _get_modulated_priority(self, rung: DecisionRung, modulation: Dict[str, float]) -> float:
        """Get a rung's priority adjusted by temporal modulation."""
        base_priority = rung.get_priority()
        if not modulation:
            return base_priority
        mod_category = self._category_modulation_map.get(rung.category, 'neutral')
        if mod_category == 'neutral':
            return base_priority
        multiplier = modulation.get(mod_category, 1.0)
        return base_priority / multiplier

    # -- Ordering management --------------------------------------------------

    def load_ordering(self, preset_name: str) -> None:
        """Load a preset ordering or custom config."""
        self.ordering_name = preset_name
        self.rungs = []

        suppress = getattr(self, '_suppress_ordering_deprecation', False)

        if preset_name in ORDERING_PRESETS:
            
            ordering = ORDERING_PRESETS[preset_name]
        else:
            ordering = self._load_custom_ordering(preset_name)
            if not ordering:
                print(f"[RUNG-SYSTEM] Warning: Unknown ordering '{preset_name}', using comprehensive")
                ordering = ORDERING_PRESETS['comprehensive']

        # Shared engine registry
        if self._engine_registry is None:
            from engines.registry import EngineRegistry
            if self.core is not None:
                self._engine_registry = _STUB_REGISTRY
            else:
                self._engine_registry = _STUB_REGISTRY

        for rung_name, priority in ordering:
            if rung_name in self.RUNG_REGISTRY:
                rung = self.RUNG_REGISTRY[rung_name](
                    core_gameplay_ref=self.core,
                    engine_registry=self._engine_registry
                )
                rung.priority_override = priority
                self.rungs.append(rung)
            else:
                print(f"[RUNG-SYSTEM] Warning: Unknown rung '{rung_name}'")

        self.rungs.sort(key=lambda r: r.get_priority())
        print(f"[RUNG-SYSTEM] Loaded ordering '{preset_name}' with {len(self.rungs)} rungs")

    def _load_custom_ordering(self, name: str) -> Optional[List[Tuple[str, int]]]:
        """Load custom ordering from config file"""
        try:
            if os.path.exists(self.config_path):
                with open(self.config_path, 'r') as f:
                    config = json.load(f)
                    return config.get(name)
        except Exception as e:
            print(f"[RUNG-SYSTEM] Error loading config: {e}")
        return None

    def save_ordering(self, name: str, ordering: List[Tuple[str, int]]) -> None:
        """Save a custom ordering to config file"""
        try:
            config = {}
            if os.path.exists(self.config_path):
                with open(self.config_path, 'r') as f:
                    config = json.load(f)
            config[name] = ordering
            os.makedirs(os.path.dirname(self.config_path), exist_ok=True)
            with open(self.config_path, 'w') as f:
                json.dump(config, f, indent=2)
            print(f"[RUNG-SYSTEM] Saved ordering '{name}'")
        except Exception as e:
            print(f"[RUNG-SYSTEM] Error saving config: {e}")

    # -- Main decide() entry point --------------------------------------------

    # Phase 0.1: keys a valid DecisionContext must carry
    _CONTEXT_EXPECTED_KEYS = frozenset({
        'available_actions', 'game_id', 'agent_id',
        'action_count', 'level_number',
    })

    def _get_wall_blocked_actions(self, context: Dict[str, Any]) -> Set[str]:
        """H16: Query the spatial_map rung for actions that lead to known walls.

        Returns a set of action names (e.g. {'ACTION1', 'ACTION3'}) that would
        move the agent into a known wall at the current position. Other rungs
        and the weighted-random fallback use this to avoid wasted actions.
        """
        # H51c: Use game_id (not game_type) to match SpatialMapRung's
        # variant-isolated keys from H51b.
        game_id = context.get('game_id', context.get('game_type', ''))
        level = context.get('level', 1)
        game_key = f"{game_id}_L{level}"

        spatial_rung = next(
            (r for r in self.rungs if r.name == 'spatial_map'), None
        )
        if spatial_rung is None:
            return set()

        pos = spatial_rung._position.get(game_key)
        if pos is None:
            return set()

        spatial_map = spatial_rung._maps.get(game_key, {})
        if not spatial_map:
            return set()

        blocked: Set[str] = set()
        for action, (dx, dy) in spatial_rung.ACTION_DELTAS.items():
            neighbor = (pos[0] + dx, pos[1] + dy)
            if spatial_map.get(neighbor) == 'wall':
                blocked.add(action)
        return blocked

    def decide(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """Make an action decision using current strategy."""
        self.total_decisions += 1

        # Phase 0.1: Warn once per session if context is a raw dict missing expected keys
        missing = self._CONTEXT_EXPECTED_KEYS - set(context.keys())
        if missing and not getattr(self, '_context_warned', False):
            import logging as _logging
            _logging.getLogger(__name__).warning(
                "[DRS] DecisionContext missing keys (raw dict?): %s", missing,
            )
            self._context_warned = True

        available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])

        # H16: Inject wall-blocked actions into context so ALL strategies
        # can avoid known walls, not just the spatial_map rung.
        context['_wall_blocked_actions'] = self._get_wall_blocked_actions(context)
        current_ordering = self.ordering_name

        target_ordering = self._select_ordering_for_context(available, context)
        if target_ordering != current_ordering:
            self._switch_ordering_temporarily(target_ordering)

        try:
            if self.strategy == DecisionStrategy.LADDER:
                return self._decide_ladder(game_state, context)
            elif self.strategy == DecisionStrategy.WEIGHTED:
                return self._decide_weighted(game_state, context)
            elif self.strategy == DecisionStrategy.PHASED:
                return self._decide_phased(game_state, context)
            elif self.strategy == DecisionStrategy.PARALLEL:
                return self._decide_parallel(game_state, context)
            elif self.strategy == DecisionStrategy.CONTEXT_ADAPTIVE:
                return self._decide_context_adaptive(game_state, context)
            elif self.strategy == DecisionStrategy.COGNITIVE:
                return self._decide_cognitive(game_state, context)
            else:
                return self._decide_ladder(game_state, context)
        finally:
            if target_ordering != current_ordering:
                self._switch_ordering_temporarily(current_ordering)

    def _select_ordering_for_context(self, available_actions: List[int], context: Dict[str, Any]) -> str:
        """Select the best ordering based on available actions, game type, and agent role.

        Fix 1.4: ACTION6-only games now get the 'action6_only' preset.
        Fix 3.2: Agent role influences ordering -- exploiters get minimal,
        optimizers get efficiency, pioneers get comprehensive.
        """
        actions_list = list(available_actions) if available_actions is not None else []

        # Fix 1.4: ACTION6-only games always use click-specialised ordering
        if actions_list == [6]:
            return 'action6_only'

        if 6 in available_actions:
            if self.ordering_name in ('action6_world', 'action6_only', 'frontier_exploration'):
                return self.ordering_name
            if context.get('frontier_mode', False):
                return 'action6_world'

        # Fix 3.2: Role-based ordering selection (archetype differentiation).
        # Exploiters use minimal proven strategies; optimizers refine
        # known patterns; pioneers explore everything.
        agent_role = context.get('agent_role', 'pioneer')
        if agent_role == 'exploiter':
            return 'efficiency'  # Only proven strategies
        elif agent_role == 'optimizer':
            return 'efficiency'  # Exploit known patterns, some exploration
        elif agent_role == 'pioneer':
            return 'comprehensive'  # Full exploration + all hypotheses

        return self.ordering_name

    def _switch_ordering_temporarily(self, target_ordering: str) -> None:
        """Switch to a different ordering without reinstantiating rungs."""
        if target_ordering not in ORDERING_PRESETS:
            return
        ordering = ORDERING_PRESETS[target_ordering]
        priority_map = {name: priority for name, priority in ordering}
        for rung in self.rungs:
            if rung.name in priority_map:
                rung.priority_override = priority_map[rung.name]
            else:
                rung.priority_override = 200
        self.rungs.sort(key=lambda r: r.get_priority())
        self.ordering_name = target_ordering

    # -- Strategy implementations ---------------------------------------------

    def _decide_ladder(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """First confident answer wins, with temporal modulation."""
        accumulated_weights = get_available_action_weights(context, 1.0)
        self.last_decision_metadata = {}
        self._last_winning_rung = None

        agent_id = context.get('agent_id', 'unknown')
        game_type = context.get('game_type', 'unknown')
        modulation = self._get_category_modulation(agent_id, game_type)

        if self.temporal_integrator is not None:
            context['exploration_appetite'] = self.temporal_integrator.get_exploration_appetite(
                agent_id=agent_id, game_type=game_type,
                current_generation=self._current_generation,
                current_action=self._current_action_in_generation
            )

        sorted_rungs = sorted(self.rungs, key=lambda r: self._get_modulated_priority(r, modulation))

        for rung in sorted_rungs:
            if not rung.enabled:
                continue
            result = rung.evaluate(game_state, context)

            if result.weights:
                for action, weight in result.weights.items():
                    if is_action_available(action, context):
                        accumulated_weights[action] = accumulated_weights.get(action, 1.0) * weight

            if result.has_suggestion(rung.confidence_threshold):
                if result.action and not is_action_available(result.action, context):
                    continue
                if result.action == 'ACTION6':
                    result = Action6CoordinateProvider.enrich_result_with_coordinates(
                        result, context, self._engine_registry, game_state
                    )
                self.rung_wins[rung.name] = self.rung_wins.get(rung.name, 0) + 1
                self._last_winning_rung = rung
                rung.record_outcome(was_accepted=True)
                self.last_decision_metadata = result.metadata or {}
                self.last_decision_metadata['rung_name'] = rung.name
                return result.action or get_random_available_action(context), f"[{rung.name}] {result.reason}"

        action, reason = self._weighted_random_choice(accumulated_weights, context), "Weighted fallback after ladder"
        self.last_decision_metadata['rung_name'] = 'weighted_fallback'
        if action == 'ACTION6':
            coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
            self.last_decision_metadata = {**coords, 'rung_name': 'weighted_fallback'}
            reason += f" [coords: ({coords['x']},{coords['y']})]"
        return action, reason

    def _decide_weighted(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """All rungs vote, weighted sum decides."""
        action_votes = get_available_action_weights(context, 0.0)
        reasons: List[str] = []
        self.last_decision_metadata = {}
        self._last_winning_rung = None

        agent_id = context.get('agent_id', 'unknown')
        game_type = context.get('game_type', 'unknown')
        modulation = self._get_category_modulation(agent_id, game_type)

        # Deliberation audit
        if self.deliberation_auditor is not None:
            game_id = context.get('game_id', context.get('scorecard_id', 'unknown'))
            level_number = context.get('level_number', context.get('level', 0))
            action_number = context.get('action_count', 0)
            self.deliberation_auditor.start_deliberation(
                game_id=game_id,
                game_type=game_type[:4] if len(game_type) >= 4 else game_type,
                level_number=level_number, action_number=action_number,
                agent_id=agent_id, context=context,
            )

        if self.temporal_integrator is not None:
            context['exploration_appetite'] = self.temporal_integrator.get_exploration_appetite(
                agent_id=agent_id, game_type=game_type,
                current_generation=self._current_generation,
                current_action=self._current_action_in_generation
            )

        rung_contributions: Dict[str, Tuple[DecisionRung, float, str]] = {}
        all_alternatives: Dict[str, Tuple[float, str, str]] = {}

        for rung in self.rungs:
            if not rung.enabled:
                continue
            result = rung.evaluate(game_state, context)

            if result.action:
                if not is_action_available(result.action, context):
                    continue
                base_weight = result.confidence * (100 - rung.get_priority()) / 100
                mod_category = self._category_modulation_map.get(rung.category, 'neutral')
                mod_multiplier = modulation.get(mod_category, 1.0) if modulation else 1.0
                weight = base_weight * mod_multiplier
                action_votes[result.action] = action_votes.get(result.action, 0) + weight
                reasons.append(f"{rung.name}:{result.action}({weight:.2f})")
                if result.action not in rung_contributions or weight > rung_contributions[result.action][1]:
                    rung_contributions[result.action] = (rung, weight, rung.name)
                if result.action not in all_alternatives or result.confidence > all_alternatives[result.action][0]:
                    all_alternatives[result.action] = (result.confidence, result.reason, rung.name)

            if result.weights:
                for action, w in result.weights.items():
                    if is_action_available(action, context):
                        action_votes[action] = action_votes.get(action, 0) + w * 0.1

        best_action = max(action_votes, key=lambda k: action_votes[k])

        # Deliberation audit recording
        if self.deliberation_auditor is not None:
            sorted_actions = sorted(action_votes.items(), key=lambda x: x[1], reverse=True)
            for action, vote_weight in sorted_actions[:5]:
                if action in all_alternatives:
                    conf, reason_txt, rung_name = all_alternatives[action]
                    why_rejected = None if action == best_action else f"lower_vote:{vote_weight:.2f}"
                    self.deliberation_auditor.add_alternative(
                        action=action, confidence=conf, reason=reason_txt,
                        rung=rung_name, why_rejected=why_rejected,
                    )
            if best_action in all_alternatives:
                conf, reason_txt, rung_name = all_alternatives[best_action]
            else:
                conf, reason_txt, rung_name = action_votes.get(best_action, 0.0), "weighted_vote", "aggregate"
            self.deliberation_auditor.record_choice(
                chosen_action=best_action, confidence=conf, reason=reason_txt, rung=rung_name,
            )
            sparse_cell_count = context.get('sparse_cell_count', 0)
            sparse_colors = context.get('sparse_colors', set())
            sparse_hash = context.get('sparse_hash', '')
            if sparse_cell_count > 0:
                self.deliberation_auditor.set_sparse_context(
                    cell_count=sparse_cell_count,
                    colors=list(sparse_colors) if isinstance(sparse_colors, set) else sparse_colors,
                    sparse_hash=sparse_hash,
                )
            self._current_deliberation = self.deliberation_auditor._current_record

        if best_action in rung_contributions:
            self._last_winning_rung = rung_contributions[best_action][0]
            self._last_winning_rung.record_outcome(was_accepted=True)
            self.last_decision_metadata['rung_name'] = rung_contributions[best_action][2]
        else:
            self.last_decision_metadata['rung_name'] = 'aggregate'

        reason = f"Weighted vote: {best_action} ({action_votes[best_action]:.2f}) from [{', '.join(reasons[:3])}]"

        if best_action == 'ACTION6':
            coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
            self.last_decision_metadata = {**self.last_decision_metadata, **coords}
            reason += f" [coords: ({coords['x']},{coords['y']})]"

        return best_action, reason

    def _decide_phased(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """Use different orderings based on budget phase"""
        budget_used: float = float(context.get('budget_used_percent', 0))
        if budget_used < 0.1:
            phase_ordering = 'phased_orientation'
        elif budget_used < 0.3:
            phase_ordering = 'phased_hypothesis'
        else:
            phase_ordering = 'phased_exploitation'
        old_rungs = self.rungs
        self.load_ordering(phase_ordering)
        action, reason = self._decide_ladder(game_state, context)
        self.rungs = old_rungs
        return action, f"[{phase_ordering}] {reason}"

    def _decide_parallel(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """Run all rungs, pick highest confidence"""
        best_result: Optional[RungResult] = None
        best_rung: Optional[DecisionRung] = None
        self._last_winning_rung = None

        for rung in self.rungs:
            if not rung.enabled:
                continue
            result = rung.evaluate(game_state, context)
            if result.action and is_action_available(result.action, context):
                if best_result is None or result.confidence > best_result.confidence:
                    best_result = result
                    best_rung = rung

        if best_result and best_rung:
            self.rung_wins[best_rung.name] = self.rung_wins.get(best_rung.name, 0) + 1
            self._last_winning_rung = best_rung
            best_rung.record_outcome(was_accepted=True)
            self.last_decision_metadata = best_result.metadata or {}
            self.last_decision_metadata['rung_name'] = best_rung.name
            return best_result.action or get_random_available_action(context), f"[{best_rung.name}] {best_result.reason}"

        self.last_decision_metadata['rung_name'] = 'no_suggestion'
        return get_random_available_action(context), "No suggestions from any rung"

    def _weighted_random_choice(self, weights: Dict[str, float], context: Optional[Dict[str, Any]] = None) -> str:
        """Make a weighted random choice from weights dict."""
        if not weights:
            if context:
                return get_random_available_action(context)
            return 'ACTION1'

        # H16: Suppress actions that lead to known walls.
        # Only filter if we have alternatives -- never block ALL actions.
        wall_blocked = context.get('_wall_blocked_actions', set()) if context else set()
        if wall_blocked:
            safe_weights = {a: w for a, w in weights.items() if a not in wall_blocked}
            if safe_weights:
                weights = safe_weights

        # H21: Modulate weights by action effectiveness.
        # Actions that rarely produce frame changes are down-weighted.
        # This causes FT09 to converge on ACTION6 (the only productive
        # action) instead of wasting 85% of its step budget on NOPs.
        action_eff = context.get('_action_effectiveness', {}) if context else {}
        if action_eff:
            modulated = {}
            for action, weight in weights.items():
                eff = action_eff.get(action)
                if eff is not None:
                    # Scale weight by effectiveness, floor at 0.1 to keep exploring
                    modulated[action] = weight * max(0.1, eff)
                else:
                    modulated[action] = weight
            if modulated:
                weights = modulated

        total = sum(max(0.05, w) for w in weights.values())
        r = random.random() * total
        cumulative = 0
        for action, weight in weights.items():
            cumulative += max(0.05, weight)
            if r <= cumulative:
                return action
        return next(iter(weights.keys()))

    # -- Outcome feedback -----------------------------------------------------

    def report_outcome(self, action: str, success: bool, is_death: bool = False,
                       score_delta: float = 0.0, context: Optional[Dict[str, Any]] = None) -> None:
        """Report the actual outcome of the last decision back to the winning rung."""
        if self._last_winning_rung is None:
            return
        self._last_outcome_context = {
            'action': action, 'success': success, 'is_death': is_death,
            'score_delta': score_delta, 'context': context or {}
        }
        try:
            self._last_winning_rung.record_outcome(was_accepted=True, success=success)
        except TypeError:
            pass
        except Exception:
            pass

        if is_death:
            outcome_value = -1.0
        elif success:
            outcome_value = min(1.0, max(0.1, score_delta)) if score_delta > 0 else 0.5
        else:
            outcome_value = -0.3

        ctx = context or {}
        self.record_outcome(
            agent_id=ctx.get('agent_id', 'unknown'),
            game_type=ctx.get('game_type', 'unknown'),
            outcome_value=outcome_value
        )

    def notify_action_complete(self, action: str, action_data: Dict[str, Any],
                               frame_before: Any, frame_after: Any,
                               context: Dict[str, Any]) -> None:
        """Notify rungs that have on_action_complete hooks.

        Gap 4D: Also adjusts rung confidence based on action outcome.
        If the action was destructive or wasted, the rung that suggested
        it should lose confidence. If productive, confidence is boosted.
        """
        # === GAP 4D: Outcome-based confidence adjustment ===
        # Find which rung was responsible for the last action
        last_rung_name = context.get('last_rung_name') or (
            self._last_winning_rung.name if self._last_winning_rung else None)
        was_productive = context.get('was_productive', False)
        was_destructive = context.get('was_destructive', False)
        was_wasted = context.get('was_wasted', False)

        if last_rung_name:
            for rung in self.rungs:
                if rung.name == last_rung_name:
                    # Adjust confidence threshold based on outcome
                    if was_destructive:
                        # Rung produced harmful action -> raise threshold (harder to fire)
                        rung.confidence_threshold = min(
                            0.95, rung.confidence_threshold + 0.05)
                    elif was_wasted:
                        # Rung produced no-effect action -> slight threshold increase
                        rung.confidence_threshold = min(
                            0.95, rung.confidence_threshold + 0.02)
                    elif was_productive:
                        # Rung produced good action -> lower threshold (easier to fire)
                        rung.confidence_threshold = max(
                            0.1, rung.confidence_threshold - 0.03)
                    break

        for rung in self.rungs:
            if hasattr(rung, 'on_action_complete'):
                try:
                    rung.on_action_complete(
                        action=action, action_data=action_data,
                        frame_before=frame_before, frame_after=frame_after,
                        context=context
                    )
                except Exception:
                    pass

    # -- Emergency rungs ------------------------------------------------------

    EMERGENCY_RUNG_NAMES = frozenset({'infinite_loop_breaker', 'coordinate_oscillation'})

    def _decide_context_adaptive(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """Context-dependent strategy selection."""
        emergency_result = self._check_emergency_rungs(game_state, context)
        if emergency_result is not None:
            return emergency_result

        effective_strategy = self._select_effective_strategy(context)
        if effective_strategy == 'weighted':
            return self._decide_weighted_non_emergency(game_state, context)
        else:
            return self._decide_ladder_non_emergency(game_state, context)

    def _check_emergency_rungs(self, game_state: Any, context: Dict[str, Any]) -> Optional[Tuple[str, str]]:
        """Check emergency rungs with LADDER semantics."""
        for rung in self.rungs:
            if not rung.enabled or rung.name not in self.EMERGENCY_RUNG_NAMES:
                continue
            result = rung.evaluate(game_state, context)
            if result.has_suggestion(rung.confidence_threshold):
                if result.action and not is_action_available(result.action, context):
                    continue
                if result.action == 'ACTION6':
                    result = Action6CoordinateProvider.enrich_result_with_coordinates(
                        result, context, self._engine_registry, game_state
                    )
                    self.last_decision_metadata = result.metadata or {}
                self.rung_wins[rung.name] = self.rung_wins.get(rung.name, 0) + 1
                rung.record_outcome(was_accepted=True)
                self.last_decision_metadata['rung_name'] = rung.name
                return result.action or get_random_available_action(context), f"[EMERGENCY:{rung.name}] {result.reason}"
        return None

    def _select_effective_strategy(self, context: Dict[str, Any]) -> str:
        """Select effective strategy based on context."""
        if context.get('replay_mode', False):
            return 'ladder'
        active_sequence = context.get('active_sequence')
        if active_sequence and context.get('sequence_position', 0) < len(active_sequence):
            return 'ladder'
        if context.get('frontier_mode', False):
            return 'weighted'
        if context.get('optimization_mode', False):
            return 'weighted'
        if context.get('game_state_mode', 'unknown') == 'exploration':
            return 'weighted'
        if context.get('has_winning_sequence', False) and not context.get('active_sequence'):
            return 'weighted'
        return 'ladder'

    def _decide_weighted_non_emergency(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """WEIGHTED strategy excluding emergency rungs."""
        action_votes = get_available_action_weights(context, 0.0)
        accumulated_weights = get_available_action_weights(context, 1.0)
        reasons: List[str] = []

        for rung in self.rungs:
            if not rung.enabled or rung.name in self.EMERGENCY_RUNG_NAMES:
                continue
            result = rung.evaluate(game_state, context)

            if result.weights:
                for action, weight in result.weights.items():
                    if is_action_available(action, context):
                        accumulated_weights[action] = accumulated_weights.get(action, 1.0) * weight

            if result.action:
                if not is_action_available(result.action, context):
                    continue
                weight = result.confidence * (100 - rung.get_priority()) / 100
                action_votes[result.action] = action_votes.get(result.action, 0) + weight
                reasons.append(f"{rung.name}:{result.action}({weight:.2f})")

        final_scores: Dict[str, float] = {}
        for action in action_votes:
            vote = action_votes[action]
            filter_weight = accumulated_weights.get(action, 1.0)
            final_scores[action] = (vote + 0.1) * filter_weight

        # H16: Remove wall-blocked actions from candidates if alternatives exist.
        wall_blocked = context.get('_wall_blocked_actions', set())
        if wall_blocked:
            safe_scores = {a: s for a, s in final_scores.items() if a not in wall_blocked}
            if safe_scores:
                final_scores = safe_scores

        best_action = max(final_scores, key=lambda k: final_scores[k])
        best_score = final_scores[best_action]

        if best_score < 0.15:
            action, reason = self._weighted_random_choice(accumulated_weights, context), "Weighted random (low confidence)"
            self.last_decision_metadata['rung_name'] = 'weighted_random'
            if action == 'ACTION6':
                coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
                self.last_decision_metadata = {**coords, 'rung_name': 'weighted_random'}
                reason += f" [coords: ({coords['x']},{coords['y']})]"
            return action, reason

        top_contributors = ', '.join(reasons[:3]) if reasons else 'filters only'
        reason = f"[WEIGHTED] {best_action} ({best_score:.2f}) from [{top_contributors}]"

        # Extract winning rung name from top contributor
        _wnr_rung = reasons[0].split(':')[0] if reasons else 'aggregate'
        self.last_decision_metadata['rung_name'] = _wnr_rung

        if best_action == 'ACTION6':
            coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
            self.last_decision_metadata = {**self.last_decision_metadata, **coords, 'rung_name': _wnr_rung}
            reason += f" [coords: ({coords['x']},{coords['y']})]"

        return best_action, reason

    def _decide_ladder_non_emergency(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """LADDER strategy excluding emergency rungs."""
        accumulated_weights = get_available_action_weights(context, 1.0)

        for rung in self.rungs:
            if not rung.enabled or rung.name in self.EMERGENCY_RUNG_NAMES:
                continue
            result = rung.evaluate(game_state, context)

            if result.weights:
                for action, weight in result.weights.items():
                    if is_action_available(action, context):
                        accumulated_weights[action] = accumulated_weights.get(action, 1.0) * weight

            if result.has_suggestion(rung.confidence_threshold):
                if result.action and not is_action_available(result.action, context):
                    continue
                # H16: Skip movement actions that lead to known walls.
                # The spatial_map rung already avoids walls, but other rungs
                # may suggest wall-hitting movements. Skip and try the next rung.
                wall_blocked = context.get('_wall_blocked_actions', set())
                if result.action in wall_blocked:
                    continue
                if result.action == 'ACTION6':
                    result = Action6CoordinateProvider.enrich_result_with_coordinates(
                        result, context, self._engine_registry, game_state
                    )
                self.rung_wins[rung.name] = self.rung_wins.get(rung.name, 0) + 1
                rung.record_outcome(was_accepted=True)
                self.last_decision_metadata = result.metadata or {}
                self.last_decision_metadata['rung_name'] = rung.name
                return result.action or get_random_available_action(context), f"[{rung.name}] {result.reason}"

        action, reason = self._weighted_random_choice(accumulated_weights, context), "Weighted fallback after ladder"
        self.last_decision_metadata['rung_name'] = 'weighted_fallback'
        if action == 'ACTION6':
            coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
            self.last_decision_metadata = {**coords, 'rung_name': 'weighted_fallback'}
            reason += f" [coords: ({coords['x']},{coords['y']})]"
        return action, reason

    def _decide_cognitive(self, game_state: Any, context: Dict[str, Any]) -> Tuple[str, str]:
        """Cognitive routing strategy - full pipeline."""
        router = self.cognitive_router
        if router is None:
            logger.warning("[RUNG-SYSTEM] CognitiveRouter unavailable, falling back to context_adaptive")
            return self._decide_context_adaptive(game_state, context)

        # Emergency rungs first (safety invariant)
        emergency_result = self._check_emergency_rungs(game_state, context)
        if emergency_result is not None:
            return emergency_result

        # H44/H51d: spatial_map priority for movement games. The cognitive
        # router may not select spatial_map due to low affinity/UK-potential,
        # but it's the ONLY rung with wall-aware BFS navigation. Without
        # this override, agents wander randomly into walls.
        # H44: fuel-limited games. H51d: ALL directional-action-only games.
        _avail = context.get('available_actions', [])
        _movement_only = (
            any(a in (1, 2, 3, 4) for a in _avail if isinstance(a, int))
            and not any(a in (5, 6, 7) for a in _avail if isinstance(a, int))
        )
        if context.get('solver_max_fuel', 999) < 999 or _movement_only:
            for rung in self.rungs:
                if rung.name == 'spatial_map' and rung.enabled:
                    try:
                        result = rung.evaluate(game_state, context)
                        if result.has_suggestion(rung.confidence_threshold):
                            if result.action and is_action_available(result.action, context):
                                self.rung_wins[rung.name] = self.rung_wins.get(rung.name, 0) + 1
                                rung.record_outcome(was_accepted=True)
                                self._last_winning_rung = rung
                                self.last_decision_metadata = result.metadata or {}
                                self.last_decision_metadata['rung_name'] = rung.name
                                return result.action, f"[COGNITIVE:{rung.name}] {result.reason}"
                    except Exception:
                        pass
                    break

        # Context-setter rungs: rungs with confidence_threshold=0.0 that
        # enrich context['world_model'] as side effects but never propose
        # actions. Must run before the cognitive router selects action rungs,
        # otherwise CLASSIFY/EXTRACT_GOAL/VERIFY chains are broken.
        for rung in sorted(self.rungs, key=lambda r: r.get_priority()):
            if not rung.enabled or rung.name in self.EMERGENCY_RUNG_NAMES:
                continue
            if getattr(rung, 'confidence_threshold', 0.1) == 0.0:
                try:
                    rung.evaluate(game_state, context)
                except Exception:
                    pass

        # Replay fast-path
        active_seq = context.get('active_sequence')
        seq_pos = context.get('sequence_position', 0)
        if active_seq and seq_pos < len(active_seq) and context.get('is_replay'):
            seq_action = active_seq[seq_pos]
            if is_action_available(seq_action, context):
                # H41: Shadow-evaluate rungs against solver action
                # Imitation learning signal -- which rungs match the solver?
                # Optimized: sample every 5th step with rotating offset so
                # all positions are covered across sessions (stratified).
                # Session A: 0,5,10...  B: 1,6,11...  C: 2,7,12... etc.
                _offset = hash(context.get('session_id', '')) % 5
                if self._rung_affinity_model is not None and seq_pos % 5 == _offset:
                    game_type = context.get('game_type', '')
                    if game_type:
                        for rung in self.rungs:
                            if not rung.enabled:
                                continue
                            if rung.name in self.EMERGENCY_RUNG_NAMES:
                                continue
                            try:
                                result = rung.evaluate(game_state, context)
                                hit = (
                                    result.action == seq_action
                                    and result.confidence > 0
                                )
                                self._rung_affinity_model.record(
                                    game_type, rung.name, hit
                                )
                            except Exception:
                                pass

                rung_name = 'three_try_sequence'
                self.rung_wins[rung_name] = self.rung_wins.get(rung_name, 0) + 1
                rung_obj = next((r for r in self.rungs if r.name == rung_name), None)
                if rung_obj:
                    rung_obj.record_outcome(was_accepted=True)
                    self._last_winning_rung = rung_obj
                self.last_decision_metadata['rung_name'] = rung_name
                return seq_action, f"[COGNITIVE:{rung_name}] Replay fast-path: step {seq_pos + 1}/{len(active_seq)}"

        # Initialize router per game
        game_id = context.get('game_id', context.get('scorecard_id', 'unknown'))
        if not hasattr(self, '_cognitive_game_id') or self._cognitive_game_id != game_id:
            nodes = {
                rung.name: {'name': rung.name, 'category': rung.category, 'priority': rung.get_priority()}
                for rung in self.rungs if rung.name not in self.EMERGENCY_RUNG_NAMES
            }
            edges: Dict[str, List[str]] = {}
            try:
                from engines.cognition.edge_inference import EdgeInferenceEngine
                edge_engine = EdgeInferenceEngine()
                rung_classes = [type(r) for r in self.rungs if r.name not in self.EMERGENCY_RUNG_NAMES]
                edge_engine.analyze_rungs(rung_classes)
                inferred = edge_engine.infer_all_edges()
                for ie in inferred:
                    if ie.source_rung not in edges:
                        edges[ie.source_rung] = []
                    if ie.target_rung not in edges[ie.source_rung]:
                        edges[ie.source_rung].append(ie.target_rung)
                logger.info(f"[RUNG-SYSTEM] Edge inference: {len(inferred)} edges for {len(nodes)} rungs")
            except Exception as edge_err:
                logger.warning(f"[RUNG-SYSTEM] Edge inference failed, using empty edges: {edge_err}")

            router.initialize(nodes, edges, game_id)
            self._cognitive_game_id = game_id
            self._cognitive_previously_successful_rungs = set()

        # Rung executor closure
        self._cognitive_last_rung_metadata: Dict[str, Any] = {}
        self._cognitive_last_winning_rung: Optional[DecisionRung] = None
        if not hasattr(self, '_cognitive_previously_successful_rungs'):
            self._cognitive_previously_successful_rungs: set = set()

        def rung_executor(rung_name: str, _game_state_dict: Dict) -> Any:
            """Execute a legacy rung and bridge to cognitive RungResult."""
            rung = next((r for r in self.rungs if r.name == rung_name), None)
            if rung is None or not rung.enabled:
                if _RungResult_Cognitive:
                    return _RungResult_Cognitive(rung_name=rung_name, confidence=0.0)
                return None

            try:
                result = rung.evaluate(game_state, context)
            except Exception as eval_err:
                logger.debug(f"[RUNG-EXECUTOR] {rung_name} evaluate() failed: {eval_err}")
                if _RungResult_Cognitive:
                    return _RungResult_Cognitive(rung_name=rung_name, confidence=0.0)
                return None

            if result.action and result.confidence > 0:
                self._cognitive_last_winning_rung = rung
                self._cognitive_last_rung_metadata = result.metadata or {}
                self._cognitive_previously_successful_rungs.add(rung_name)

            metadata = result.metadata or {}
            contradiction_detected = metadata.get('contradiction_detected', False)
            surprise_level = 0.0

            if not result.action and rung_name in self._cognitive_previously_successful_rungs:
                contradiction_detected = True
                surprise_level = 0.8

            slot_name = metadata.get('slot_name')
            if not slot_name and result.action:
                slot_name = rung_name

            questions_raised = []
            answers = []
            if not result.action and not self._cognitive_last_winning_rung:
                try:
                    from engines.cognition.blackboard import Question
                    questions_raised.append(Question(
                        question_id=f"what_works_for_{game_id}",
                        description="What action works in current game state?",
                        answerable_by=[r.name for r in self.rungs if r.name not in self.EMERGENCY_RUNG_NAMES],
                        priority=0.6,
                    ))
                except ImportError:
                    pass

            if contradiction_detected and rung_name in self._cognitive_previously_successful_rungs:
                try:
                    from engines.cognition.blackboard import Question
                    questions_raised.append(Question(
                        question_id=f"why_failed_{rung_name}_{game_id}",
                        description=f"Why did {rung_name} stop working?",
                        answerable_by=[r.name for r in self.rungs if r.name != rung_name and r.name not in self.EMERGENCY_RUNG_NAMES],
                        priority=0.8,
                    ))
                except ImportError:
                    pass

            if result.action and result.confidence > 0.3:
                answers.append(f"what_works_for_{game_id}")

            if _RungResult_Cognitive:
                return _RungResult_Cognitive(
                    rung_name=rung_name, slot_name=slot_name,
                    value=result.action, confidence=result.confidence,
                    raises_questions=questions_raised, answers_questions=answers,
                    surprise_level=surprise_level,
                    contradiction_detected=contradiction_detected,
                    contradiction_with=metadata.get('contradiction_with'),
                )
            return result

        # Run the cognitive router
        try:
            decision_result = router.decide(
                game_state={'frame': getattr(game_state, 'frame', None), **context},
                rung_executor=rung_executor
            )

            action = decision_result.action_value
            rung_name = decision_result.action
            confidence = decision_result.confidence
            reasoning = f"[COGNITIVE:{rung_name}] {decision_result.reasoning}"

            if self._cognitive_last_winning_rung is not None:
                self._last_winning_rung = self._cognitive_last_winning_rung
                self.rung_wins[rung_name] = self.rung_wins.get(rung_name, 0) + 1
                self._cognitive_last_winning_rung.record_outcome(was_accepted=True)

            self.last_decision_metadata['rung_name'] = rung_name

            if not action or not is_action_available(action, context):
                weighted_action, weighted_reason = self._decide_weighted_non_emergency(game_state, context)
                action = weighted_action
                reasoning = f"[COGNITIVE:{rung_name}] Weighted fallback (router rung had no action) -> {weighted_reason}"

            if action not in {f'ACTION{i}' for i in range(1, 8)}:
                action = get_random_available_action(context)
                reasoning = "[COGNITIVE] Random fallback: invalid action format"

            if action == 'ACTION6':
                coords = self._cognitive_last_rung_metadata
                if 'x' not in coords or 'y' not in coords:
                    coords = Action6CoordinateProvider.get_coordinates(context, self._engine_registry, game_state)
                self.last_decision_metadata = {**coords, 'rung_name': rung_name}
                reasoning += f" [coords: ({coords.get('x', 32)},{coords.get('y', 32)})]"

            # Record trace
            if self._routing_trace_store is not None:
                try:
                    trace_id = self._routing_trace_store.record_trace(
                        game_id=game_id, agent_id=context.get('agent_id', 'unknown'),
                        path=decision_result.path,
                        algorithm_used=getattr(decision_result, 'algorithm_name', decision_result.final_quadrant),
                        final_action=action, final_confidence=confidence,
                        initial_quadrant=getattr(decision_result, 'initial_quadrant', decision_result.final_quadrant),
                        final_quadrant=decision_result.final_quadrant,
                        quadrant_transitions=getattr(decision_result, 'quadrant_transitions', []),
                        algorithms_history=getattr(decision_result, 'algorithms_history', []),
                        backtrack_count=getattr(decision_result, 'backtrack_count', 0),
                        iterations=decision_result.iterations,
                        decision_latency_ms=decision_result.time_elapsed * 1000
                    )
                    self.last_decision_metadata['trace_id'] = trace_id
                except Exception as trace_err:
                    logger.warning(f"[COGNITIVE] Failed to record trace: {trace_err}")

            if self.total_decisions % 100 == 0:
                stats = router.get_statistics()
                logger.info(
                    f"[COGNITIVE] Stats: {stats['total_decisions']} decisions, "
                    f"{stats['total_fallbacks']} fallbacks ({stats['fallback_rate']:.1%})"
                )

            return action, reasoning

        except Exception as e:
            import traceback
            logger.error(f"[COGNITIVE] Router error: {e}, falling back to context_adaptive\n{traceback.format_exc()}")
            return self._decide_context_adaptive(game_state, context)

    # -- H41: Online credit assignment ----------------------------------------

    def record_game_outcome(self, game_type: str, won: bool) -> None:
        """Credit winning rungs from this game to the affinity model.

        Called at end of cognitive (non-replay) games. Winning rungs
        get 'hit' credit; this complements shadow evaluation during replay.
        """
        if self._rung_affinity_model is None or not won or not game_type:
            return
        for rung_name, count in self.rung_wins.items():
            if rung_name == 'three_try_sequence':
                continue  # Skip replay rung -- not a cognitive rung
            for _ in range(count):
                self._rung_affinity_model.record(game_type, rung_name, hit=True)

    # -- Stats & experiments --------------------------------------------------

    def get_stats(self) -> Dict[str, Any]:
        """Get decision statistics"""
        return {
            'total_decisions': self.total_decisions,
            'ordering': self.ordering_name,
            'strategy': self.strategy.value,
            'rung_wins': self.rung_wins,
            'rung_count': len(self.rungs),
            'rungs': [{'name': r.name, 'priority': r.get_priority(), 'enabled': r.enabled} for r in self.rungs]
        }

    def experiment_orderings(self, game_state: Any, context: Dict[str, Any],
                             orderings: List[str]) -> Dict[str, Tuple[str, str]]:
        """Test multiple orderings on same state (for analysis)."""
        results: Dict[str, Tuple[str, str]] = {}
        original = self.ordering_name
        for ordering in orderings:
            self.load_ordering(ordering)
            action, reason = self._decide_ladder(game_state, context)
            results[ordering] = (action, reason)
        self.load_ordering(original)
        return results


# =============================================================================
# INTEGRATION ADAPTER
# =============================================================================

class CoreGameplayAdapter:
    """
    Adapter to integrate DecisionRungSystem with existing CoreGameplay._select_action().

    Enables phased migration:
    - PHASE 1: Shadow mode (run both, compare)
    - PHASE 2: Category takeover (rung system handles specific categories)
    - PHASE 3: Full replacement
    """

    def __init__(self, core_gameplay_ref: Any, ordering: str = 'comprehensive'):
        self.core: Any = core_gameplay_ref
        self.rung_system = DecisionRungSystem(
            strategy='ladder', core_gameplay_ref=core_gameplay_ref
        )
        self.rung_system.load_ordering(ordering)

        self.shadow_mode = False
        self.shadow_log: List[Dict[str, Any]] = []
        self.divergence_count = 0
        self.agreement_count = 0
        self._shadow_tester: Any = None
        self.category_enabled: Dict[str, bool] = {
            'emergency': False, 'filter': False, 'orientation': False,
            'hypothesis': False, 'exploitation': False, 'fallback': False,
        }

    def enable_shadow_mode(self, log_limit: int = 1000):
        """Enable shadow mode."""
        self.shadow_mode = True
        self.shadow_log = []
        self._shadow_log_limit = log_limit
        logger.info("[RUNG-ADAPTER] Shadow mode ENABLED")

    def get_shadow_tester(self) -> Any:
        """Lazy-load ShadowTester."""
        if self._shadow_tester is None:
            try:
                from engines.cognition.shadow_testing import ShadowTester
                self._shadow_tester = ShadowTester()
            except ImportError:
                logger.warning("[RUNG-ADAPTER] ShadowTester not available")
        return self._shadow_tester

    def disable_shadow_mode(self) -> Dict[str, Any]:
        """Disable shadow mode and return stats."""
        self.shadow_mode = False
        total = self.divergence_count + self.agreement_count
        agreement_rate = self.agreement_count / total if total > 0 else 0
        stats: Dict[str, Any] = {
            'total_comparisons': total, 'agreements': self.agreement_count,
            'divergences': self.divergence_count, 'agreement_rate': agreement_rate,
            'divergence_samples': self.shadow_log[-10:],
        }
        logger.info(f"[RUNG-ADAPTER] Shadow mode DISABLED - agreement rate: {agreement_rate:.1%}")
        return stats

    def shadow_compare(self, game_state: Any, context: Dict[str, Any], old_action: str) -> Dict[str, Any]:
        """Compare rung system decision with old system decision."""
        if not self.shadow_mode:
            return {}
        rung_action, rung_reason = self.rung_system.decide(game_state, context)
        agrees = rung_action == old_action
        if agrees:
            self.agreement_count += 1
        else:
            self.divergence_count += 1
            if len(self.shadow_log) < self._shadow_log_limit:
                self.shadow_log.append({
                    'old_action': old_action, 'rung_action': rung_action,
                    'rung_reason': rung_reason, 'ordering': self.rung_system.ordering_name,
                    'game_type': context.get('game_type'), 'level': context.get('level'),
                })
        return {'agrees': agrees, 'old_action': old_action, 'rung_action': rung_action, 'rung_reason': rung_reason}

    def enable_category(self, category: str):
        """Enable rung system for a specific category."""
        if category in self.category_enabled:
            self.category_enabled[category] = True

    def disable_category(self, category: str):
        """Disable rung system for a specific category."""
        if category in self.category_enabled:
            self.category_enabled[category] = False

    def decide_category(self, category: str, game_state: Any, context: Dict[str, Any]) -> RungResult:
        """Get decision from only rungs in a specific category."""
        if not self.category_enabled.get(category, False):
            return RungResult()
        category_rungs = [r for r in self.rung_system.rungs if r.category == category]
        if not category_rungs:
            return RungResult()
        for rung in sorted(category_rungs, key=lambda r: r.get_priority()):
            result = rung.evaluate(game_state, context)
            if result.has_suggestion(rung.confidence_threshold):
                return result
        return RungResult()

    def build_context_from_core(self, game_state: Any, loop_state: Any = None) -> Dict[str, Any]:
        """Build rung context from CoreGameplay state."""
        context: Dict[str, Any] = {}
        try:
            if hasattr(self.core, 'session_manager') and self.core.session_manager:
                game_id = self.core.session_manager.current_game_id
                context['game_id'] = game_id
                context['game_type'] = game_id[:4] if game_id and len(game_id) >= 4 else None

            if hasattr(game_state, 'score'):
                context['level'] = int(game_state.score) + 1
                context['score'] = game_state.score

            if loop_state:
                context['action_count'] = getattr(loop_state, 'action_count', 0)
                action_budget = context.get('action_budget') or getattr(
                    getattr(self.core, '_loop_config', None), 'max_actions', 400
                )
                context.setdefault('action_budget', action_budget)
                context['budget_used_percent'] = context['action_count'] / max(action_budget, 1)

            if hasattr(self.core, 'game_config'):
                context['agent_id'] = self.core.game_config.get('agent_id')
                context['agent_role'] = self.core.game_config.get('agent_role')

            context['w_A'] = getattr(self.core, '_current_wA', 0.5)
            context['w_B'] = getattr(self.core, '_current_wB', 0.5)
            context['agent_position'] = getattr(self.core, '_current_agent_position', None)

            available = context.get('available_actions', [1, 2, 3, 4, 5, 6, 7])
            default_safety = {a: 1.0 for a in available}
            context['action_safety_weights'] = getattr(self.core, '_action_safety_weights', default_safety)
            context['recent_actions'] = getattr(self.core, '_recent_actions', [])[-10:]
            context['game_type'] = context.get('game_id', '').split('-')[0] if context.get('game_id') else None

            context['is_frontier'] = self.core._is_frontier_level(
                context.get('game_id', ''), context.get('level', 1)
            ) if hasattr(self.core, '_is_frontier_level') else False

        except Exception as e:
            logger.debug(f"[RUNG-ADAPTER] Context build partial failure: {e}")

        return context

    def full_decide(self, game_state: Any, loop_state: Any = None) -> Tuple[str, str]:
        """Full replacement for _select_action() - PHASE 3."""
        context = self.build_context_from_core(game_state, loop_state)
        return self.rung_system.decide(game_state, context)


# =============================================================================
# HELPER: Create custom ordering interactively
# =============================================================================

def create_custom_ordering(name: str, rung_priorities: Dict[str, int]) -> List[Tuple[str, int]]:
    """Create a custom ordering."""
    return [(rung, priority) for rung, priority in sorted(rung_priorities.items(), key=lambda x: x[1])]


if __name__ == '__main__':
    print("=" * 60)
    print("DECISION RUNG SYSTEM - MODULAR ACTION ARCHITECTURE")
    print("=" * 60)

    print("\nAvailable Rungs:")
    for name, cls in DecisionRungSystem.RUNG_REGISTRY.items():
        category = getattr(cls, 'category', 'unknown')
        priority = getattr(cls, 'default_priority', 50)
        print(f"  - {name}: {category} (default priority: {priority})")

    print("\nAvailable Orderings:")
    for name, ordering in ORDERING_PRESETS.items():
        rungs_list = [r[0] for r in ordering]
        print(f"  - {name}: {len(ordering)} rungs")
        print(f"    Order: {' -> '.join(rungs_list[:5])}...")

    print("=" * 60)

## Cognitive Phases & Rung Roles

Every rung maps to a primary cognitive phase in the 7-phase solver pipeline:

```
OBSERVE -> CLASSIFY -> EXTRACT_GOAL -> MAP_EFFECTS -> PLAN -> EXECUTE -> VERIFY
```

And a problem-solving role:

| Role | Confidence Range | Purpose |
|------|-----------------|---------|
| **ENTRY** | 0.3 - 0.6 | Low-friction starting points |
| **LEVERAGE** | 0.5 - 0.75 | Build on entry points |
| **COMPOUNDING** | 0.6 - 0.85 | Knowledge spreads, connections multiply |
| **RESOLUTION** | 0.8 - 1.0 | Path crystallizes, commit to action |

In [ ]:
"""
Rung Role Taxonomy - Phase 7.2 + H41 Cognitive Phase Mapping.

Two complementary taxonomies:

1. **RungRole** (4-tier): Problem-solving progression
   - ENTRY -> LEVERAGE -> COMPOUNDING -> RESOLUTION

2. **CognitivePhase** (7-phase): Solver cognitive pipeline
   - OBSERVE -> CLASSIFY -> EXTRACT_GOAL -> MAP_EFFECTS -> PLAN -> EXECUTE -> VERIFY
   - Mirrors the universal pattern behind all solver solutions.
   - Used by H41 rung affinity to group learned affinities by phase.

Usage:
    from engines.cognition.rung_roles import (
        RungRole, get_rung_role,
        CognitivePhase, get_cognitive_phase, get_rungs_by_phase,
    )

    role = get_rung_role("survey")           # RungRole.ENTRY
    phase = get_cognitive_phase("survey")    # CognitivePhase.OBSERVE
    plan_rungs = get_rungs_by_phase(CognitivePhase.PLAN)  # [...]
"""
# RULE 1: PYTHONDONTWRITEBYTECODE=1 (no .pyc files)
import sys


# =============================================================================
# RUNG ROLE ENUM
# =============================================================================

class RungRole(Enum):
    """
    Roles rungs play in the universal problem-solving pattern.

    From Part 4: Problem-solving follows a universal pattern:
    1. Entry - Find easy starting points
    2. Leverage - Use entry points to reach harder targets
    3. Compounding - Knowledge spreads, connections multiply
    4. Resolution - Path becomes obvious, commit to action
    """
    ENTRY = "entry"              # Low-friction starting points
    LEVERAGE = "leverage"        # Build on entry points to reach harder targets
    COMPOUNDING = "compounding"  # Connections multiply, knowledge spreads
    RESOLUTION = "resolution"    # Path becomes obvious, commit to action

    def __str__(self) -> str:
        return self.value

    @property
    def description(self) -> str:
        """Human-readable description of this role."""
        descriptions = {
            RungRole.ENTRY: "Low-friction starting points for orientation",
            RungRole.LEVERAGE: "Build on entry points for deeper understanding",
            RungRole.COMPOUNDING: "Knowledge spreads, connections multiply",
            RungRole.RESOLUTION: "Path crystallizes, commit to action",
        }
        return descriptions[self]

    @property
    def typical_confidence_range(self) -> tuple:
        """Typical confidence range for rungs in this role."""
        ranges = {
            RungRole.ENTRY: (0.3, 0.6),        # Exploratory, lower confidence
            RungRole.LEVERAGE: (0.5, 0.75),    # Building understanding
            RungRole.COMPOUNDING: (0.6, 0.85), # Confidence growing
            RungRole.RESOLUTION: (0.8, 1.0),   # High confidence for action
        }
        return ranges[self]


# =============================================================================
# COGNITIVE PHASE ENUM -- The 7-Phase Solver Pipeline
# =============================================================================

class CognitivePhase(Enum):
    """
    The 7-phase cognitive pipeline behind all solver solutions.

    OBSERVE -> CLASSIFY -> EXTRACT_GOAL -> MAP_EFFECTS -> PLAN -> EXECUTE -> VERIFY

    Each rung participates primarily in one phase. Context-setter rungs
    (confidence=0.0) are pure phase contributions. Action-proposing rungs
    participate in their phase AND implicitly in EXECUTE.

    Used by H41 rung affinity to report phase-level coverage per game type.
    """
    OBSERVE = "observe"            # Perceive state, detect features
    CLASSIFY = "classify"          # Determine game type, mechanics, problem class
    EXTRACT_GOAL = "extract_goal"  # Identify target state, constraints
    MAP_EFFECTS = "map_effects"    # Learn action -> outcome mappings
    PLAN = "plan"                  # Compute action sequence toward goal
    EXECUTE = "execute"            # Select and commit to next action
    VERIFY = "verify"              # Check outcome, track progress, detect errors

    def __str__(self) -> str:
        return self.value

    @property
    def description(self) -> str:
        descriptions = {
            CognitivePhase.OBSERVE: "Perceive game state, extract visual features",
            CognitivePhase.CLASSIFY: "Determine game type and problem class",
            CognitivePhase.EXTRACT_GOAL: "Identify target state and constraints",
            CognitivePhase.MAP_EFFECTS: "Learn what each action does",
            CognitivePhase.PLAN: "Compute optimal action sequence",
            CognitivePhase.EXECUTE: "Select and commit to action",
            CognitivePhase.VERIFY: "Check outcome against prediction",
        }
        return descriptions[self]

    @property
    def maps_to_role(self) -> RungRole:
        """Which RungRole this phase most naturally maps to."""
        mapping = {
            CognitivePhase.OBSERVE: RungRole.ENTRY,
            CognitivePhase.CLASSIFY: RungRole.ENTRY,
            CognitivePhase.EXTRACT_GOAL: RungRole.LEVERAGE,
            CognitivePhase.MAP_EFFECTS: RungRole.LEVERAGE,
            CognitivePhase.PLAN: RungRole.COMPOUNDING,
            CognitivePhase.EXECUTE: RungRole.RESOLUTION,
            CognitivePhase.VERIFY: RungRole.COMPOUNDING,
        }
        return mapping[self]


# =============================================================================
# COGNITIVE PHASE MAP -- Every real rung -> its primary phase
# =============================================================================

# Maps all actual rung names (from rung registries) to their primary cognitive
# phase. Rungs may contribute to multiple phases, but this records the PRIMARY
# phase where the rung's core logic lives.
COGNITIVE_PHASE_MAP: Dict[str, CognitivePhase] = {
    # -------------------------------------------------------------------------
    # OBSERVE: Perceive state, detect features, scan environment
    # -------------------------------------------------------------------------
    "survey": CognitivePhase.OBSERVE,
    "frame_interpretation": CognitivePhase.OBSERVE,
    "sparse_grid": CognitivePhase.OBSERVE,
    "palette_detection": CognitivePhase.OBSERVE,
    "affordance_detection": CognitivePhase.OBSERVE,
    "visual_analyzer": CognitivePhase.OBSERVE,
    "network_object_inventory": CognitivePhase.OBSERVE,
    "action6_object_exploration": CognitivePhase.OBSERVE,
    "exploration_phase": CognitivePhase.OBSERVE,
    "grid_exploration": CognitivePhase.OBSERVE,
    "control_tracker": CognitivePhase.OBSERVE,
    "questioning_engine": CognitivePhase.OBSERVE,
    "network_exploration_stats": CognitivePhase.OBSERVE,
    "self_trust_boost": CognitivePhase.OBSERVE,

    # -------------------------------------------------------------------------
    # CLASSIFY: Determine game type, problem class, click semantics
    # -------------------------------------------------------------------------
    "game_classifier": CognitivePhase.CLASSIFY,
    "sensation_engine": CognitivePhase.CLASSIFY,
    "two_streams": CognitivePhase.CLASSIFY,

    # -------------------------------------------------------------------------
    # EXTRACT_GOAL: Identify target state, constraints, subgoals
    # -------------------------------------------------------------------------
    "solver_goal_extraction": CognitivePhase.EXTRACT_GOAL,
    "constraint_decoder": CognitivePhase.EXTRACT_GOAL,
    "interactable_tile_discovery": CognitivePhase.EXTRACT_GOAL,
    "goal_relationship_modeling": CognitivePhase.EXTRACT_GOAL,
    "valence_goals": CognitivePhase.EXTRACT_GOAL,
    "subgoal_planning": CognitivePhase.EXTRACT_GOAL,

    # -------------------------------------------------------------------------
    # MAP_EFFECTS: Learn action -> outcome, build causal model
    # -------------------------------------------------------------------------
    "causal_click_mapping": CognitivePhase.MAP_EFFECTS,
    "click_behavior_learning": CognitivePhase.MAP_EFFECTS,
    "spatial_relationship": CognitivePhase.MAP_EFFECTS,
    "event_understanding": CognitivePhase.MAP_EFFECTS,
    "object_color_targeting": CognitivePhase.MAP_EFFECTS,
    "hypothesis_system": CognitivePhase.MAP_EFFECTS,
    "hypothesis_testing": CognitivePhase.MAP_EFFECTS,
    "scientific_method": CognitivePhase.MAP_EFFECTS,
    "assumption_formation": CognitivePhase.MAP_EFFECTS,
    "belief_system": CognitivePhase.MAP_EFFECTS,
    "resonance_detector": CognitivePhase.MAP_EFFECTS,
    "symbolic_tracker": CognitivePhase.MAP_EFFECTS,
    "map_intel_collision": CognitivePhase.MAP_EFFECTS,
    "effect_prediction": CognitivePhase.MAP_EFFECTS,

    # -------------------------------------------------------------------------
    # PLAN: Compute action sequence, solve constraints, navigate
    # -------------------------------------------------------------------------
    "constraint_satisfaction": CognitivePhase.PLAN,
    "wall_aware_navigation": CognitivePhase.PLAN,
    "spatial_map": CognitivePhase.PLAN,
    "controlled_movement_planning": CognitivePhase.PLAN,
    "trigger_sequences": CognitivePhase.PLAN,
    "theory_gate": CognitivePhase.PLAN,
    "metacognitive_prediction": CognitivePhase.PLAN,
    "deliberation_system": CognitivePhase.PLAN,
    "i_thread": CognitivePhase.PLAN,
    "budget_aware_planning": CognitivePhase.PLAN,
    "distance_guided_click": CognitivePhase.PLAN,

    # -------------------------------------------------------------------------
    # EXECUTE: Select action, commit, fallback strategies
    # -------------------------------------------------------------------------
    "smart_action_selection": CognitivePhase.EXECUTE,
    "three_try_sequence": CognitivePhase.EXECUTE,
    "discovery_exploitation": CognitivePhase.EXECUTE,
    "embedding_suggestion": CognitivePhase.EXECUTE,
    "embedding_matcher": CognitivePhase.EXECUTE,
    "network_wisdom": CognitivePhase.EXECUTE,
    "network_sharing": CognitivePhase.EXECUTE,
    "primitive_suggester": CognitivePhase.EXECUTE,
    "state_matching": CognitivePhase.EXECUTE,
    "multi_stage_matching": CognitivePhase.EXECUTE,
    "replay_learning": CognitivePhase.EXECUTE,
    "few_shot_invariants": CognitivePhase.EXECUTE,
    "few_shot_relations": CognitivePhase.EXECUTE,
    "abstraction_templates": CognitivePhase.EXECUTE,

    # -------------------------------------------------------------------------
    # VERIFY: Check outcome, track progress, detect errors
    # -------------------------------------------------------------------------
    "goal_progress": CognitivePhase.VERIFY,
    "near_miss_analyzer": CognitivePhase.VERIFY,
    "completion_prediction": CognitivePhase.VERIFY,
    "rule_transfer": CognitivePhase.VERIFY,
    "frontier_topology": CognitivePhase.VERIFY,
    "frontier_checkpoint": CognitivePhase.VERIFY,
    "theory_contradiction": CognitivePhase.VERIFY,
    "metacognitive_elimination": CognitivePhase.VERIFY,
    "contextual_failure": CognitivePhase.VERIFY,
    "action_outcome_verifier": CognitivePhase.VERIFY,

    # -------------------------------------------------------------------------
    # SAFETY RUNGS: Cross-cutting (mapped to VERIFY -- they check/filter)
    # -------------------------------------------------------------------------
    "death_avoidance": CognitivePhase.VERIFY,
    "prior_lessons": CognitivePhase.VERIFY,
    "three_layer_filter": CognitivePhase.VERIFY,
    "pariah_avoidance": CognitivePhase.VERIFY,
    "terminal_pattern": CognitivePhase.VERIFY,
    "destructive_action_detection": CognitivePhase.VERIFY,
    "viral_package_weights": CognitivePhase.EXECUTE,

    # -------------------------------------------------------------------------
    # EMERGENCY: Always-first (mapped to OBSERVE -- they perceive danger)
    # -------------------------------------------------------------------------
    "infinite_loop_breaker": CognitivePhase.OBSERVE,
    "coordinate_oscillation": CognitivePhase.OBSERVE,

    # -------------------------------------------------------------------------
    # BUDGET/META: Resource tracking (mapped to PLAN -- they constrain)
    # -------------------------------------------------------------------------
    "frustration_detection": CognitivePhase.VERIFY,
    "breakthrough_budget": CognitivePhase.PLAN,
    "regulatory_signal": CognitivePhase.PLAN,
    "imagination_budget": CognitivePhase.PLAN,
}


# =============================================================================
# RUNG ROLE MAPPING -- Grounded to actual rung names
# =============================================================================

# Map rungs to their primary role in problem-solving.
# All entries correspond to actual registered rung names.
RUNG_ROLE_MAP: Dict[str, RungRole] = {
    # ENTRY: Orientation, perception, broad survey
    "survey": RungRole.ENTRY,
    "frame_interpretation": RungRole.ENTRY,
    "sparse_grid": RungRole.ENTRY,
    "palette_detection": RungRole.ENTRY,
    "affordance_detection": RungRole.ENTRY,
    "visual_analyzer": RungRole.ENTRY,
    "network_object_inventory": RungRole.ENTRY,
    "action6_object_exploration": RungRole.ENTRY,
    "exploration_phase": RungRole.ENTRY,
    "grid_exploration": RungRole.ENTRY,
    "questioning_engine": RungRole.ENTRY,
    "game_classifier": RungRole.ENTRY,
    "control_tracker": RungRole.ENTRY,
    "network_exploration_stats": RungRole.ENTRY,
    "self_trust_boost": RungRole.ENTRY,
    "infinite_loop_breaker": RungRole.ENTRY,
    "coordinate_oscillation": RungRole.ENTRY,
    "sensation_engine": RungRole.ENTRY,
    "two_streams": RungRole.ENTRY,

    # LEVERAGE: Build understanding, test hypotheses, map effects
    "solver_goal_extraction": RungRole.LEVERAGE,
    "constraint_decoder": RungRole.LEVERAGE,
    "interactable_tile_discovery": RungRole.LEVERAGE,
    "goal_relationship_modeling": RungRole.LEVERAGE,
    "causal_click_mapping": RungRole.LEVERAGE,
    "click_behavior_learning": RungRole.LEVERAGE,
    "spatial_relationship": RungRole.LEVERAGE,
    "event_understanding": RungRole.LEVERAGE,
    "object_color_targeting": RungRole.LEVERAGE,
    "hypothesis_system": RungRole.LEVERAGE,
    "hypothesis_testing": RungRole.LEVERAGE,
    "scientific_method": RungRole.LEVERAGE,
    "assumption_formation": RungRole.LEVERAGE,
    "belief_system": RungRole.LEVERAGE,
    "resonance_detector": RungRole.LEVERAGE,
    "symbolic_tracker": RungRole.LEVERAGE,
    "map_intel_collision": RungRole.LEVERAGE,
    "effect_prediction": RungRole.LEVERAGE,
    "valence_goals": RungRole.LEVERAGE,
    "subgoal_planning": RungRole.LEVERAGE,

    # COMPOUNDING: Connect knowledge, plan sequences
    "constraint_satisfaction": RungRole.COMPOUNDING,
    "wall_aware_navigation": RungRole.COMPOUNDING,
    "spatial_map": RungRole.COMPOUNDING,
    "controlled_movement_planning": RungRole.COMPOUNDING,
    "trigger_sequences": RungRole.COMPOUNDING,
    "theory_gate": RungRole.COMPOUNDING,
    "metacognitive_prediction": RungRole.COMPOUNDING,
    "deliberation_system": RungRole.COMPOUNDING,
    "i_thread": RungRole.COMPOUNDING,
    "rule_transfer": RungRole.COMPOUNDING,
    "near_miss_analyzer": RungRole.COMPOUNDING,
    "completion_prediction": RungRole.COMPOUNDING,
    "frontier_topology": RungRole.COMPOUNDING,
    "frontier_checkpoint": RungRole.COMPOUNDING,
    "goal_progress": RungRole.COMPOUNDING,
    "action_outcome_verifier": RungRole.COMPOUNDING,
    "frustration_detection": RungRole.COMPOUNDING,
    "distance_guided_click": RungRole.COMPOUNDING,

    # RESOLUTION: Select action, commit, exploit knowledge
    "smart_action_selection": RungRole.RESOLUTION,
    "three_try_sequence": RungRole.RESOLUTION,
    "discovery_exploitation": RungRole.RESOLUTION,
    "embedding_suggestion": RungRole.RESOLUTION,
    "embedding_matcher": RungRole.RESOLUTION,
    "network_wisdom": RungRole.RESOLUTION,
    "network_sharing": RungRole.RESOLUTION,
    "primitive_suggester": RungRole.RESOLUTION,
    "state_matching": RungRole.RESOLUTION,
    "multi_stage_matching": RungRole.RESOLUTION,
    "replay_learning": RungRole.RESOLUTION,
    "few_shot_invariants": RungRole.RESOLUTION,
    "few_shot_relations": RungRole.RESOLUTION,
    "abstraction_templates": RungRole.RESOLUTION,
    "death_avoidance": RungRole.RESOLUTION,
    "prior_lessons": RungRole.RESOLUTION,
    "three_layer_filter": RungRole.RESOLUTION,
    "pariah_avoidance": RungRole.RESOLUTION,
    "terminal_pattern": RungRole.RESOLUTION,
    "destructive_action_detection": RungRole.RESOLUTION,
    "budget_aware_planning": RungRole.RESOLUTION,
    "theory_contradiction": RungRole.RESOLUTION,
    "viral_package_weights": RungRole.RESOLUTION,
    "metacognitive_elimination": RungRole.RESOLUTION,
    "contextual_failure": RungRole.RESOLUTION,
    "breakthrough_budget": RungRole.RESOLUTION,
    "regulatory_signal": RungRole.RESOLUTION,
    "imagination_budget": RungRole.RESOLUTION,
}


# =============================================================================
# PHASE TO ROLE MAPPING
# =============================================================================

PHASE_ROLE_MAP: Dict[str, RungRole] = {
    "exploration": RungRole.ENTRY,
    "building": RungRole.LEVERAGE,
    "connecting": RungRole.COMPOUNDING,
    "resolving": RungRole.RESOLUTION,
    # Aliases
    "orient": RungRole.ENTRY,
    "investigate": RungRole.LEVERAGE,
    "synthesize": RungRole.COMPOUNDING,
    "act": RungRole.RESOLUTION,
}


# =============================================================================
# ROLE TRANSITIONS
# =============================================================================

# Valid role transitions (from -> to)
# Some transitions are natural progressions, others are backtracking
VALID_ROLE_TRANSITIONS: Dict[RungRole, Set[RungRole]] = {
    RungRole.ENTRY: {RungRole.ENTRY, RungRole.LEVERAGE},
    RungRole.LEVERAGE: {RungRole.ENTRY, RungRole.LEVERAGE, RungRole.COMPOUNDING},
    RungRole.COMPOUNDING: {RungRole.LEVERAGE, RungRole.COMPOUNDING, RungRole.RESOLUTION},
    RungRole.RESOLUTION: {RungRole.COMPOUNDING, RungRole.RESOLUTION},
}

# Transitions that indicate backtracking (need more information)
BACKTRACK_TRANSITIONS: Set[tuple] = {
    (RungRole.LEVERAGE, RungRole.ENTRY),
    (RungRole.COMPOUNDING, RungRole.LEVERAGE),
    (RungRole.COMPOUNDING, RungRole.ENTRY),
    (RungRole.RESOLUTION, RungRole.COMPOUNDING),
    (RungRole.RESOLUTION, RungRole.LEVERAGE),
}


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_rung_role(rung_name: str) -> RungRole:
    """
    Get the role for a rung.

    Args:
        rung_name: Name of the rung

    Returns:
        RungRole for the rung, defaults to ENTRY if unknown
    """
    return RUNG_ROLE_MAP.get(rung_name, RungRole.ENTRY)


def get_cognitive_phase(rung_name: str) -> CognitivePhase:
    """Get the primary cognitive phase for a rung.

    Falls back to EXECUTE if rung is unknown (action-proposing by default).
    """
    return COGNITIVE_PHASE_MAP.get(rung_name, CognitivePhase.EXECUTE)


def get_rungs_by_phase(phase: CognitivePhase) -> List[str]:
    """Get all rungs in a specific cognitive phase."""
    return [rung for rung, p in COGNITIVE_PHASE_MAP.items() if p == phase]


def get_phase_coverage() -> Dict[str, int]:
    """Get rung count per cognitive phase -- used by H41 affinity summary."""
    counts: Dict[str, int] = {}
    for phase in CognitivePhase:
        counts[phase.value] = sum(
            1 for p in COGNITIVE_PHASE_MAP.values() if p == phase
        )
    return counts


def get_role_for_phase(phase: str) -> RungRole:
    """
    Map problem-solving phase to appropriate rung role.

    Args:
        phase: Phase name (exploration, building, connecting, resolving)

    Returns:
        RungRole for the phase
    """
    return PHASE_ROLE_MAP.get(phase.lower(), RungRole.ENTRY)


def get_rungs_by_role(role: RungRole) -> List[str]:
    """
    Get all rungs with a specific role.

    Args:
        role: The RungRole to filter by

    Returns:
        List of rung names with that role
    """
    return [rung for rung, r in RUNG_ROLE_MAP.items() if r == role]


def is_valid_transition(from_role: RungRole, to_role: RungRole) -> bool:
    """
    Check if a role transition is valid.

    Args:
        from_role: Current role
        to_role: Target role

    Returns:
        True if transition is valid
    """
    return to_role in VALID_ROLE_TRANSITIONS.get(from_role, set())


def is_backtrack_transition(from_role: RungRole, to_role: RungRole) -> bool:
    """
    Check if a transition represents backtracking.

    Args:
        from_role: Current role
        to_role: Target role

    Returns:
        True if this is a backtrack transition
    """
    return (from_role, to_role) in BACKTRACK_TRANSITIONS


def extract_role_sequence(path: List[str]) -> List[RungRole]:
    """
    Extract the role sequence from a concrete path.

    Args:
        path: List of rung names

    Returns:
        List of RungRoles
    """
    return [get_rung_role(rung) for rung in path]


def role_sequence_to_id(roles: List[RungRole]) -> str:
    """
    Convert a role sequence to a pattern ID string.

    Args:
        roles: List of RungRoles

    Returns:
        Pattern ID string (e.g., "ENTRY->LEVERAGE->RESOLUTION")
    """
    return "->".join(role.name for role in roles)


def count_backtrack_transitions(path: List[str]) -> int:
    """
    Count backtrack transitions in a path.

    Args:
        path: List of rung names

    Returns:
        Number of backtrack transitions
    """
    if len(path) < 2:
        return 0

    count = 0
    roles = extract_role_sequence(path)

    for i in range(len(roles) - 1):
        if is_backtrack_transition(roles[i], roles[i + 1]):
            count += 1

    return count


def analyze_path_structure(path: List[str]) -> Dict:
    """
    Analyze the structure of a path.

    Args:
        path: List of rung names

    Returns:
        Dictionary with path analysis
    """
    if not path:
        return {
            'length': 0,
            'roles': [],
            'pattern_id': '',
            'backtrack_count': 0,
            'role_distribution': {},
            'is_progressive': True,
        }

    roles = extract_role_sequence(path)
    pattern_id = role_sequence_to_id(roles)
    backtrack_count = count_backtrack_transitions(path)

    # Count role distribution
    role_distribution = {}
    for role in RungRole:
        role_distribution[role.name] = sum(1 for r in roles if r == role)

    # Check if progressive (no backtracks)
    is_progressive = backtrack_count == 0

    return {
        'length': len(path),
        'roles': [r.name for r in roles],
        'pattern_id': pattern_id,
        'backtrack_count': backtrack_count,
        'role_distribution': role_distribution,
        'is_progressive': is_progressive,
    }


# =============================================================================
# ROLE COMPATIBILITY
# =============================================================================

def get_compatible_roles(epistemic_quadrant: str) -> List[RungRole]:
    """
    Get roles that are compatible with an epistemic quadrant.

    Args:
        epistemic_quadrant: KK, KU, UK, or UU

    Returns:
        List of compatible roles (ordered by preference)
    """
    compatibility = {
        'KK': [RungRole.RESOLUTION, RungRole.COMPOUNDING],  # High confidence, resolve
        'KU': [RungRole.LEVERAGE, RungRole.COMPOUNDING],    # Building understanding
        'UK': [RungRole.ENTRY, RungRole.LEVERAGE],          # Need to explore
        'UU': [RungRole.ENTRY],                             # Start fresh
    }
    return compatibility.get(epistemic_quadrant, [RungRole.ENTRY])


def suggest_next_role(current_role: RungRole, confidence: float) -> RungRole:
    """
    Suggest the next role based on current role and confidence.

    Args:
        current_role: Current role
        confidence: Current confidence level (0-1)

    Returns:
        Suggested next role
    """
    # Low confidence: stay in current phase or backtrack
    if confidence < 0.4:
        if current_role in (RungRole.RESOLUTION, RungRole.COMPOUNDING):
            return RungRole.LEVERAGE  # Backtrack
        return current_role  # Stay

    # Medium confidence: progress one step
    if confidence < 0.7:
        progressions = {
            RungRole.ENTRY: RungRole.LEVERAGE,
            RungRole.LEVERAGE: RungRole.COMPOUNDING,
            RungRole.COMPOUNDING: RungRole.COMPOUNDING,  # Build more
            RungRole.RESOLUTION: RungRole.RESOLUTION,
        }
        return progressions.get(current_role, current_role)

    # High confidence: progress toward resolution
    progressions = {
        RungRole.ENTRY: RungRole.LEVERAGE,
        RungRole.LEVERAGE: RungRole.COMPOUNDING,
        RungRole.COMPOUNDING: RungRole.RESOLUTION,
        RungRole.RESOLUTION: RungRole.RESOLUTION,
    }
    return progressions.get(current_role, RungRole.RESOLUTION)


logger.info(
    f"[RUNG-ROLES] Loaded {len(RUNG_ROLE_MAP)} role + "
    f"{len(COGNITIVE_PHASE_MAP)} phase mappings"
)

## Game-Playing Loop -- Cognitive Rungs vs Real ARC Games

The DecisionRungSystem plays every ARC game. For each game step:
1. The rung cascade evaluates the current observation
2. The winning rung selects an action (with coordinates for ACTION6)
3. The action is executed via the game environment
4. Results feed back into the next decision cycle

This is the full PTMA loop running against real games -- no fake decisions.

In [ ]:
# -- THREADED SWARM: Play ALL games in parallel ----------------------------
# Mirrors the official ARC-AGI-3 Swarm pattern (random agent scored 0.18):
#   1. ALL games run in parallel threads (I/O overlap on HTTP calls)
#   2. Each game gets the FULL time budget, not 1/Nth
#   3. RESET on GAME_OVER -- retry until WIN or time expires
#   4. No action cap -- time is the only constraint
import time, copy, traceback, random as _rnd, threading

T_START = time.time()
T_LIMIT_HOURS = 5.0          # leave 0.5h buffer for setup/teardown
T_LIMIT_SECS = T_LIMIT_HOURS * 3600

# Create scorecard
scorecard_id = arcade.create_scorecard(
    source_url='https://github.com/BitterTruth-AI',
    tags=['bittertruth', 'cognitive-rungs', 'swarm-v46'],
)
print(f'Scorecard: {scorecard_id}')
print(f'Playing {len(games)} games in PARALLEL with {T_LIMIT_HOURS}h budget each')
print()

# -- Helper: parse action string from decide() -> GameAction + data --------
def _parse_action(action_str, metadata):
    """Convert decide() output to (GameAction, data_dict)."""
    if isinstance(action_str, int):
        action_num = action_str
    elif isinstance(action_str, str) and action_str.startswith('ACTION'):
        try:
            action_num = int(action_str.replace('ACTION', ''))
        except ValueError:
            action_num = 1
    else:
        action_num = 1

    game_action = getattr(GameAction, f'ACTION{action_num}', GameAction.ACTION1)

    data = None
    if action_num == 6 and metadata:
        if 'pixel_position' in metadata:
            px, py = metadata['pixel_position']
            data = {'x': int(px), 'y': int(py)}
        elif 'target' in metadata and isinstance(metadata['target'], dict):
            t = metadata['target']
            data = {'x': int(t.get('x', 32)), 'y': int(t.get('y', 32))}
        elif 'x' in metadata and 'y' in metadata:
            data = {'x': int(metadata['x']), 'y': int(metadata['y'])}
        else:
            data = {'x': _rnd.randint(0, 63), 'y': _rnd.randint(0, 63)}

    return game_action, data


# -- Thread-safe results list ----------------------------------------------
_results_lock = threading.Lock()
all_results = []


def _play_one_game(game_info, game_idx, total_games):
    """Play a single game in its own thread. Retries on GAME_OVER until WIN or timeout."""
    game_id = game_info.game_id
    result = {'game_id': game_id, 'score': 0.0, 'levels': 0,
              'actions': 0, 'resets': 0, 'elapsed': 0}
    try:
        # Each thread gets its own rung system instance (thread safety)
        system = DecisionRungSystem(strategy='ladder')
        system.load_ordering('efficiency')

        env = arcade.make(game_id, scorecard_id=scorecard_id)
        obs = env.reset()

        available_actions = getattr(obs, 'available_actions', None)                             or getattr(game_info, 'available_actions', [1, 2, 3, 4, 6])
        win_levels = getattr(obs, 'win_levels', None)                      or getattr(game_info, 'win_levels', 5)

        has_click = 6 in available_actions
        has_move = bool(set(available_actions) & {1, 2, 3, 4})
        if has_click and not has_move:
            try:
                system.load_ordering('action6_only')
            except Exception:
                system.load_ordering('efficiency')
        else:
            system.load_ordering('efficiency')

        print(f'[{game_idx+1}/{total_games}] {game_id} actions={available_actions} '
              f'win_levels={win_levels} ordering={system.ordering_name}')

        score = 0.0
        actions_taken = 0
        best_levels = 0
        prev_frame = None
        stuck_count = 0
        resets = 0

        game_state = {
            'frame': obs.frame.tolist() if hasattr(getattr(obs, 'frame', None), 'tolist') else getattr(obs, 'frame', [[0]*64 for _ in range(64)]),
            'score': getattr(obs, 'score', 0),
            'level': (getattr(obs, 'levels_completed', 0) or 0) + 1,
        }

        context = {
            'available_actions': available_actions,
            'game_id': game_id,
            'game_type': 'click' if (has_click and not has_move) else 'movement',
            'agent_id': 'rung_player',
            'action_count': 0,
            'level_number': 1,
            'level': 1,
            'recent_stuck_count': 0,
            'frame_changed': True,
            'fallback_strategy': 'balanced',
            'win_levels': win_levels,
        }

        t_game_start = time.time()

        while True:
            # Global time budget check
            if (time.time() - T_START) > T_LIMIT_SECS:
                break

            # ---- Handle terminal states ----
            obs_state = getattr(obs, 'state', None)

            if obs_state == GameState.WIN:
                best_levels = getattr(obs, 'levels_completed', win_levels) or win_levels
                score = best_levels / win_levels if win_levels > 0 else 1.0
                break

            if obs_state == GameState.GAME_OVER or (
                hasattr(GameState, 'NOT_PLAYED') and obs_state == GameState.NOT_PLAYED
            ):
                cur_lv = getattr(obs, 'levels_completed', 0) or 0
                best_levels = max(best_levels, cur_lv)
                try:
                    obs = env.step(GameAction.RESET)
                    actions_taken += 1
                    resets += 1
                except Exception:
                    break
                new_frame = getattr(obs, 'frame', None)
                prev_frame = new_frame.copy() if hasattr(new_frame, 'copy') else new_frame
                game_state = {
                    'frame': new_frame.tolist() if hasattr(new_frame, 'tolist') else (new_frame or game_state['frame']),
                    'score': getattr(obs, 'score', 0),
                    'level': (getattr(obs, 'levels_completed', 0) or 0) + 1,
                }
                context['level_number'] = game_state['level']
                context['level'] = game_state['level']
                context['action_count'] = actions_taken
                stuck_count = 0
                continue

            # ---- DECIDE using rung cascade ----
            context['action_count'] = actions_taken
            context['recent_stuck_count'] = stuck_count

            try:
                action_str, reason = system.decide(game_state, context)
                metadata = getattr(system, 'last_decision_metadata', {}) or {}
            except Exception:
                an = _rnd.choice(available_actions)
                action_str = f'ACTION{an}'
                reason = '[fallback] rung cascade error'
                metadata = {}
                if an == 6:
                    metadata = {'x': _rnd.randint(0, 63), 'y': _rnd.randint(0, 63)}

            # ---- EXECUTE action ----
            game_action, action_data = _parse_action(action_str, metadata)

            try:
                obs = env.step(game_action, data=action_data)
                actions_taken += 1
            except Exception:
                actions_taken += 1
                continue

            if obs is None:
                break

            # ---- UPDATE state ----
            new_frame = getattr(obs, 'frame', None)
            try:
                if new_frame is not None and prev_frame is not None:
                    if hasattr(new_frame, 'tolist'):
                        frame_changed = (new_frame.tolist() != prev_frame.tolist()
                                         if hasattr(prev_frame, 'tolist')
                                         else new_frame.tolist() != prev_frame)
                    else:
                        frame_changed = (new_frame != prev_frame)
                        if hasattr(frame_changed, '__len__'):
                            frame_changed = True
                else:
                    frame_changed = True
            except Exception:
                frame_changed = True
            prev_frame = new_frame if not hasattr(new_frame, 'copy') else new_frame.copy()

            if frame_changed:
                stuck_count = 0
            else:
                stuck_count += 1
            context['frame_changed'] = frame_changed

            cur_levels = (getattr(obs, 'levels_completed', 0) or 0)
            best_levels = max(best_levels, cur_levels)
            game_state = {
                'frame': new_frame.tolist() if hasattr(new_frame, 'tolist') else (new_frame or game_state['frame']),
                'score': getattr(obs, 'score', 0),
                'level': cur_levels + 1,
            }
            context['level_number'] = game_state['level']
            context['level'] = game_state['level']

        # Finalize
        if score == 0.0 and best_levels > 0:
            score = best_levels / win_levels if win_levels > 0 else 0.0

        result = {
            'game_id': game_id,
            'score': score,
            'levels': best_levels,
            'actions': actions_taken,
            'resets': resets,
            'elapsed': time.time() - t_game_start,
        }

    except Exception as _game_err:
        print(f'  [{game_id}] ERROR: {_game_err}')
        traceback.print_exc()

    with _results_lock:
        all_results.append(result)

    print(f'  [{game_id}] score={result["score"]:.3f} levels={result["levels"]} '
          f'actions={result["actions"]} resets={result["resets"]} '
          f'time={result["elapsed"]:.1f}s')


# -- Launch all game threads -----------------------------------------------
threads = []
for i, game_info in enumerate(games):
    t = threading.Thread(
        target=_play_one_game,
        args=(game_info, i, len(games)),
        daemon=True,
    )
    threads.append(t)

print(f'Launching {len(threads)} game threads...')
for t in threads:
    t.start()

# Wait for all to finish
for t in threads:
    t.join(timeout=T_LIMIT_SECS + 60)

# Kill any stragglers still running past the budget
print()

# -- Finalize scorecard + write submission.parquet -------------------------
print('=' * 60)
print(f'DONE. Total time: {(time.time()-T_START)/60:.1f} min')
print(f'Games played: {len(all_results)}/{len(games)}')
avg_score = sum(r["score"] for r in all_results) / max(len(all_results), 1)
total_resets = sum(r.get("resets", 0) for r in all_results)
total_actions = sum(r.get("actions", 0) for r in all_results)
print(f'Average score: {avg_score:.4f}')
print(f'Total actions: {total_actions}  Total resets: {total_resets}')
print()

try:
    _final_scorecard = arcade.close_scorecard(scorecard_id)
    if _final_scorecard:
        print(f'Scorecard closed. Official score: {getattr(_final_scorecard, "score", "?")}')
except Exception as _sce:
    print(f'close_scorecard warning: {_sce}')

if not _IS_COMP_RERUN:
    try:
        import pandas as _pd
        _results_map = {r["game_id"]: r["score"] for r in all_results}
        _rows = []
        for _i, _g in enumerate(games):
            _score = float(_results_map.get(_g.game_id, 0.0))
            _rows.append({"row_id": f"{_i}_0", "game_id": _g.game_id,
                           "end_of_game": True, "score": _score})
        _sub_df = _pd.DataFrame(_rows)
        _sub_path = '/kaggle/working/submission.parquet' if KAGGLE else 'submission.parquet'
        _sub_df.to_parquet(_sub_path, index=False)
        print(f'submission.parquet written: {len(_sub_df)} rows, '
              f'avg_score={_sub_df["score"].mean():.4f}')
    except Exception as _e:
        print(f'WARNING: failed to write submission.parquet: {_e}')

# -- Per-game summary table ------------------------------------------------
print()
all_results.sort(key=lambda r: r.get('score', 0), reverse=True)
print('Per-Game Results (sorted by score):')
print(f'  {"Game":30s} | {"Score":>6s} | {"Levels":>6s} | {"Actions":>7s} | {"Resets":>6s} | {"Time":>6s}')
print(f'  {"-"*30}-+-{"-"*6}-+-{"-"*6}-+-{"-"*7}-+-{"-"*6}-+-{"-"*6}')
for r in all_results:
    print(f'  {r["game_id"]:30s} | {r["score"]:6.3f} | {r.get("levels",0):6d} | '
          f'{r.get("actions",0):7d} | {r.get("resets",0):6d} | {r.get("elapsed",0):5.1f}s')

In [ ]:
# -- Cognitive Phase Analysis -------------------------------------------------
try:
    phase_counts = Counter()
    for rung_name, phase in COGNITIVE_PHASE_MAP.items():
        phase_counts[phase.value] += 1

    print("Cognitive Phase Distribution")
    print("=" * 55)
    for phase in ['observe', 'classify', 'extract_goal', 'map_effects', 'plan', 'execute', 'verify']:
        count = phase_counts.get(phase, 0)
        bar = '#' * count
        print(f"  {phase:15s} | {count:2d} | {bar}")

    print()
    print("Rung Role Distribution")
    print("=" * 55)
    role_counts = Counter()
    for rung_name in COGNITIVE_PHASE_MAP:
        role = get_rung_role(rung_name)
        if role:
            role_counts[role.value] += 1

    for role in ['entry', 'leverage', 'compounding', 'resolution']:
        count = role_counts.get(role, 0)
        bar = '#' * count
        print(f"  {role:15s} | {count:2d} | {bar}")
except NameError:
    print("[WARN] COGNITIVE_PHASE_MAP not available (rung_roles cell may not have run)")

## The Dual Matrices -- Navigating the Rung Cascade

Two complementary decision matrices govern how the rung system prioritizes what to act on. These are not metaphors -- they are the actual triage logic the orchestrator uses before committing compute to any hypothesis or rung evaluation.

---

### Matrix 1: The Rumsfeld Matrix (What do we actually know?)

Classifies every observation into an epistemic quadrant before acting on it.

|                    | **Known**                                                        | **Unknown**                                                      |
| :----------------- | :--------------------------------------------------------------- | :--------------------------------------------------------------- |
| **Known**          | **Known Knowns**: Measured metrics, confirmed patterns, verified action-outcome mappings. Rungs in this quadrant have high confidence (`0.7+`) and `win_validated` provenance. **ACT on these directly** -- exploitation rungs fire. | **Known Unknowns**: Identified hypotheses not yet tested. The agent knows it doesn't know something specific. **DESIGN experiments** -- hypothesis rungs activate (scientific_method, hypothesis_testing). |
| **Unknown**        | **Unknown Knowns**: Data already collected but never queried. Patterns in traces that no rung has analyzed yet. **MINE these first** -- orientation rungs (survey, frame_interpretation) surface latent knowledge before generating new hypotheses. | **Unknown Unknowns**: Emergent interaction effects, game mechanics not yet encountered. **Cannot target directly** -- these surface through exploration rungs and the smart_action_selection fallback. Serendipitous discovery, not planned investigation. |

**How rungs map to the Rumsfeld quadrants:**

| Quadrant | Primary Rungs | Strategy |
|----------|--------------|----------|
| Known-Known | exploitation (discovery_exploitation, replay_learning, embedding_suggestion) | High confidence, direct action |
| Known-Unknown | hypothesis (scientific_method, hypothesis_testing, belief_system, theory_gate) | Structured experimentation |
| Unknown-Known | orientation (survey, palette_detection, frame_interpretation, sparse_grid) | Data mining, context enrichment |
| Unknown-Unknown | exploration + fallback (smart_action_selection, action6_object_exploration) | Random exploration, serendipity |

The `KnowledgeProvenance` system tracks which quadrant each piece of knowledge came from, using `detection_source`, `validation_type`, and `crystallization_stage` to separate "widely known" from "actually true."

---

### Matrix 2: The Eisenhower Matrix (Is this worth doing now?)

After classifying WHAT a question is (Rumsfeld), this matrix decides WHEN to pursue it based on urgency and importance.

|                      | **Urgent** (blocking current progress)                           | **Not Urgent** (would improve future progress)                   |
| :------------------- | :--------------------------------------------------------------- | :--------------------------------------------------------------- |
| **Important** (moves the benchmarks) | **DO NOW.** Emergency rungs fire immediately: infinite_loop_breaker (p=1), coordinate_oscillation (p=3), death_avoidance (p=15). These address stuck states, score loss, and oscillation that block ALL downstream reasoning. | **SCHEDULE.** Higher-priority exploitation and hypothesis rungs: scientific_method (p=12), discovery_exploitation (p=20), two_streams (p=30). These improve performance but don't block the current action. |
| **Not Important** (doesn't move benchmarks) | **DELEGATE or DISMISS.** Filter rungs that modulate weights but rarely change outcomes: viral_package_weights (p=20), three_layer_filter (p=55), pariah_avoidance (p=17). They run but with low priority override. | **DROP.** Context-setting rungs that already fired: survey (runs once per level), palette_detection (sets context then exits). After their initial contribution, they return `RungResult()` with zero confidence. |

**How the priority system implements Eisenhower:**

```
Priority 1-5:   Urgent + Important    (emergency, hard safety)
Priority 5-20:  Important + Scheduled (orientation, early exploitation)
Priority 20-50: Scheduled work        (hypothesis, deep exploitation, filters)
Priority 50-99: Low priority / drops   (fallback, late filters)
```

The `context_adaptive` strategy dynamically shifts these boundaries based on:
- **Agent role**: Pioneers explore (more hypothesis rungs), Exploiters optimize (more exploitation rungs)
- **Game type**: Click-only games shift to `action6_only` ordering (42 specialized rungs)
- **Budget phase**: Early game = orientation-heavy, Late game = exploitation-heavy

---

### How the Matrices Work Together

```
Observation arrives
    |
    v
[Rumsfeld] What quadrant is this?
    |
    +--> Known-Known     --> [Eisenhower] Important+Urgent?    --> Exploit immediately (p=5-20)
    +--> Known-Unknown   --> [Eisenhower] Important+Scheduled? --> Hypothesize next cycle (p=12-35)
    +--> Unknown-Known   --> [Eisenhower] Mine first           --> Orient before acting (p=3-10)  
    +--> Unknown-Unknown --> [Eisenhower] Cannot target         --> Let exploration surface it (p=99)
```

This is why the rung priority ordering is not arbitrary -- it encodes both epistemic status (what do we know?) and temporal urgency (when should we act?) into a single evaluation cascade.

## Architecture Summary

### What makes this different from a typical AI decision system:

1. **No LLM** -- Pure algorithmic reasoning with 85 specialized modules
2. **Knowledge provenance** -- Every decision tracks HOW it knows what it knows
3. **Dual economy** -- Actions (ATP) and social capital (Prestige) are separate currencies
4. **Evolutionary selection** -- Rung orderings mutate and compete across generations
5. **Epistemic humility** -- Unknown-Unknown quadrant handled explicitly, not ignored
6. **Cross-game transfer** -- Rungs learn patterns that generalize across game types

### The 7 Cognitive Categories

| Category | Role | Example |
|----------|------|---------|
| **Emergency** | Safety circuit breaker | Break infinite loops, stop oscillation |
| **Orientation** | Understand the world | Survey grid, detect affordances, interpret frames |
| **Hypothesis** | Form and test theories | Scientific method, two-stream consciousness |
| **Exploitation** | Use known knowledge | Replay winning sequences, exploit discoveries |
| **Filter** | Safety gates | Avoid death patterns, budget-aware planning |
| **Exploration** | Systematic search | Object exploration for click-based games |
| **Fallback** | Last resort | Informed random selection using accumulated weights |

---
*BitterTruth-AI: 85 cognitive rungs, 0 LLMs, pure algorithmic reasoning for ARC-AGI-3.*